# GRAPE Universal Metadata Model

Goal: build a clean experiment notebook for the universal prediction setting.

Part 1: Any number of VF visits -> one future VF prediction.

Reference backbone: Bern fine-tuned + region-specific VF-only model on GRAPE.

We first verify the VF-only reference baseline under both consecutive and all-future temporal sampling. Then we will add metadata branches and compare each metadata setting against the VF-only reference.

## 0. Setup

Use fixed seeds rather than fresh random seeds. This keeps the five-run average reproducible and fair across variants.

In [ ]:
import sys
from pathlib import Path
import importlib
import numpy as np
import pandas as pd

project_root = Path(".")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import notebooks2.grape_history_region_transfer_experiment as transfer_module
transfer_module = importlib.reload(transfer_module)

import notebooks2.grape_history_global_region_experiment as hist_module
hist_module = importlib.reload(hist_module)

HistoryRegionTransferConfig = transfer_module.HistoryRegionTransferConfig
run_history_region_transfer_experiment = transfer_module.run_history_region_transfer_experiment

FIXED_SEEDS = (0, 1, 2, 3, 4)
print("Using fixed seeds:", FIXED_SEEDS)

In [ ]:
# Bern-order Garway-Heath-style 6-region mapping, 0-based over 59 TD points.
cluster_mapping = {
    "Supero_Nasal": [1, 14, 15, 21, 24, 28, 34, 38, 42, 46, 50, 53, 56],
    "Supero_Temporal": [0, 5, 20, 30, 33, 37, 45, 49, 55],
    "Macular": [4, 7, 10, 13, 16, 17, 18, 19, 32],
    "Infero_Nasal": [2, 11, 12, 22, 25, 29, 35, 39, 43, 47, 51, 54, 57],
    "Infero_Temporal": [3, 8, 23, 31, 36, 40, 48, 52, 58],
    "Temporal": [6, 9, 26, 27, 41, 44],
}
all_idx = sorted([i for indices in cluster_mapping.values() for i in indices])
assert all_idx == list(range(59))
print("Cluster mapping ready:", {k: len(v) for k, v in cluster_mapping.items()})

## 1. Load and Prepare GRAPE Data

This makes the notebook self-contained. It rebuilds the Bern-aligned GRAPE TD dataframe and the eye-level train/val/test split used by the experiment modules.

In [ ]:
from sklearn.linear_model import LinearRegression

bern_path = project_root / "data" / "Bern" / "bern_59_master_dataset_splitIDs.csv"
grape_xlsx_path = project_root / "GRAPE" / "VF and clinical information.xlsx"
coord_path = project_root / "GRAPE" / "GRAPE coordinate.xlsx"

# ------------------------------------------------------------
# 1) Fit Bern pointwise normative models: Norm_i ~ age
# ------------------------------------------------------------
bern = pd.read_csv(bern_path, low_memory=False)
norm_models = {}
for i in range(1, 60):
    age_col = "age_at_exam"
    norm_col = f"Norm_{i}"
    sub = bern[[age_col, norm_col]].copy()
    sub[age_col] = pd.to_numeric(sub[age_col], errors="coerce")
    sub[norm_col] = pd.to_numeric(sub[norm_col], errors="coerce")
    sub = sub.dropna()
    model = LinearRegression()
    model.fit(sub[[age_col]].values, sub[norm_col].values)
    norm_models[i] = model

print("Bern normative models fitted:", len(norm_models))

# ------------------------------------------------------------
# 2) Load GRAPE baseline and follow-up tables
# ------------------------------------------------------------
baseline = pd.read_excel(grape_xlsx_path, sheet_name="Baseline", header=[0, 1])
baseline.columns = [
    "_".join([str(x).strip() for x in col if str(x).strip() and str(x).strip().lower() != "nan"])
    for col in baseline.columns
]
baseline = baseline.rename(
    columns={
        "Subject Number_Unnamed: 0_level_1": "Subject Number",
        "Laterality_Unnamed: 1_level_1": "Laterality",
        "Age_Unnamed: 2_level_1": "Age",
        "Gender_Unnamed: 3_level_1": "Gender",
        "IOP_Unnamed: 4_level_1": "baseline_IOP",
        "CCT_Unnamed: 5_level_1": "CCT",
        "Total Visits_Unnamed: 6_level_1": "Total Visits",
        "Category of Glaucoma_Unnamed: 10_level_1": "Category of Glaucoma",
        "OCT RNFL thickness_Mean": "RNFL_Mean",
        "OCT RNFL thickness_S": "RNFL_S",
        "OCT RNFL thickness_N": "RNFL_N",
        "OCT RNFL thickness_I": "RNFL_I",
        "OCT RNFL thickness_T": "RNFL_T",
    }
)

followup = pd.read_excel(grape_xlsx_path, sheet_name="Follow-up", header=1)
followup.columns = [str(c).strip() for c in followup.columns]
followup = followup.rename(
    columns={
        "Unnamed: 0": "Subject Number",
        "Unnamed: 1": "Laterality",
        "Unnamed: 2": "Visit Number",
        "Unnamed: 3": "Interval Years",
        "Unnamed: 4": "IOP",
        "Unnamed: 5": "Corresponding CFP",
        "Unnamed: 6": "Acquisition Device",
        "Unnamed: 7": "Resolution",
    }
)
vf_cols_followup_raw = [c for c in followup.columns if str(c).isdigit()]
followup = followup.rename(columns={c: f"VF_{c}" for c in vf_cols_followup_raw})

baseline_meta_cols = [
    "Subject Number",
    "Laterality",
    "Age",
    "baseline_IOP",
    "RNFL_Mean",
    "RNFL_S",
    "RNFL_N",
    "RNFL_I",
    "RNFL_T",
]
baseline_meta = baseline[[c for c in baseline_meta_cols if c in baseline.columns]].drop_duplicates()
grape_fu = followup.merge(baseline_meta, on=["Subject Number", "Laterality"], how="left")

# ------------------------------------------------------------
# 3) Clean GRAPE VF values and remove blind spots
# ------------------------------------------------------------
vf_cols_all = [c for c in grape_fu.columns if str(c).startswith("VF_")]
vf_cols_all = sorted(vf_cols_all, key=lambda x: int(x.split("_")[1]))
blindspot_cols = ["VF_21", "VF_32"]
vf_cols_59_raw = [c for c in vf_cols_all if c not in blindspot_cols]

grape_fu_59 = grape_fu.drop(columns=[c for c in blindspot_cols if c in grape_fu.columns]).copy()
grape_fu_59[vf_cols_59_raw] = (
    grape_fu_59[vf_cols_59_raw].apply(pd.to_numeric, errors="coerce").replace(-1, np.nan)
)

# ------------------------------------------------------------
# 4) Reorder GRAPE VF columns into Bern order, then compute TD
# ------------------------------------------------------------
coord_df = pd.read_excel(coord_path)
coord_df.columns = [str(c).strip() for c in coord_df.columns]
coord_df = coord_df[~coord_df["ids Grape"].isin([21, 32])].copy()
coord_df["bern_idx"] = coord_df["Ids G"] + 1
coord_df = coord_df.sort_values("bern_idx").reset_index(drop=True)
assert len(coord_df) == 59

grape_reordered = grape_fu_59.copy()
for grape_id, bern_id in zip(coord_df["ids Grape"].tolist(), coord_df["bern_idx"].tolist()):
    grape_reordered[f"Sens_{bern_id}"] = pd.to_numeric(
        grape_reordered[f"VF_{grape_id}"], errors="coerce"
    )

for i in range(1, 60):
    grape_reordered[f"Norm_{i}"] = norm_models[i].predict(
        grape_reordered[["Age"]].astype(float).values
    )
    grape_reordered[f"TD_{i}"] = grape_reordered[f"Sens_{i}"] - grape_reordered[f"Norm_{i}"]

grape_reordered["eye_id"] = (
    grape_reordered["Subject Number"].astype(str).str.strip()
    + "_"
    + grape_reordered["Laterality"].astype(str).str.strip()
)
grape_reordered["patient_uid"] = grape_reordered["Subject Number"].astype(str).str.strip()
grape_reordered["Eye"] = grape_reordered["Laterality"].astype(str).str.strip()
grape_reordered["VisitN"] = pd.to_numeric(grape_reordered["Visit Number"], errors="coerce").astype("Int64")
grape_reordered["Time_from_Baseline"] = pd.to_numeric(grape_reordered["Interval Years"], errors="coerce")

grape_longitudinal_td_aligned_df = grape_reordered.copy()
print("Aligned GRAPE dataframe:", grape_longitudinal_td_aligned_df.shape)
print("TD columns:", len([c for c in grape_longitudinal_td_aligned_df.columns if c.startswith("TD_")]))

In [ ]:
from itertools import combinations
from sklearn.preprocessing import StandardScaler

# Rebuild the original notebook-style sequence objects and split.
# Main task: consecutive k-history visits -> next VF visit.
df = grape_longitudinal_td_aligned_df.copy()

if "eye_id" not in df.columns:
    df["eye_id"] = df["Subject Number"].astype(str).str.strip() + "_" + df["Laterality"].astype(str).str.strip()
if "patient_uid" not in df.columns:
    df["patient_uid"] = df["Subject Number"].astype(str).str.strip()
if "Eye" not in df.columns:
    df["Eye"] = df["Laterality"].astype(str).str.strip()

# Match old notebook assumptions.
df = df.dropna(subset=["eye_id", "Visit Number", "Interval Years"]).copy()
df = df.sort_values(["eye_id", "Visit Number"]).reset_index(drop=True)
df["VisitN"] = pd.to_numeric(df["Visit Number"], errors="coerce").astype(int)
df["Time_from_Baseline"] = pd.to_numeric(df["Interval Years"], errors="coerce").astype(float)

td_keep_grape = [c for c in df.columns if c.startswith("TD_")]
td_keep_grape = sorted(td_keep_grape, key=lambda x: int(x.split("_")[1]))
if len(td_keep_grape) != 59:
    raise ValueError(f"Expected 59 TD columns, got {len(td_keep_grape)}")
for c in td_keep_grape:
    df[c] = pd.to_numeric(df[c], errors="coerce")

def prepare_td_vector(row):
    vals = row[td_keep_grape].to_numpy(dtype=np.float32)
    if not np.all(np.isfinite(vals)):
        return None
    return vals

def build_sequences_generic_old_style(
    df,
    prep_fn,
    feature_cols,
    k_in=2,
    k_out=1,
    mode="consecutive",
    min_tests=None,
    min_span_years=0.0,
    group_col="eye_id",
    visit_col="VisitN",
    time_col="Time_from_Baseline",
):
    assert k_out == 1, "Only k_out=1 is currently supported."
    need = k_in + k_out
    if min_tests is None:
        min_tests = need

    df_sorted = df.sort_values([group_col, visit_col]).reset_index(drop=True)
    X_list, Y_list, T_list, meta = [], [], [], []
    L = len(feature_cols)

    for group_id, g in df_sorted.groupby(group_col):
        g = g.sort_values(visit_col).reset_index(drop=True)
        if len(g) < min_tests:
            continue

        idx = list(range(len(g)))
        if mode == "consecutive":
            windows = [tuple(range(s, s + need)) for s in range(0, len(g) - need + 1)]
        elif mode == "all":
            windows = combinations(idx, need)
        else:
            raise ValueError("mode must be 'consecutive' or 'all'")

        for win in windows:
            rows = [g.iloc[w] for w in win]
            times = [float(r[time_col]) for r in rows]

            if not all(times[i] <= times[i + 1] for i in range(len(times) - 1)):
                continue
            if (times[-1] - times[0]) < min_span_years:
                continue

            inputs = rows[:k_in]
            target = rows[-1]
            vec_in = [prep_fn(r) for r in inputs]
            vec_out = prep_fn(target)
            if any(v is None for v in vec_in) or vec_out is None:
                continue

            X_list.append(np.stack(vec_in, axis=0).astype(np.float32))
            Y_list.append(vec_out.astype(np.float32))
            T_list.append(np.array(times[:k_in], dtype=np.float32))
            meta.append({
                "eye_id": group_id,
                "patient_uid": g.loc[0, "patient_uid"] if "patient_uid" in g.columns else None,
                "Eye": g.loc[0, "Eye"] if "Eye" in g.columns else None,
                "VisitNs": [int(r[visit_col]) for r in rows],
                "Times": times,
                "InputTimes": times[:k_in],
                "TargetTime": times[-1],
                "TimeGapToTarget": times[-1] - times[k_in - 1],
                "HistoryLength": k_in,
                "TotalVisitsForEye": len(g),
            })

    if not X_list:
        return (
            np.empty((0, k_in, L), dtype=np.float32),
            np.empty((0, L), dtype=np.float32),
            np.empty((0, k_in), dtype=np.float32),
            pd.DataFrame(),
        )

    return (
        np.stack(X_list).astype(np.float32),
        np.stack(Y_list).astype(np.float32),
        np.stack(T_list).astype(np.float32),
        pd.DataFrame(meta),
    )

sequence_groups = []
for k in (2, 3, 4):
    Xk, Yk, Tk, Mk = build_sequences_generic_old_style(
        df=df,
        prep_fn=prepare_td_vector,
        feature_cols=td_keep_grape,
        k_in=k,
        k_out=1,
        mode="consecutive",
        min_tests=k + 1,
        min_span_years=0.0,
        group_col="eye_id",
        visit_col="VisitN",
        time_col="Time_from_Baseline",
    )
    sequence_groups.append((Xk, Yk, Tk, Mk, k))
    print(f"k={k}:", Xk.shape, Yk.shape, Tk.shape, Mk.shape)

X_hist_list = []
Y_list = []
time_hist_list = []
eye_list = []
meta_list = []
history_len_list = []

for Xk, Yk, Tk, Mk, k in sequence_groups:
    for i in range(Xk.shape[0]):
        X_hist_list.append(Xk[i])
        Y_list.append(Yk[i])
        time_hist_list.append(Tk[i])
        eye_list.append(Mk.iloc[i]["eye_id"])
        meta_list.append(Mk.iloc[i].to_dict())
        history_len_list.append(k)

Y_array = np.stack(Y_list, axis=0).astype(np.float32)
eye_array = np.array(eye_list)
history_len_array = np.array(history_len_list)

unique_eyes = np.unique(eye_array)
rng = np.random.default_rng(42)
rng.shuffle(unique_eyes)

n_trainval_eye = int(0.85 * len(unique_eyes))
trainval_eyes = unique_eyes[:n_trainval_eye]
test_eyes = unique_eyes[n_trainval_eye:]

rng.shuffle(trainval_eyes)
n_val_eye = int(0.10 * len(trainval_eyes))
val_eyes = trainval_eyes[:n_val_eye]
real_train_eyes = trainval_eyes[n_val_eye:]

train_mask = np.isin(eye_array, real_train_eyes)
val_mask = np.isin(eye_array, val_eyes)
test_mask = np.isin(eye_array, test_eyes)

train_indices = np.where(train_mask)[0]
val_indices = np.where(val_mask)[0]
test_indices = np.where(test_mask)[0]

def select_by_indices(seq_list, time_list, Y, meta_list, hist_len_array, indices):
    return (
        [seq_list[i] for i in indices],
        [time_list[i] for i in indices],
        Y[indices],
        [meta_list[i] for i in indices],
        hist_len_array[indices],
    )

X_train_list, time_train_list, Y_train, meta_train_grape, histlen_train = select_by_indices(
    X_hist_list, time_hist_list, Y_array, meta_list, history_len_array, train_indices
)
X_val_list, time_val_list, Y_val, meta_val_grape, histlen_val = select_by_indices(
    X_hist_list, time_hist_list, Y_array, meta_list, history_len_array, val_indices
)
X_test_list, time_test_list, Y_test, meta_test_grape, histlen_test = select_by_indices(
    X_hist_list, time_hist_list, Y_array, meta_list, history_len_array, test_indices
)

# Keep the same convenient objects as the old notebook.
scaler_grape = StandardScaler()
scaler_grape.fit(Y_train)

X_train_s_grape = [scaler_grape.transform(s).astype(np.float32) for s in X_train_list]
X_val_s_grape = [scaler_grape.transform(s).astype(np.float32) for s in X_val_list]
X_test_s_grape = [scaler_grape.transform(s).astype(np.float32) for s in X_test_list]
Y_train_s_grape = scaler_grape.transform(Y_train).astype(np.float32)
Y_val_s_grape = scaler_grape.transform(Y_val).astype(np.float32)
Y_test_s_grape = scaler_grape.transform(Y_test).astype(np.float32)
time_train_grape = [np.asarray(t, dtype=np.float32) for t in time_train_list]
time_val_grape = [np.asarray(t, dtype=np.float32) for t in time_val_list]
time_test_grape = [np.asarray(t, dtype=np.float32) for t in time_test_list]

print("\nOld-style consecutive split:")
print("Train samples:", len(X_train_s_grape), "| Val samples:", len(X_val_s_grape), "| Test samples:", len(X_test_s_grape))
print("Train eyes:", len(set(m["eye_id"] for m in meta_train_grape)))
print("Val eyes:", len(set(m["eye_id"] for m in meta_val_grape)))
print("Test eyes:", len(set(m["eye_id"] for m in meta_test_grape)))
print("History length train:", pd.Series(histlen_train).value_counts().sort_index().to_dict())
print("History length val:", pd.Series(histlen_val).value_counts().sort_index().to_dict())
print("History length test:", pd.Series(histlen_test).value_counts().sort_index().to_dict())


## 2. Check Available Transfer Weights

These are the Bern pretrained region-specific weights and the already saved GRAPE fine-tuned region-specific weights. The experiment runner below initializes from Bern weights and fine-tunes on GRAPE.

In [ ]:
region_names = list(cluster_mapping.keys())

weight_rows = []
for region_name in region_names:
    bern_ckpt = project_root / "models" / f"bern_seq_lstm_timePE_{region_name}_best.pth"
    grape_ft_ckpt = project_root / "models" / f"grape_region_transfer_{region_name}_finetune_best.pth"
    weight_rows.append(
        {
            "region": region_name,
            "bern_pretrained_exists": bern_ckpt.exists(),
            "bern_pretrained_path": str(bern_ckpt),
            "grape_finetuned_exists": grape_ft_ckpt.exists(),
            "grape_finetuned_path": str(grape_ft_ckpt),
        }
    )

weights_df = pd.DataFrame(weight_rows)
display(weights_df)

if not weights_df["bern_pretrained_exists"].all():
    missing = weights_df.loc[~weights_df["bern_pretrained_exists"], "bern_pretrained_path"].tolist()
    raise FileNotFoundError(f"Missing Bern pretrained weights: {missing}")

## 3. Training Sweep: Does All-future Help as Training Augmentation?

All variants below are evaluated on the same consecutive k=2,3,4 next-visit test set.

Here, `all_future` only changes the training/validation sample construction: for each eye, it adds non-adjacent future targets as extra supervision. The test set remains the clinically relevant next-visit prediction task.

In [ ]:
import copy
import time
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from notebooks2.grape_region_full_experiment import (
    RegionHistoryDataset,
    PEBlock,
    MetaEncoder,
    Decoder,
    Encoder,
    VFOnlyModel,
    LateFusionModel,
    _apply_mask,
    _df_to_arrays_with_history,
    _finite_mask,
    _inverse_scale,
    _predict_model,
    _scale_x_list,
    _summarize_abs_prediction_matrix,
    _train_model,
    region_collate,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def run_bern_ft_region_train_eval(
    *,
    train_mode,
    eval_mode="consecutive",
    k_list=(2, 3, 4),
    seeds=(0, 1, 2),
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    label=None,
):
    label = label or f"Bern FT RS | train={train_mode} | eval={eval_mode} | lr={lr}"
    print("\n" + "#" * 100)
    print(label)
    print("#" * 100)

    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)

    train_df, val_df, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, eval_df = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    print("Train samples:", len(train_df), "| Val samples:", len(val_df), "| Eval samples:", len(eval_df))

    arrays = hist_module._prepare_arrays(train_df, val_df, eval_df)
    (
        x_train_s, y_train, t_train, m_train,
        x_val_s, y_val, t_val, m_val,
        x_eval_s, y_eval, t_eval, m_eval,
    ) = arrays

    seed_results = []
    run_rows = []

    for seed in seeds:
        print(f"-- seed={seed}")
        hist_module._set_seed(seed)

        pred_full_abs = np.zeros((len(y_eval), len(td_cols_exp)), dtype=np.float32)
        true_full_abs = np.zeros((len(y_eval), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            print(f"{label} | seed={seed} | region={region_name}")

            y_train_r = y_train[:, idx]
            y_val_r = y_val[:, idx]
            y_eval_r = y_eval[:, idx]

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
            y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
            y_eval_s = y_scaler.transform(y_eval_r).astype(np.float32)

            train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, m_train)
            val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, m_val)
            eval_ds = RegionHistoryDataset(x_eval_s, y_eval_s, t_eval, m_eval)

            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
            eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

            model = VFOnlyModel(output_dim=len(idx))
            bern_ckpt = project_root / "models" / f"bern_seq_lstm_timePE_{region_name}_best.pth"
            model.load_state_dict(torch.load(bern_ckpt, map_location=device))

            model, hist_df, summary = _train_model(
                model=model,
                run_name=f"sweep_bern_ft_{train_mode}_eval_{eval_mode}_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader,
                val_loader=val_loader,
                train_ds=train_ds,
                val_ds=val_ds,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs,
                lr=lr,
            )

            pred_s, true_s = _predict_model(model, eval_loader, device)
            pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
            pred_full_abs[:, idx] = pred_abs
            true_full_abs[:, idx] = true_abs

            summary.update({"seed": seed, "region": region_name, "label": label})
            run_rows.append(summary)

        seed_df = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, label)
        seed_df["seed"] = seed
        seed_results.append(seed_df)

    raw_long = pd.concat(seed_results, ignore_index=True)
    summary_long = (
        raw_long.groupby(["Model", "Region"], as_index=False)
        .agg(
            MAE_mean_db=("mae_mean", "mean"),
            MAE_seed_std_db=("mae_mean", "std"),
            MAE_sample_std_db=("mae_std", "mean"),
            Samples=("Samples", "first"),
        )
    )
    summary_long["MAE ± SD (dB)"] = summary_long.apply(
        lambda r: f"{r['MAE_mean_db']:.2f} ± {r['MAE_sample_std_db']:.2f}",
        axis=1,
    )
    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "runs_df": pd.DataFrame(run_rows),
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "eval_samples": len(eval_df),
    }

def get_overall_row(outputs, label):
    row = outputs["summary_long"].loc[outputs["summary_long"]["Region"].eq("Overall")].iloc[0].copy()
    row["setting"] = label
    row["train_samples"] = outputs["train_samples"]
    row["val_samples"] = outputs["val_samples"]
    row["eval_samples"] = outputs["eval_samples"]
    return row

In [ ]:
# Random seeds each run, so you can rerun this cell several times and inspect variability.
N_RANDOM_SEEDS = 5
RANDOM_SEED_GENERATOR = np.random.default_rng()
SWEEP_SEEDS = tuple(RANDOM_SEED_GENERATOR.integers(0, 1_000_000, size=N_RANDOM_SEEDS).tolist())
print("Random seeds for this run:", SWEEP_SEEDS)

# ------------------------------------------------------------
# Temporal augmentation diagnostics
# ------------------------------------------------------------
df_aug_stats, td_cols_aug_stats = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
consec_pairs_all = hist_module._build_history_pairs(
    df=df_aug_stats,
    td_cols=td_cols_aug_stats,
    k_list=(2, 3, 4),
    mode="consecutive",
)
allfuture_pairs_all = hist_module._build_history_pairs(
    df=df_aug_stats,
    td_cols=td_cols_aug_stats,
    k_list=(2, 3, 4),
    mode="all_future",
)

consec_train_df, consec_val_df, consec_test_df = hist_module._split_by_eye(
    consec_pairs_all, meta_train_grape, meta_val_grape, meta_test_grape
)
aug_train_df, aug_val_df, aug_test_df = hist_module._split_by_eye(
    allfuture_pairs_all, meta_train_grape, meta_val_grape, meta_test_grape
)

def horizon_stats(label, df_part, split_name):
    return {
        "protocol": label,
        "split": split_name,
        "samples": int(len(df_part)),
        "mean_delta_t": float(df_part["delta_t"].mean()) if len(df_part) else np.nan,
        "median_delta_t": float(df_part["delta_t"].median()) if len(df_part) else np.nan,
        "max_delta_t": float(df_part["delta_t"].max()) if len(df_part) else np.nan,
    }

augmentation_stats = pd.DataFrame(
    [
        horizon_stats("Original consecutive", consec_train_df, "train"),
        horizon_stats("Temporal augmentation all_future", aug_train_df, "train"),
        horizon_stats("Original consecutive", consec_val_df, "val"),
        horizon_stats("Temporal augmentation all_future", aug_val_df, "val"),
        horizon_stats("Fixed next-visit evaluation", consec_test_df, "test"),
    ]
)

train_growth_factor = len(aug_train_df) / len(consec_train_df)
val_growth_factor = len(aug_val_df) / len(consec_val_df)

print("Temporal augmentation sample statistics")
display(augmentation_stats)
print(f"Training sample growth factor: {train_growth_factor:.2f}x")
print(f"Validation sample growth factor: {val_growth_factor:.2f}x")
print("Important: final evaluation remains fixed next-visit/consecutive test set.")

# ------------------------------------------------------------
# Performance sweep
# ------------------------------------------------------------
# Keep this small and interpretable: compare original training vs all-future training,
# with the exact same final next-visit evaluation set.
sweep_specs = [
    {
        "setting_id": "original_lr5e4",
        "display_name": "Original training (consecutive)",
        "train_mode": "consecutive",
        "lr": 5e-4,
        "num_epochs": 20,
        "compare_to": None,
    },
    {
        "setting_id": "aug_lr5e4",
        "display_name": "Temporal augmentation (all_future)",
        "train_mode": "all_future",
        "lr": 5e-4,
        "num_epochs": 20,
        "compare_to": "original_lr5e4",
    },
    {
        "setting_id": "original_lr1e3",
        "display_name": "Original training (consecutive), lr=1e-3",
        "train_mode": "consecutive",
        "lr": 1e-3,
        "num_epochs": 20,
        "compare_to": None,
    },
    {
        "setting_id": "aug_lr1e3",
        "display_name": "Temporal augmentation (all_future), lr=1e-3",
        "train_mode": "all_future",
        "lr": 1e-3,
        "num_epochs": 20,
        "compare_to": "original_lr1e3",
    },
]

baseline_sweep_outputs = {}
baseline_sweep_rows = []

for spec in sweep_specs:
    label = spec["display_name"]
    out = run_bern_ft_region_train_eval(
        train_mode=spec["train_mode"],
        eval_mode="consecutive",
        k_list=(2, 3, 4),
        seeds=SWEEP_SEEDS,
        num_epochs=spec["num_epochs"],
        batch_size=64,
        lr=spec["lr"],
        label=label,
    )
    baseline_sweep_outputs[spec["setting_id"]] = out
    row = get_overall_row(out, label)
    row["setting_id"] = spec["setting_id"]
    row["training_protocol"] = "All-future augmentation" if spec["train_mode"] == "all_future" else "Original consecutive"
    row["lr"] = spec["lr"]
    row["num_epochs"] = spec["num_epochs"]
    row["random_seeds"] = str(SWEEP_SEEDS)
    row["compare_to"] = spec["compare_to"]
    baseline_sweep_rows.append(row)

baseline_sweep_overall = pd.DataFrame(baseline_sweep_rows)

# Add augmentation gain relative to the matched original-training setting.
mae_lookup = baseline_sweep_overall.set_index("setting_id")["MAE_mean_db"].to_dict()
baseline_sweep_overall["delta_vs_matched_original"] = baseline_sweep_overall.apply(
    lambda r: r["MAE_mean_db"] - mae_lookup[r["compare_to"]] if pd.notna(r["compare_to"]) and r["compare_to"] in mae_lookup else np.nan,
    axis=1,
)
baseline_sweep_overall["augmentation_improved"] = baseline_sweep_overall["delta_vs_matched_original"] < 0
baseline_sweep_overall["MAE mean ± sample SD (dB)"] = baseline_sweep_overall.apply(
    lambda r: f"{r['MAE_mean_db']:.2f} ± {r['MAE_sample_std_db']:.2f}",
    axis=1,
)
baseline_sweep_overall["delta_vs_matched_original_display"] = baseline_sweep_overall["delta_vs_matched_original"].apply(
    lambda x: "" if pd.isna(x) else f"{x:+.3f} dB"
)

baseline_sweep_display = baseline_sweep_overall[
    [
        "training_protocol",
        "lr",
        "num_epochs",
        "MAE mean ± sample SD (dB)",
        "MAE_mean_db",
        "delta_vs_matched_original_display",
        "augmentation_improved",
        "train_samples",
        "val_samples",
        "eval_samples",
    ]
].copy()

print("Performance comparison: all rows use the same fixed next-visit test set")
display(baseline_sweep_display)

# Compact interpretation table: augmentation rows only.
augmentation_effect_table = baseline_sweep_overall[
    baseline_sweep_overall["training_protocol"].eq("All-future augmentation")
][
    [
        "setting",
        "MAE mean ± sample SD (dB)",
        "delta_vs_matched_original_display",
        "augmentation_improved",
        "random_seeds",
    ]
].reset_index(drop=True)

print("Augmentation effect summary")
display(augmentation_effect_table)


## 4. Best VF-only Sweep Model by Region

Select the sweep setting with the lowest overall MAE, then show its region-level performance table.

In [ ]:
if "baseline_sweep_overall" not in globals() or "baseline_sweep_outputs" not in globals():
    raise RuntimeError("Run the training sweep cell first so baseline_sweep_overall and baseline_sweep_outputs exist.")

best_overall_row = baseline_sweep_overall.sort_values("MAE_mean_db", ascending=True).iloc[0]
best_setting_id = best_overall_row["setting_id"]
best_setting_name = best_overall_row["setting"]

best_region_table = baseline_sweep_outputs[best_setting_id]["summary_long"].copy()
best_region_table = best_region_table.rename(
    columns={
        "MAE_mean_db": "mae_mean",
        "MAE_sample_std_db": "mae_std",
    }
)
best_region_table["#Pts"] = best_region_table["Region"].map(
    lambda r: 59 if r == "Overall" else len(cluster_mapping[r])
)
best_region_table["Samples"] = best_region_table["Samples"].astype(int)
best_region_table["Model"] = best_setting_name
best_region_table["MAE ± SD (dB)"] = best_region_table.apply(
    lambda r: f"{r['mae_mean']:.2f} ± {r['mae_std']:.2f}",
    axis=1,
)

region_order = ["Overall"] + list(cluster_mapping.keys())
best_region_table["Region"] = pd.Categorical(
    best_region_table["Region"],
    categories=region_order,
    ordered=True,
)
best_region_table = best_region_table.sort_values("Region").reset_index(drop=True)

best_region_table_display = best_region_table[
    ["Model", "Region", "#Pts", "mae_mean", "mae_std", "Samples", "MAE ± SD (dB)"]
].copy()
best_region_table_display["Region"] = best_region_table_display["Region"].astype(str)

print("Best overall VF-only sweep setting:")
print(best_setting_id, "|", best_setting_name)
print(f"Overall MAE: {best_overall_row['MAE_mean_db']:.3f} dB")
display(best_region_table_display)


## 5. Metadata Experiment Plan

We will test metadata in stages, always using the same split and fixed next-visit evaluation protocol as the VF-only reference baseline. A metadata setting is considered useful only if it lowers MAE relative to the VF-only baseline.

Stage 1 runs a compact set of clinically interpretable metadata inputs. Stage 2 expands RNFL into all S/N/I/T subset combinations if Stage 1 suggests RNFL is useful. Stage 3 can add region-aware RNFL variants after the best global RNFL subsets are identified.

In [ ]:
from itertools import combinations

# Stage 1: compact metadata screen.
# These are the first variants to run because they answer the main question quickly:
# does IOP, RNFL mean, or full SNIT RNFL help beyond VF-only?
stage1_metadata_plan = pd.DataFrame(
    [
        {"stage": 1, "variant": "VF only", "uses_iop": False, "rnfl_cols": [], "rnfl_mode": "none", "fusion": "none"},
        {"stage": 1, "variant": "VF + IOP", "uses_iop": True, "rnfl_cols": [], "rnfl_mode": "none", "fusion": "late"},
        {"stage": 1, "variant": "VF + RNFL mean", "uses_iop": False, "rnfl_cols": ["RNFL_Mean"], "rnfl_mode": "mean", "fusion": "late"},
        {"stage": 1, "variant": "VF + RNFL SNIT", "uses_iop": False, "rnfl_cols": ["RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"], "rnfl_mode": "snit", "fusion": "late"},
        {"stage": 1, "variant": "VF + IOP + RNFL mean", "uses_iop": True, "rnfl_cols": ["RNFL_Mean"], "rnfl_mode": "mean", "fusion": "late"},
        {"stage": 1, "variant": "VF + IOP + RNFL SNIT", "uses_iop": True, "rnfl_cols": ["RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"], "rnfl_mode": "snit", "fusion": "late"},
    ]
)

# Stage 2: exhaustive RNFL subset screen over S/N/I/T.
# This tests whether one sector or a small sector combination is better than using all four sectors.
rnfl_sector_cols = ["RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
rnfl_subset_rows = []
for r in range(1, len(rnfl_sector_cols) + 1):
    for subset in combinations(rnfl_sector_cols, r):
        subset = list(subset)
        suffix = "+".join(c.replace("RNFL_", "") for c in subset)
        rnfl_subset_rows.append({"stage": 2, "variant": f"VF + RNFL {suffix}", "uses_iop": False, "rnfl_cols": subset, "rnfl_mode": "subset", "fusion": "late"})
        rnfl_subset_rows.append({"stage": 2, "variant": f"VF + IOP + RNFL {suffix}", "uses_iop": True, "rnfl_cols": subset, "rnfl_mode": "subset", "fusion": "late"})

stage2_metadata_plan = pd.DataFrame(rnfl_subset_rows)

print("Stage 1 compact metadata screen:")
display(stage1_metadata_plan)

print("Stage 2 RNFL subset expansion candidate list:")
display(stage2_metadata_plan)

print(f"Stage 1 variants: {len(stage1_metadata_plan)}")
print(f"Stage 2 variants: {len(stage2_metadata_plan)}")


## 6. Stage 1 Metadata Runner

Run the compact metadata screen using Bern-initialized region-specific models, original consecutive training, and fixed next-visit evaluation.


In [ ]:
def _attach_history_metadata(pooled_df, df_source, rnfl_cols, max_k=4):
    """Attach IOP history plus static RNFL from the last input visit.

    IOP is visit-varying, so we keep the full input-history IOP sequence and pad it to max_k.
    RNFL is treated as static structural metadata and is taken from the last input visit.
    """
    df_meta = df_source.copy()
    if "eye_id" not in df_meta.columns:
        df_meta["eye_id"] = (
            df_meta["Subject Number"].astype(str).str.strip()
            + "_"
            + df_meta["Laterality"].astype(str).str.strip()
        )
    if "VisitN" not in df_meta.columns:
        df_meta["VisitN"] = pd.to_numeric(df_meta["Visit Number"], errors="coerce")

    needed_cols = ["eye_id", "VisitN", "IOP"] + rnfl_cols
    for col in ["VisitN", "IOP"] + rnfl_cols:
        df_meta[col] = pd.to_numeric(df_meta[col], errors="coerce")

    df_meta = df_meta[needed_cols].drop_duplicates(subset=["eye_id", "VisitN"], keep="last")
    eye_to_meta = {
        eye_id: g.sort_values("VisitN").reset_index(drop=True)
        for eye_id, g in df_meta.groupby("eye_id")
    }

    iop_hist_padded = []
    rnfl_static = []
    input_visit_lists = []

    for _, row in pooled_df.iterrows():
        eye_id = row["eye_id"]
        k = int(row["k"])
        last_visit = int(row["visit_in_last"])
        g = eye_to_meta.get(eye_id)

        if g is None:
            iop_values = np.full(k, np.nan, dtype=np.float32)
            rnfl_values = np.full(len(rnfl_cols), np.nan, dtype=np.float32)
            visit_values = []
        else:
            hist_g = g[g["VisitN"] <= last_visit].tail(k)
            iop_values = hist_g["IOP"].to_numpy(dtype=np.float32)
            visit_values = hist_g["VisitN"].astype(int).tolist()
            if len(hist_g) > 0:
                rnfl_values = hist_g.iloc[-1][rnfl_cols].to_numpy(dtype=np.float32)
            else:
                rnfl_values = np.full(len(rnfl_cols), np.nan, dtype=np.float32)

        padded = np.full(max_k, np.nan, dtype=np.float32)
        if len(iop_values) > 0:
            padded[-len(iop_values):] = iop_values[-max_k:]

        iop_hist_padded.append(padded)
        rnfl_static.append(rnfl_values.astype(np.float32))
        input_visit_lists.append(visit_values)

    out = pooled_df.copy()
    out["iop_hist_padded"] = iop_hist_padded
    out["rnfl_static"] = rnfl_static
    out["input_visit_list"] = input_visit_lists
    return out


def _impute_history_metadata_by_train(train_df, val_df, test_df, max_k=4, rnfl_cols=None):
    rnfl_cols = rnfl_cols or ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    iop_hist_cols = [f"IOP_hist_{i + 1}" for i in range(max_k)]
    iop_cols = ["IOP_last"] + iop_hist_cols

    def _iop_frame(df_part):
        hist_df = pd.DataFrame(df_part["iop_hist_padded"].tolist(), columns=iop_hist_cols)
        hist_df.insert(0, "IOP_last", hist_df[iop_hist_cols[-1]])
        return hist_df

    train_iop = _iop_frame(train_df)
    train_rnfl = pd.DataFrame(train_df["rnfl_static"].tolist(), columns=rnfl_cols)
    train_means = pd.concat([train_iop, train_rnfl], axis=1).mean(axis=0)

    def _apply(df_part):
        iop_df = _iop_frame(df_part)
        rnfl_df = pd.DataFrame(df_part["rnfl_static"].tolist(), columns=rnfl_cols)
        meta_df = pd.concat([iop_df, rnfl_df], axis=1).fillna(train_means)
        out = df_part.copy()
        out["meta_raw"] = list(meta_df.to_numpy(dtype=np.float32))
        return out

    return _apply(train_df), _apply(val_df), _apply(test_df), iop_cols + rnfl_cols


def _set_meta_sub_by_cols(df_part, meta_master_cols, uses_iop, rnfl_cols, iop_mode="last"):
    out = df_part.copy()
    selected_cols = []
    if uses_iop:
        if iop_mode == "last":
            selected_cols.append("IOP_last")
        elif iop_mode == "history":
            selected_cols.extend([c for c in meta_master_cols if c.startswith("IOP_hist_")])
        else:
            raise ValueError("iop_mode must be 'last' or 'history'")
    selected_cols.extend(rnfl_cols)

    if len(selected_cols) == 0:
        out["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(out))]
    else:
        keep_indices = [meta_master_cols.index(c) for c in selected_cols]
        out["meta_sub"] = [np.asarray(v, dtype=np.float32)[keep_indices] for v in out["meta_raw"].values]
    return out, selected_cols



class ResidualLateFusionModel(nn.Module):
    """VF-only Bern backbone plus metadata residual correction.

    This keeps the pretrained VF decoder path intact. Metadata only learns an additive
    correction in the same scaled target space, so the model starts from the VF-only
    prediction rather than replacing the output head.
    """
    def __init__(
        self,
        meta_dim,
        input_dim=59,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
        meta_hidden_dim=32,
        meta_emb_dim=16,
        correction_hidden_dim=64,
    ):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm = nn.LSTM(
            input_size=latent_dim + pe_output_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)
        self.meta_encoder = MetaEncoder(input_dim=meta_dim, hidden_dim=meta_hidden_dim, output_dim=meta_emb_dim)
        self.correction_head = nn.Sequential(
            nn.Linear(latent_dim + meta_emb_dim, correction_hidden_dim),
            nn.ReLU(),
            nn.Linear(correction_hidden_dim, output_dim),
        )
        # Start exactly as the VF-only path; metadata correction must earn its contribution.
        nn.init.zeros_(self.correction_head[-1].weight)
        nn.init.zeros_(self.correction_head[-1].bias)

    def forward(self, padded_seqs, padded_times, lengths, meta_input):
        enc_out = self.encoder(padded_seqs)
        time_emb = self.time_pe(padded_times)
        lstm_in = torch.cat([enc_out, time_emb], dim=-1)
        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        h_vf = h_n[-1]

        base_pred = self.decoder(h_vf)
        h_meta = self.meta_encoder(meta_input)
        correction = self.correction_head(torch.cat([h_vf, h_meta], dim=-1))
        return base_pred + correction


def _load_bern_weights_into_model(model, region_name):
    """Load matching Bern VF weights. Metadata-specific layers stay randomly initialized."""
    ckpt = project_root / "models" / f"bern_seq_lstm_timePE_{region_name}_best.pth"
    if not ckpt.exists():
        raise FileNotFoundError(ckpt)
    source_state = torch.load(ckpt, map_location=device)
    target_state = model.state_dict()
    matched = {
        k: v
        for k, v in source_state.items()
        if k in target_state and tuple(v.shape) == tuple(target_state[k].shape)
    }
    target_state.update(matched)
    model.load_state_dict(target_state)
    return model, sorted(matched.keys())


def _metadata_summary_agg(raw_long):
    out = (
        raw_long.groupby(["Model", "Region"], as_index=False)
        .agg(
            MAE_mean_db=("mae_mean", "mean"),
            MAE_seed_std_db=("mae_mean", "std"),
            MAE_sample_std_db=("mae_std", "mean"),
            Samples=("Samples", "first"),
            **{"#Pts": ("#Pts", "first")},
        )
    )
    out["MAE ± SD (dB)"] = out.apply(
        lambda r: f"{r['MAE_mean_db']:.2f} ± {r['MAE_sample_std_db']:.2f}",
        axis=1,
    )
    return out


def run_stage1_metadata_screen(
    metadata_plan,
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="last",
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)

    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    train_df_all, val_df_all, test_df_all, meta_master_cols = _impute_history_metadata_by_train(
        train_df_all, val_df_all, test_df_all, max_k=max_k, rnfl_cols=rnfl_master_cols
    )

    raw_results = []
    run_summaries = []

    print("Metadata train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Metadata seeds:", seeds)
    print("IOP mode:", iop_mode)
    print("Available IOP metadata columns:", [c for c in meta_master_cols if c.startswith("IOP")])

    for _, exp in metadata_plan.iterrows():
        exp_name = exp["variant"]
        uses_iop = bool(exp["uses_iop"])
        rnfl_cols = list(exp["rnfl_cols"])

        print("\n" + "#" * 100)
        print(f"STAGE 1 METADATA: {exp_name} | residual late fusion | iop_mode={iop_mode} | uses_iop={uses_iop} | rnfl_cols={rnfl_cols}")
        print("#" * 100)

        train_df, selected_meta_cols = _set_meta_sub_by_cols(train_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
        val_df, _ = _set_meta_sub_by_cols(val_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
        test_df, _ = _set_meta_sub_by_cols(test_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
        is_vf_only = len(selected_meta_cols) == 0

        x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df)
        x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df)
        x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df)

        train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
        val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
        test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

        x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
        x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
        x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)

        x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

        if meta_train.shape[1] > 0:
            meta_scaler = StandardScaler()
            meta_train_s = meta_scaler.fit_transform(meta_train).astype(np.float32)
            meta_val_s = meta_scaler.transform(meta_val).astype(np.float32)
            meta_test_s = meta_scaler.transform(meta_test).astype(np.float32)
        else:
            meta_train_s = meta_train.astype(np.float32)
            meta_val_s = meta_val.astype(np.float32)
            meta_test_s = meta_test.astype(np.float32)

        for seed in seeds:
            print(f"-- seed={seed}")
            hist_module._set_seed(int(seed))

            pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                print(f"{exp_name} | seed={seed} | region={region_name}")

                y_train_r = y_train[:, idx]
                y_val_r = y_val[:, idx]
                y_test_r = y_test[:, idx]

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
                y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
                y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)

                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                if is_vf_only:
                    model = VFOnlyModel(output_dim=len(idx))
                else:
                    model = ResidualLateFusionModel(meta_dim=meta_train_s.shape[1], output_dim=len(idx))

                model, loaded_keys = _load_bern_weights_into_model(model, region_name)

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"stage1_{exp_name.replace(' ', '_').replace('+', 'plus')}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "seed": int(seed),
                    "region": region_name,
                    "selected_meta_cols": selected_meta_cols,
                    "loaded_bern_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["selected_meta_cols"] = str(selected_meta_cols)
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)

    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_mae = float(overall.loc[overall["Model"].eq("VF only"), "MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(train_df_all),
        "val_samples": len(val_df_all),
        "test_samples": len(test_df_all),
        "seeds": seeds,
        "meta_master_cols": meta_master_cols,
        "iop_mode": iop_mode,
    }


## 7. Run Stage 1 Metadata Screen

This cell trains the compact metadata variants. It is intentionally separated from the function definition so rerunning the plan table does not retrain models.


In [ ]:
# Random seeds each run. Use 1-2 seeds for quick debugging; use 5 for the reported table.
N_METADATA_SEEDS = 5
METADATA_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_METADATA_SEEDS).tolist())
print("Random metadata seeds for this run:", METADATA_SEEDS)

stage1_metadata_outputs_by_iop_mode = {}
stage1_metadata_overall_tables = []

for IOP_MODE in ["last", "history"]:
    print("\n" + "=" * 120)
    print(f"RUNNING STAGE 1 METADATA SCREEN | IOP_MODE={IOP_MODE}")
    print("=" * 120)

    out = run_stage1_metadata_screen(
        stage1_metadata_plan,
        seeds=METADATA_SEEDS,
        k_list=(2, 3, 4),
        train_mode="consecutive",
        eval_mode="consecutive",
        num_epochs=20,
        batch_size=64,
        lr=1e-3,
        iop_mode=IOP_MODE,
    )
    stage1_metadata_outputs_by_iop_mode[IOP_MODE] = out

    overall = out["overall"].copy()
    overall["iop_mode"] = IOP_MODE
    stage1_metadata_overall_tables.append(overall)

stage1_metadata_overall_all = pd.concat(stage1_metadata_overall_tables, ignore_index=True)
stage1_metadata_overall_display = stage1_metadata_overall_all[
    [
        "iop_mode",
        "Model",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
    ]
].sort_values(["iop_mode", "MAE_mean_db"]).reset_index(drop=True)

print("Stage 1 metadata overall results, sorted by IOP mode and MAE:")
display(stage1_metadata_overall_display)

print("Stage 1 best model per IOP mode:")
display(
    stage1_metadata_overall_display
    .sort_values("MAE_mean_db")
    .groupby("iop_mode", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# Convenience aliases for downstream inspection. Use history IOP as the main line.
stage1_metadata_outputs = stage1_metadata_outputs_by_iop_mode["history"]
stage1_metadata_region_table = stage1_metadata_outputs["summary_long"].copy()
print("Stage 1 region-level results for IOP_MODE='history':")
display(stage1_metadata_region_table)


## 8. Stage 2 RNFL Subset Screen

Use the main protocol from Stage 1 (`iop_mode='history'`) and test all RNFL sector subsets. `VF only` and `VF + IOP` are included as matched baselines, so every RNFL combination can be compared against both the no-metadata baseline and the IOP-only baseline.


In [ ]:
# Full Stage 2 is heavier than Stage 1: 32 variants x seeds x 6 regions.
# Use 1-2 seeds for quick debugging; use 5 for the final reported table.
N_STAGE2_SEEDS = 5
STAGE2_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_STAGE2_SEEDS).tolist())
print("Random Stage 2 seeds for this run:", STAGE2_SEEDS)

stage2_history_plan = pd.concat(
    [
        stage1_metadata_plan[stage1_metadata_plan["variant"].isin(["VF only", "VF + IOP"])],
        stage2_metadata_plan,
    ],
    ignore_index=True,
)

print("Stage 2 plan with matched baselines:")
display(stage2_history_plan)

stage2_history_outputs = run_stage1_metadata_screen(
    stage2_history_plan,
    seeds=STAGE2_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
)

stage2_history_overall = stage2_history_outputs["overall"].copy()
iop_baseline_mae = float(
    stage2_history_overall.loc[stage2_history_overall["Model"].eq("VF + IOP"), "MAE_mean_db"].iloc[0]
)
stage2_history_overall["delta_vs_iop"] = stage2_history_overall["MAE_mean_db"] - iop_baseline_mae
stage2_history_overall["improved_vs_iop"] = stage2_history_overall["delta_vs_iop"] < 0
stage2_history_overall["delta_vs_iop_display"] = stage2_history_overall["delta_vs_iop"].map(lambda x: f"{x:+.3f} dB")

stage2_history_overall_display = stage2_history_overall[
    [
        "Model",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "delta_vs_iop_display",
        "improved_vs_iop",
        "Samples",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True)

print("Stage 2 overall results, sorted by MAE:")
display(stage2_history_overall_display)

print("Stage 2 candidates that improve over VF + IOP:")
display(stage2_history_overall_display[stage2_history_overall_display["improved_vs_iop"]].reset_index(drop=True))

stage2_best_model = stage2_history_overall_display.iloc[0]["Model"]
print("Best Stage 2 overall model:", stage2_best_model)

stage2_best_region_table = stage2_history_outputs["summary_long"][
    stage2_history_outputs["summary_long"]["Model"].eq(stage2_best_model)
].copy()
print("Region-level table for best Stage 2 model:")
display(stage2_best_region_table)


## 9. Stage 3 Region-aware RNFL Screen

Test whether RNFL helps more when each VF region receives anatomically related RNFL sectors instead of the same global RNFL subset. This is still evaluated under the main protocol: Bern fine-tuned VF backbone, residual late fusion, IOP history when IOP is used, and fixed next-visit evaluation.


In [ ]:
# Two region-aware RNFL designs:
# 1) broad: old/existing anatomy-inspired mapping from VF regions to related RNFL sectors.
# 2) single: stricter one-dominant-sector mapping to reduce RNFL noise.
region_aware_rnfl_broad = {
    "Supero_Nasal": ["RNFL_I", "RNFL_N"],
    "Supero_Temporal": ["RNFL_I", "RNFL_T"],
    "Infero_Nasal": ["RNFL_S", "RNFL_N"],
    "Infero_Temporal": ["RNFL_S", "RNFL_T"],
    "Temporal": ["RNFL_T"],
    "Macular": ["RNFL_T"],
}

region_aware_rnfl_single = {
    "Supero_Nasal": ["RNFL_I"],
    "Supero_Temporal": ["RNFL_I"],
    "Infero_Nasal": ["RNFL_S"],
    "Infero_Temporal": ["RNFL_S"],
    "Temporal": ["RNFL_T"],
    "Macular": ["RNFL_T"],
}

stage3_region_aware_plan = pd.DataFrame(
    [
        {"variant": "VF only", "uses_iop": False, "region_rnfl_map": {}, "rnfl_mode": "none"},
        {"variant": "VF + IOP", "uses_iop": True, "region_rnfl_map": {}, "rnfl_mode": "none"},
        {"variant": "VF + RegionAware RNFL broad", "uses_iop": False, "region_rnfl_map": region_aware_rnfl_broad, "rnfl_mode": "region_broad"},
        {"variant": "VF + IOP + RegionAware RNFL broad", "uses_iop": True, "region_rnfl_map": region_aware_rnfl_broad, "rnfl_mode": "region_broad"},
        {"variant": "VF + RegionAware RNFL single", "uses_iop": False, "region_rnfl_map": region_aware_rnfl_single, "rnfl_mode": "region_single"},
        {"variant": "VF + IOP + RegionAware RNFL single", "uses_iop": True, "region_rnfl_map": region_aware_rnfl_single, "rnfl_mode": "region_single"},
    ]
)

print("Stage 3 region-aware RNFL plan:")
display(stage3_region_aware_plan)


def _set_model_time_inputs(pooled_df, mode="input_times"):
    """Choose what the LSTM time encoder receives.

    input_times: original visit times for the input VF sequence.
    delta_t: prediction horizon to the target VF, repeated to match sequence length.
    """
    if mode == "input_times":
        return pooled_df
    if mode != "delta_t":
        raise ValueError("time_input_mode must be 'input_times' or 'delta_t'")

    out = pooled_df.copy()
    out["input_times"] = [
        np.repeat(float(delta_t), int(k)).astype(np.float32)
        for delta_t, k in zip(out["delta_t"], out["k"])
    ]
    return out


def run_region_aware_metadata_screen(
    metadata_plan,
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
    time_input_mode="input_times",
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)
    train_pooled = _set_model_time_inputs(train_pooled, mode=time_input_mode)
    eval_pooled = _set_model_time_inputs(eval_pooled, mode=time_input_mode)

    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    train_df_all, val_df_all, test_df_all, meta_master_cols = _impute_history_metadata_by_train(
        train_df_all, val_df_all, test_df_all, max_k=max_k, rnfl_cols=rnfl_master_cols
    )

    raw_results = []
    run_summaries = []

    print("Region-aware metadata train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Region-aware metadata seeds:", seeds)
    print("IOP mode:", iop_mode)
    print("Time input mode:", time_input_mode)

    for _, exp in metadata_plan.iterrows():
        exp_name = exp["variant"]
        uses_iop = bool(exp["uses_iop"])
        region_rnfl_map = dict(exp["region_rnfl_map"])

        print("\n" + "#" * 100)
        print(f"STAGE 3 METADATA: {exp_name} | region-aware RNFL | iop_mode={iop_mode}")
        print("#" * 100)

        for seed in seeds:
            print(f"-- seed={seed}")
            hist_module._set_seed(int(seed))

            pred_full_abs = None
            true_full_abs = None

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = list(region_rnfl_map.get(region_name, []))
                print(f"{exp_name} | seed={seed} | region={region_name} | rnfl_cols={rnfl_cols}")

                train_df, selected_meta_cols = _set_meta_sub_by_cols(train_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
                val_df, _ = _set_meta_sub_by_cols(val_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
                test_df, _ = _set_meta_sub_by_cols(test_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
                is_vf_only = len(selected_meta_cols) == 0

                x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df)
                x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df)
                x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df)

                train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
                val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
                test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

                x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
                x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
                x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)

                x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

                if meta_train.shape[1] > 0:
                    meta_scaler = StandardScaler()
                    meta_train_s = meta_scaler.fit_transform(meta_train).astype(np.float32)
                    meta_val_s = meta_scaler.transform(meta_val).astype(np.float32)
                    meta_test_s = meta_scaler.transform(meta_test).astype(np.float32)
                else:
                    meta_train_s = meta_train.astype(np.float32)
                    meta_val_s = meta_val.astype(np.float32)
                    meta_test_s = meta_test.astype(np.float32)

                if pred_full_abs is None:
                    pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                    true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

                y_train_r = y_train[:, idx]
                y_val_r = y_val[:, idx]
                y_test_r = y_test[:, idx]

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
                y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
                y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)

                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                if is_vf_only:
                    model = VFOnlyModel(output_dim=len(idx))
                else:
                    model = ResidualLateFusionModel(meta_dim=meta_train_s.shape[1], output_dim=len(idx))

                model, loaded_keys = _load_bern_weights_into_model(model, region_name)
                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"stage3_{exp_name.replace(' ', '_').replace('+', 'plus')}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "seed": int(seed),
                    "region": region_name,
                    "region_rnfl_cols": rnfl_cols,
                    "selected_meta_cols": selected_meta_cols,
                    "loaded_bern_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["rnfl_mode"] = exp["rnfl_mode"]
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()

    vf_baseline_mae = float(overall.loc[overall["Model"].eq("VF only"), "MAE_mean_db"].iloc[0])
    iop_baseline_rows = overall[overall["Model"].isin(["VF + IOP", "VF + IOP history"])]
    if len(iop_baseline_rows) == 0:
        raise ValueError("No IOP-only baseline found. Expected model name 'VF + IOP' or 'VF + IOP history'.")
    iop_baseline_mae = float(iop_baseline_rows.sort_values("MAE_mean_db")["MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - vf_baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall["delta_vs_iop"] = overall["MAE_mean_db"] - iop_baseline_mae
    overall["improved_vs_iop"] = overall["delta_vs_iop"] < 0
    overall["delta_vs_iop_display"] = overall["delta_vs_iop"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(train_df_all),
        "val_samples": len(val_df_all),
        "test_samples": len(test_df_all),
        "seeds": seeds,
        "iop_mode": iop_mode,
        "time_input_mode": time_input_mode,
    }


# This is much smaller than Stage 2: 6 variants x seeds x 6 regions.
N_STAGE3_SEEDS = 5
STAGE3_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_STAGE3_SEEDS).tolist())
print("Random Stage 3 seeds for this run:", STAGE3_SEEDS)

stage3_region_aware_outputs = run_region_aware_metadata_screen(
    stage3_region_aware_plan,
    seeds=STAGE3_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
)

stage3_region_aware_overall_display = stage3_region_aware_outputs["overall"][
    [
        "Model",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "delta_vs_iop_display",
        "improved_vs_iop",
        "Samples",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True)

print("Stage 3 region-aware RNFL overall results:")
display(stage3_region_aware_overall_display)

stage3_best_model = stage3_region_aware_overall_display.iloc[0]["Model"]
print("Best Stage 3 overall model:", stage3_best_model)

stage3_best_region_table = stage3_region_aware_outputs["summary_long"][
    stage3_region_aware_outputs["summary_long"]["Model"].eq(stage3_best_model)
].copy()
print("Region-level table for best Stage 3 model:")
display(stage3_best_region_table)


## 10. Universal Metadata Availability Screen

This is the practical universal-model table. It treats metadata availability as the deployment condition: some patients may have only IOP, only RNFL mean, one or more RNFL sectors, or IOP plus partial RNFL sectors. RNFL sector subsets are spatialized by intersecting the available sectors with the broad region-aware RNFL map for each VF region.


In [ ]:
def _spatialize_available_rnfl_subset(available_cols, broad_map):
    """For each VF region, keep only the available RNFL sectors from the broad anatomical map."""
    available = set(available_cols)
    return {
        region_name: [col for col in rnfl_cols if col in available]
        for region_name, rnfl_cols in broad_map.items()
    }


universal_rows = [
    {
        "variant": "VF only",
        "uses_iop": False,
        "rnfl_available": "none",
        "region_rnfl_map": {},
        "rnfl_mode": "none",
    },
    {
        "variant": "VF + IOP",
        "uses_iop": True,
        "rnfl_available": "none",
        "region_rnfl_map": {},
        "rnfl_mode": "none",
    },
    {
        "variant": "VF + RNFL mean",
        "uses_iop": False,
        "rnfl_available": "mean",
        "region_rnfl_map": {region_name: ["RNFL_Mean"] for region_name in cluster_mapping},
        "rnfl_mode": "mean",
    },
    {
        "variant": "VF + IOP + RNFL mean",
        "uses_iop": True,
        "rnfl_available": "mean",
        "region_rnfl_map": {region_name: ["RNFL_Mean"] for region_name in cluster_mapping},
        "rnfl_mode": "mean",
    },
]

for r in range(1, len(rnfl_sector_cols) + 1):
    for subset in combinations(rnfl_sector_cols, r):
        subset = list(subset)
        suffix = "+".join(c.replace("RNFL_", "") for c in subset)
        spatial_map = _spatialize_available_rnfl_subset(subset, region_aware_rnfl_broad)
        universal_rows.append(
            {
                "variant": f"VF + Spatial RNFL {suffix}",
                "uses_iop": False,
                "rnfl_available": suffix,
                "region_rnfl_map": spatial_map,
                "rnfl_mode": "spatial_subset_broad",
            }
        )
        universal_rows.append(
            {
                "variant": f"VF + IOP + Spatial RNFL {suffix}",
                "uses_iop": True,
                "rnfl_available": suffix,
                "region_rnfl_map": spatial_map,
                "rnfl_mode": "spatial_subset_broad",
            }
        )

universal_metadata_plan = pd.DataFrame(universal_rows)
print("Universal metadata availability plan:")
display(universal_metadata_plan[["variant", "uses_iop", "rnfl_available", "rnfl_mode", "region_rnfl_map"]])
print(f"Universal metadata variants: {len(universal_metadata_plan)}")

# This is the broad deployment screen: 34 variants x seeds x 6 regions.
# Use 1 seed for debugging; use 5 for the reported universal table.
N_UNIVERSAL_SEEDS = 5
UNIVERSAL_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_UNIVERSAL_SEEDS).tolist())
print("Random universal metadata seeds for this run:", UNIVERSAL_SEEDS)

universal_metadata_outputs = run_region_aware_metadata_screen(
    universal_metadata_plan,
    seeds=UNIVERSAL_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
)

universal_overall = universal_metadata_outputs["overall"].copy()
universal_overall = universal_overall.merge(
    universal_metadata_plan[["variant", "rnfl_available", "rnfl_mode", "uses_iop"]],
    left_on="Model",
    right_on="variant",
    how="left",
).drop(columns=["variant"])

universal_display = universal_overall[
    [
        "Model",
        "uses_iop",
        "rnfl_available",
        "rnfl_mode",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "delta_vs_iop_display",
        "improved_vs_iop",
        "Samples",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True)

print("Universal metadata overall table, sorted by MAE:")
display(universal_display)

print("Universal candidates improving over IOP-history baseline:")
display(universal_display[universal_display["improved_vs_iop"]].reset_index(drop=True))

# For deployment, choose the best model available for each metadata availability pattern.
universal_best_by_availability = (
    universal_display
    .sort_values("MAE_mean_db")
    .groupby(["uses_iop", "rnfl_available"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
print("Best model for each metadata availability pattern:")
display(universal_best_by_availability)

universal_best_model = universal_display.iloc[0]["Model"]
print("Best universal-screen overall model:", universal_best_model)
universal_best_region_table = universal_metadata_outputs["summary_long"][
    universal_metadata_outputs["summary_long"]["Model"].eq(universal_best_model)
].copy()
print("Region-level table for best universal-screen model:")
display(universal_best_region_table)


## 11. Single-Visit Universal Metadata Screen

This repeats the universal metadata availability screen for the `1 VF + Δt -> 1 future VF` setting. The input contains one VF visit, and the model is explicitly given the prediction horizon `Δt` to the target visit. IOP history therefore reduces to the IOP from that single input visit, while RNFL remains baseline/static structural metadata.

Keep the same train/val/test eye split, residual late fusion design, fixed next-visit evaluation, and five-random-seed averaging logic so the metadata variants are comparable to the single-visit VF-only baseline. This matches the slide definition: `VF + Δt (+ metadata) -> VF(t + Δt)`.


In [ ]:
if "_spatialize_available_rnfl_subset" not in globals():
    def _spatialize_available_rnfl_subset(available_cols, broad_map):
        """For each VF region, keep only the available RNFL sectors from the broad anatomical map."""
        available = set(available_cols)
        return {
            region_name: [col for col in rnfl_cols if col in available]
            for region_name, rnfl_cols in broad_map.items()
        }


def build_universal_metadata_plan():
    """Build the universal metadata availability table without rerunning the multi-visit screen."""
    rows = [
        {
            "variant": "VF only",
            "uses_iop": False,
            "rnfl_available": "none",
            "region_rnfl_map": {},
            "rnfl_mode": "none",
        },
        {
            "variant": "VF + IOP",
            "uses_iop": True,
            "rnfl_available": "none",
            "region_rnfl_map": {},
            "rnfl_mode": "none",
        },
        {
            "variant": "VF + RNFL mean",
            "uses_iop": False,
            "rnfl_available": "mean",
            "region_rnfl_map": {region_name: ["RNFL_Mean"] for region_name in cluster_mapping},
            "rnfl_mode": "mean",
        },
        {
            "variant": "VF + IOP + RNFL mean",
            "uses_iop": True,
            "rnfl_available": "mean",
            "region_rnfl_map": {region_name: ["RNFL_Mean"] for region_name in cluster_mapping},
            "rnfl_mode": "mean",
        },
    ]

    for r in range(1, len(rnfl_sector_cols) + 1):
        for subset in combinations(rnfl_sector_cols, r):
            subset = list(subset)
            suffix = "+".join(c.replace("RNFL_", "") for c in subset)
            spatial_map = _spatialize_available_rnfl_subset(subset, region_aware_rnfl_broad)
            rows.append(
                {
                    "variant": f"VF + Spatial RNFL {suffix}",
                    "uses_iop": False,
                    "rnfl_available": suffix,
                    "region_rnfl_map": spatial_map,
                    "rnfl_mode": "spatial_subset_broad",
                }
            )
            rows.append(
                {
                    "variant": f"VF + IOP + Spatial RNFL {suffix}",
                    "uses_iop": True,
                    "rnfl_available": suffix,
                    "region_rnfl_map": spatial_map,
                    "rnfl_mode": "spatial_subset_broad",
                }
            )

    return pd.DataFrame(rows)


if "universal_metadata_plan" not in globals():
    universal_metadata_plan = build_universal_metadata_plan()

single_vf_metadata_plan = universal_metadata_plan.copy()
print("Single-visit universal metadata plan:")
display(single_vf_metadata_plan[["variant", "uses_iop", "rnfl_available", "rnfl_mode", "region_rnfl_map"]])
print(f"Single-visit universal metadata variants: {len(single_vf_metadata_plan)}")

# Use 1 seed for a quick smoke test; use 5 for the reported table, or 10 for a stability check.
N_SINGLE_VF_SEEDS = 5
SINGLE_VF_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_SINGLE_VF_SEEDS).tolist())
print("Random single-VF metadata seeds:", SINGLE_VF_SEEDS)

single_vf_metadata_outputs = run_region_aware_metadata_screen(
    single_vf_metadata_plan,
    seeds=SINGLE_VF_SEEDS,
    k_list=(1,),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
    time_input_mode="delta_t",
)

single_vf_overall = single_vf_metadata_outputs["overall"].copy()
single_vf_overall = single_vf_overall.merge(
    single_vf_metadata_plan[["variant", "rnfl_available", "rnfl_mode", "uses_iop"]],
    left_on="Model",
    right_on="variant",
    how="left",
).drop(columns=["variant"])

single_vf_display = single_vf_overall[
    [
        "Model",
        "uses_iop",
        "rnfl_available",
        "rnfl_mode",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "delta_vs_iop_display",
        "improved_vs_iop",
        "Samples",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True)

print("Single-visit delta_t-aware universal metadata overall table, sorted by MAE:")
display(single_vf_display)

print("Single-visit delta_t-aware candidates improving over IOP-only baseline:")
display(single_vf_display[single_vf_display["improved_vs_iop"]].reset_index(drop=True))

single_vf_best_by_availability = (
    single_vf_display
    .sort_values("MAE_mean_db")
    .groupby(["uses_iop", "rnfl_available"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
print("Best single-visit delta_t-aware model for each metadata availability pattern:")
display(single_vf_best_by_availability)

single_vf_best_model = single_vf_display.iloc[0]["Model"]
print("Best single-visit delta_t-aware universal-screen overall model:", single_vf_best_model)
single_vf_best_region_table = single_vf_metadata_outputs["summary_long"][
    single_vf_metadata_outputs["summary_long"]["Model"].eq(single_vf_best_model)
].copy()
print("Region-level table for best single-visit delta_t-aware universal-screen model:")
display(single_vf_best_region_table)


## 12. Baseline-only Single-Visit Universal Metadata Screen

This is the stricter baseline-visit setting: for each eye, use only the first available VF visit as input, explicitly provide the future horizon `Δt`, and predict one or more future visits. Metadata is also taken from the baseline input visit. This better matches the clinical setting where RNFL is usually available only at baseline.

The default target mode is `all_future`, so one baseline visit can generate multiple future targets. Change `BASELINE_ONLY_TARGET_MODE` to `next` if you want only baseline -> next visit.


In [ ]:
def build_baseline_only_pairs(df, td_cols, *, target_mode="all_future"):
    """Build baseline-visit forecasting samples.

    For each eye, the first visit is the only input. Targets are either the next visit
    or all future visits. The model time input is delta_t, matching VF + delta_t -> VF(t+delta_t).
    """
    if target_mode not in {"next", "all_future"}:
        raise ValueError("target_mode must be 'next' or 'all_future'")

    rows = []
    for eye_id, g in df.groupby("eye_id"):
        g = g.sort_values(["Time_from_Baseline", "VisitN"]).reset_index(drop=True)
        if len(g) < 2:
            continue

        baseline = g.iloc[0]
        target_positions = [1] if target_mode == "next" else range(1, len(g))
        for target_pos in target_positions:
            target = g.iloc[target_pos]
            delta_t = float(target["Time_from_Baseline"] - baseline["Time_from_Baseline"])
            rows.append(
                {
                    "eye_id": eye_id,
                    "k": 1,
                    "visit_in_last": int(baseline["VisitN"]),
                    "visit_out": int(target["VisitN"]),
                    "time_in_last": float(baseline["Time_from_Baseline"]),
                    "time_out": float(target["Time_from_Baseline"]),
                    "delta_t": delta_t,
                    "input_times": np.array([delta_t], dtype=np.float32),
                    "X": baseline[td_cols].to_numpy(dtype=np.float32)[None, :],
                    "Y": target[td_cols].to_numpy(dtype=np.float32),
                    "meta_sub": np.zeros((0,), dtype=np.float32),
                }
            )
    return pd.DataFrame(rows)


def run_region_aware_baseline_only_metadata_screen(
    metadata_plan,
    *,
    seeds,
    target_mode="all_future",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = 1

    pooled = build_baseline_only_pairs(df_exp, td_cols_exp, target_mode=target_mode)
    pooled = _attach_history_metadata(pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, test_df_all = hist_module._split_by_eye(
        pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    train_df_all, val_df_all, test_df_all, meta_master_cols = _impute_history_metadata_by_train(
        train_df_all, val_df_all, test_df_all, max_k=max_k, rnfl_cols=rnfl_master_cols
    )

    raw_results = []
    run_summaries = []

    print("Baseline-only metadata train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Baseline-only metadata seeds:", seeds)
    print("Target mode:", target_mode)
    print("IOP mode:", iop_mode)
    print("Time input mode: delta_t")

    for _, exp in metadata_plan.iterrows():
        exp_name = exp["variant"]
        uses_iop = bool(exp["uses_iop"])
        region_rnfl_map = dict(exp["region_rnfl_map"])

        print("\n" + "#" * 100)
        print(f"BASELINE-ONLY METADATA: {exp_name} | target_mode={target_mode} | iop_mode={iop_mode}")
        print("#" * 100)

        for seed in seeds:
            print(f"-- seed={seed}")
            hist_module._set_seed(int(seed))

            pred_full_abs = None
            true_full_abs = None

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = list(region_rnfl_map.get(region_name, []))
                print(f"{exp_name} | seed={seed} | region={region_name} | rnfl_cols={rnfl_cols}")

                train_df, selected_meta_cols = _set_meta_sub_by_cols(train_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
                val_df, _ = _set_meta_sub_by_cols(val_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
                test_df, _ = _set_meta_sub_by_cols(test_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode)
                is_vf_only = len(selected_meta_cols) == 0

                x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df)
                x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df)
                x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df)

                train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
                val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
                test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

                x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
                x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
                x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)

                x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

                if meta_train.shape[1] > 0:
                    meta_scaler = StandardScaler()
                    meta_train_s = meta_scaler.fit_transform(meta_train).astype(np.float32)
                    meta_val_s = meta_scaler.transform(meta_val).astype(np.float32)
                    meta_test_s = meta_scaler.transform(meta_test).astype(np.float32)
                else:
                    meta_train_s = meta_train.astype(np.float32)
                    meta_val_s = meta_val.astype(np.float32)
                    meta_test_s = meta_test.astype(np.float32)

                if pred_full_abs is None:
                    pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                    true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

                y_train_r = y_train[:, idx]
                y_val_r = y_val[:, idx]
                y_test_r = y_test[:, idx]

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
                y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
                y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)

                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                if is_vf_only:
                    model = VFOnlyModel(output_dim=len(idx))
                else:
                    model = ResidualLateFusionModel(meta_dim=meta_train_s.shape[1], output_dim=len(idx))

                model, loaded_keys = _load_bern_weights_into_model(model, region_name)
                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"baseline_only_{exp_name.replace(' ', '_').replace('+', 'plus')}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "seed": int(seed),
                    "region": region_name,
                    "region_rnfl_cols": rnfl_cols,
                    "selected_meta_cols": selected_meta_cols,
                    "loaded_bern_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["rnfl_mode"] = exp["rnfl_mode"]
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()

    vf_baseline_mae = float(overall.loc[overall["Model"].eq("VF only"), "MAE_mean_db"].iloc[0])
    iop_baseline_rows = overall[overall["Model"].isin(["VF + IOP", "VF + IOP history"])]
    if len(iop_baseline_rows) == 0:
        raise ValueError("No IOP-only baseline found. Expected model name 'VF + IOP' or 'VF + IOP history'.")
    iop_baseline_mae = float(iop_baseline_rows.sort_values("MAE_mean_db")["MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - vf_baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall["delta_vs_iop"] = overall["MAE_mean_db"] - iop_baseline_mae
    overall["improved_vs_iop"] = overall["delta_vs_iop"] < 0
    overall["delta_vs_iop_display"] = overall["delta_vs_iop"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(train_df_all),
        "val_samples": len(val_df_all),
        "test_samples": len(test_df_all),
        "seeds": seeds,
        "iop_mode": iop_mode,
        "target_mode": target_mode,
        "time_input_mode": "delta_t",
    }


if "universal_metadata_plan" not in globals():
    universal_metadata_plan = build_universal_metadata_plan()

baseline_only_metadata_plan = universal_metadata_plan.copy()
print("Baseline-only universal metadata plan:")
display(baseline_only_metadata_plan[["variant", "uses_iop", "rnfl_available", "rnfl_mode", "region_rnfl_map"]])
print(f"Baseline-only universal metadata variants: {len(baseline_only_metadata_plan)}")

# Use 1 seed for a quick smoke test; use 5 for the reported table, or 10 for a stability check.
N_BASELINE_ONLY_SEEDS = 5
BASELINE_ONLY_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_BASELINE_ONLY_SEEDS).tolist())
BASELINE_ONLY_TARGET_MODE = "all_future"
print("Random baseline-only metadata seeds:", BASELINE_ONLY_SEEDS)

baseline_only_metadata_outputs = run_region_aware_baseline_only_metadata_screen(
    baseline_only_metadata_plan,
    seeds=BASELINE_ONLY_SEEDS,
    target_mode=BASELINE_ONLY_TARGET_MODE,
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
)

baseline_only_overall = baseline_only_metadata_outputs["overall"].copy()
baseline_only_overall = baseline_only_overall.merge(
    baseline_only_metadata_plan[["variant", "rnfl_available", "rnfl_mode", "uses_iop"]],
    left_on="Model",
    right_on="variant",
    how="left",
).drop(columns=["variant"])

baseline_only_display = baseline_only_overall[
    [
        "Model",
        "uses_iop",
        "rnfl_available",
        "rnfl_mode",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "delta_vs_iop_display",
        "improved_vs_iop",
        "Samples",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True)

print("Baseline-only delta_t-aware universal metadata overall table, sorted by MAE:")
display(baseline_only_display)

print("Baseline-only delta_t-aware candidates improving over IOP-only baseline:")
display(baseline_only_display[baseline_only_display["improved_vs_iop"]].reset_index(drop=True))

baseline_only_best_by_availability = (
    baseline_only_display
    .sort_values("MAE_mean_db")
    .groupby(["uses_iop", "rnfl_available"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
print("Best baseline-only model for each metadata availability pattern:")
display(baseline_only_best_by_availability)

baseline_only_best_model = baseline_only_display.iloc[0]["Model"]
print("Best baseline-only universal-screen overall model:", baseline_only_best_model)
baseline_only_best_region_table = baseline_only_metadata_outputs["summary_long"][
    baseline_only_metadata_outputs["summary_long"]["Model"].eq(baseline_only_best_model)
].copy()
print("Region-level table for best baseline-only universal-screen model:")
display(baseline_only_best_region_table)



## 13. Legacy-compatible Final Metadata Benchmark

This block reproduces the final metadata-model logic from `4_GRAPE_new.ipynb`: GRAPE-trained region-specific models with late fusion and region-aware RNFL, using `k=1,2,3,4`. This is intentionally separate from the Bern fine-tuned VF-only baseline experiment above.


In [ ]:
import importlib
import notebooks2.grape_region_full_experiment as grape_region_full_module
grape_region_full_module = importlib.reload(grape_region_full_module)

RegionFullConfig = grape_region_full_module.RegionFullConfig
run_region_full_experiment = grape_region_full_module.run_region_full_experiment

# Random seeds each run. Set to 3 if you want to match the old notebook more closely.
N_LEGACY_METADATA_SEEDS = 5
LEGACY_METADATA_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_LEGACY_METADATA_SEEDS).tolist())
print("Random legacy-compatible metadata seeds:", LEGACY_METADATA_SEEDS)

legacy_final_model_names = [
    "VF only",
    "LateFusion IOP",
    "RegionAware LateFusion RNFL SNIT",
    "RegionAware LateFusion RNFL SNIT + IOP",
]

legacy_metadata_config = RegionFullConfig(
    k_list=(1, 2, 3, 4),
    seeds=LEGACY_METADATA_SEEDS,
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    khist_mode="consecutive",
)

legacy_metadata_outputs = run_region_full_experiment(
    df_source=grape_longitudinal_td_aligned_df,
    meta_train_grape=meta_train_grape,
    meta_val_grape=meta_val_grape,
    meta_test_grape=meta_test_grape,
    cluster_mapping=cluster_mapping,
    build_sequences_generic_fn=build_sequences_generic_old_style,
    prepare_td_vector_fn=prepare_td_vector,
    config=legacy_metadata_config,
    selected_model_names=legacy_final_model_names,
)

legacy_metadata_summary_long = legacy_metadata_outputs["summary_long"]
legacy_metadata_summary_wide = legacy_metadata_outputs["summary_wide"]
legacy_metadata_runs_df = legacy_metadata_outputs["runs_df"]
legacy_metadata_history_df = legacy_metadata_outputs["history_df"]

print("Legacy-compatible metadata overall results:")
display(legacy_metadata_summary_long[legacy_metadata_summary_long["Region"].eq("Overall")].reset_index(drop=True))

print("Legacy-compatible metadata wide table:")
display(legacy_metadata_summary_wide)


### Normalization diagnostic for architecture review

Checks scaled VF input, metadata, and target ranges using train-fit scalers and val/test transforms.


In [ ]:
def describe_scaled_array(name, arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.size == 0:
        return {
            "name": name,
            "shape": tuple(arr.shape),
            "min": np.nan,
            "max": np.nan,
            "mean": np.nan,
            "std": np.nan,
        }
    return {
        "name": name,
        "shape": tuple(arr.shape),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "mean": float(np.nanmean(arr)),
        "std": float(np.nanstd(arr)),
    }


def flatten_sequence_list(x_list):
    return np.concatenate([np.asarray(x, dtype=np.float32) for x in x_list], axis=0)


def normalization_diagnostic_for_teacher(
    metadata_plan,
    *,
    variant="VF + IOP + Spatial RNFL S+N+I+T",
    region_name="Supero_Nasal",
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    iop_mode="history",
    time_input_mode="input_times",
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)
    train_pooled = _set_model_time_inputs(train_pooled, mode=time_input_mode)
    eval_pooled = _set_model_time_inputs(eval_pooled, mode=time_input_mode)

    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    train_df_all, val_df_all, test_df_all, meta_master_cols = _impute_history_metadata_by_train(
        train_df_all, val_df_all, test_df_all, max_k=max_k, rnfl_cols=rnfl_master_cols
    )

    exp_row = metadata_plan.loc[metadata_plan["variant"].eq(variant)].iloc[0]
    uses_iop = bool(exp_row["uses_iop"])
    region_rnfl_map = dict(exp_row["region_rnfl_map"])
    rnfl_cols = list(region_rnfl_map.get(region_name, []))
    idx = np.asarray(cluster_mapping[region_name], dtype=int)

    train_df, selected_meta_cols = _set_meta_sub_by_cols(
        train_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode
    )
    val_df, _ = _set_meta_sub_by_cols(
        val_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode
    )
    test_df, _ = _set_meta_sub_by_cols(
        test_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode
    )

    x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df)
    x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df)
    x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

    x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
    x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
    x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    if meta_train.shape[1] > 0:
        meta_scaler = StandardScaler()
        meta_train_s = meta_scaler.fit_transform(meta_train).astype(np.float32)
        meta_val_s = meta_scaler.transform(meta_val).astype(np.float32)
        meta_test_s = meta_scaler.transform(meta_test).astype(np.float32)
    else:
        meta_train_s = meta_train.astype(np.float32)
        meta_val_s = meta_val.astype(np.float32)
        meta_test_s = meta_test.astype(np.float32)

    y_train_r = y_train[:, idx]
    y_val_r = y_val[:, idx]
    y_test_r = y_test[:, idx]

    y_scaler = StandardScaler()
    y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
    y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
    y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

    stats = pd.DataFrame([
        describe_scaled_array("VF input train scaled", flatten_sequence_list(x_train_s)),
        describe_scaled_array("VF input val scaled", flatten_sequence_list(x_val_s)),
        describe_scaled_array("VF input test scaled", flatten_sequence_list(x_test_s)),
        describe_scaled_array("metadata train scaled", meta_train_s),
        describe_scaled_array("metadata val scaled", meta_val_s),
        describe_scaled_array("metadata test scaled", meta_test_s),
        describe_scaled_array("target train scaled", y_train_s),
        describe_scaled_array("target val scaled", y_val_s),
        describe_scaled_array("target test scaled", y_test_s),
    ])

    return {
        "variant": variant,
        "region_name": region_name,
        "selected_meta_cols": selected_meta_cols,
        "stats": stats,
    }


normalization_check = normalization_diagnostic_for_teacher(universal_metadata_plan)
print("Variant:", normalization_check["variant"])
print("Region:", normalization_check["region_name"])
print("Selected metadata columns:", normalization_check["selected_meta_cols"])
display(normalization_check["stats"])


### Per-column metadata normalization diagnostic

Checks which metadata columns contribute to the largest scaled values.


In [ ]:
def metadata_column_diagnostic_for_teacher(
    metadata_plan,
    *,
    variant="VF + IOP + Spatial RNFL S+N+I+T",
    region_name="Supero_Nasal",
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    iop_mode="history",
    time_input_mode="input_times",
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)
    train_pooled = _set_model_time_inputs(train_pooled, mode=time_input_mode)
    eval_pooled = _set_model_time_inputs(eval_pooled, mode=time_input_mode)

    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    train_df_all, val_df_all, test_df_all, meta_master_cols = _impute_history_metadata_by_train(
        train_df_all, val_df_all, test_df_all, max_k=max_k, rnfl_cols=rnfl_master_cols
    )

    exp_row = metadata_plan.loc[metadata_plan["variant"].eq(variant)].iloc[0]
    uses_iop = bool(exp_row["uses_iop"])
    region_rnfl_map = dict(exp_row["region_rnfl_map"])
    rnfl_cols = list(region_rnfl_map.get(region_name, []))

    train_df, selected_meta_cols = _set_meta_sub_by_cols(
        train_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode
    )
    val_df, _ = _set_meta_sub_by_cols(
        val_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode
    )
    test_df, _ = _set_meta_sub_by_cols(
        test_df_all, meta_master_cols, uses_iop, rnfl_cols, iop_mode=iop_mode
    )

    _, _, _, meta_train = _df_to_arrays_with_history(train_df)
    _, _, _, meta_val = _df_to_arrays_with_history(val_df)
    _, _, _, meta_test = _df_to_arrays_with_history(test_df)

    if meta_train.shape[1] == 0:
        return pd.DataFrame()

    meta_scaler = StandardScaler()
    meta_train_s = meta_scaler.fit_transform(meta_train).astype(np.float32)
    meta_val_s = meta_scaler.transform(meta_val).astype(np.float32)
    meta_test_s = meta_scaler.transform(meta_test).astype(np.float32)

    rows = []
    for split_name, arr in [
        ("train", meta_train_s),
        ("val", meta_val_s),
        ("test", meta_test_s),
    ]:
        for j, col in enumerate(selected_meta_cols):
            values = arr[:, j]
            rows.append({
                "split": split_name,
                "column": col,
                "min": float(np.nanmin(values)),
                "max": float(np.nanmax(values)),
                "mean": float(np.nanmean(values)),
                "std": float(np.nanstd(values)),
                "abs_max": float(np.nanmax(np.abs(values))),
            })

    return pd.DataFrame(rows).sort_values(["split", "abs_max"], ascending=[True, False]).reset_index(drop=True)


metadata_column_check = metadata_column_diagnostic_for_teacher(universal_metadata_plan)
display(metadata_column_check)


### Per-visit IOP early fusion diagnostic

This is the first test for the teacher's per-visit metadata alignment suggestion. IOP is visit-specific, so here each VF timestep receives the IOP value from the same visit before the LSTM. RNFL is not included in this first version because it is treated as static structural metadata in our current data.


In [ ]:
# ---------------------------------------------------------------------------
# Per-visit IOP early fusion
# ---------------------------------------------------------------------------
from torch.nn.utils.rnn import pad_sequence


class PerVisitIOPHistoryDataset(torch.utils.data.Dataset):
    def __init__(self, x_list, y_array, t_list, iop_seq_list):
        self.x_list = x_list
        self.y_array = y_array
        self.t_list = t_list
        self.iop_seq_list = iop_seq_list

    def __len__(self):
        return len(self.x_list)

    def __getitem__(self, idx):
        x = torch.tensor(self.x_list[idx], dtype=torch.float32)
        y = torch.tensor(self.y_array[idx], dtype=torch.float32)
        t = torch.tensor(self.t_list[idx], dtype=torch.float32)
        iop_seq = torch.tensor(self.iop_seq_list[idx], dtype=torch.float32)
        length = torch.tensor(len(self.x_list[idx]), dtype=torch.long)
        return x, t, length, y, iop_seq


def per_visit_iop_collate(batch):
    xs, ts, lens, ys, iop_seqs = zip(*batch)
    lengths = torch.stack(lens, dim=0)
    order = torch.argsort(lengths, descending=True)

    xs = [xs[i] for i in order]
    ts = [ts[i] for i in order]
    ys = [ys[i] for i in order]
    iop_seqs = [iop_seqs[i] for i in order]
    lengths = lengths[order]

    padded_x = pad_sequence(xs, batch_first=True)
    padded_t = pad_sequence(ts, batch_first=True)
    padded_iop = pad_sequence(iop_seqs, batch_first=True)
    targets = torch.stack(ys, dim=0)
    return padded_x, padded_t, lengths, targets, padded_iop


class PerVisitIOPPreEncoderFusionModel(nn.Module):
    """Exact early fusion: append visit-aligned IOP to VF input before the encoder.

    This matches the teacher's suggestion more directly: each timestep is encoded from
    joint raw features [VF_t, IOP_t]. The VF encoder input dimension changes from 59 to
    60, so the original Bern encoder weights cannot be loaded for this variant.
    TimePE, LSTM, and decoder weights are still loaded when shapes match.
    """
    def __init__(
        self,
        input_dim=59,
        iop_dim=1,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
    ):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim + iop_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm = nn.LSTM(
            input_size=latent_dim + pe_output_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)

    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        seq_plus_iop = torch.cat([padded_seqs, iop_seq], dim=-1)
        vf_meta_emb = self.encoder(seq_plus_iop)
        time_emb = self.time_pe(padded_times)
        lstm_in = torch.cat([vf_meta_emb, time_emb], dim=-1)

        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        return self.decoder(h_n[-1])


class PerVisitIOPEarlyFusionModel(nn.Module):
    """Bern VF-time backbone plus visit-aligned IOP adapter before the LSTM.

    The metadata adapter is initialized at zero, so the initial forward pass is exactly
    the Bern VF-only path. During fine-tuning, the model can learn how much each
    visit-specific IOP value should perturb the LSTM input at the matching timestep.
    """
    def __init__(
        self,
        iop_dim=1,
        input_dim=59,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
        meta_hidden_dim=32,
        meta_emb_dim=16,
    ):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm_input_dim = latent_dim + pe_output_dim
        self.iop_encoder = MetaEncoder(input_dim=iop_dim, hidden_dim=meta_hidden_dim, output_dim=meta_emb_dim)
        self.iop_adapter = nn.Linear(meta_emb_dim, self.lstm_input_dim)
        self.lstm = nn.LSTM(
            input_size=self.lstm_input_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)

        nn.init.zeros_(self.iop_adapter.weight)
        nn.init.zeros_(self.iop_adapter.bias)

    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        vf_emb = self.encoder(padded_seqs)
        time_emb = self.time_pe(padded_times)
        base_lstm_in = torch.cat([vf_emb, time_emb], dim=-1)

        iop_emb = self.iop_encoder(iop_seq)
        lstm_in = base_lstm_in + self.iop_adapter(iop_emb)

        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        return self.decoder(h_n[-1])


class PerVisitIOPGatedEarlyFusionModel(nn.Module):
    """Per-visit IOP fusion with a learned sigmoid gate before the LSTM.

    The value adapter and gate input weights are initialized at zero. The gate bias is
    initialized negative, so the model starts close to the VF-only Bern path and can
    learn to open the IOP gate only if it helps during fine-tuning.
    """
    def __init__(
        self,
        iop_dim=1,
        input_dim=59,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
        meta_hidden_dim=32,
        meta_emb_dim=16,
        gate_init_bias=-4.0,
    ):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm_input_dim = latent_dim + pe_output_dim
        self.iop_encoder = MetaEncoder(input_dim=iop_dim, hidden_dim=meta_hidden_dim, output_dim=meta_emb_dim)
        self.iop_value = nn.Linear(meta_emb_dim, self.lstm_input_dim)
        self.iop_gate = nn.Linear(meta_emb_dim, self.lstm_input_dim)
        self.lstm = nn.LSTM(
            input_size=self.lstm_input_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)

        nn.init.zeros_(self.iop_value.weight)
        nn.init.zeros_(self.iop_value.bias)
        nn.init.zeros_(self.iop_gate.weight)
        nn.init.constant_(self.iop_gate.bias, gate_init_bias)

    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        vf_emb = self.encoder(padded_seqs)
        time_emb = self.time_pe(padded_times)
        base_lstm_in = torch.cat([vf_emb, time_emb], dim=-1)

        iop_emb = self.iop_encoder(iop_seq)
        iop_update = self.iop_value(iop_emb)
        iop_gate = torch.sigmoid(self.iop_gate(iop_emb))
        lstm_in = base_lstm_in + iop_gate * iop_update

        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        return self.decoder(h_n[-1])


class PerVisitIOPCrossAttentionFusionModel(nn.Module):
    """Minimal per-visit IOP cross-attention before the LSTM.

    VF-time tokens are queries. IOP-time tokens are keys/values. The attention update is
    projected back to the Bern LSTM input size with a zero-initialized adapter, so the
    model starts from the VF-only Bern path and learns attention contribution only if useful.
    """
    def __init__(
        self,
        iop_dim=1,
        input_dim=59,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
        meta_hidden_dim=32,
        meta_emb_dim=16,
        attn_dim=80,
        num_heads=4,
    ):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm_input_dim = latent_dim + pe_output_dim
        self.iop_encoder = MetaEncoder(input_dim=iop_dim, hidden_dim=meta_hidden_dim, output_dim=meta_emb_dim)
        self.iop_time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.iop_token_proj = nn.Sequential(
            nn.Linear(meta_emb_dim + pe_output_dim, attn_dim),
            nn.ReLU(),
        )
        self.cross_attn = nn.MultiheadAttention(embed_dim=attn_dim, num_heads=num_heads, batch_first=True)
        self.attn_adapter = nn.Linear(attn_dim, self.lstm_input_dim)
        self.lstm = nn.LSTM(
            input_size=self.lstm_input_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)

        nn.init.zeros_(self.attn_adapter.weight)
        nn.init.zeros_(self.attn_adapter.bias)

    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        vf_emb = self.encoder(padded_seqs)
        time_emb = self.time_pe(padded_times)
        vf_tokens = torch.cat([vf_emb, time_emb], dim=-1)

        iop_emb = self.iop_encoder(iop_seq)
        iop_time_emb = self.iop_time_pe(padded_times)
        iop_tokens = self.iop_token_proj(torch.cat([iop_emb, iop_time_emb], dim=-1))

        batch_size, seq_len, _ = vf_tokens.shape
        step_ids = torch.arange(seq_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
        key_padding_mask = step_ids >= lengths.unsqueeze(1)

        attn_out, _ = self.cross_attn(
            query=vf_tokens,
            key=iop_tokens,
            value=iop_tokens,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        lstm_in = vf_tokens + self.attn_adapter(attn_out)

        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        return self.decoder(h_n[-1])


class PerVisitIOPMetaQueryCrossAttentionFusionModel(nn.Module):
    """Per-visit IOP cross-attention with metadata as query.

    IOP-time tokens are queries. VF-time tokens are keys/values. This tests the
    supervisor-suggested direction where metadata conditions which VF-history
    information is selected before the LSTM.
    """
    def __init__(
        self,
        iop_dim=1,
        input_dim=59,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
        meta_hidden_dim=32,
        meta_emb_dim=16,
        attn_dim=80,
        num_heads=4,
    ):
        super().__init__()
        self.encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm_input_dim = latent_dim + pe_output_dim
        self.iop_encoder = MetaEncoder(input_dim=iop_dim, hidden_dim=meta_hidden_dim, output_dim=meta_emb_dim)
        self.iop_time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.iop_query_proj = nn.Sequential(
            nn.Linear(meta_emb_dim + pe_output_dim, attn_dim),
            nn.ReLU(),
        )
        self.cross_attn = nn.MultiheadAttention(embed_dim=attn_dim, num_heads=num_heads, batch_first=True)
        self.attn_adapter = nn.Linear(attn_dim, self.lstm_input_dim)
        self.lstm = nn.LSTM(
            input_size=self.lstm_input_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)

        nn.init.zeros_(self.attn_adapter.weight)
        nn.init.zeros_(self.attn_adapter.bias)

    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        vf_emb = self.encoder(padded_seqs)
        time_emb = self.time_pe(padded_times)
        vf_tokens = torch.cat([vf_emb, time_emb], dim=-1)

        iop_emb = self.iop_encoder(iop_seq)
        iop_time_emb = self.iop_time_pe(padded_times)
        iop_queries = self.iop_query_proj(torch.cat([iop_emb, iop_time_emb], dim=-1))

        batch_size, seq_len, _ = vf_tokens.shape
        step_ids = torch.arange(seq_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
        key_padding_mask = step_ids >= lengths.unsqueeze(1)

        attn_out, _ = self.cross_attn(
            query=iop_queries,
            key=vf_tokens,
            value=vf_tokens,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        lstm_in = vf_tokens + self.attn_adapter(attn_out)

        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        return self.decoder(h_n[-1])


class PointLevelSpatioTemporalTransformerModel(nn.Module):
    """Point-level spatio-temporal Transformer for VF forecasting.

    Each input token is one VF test location at one visit. The token contains:
    VF value + point identity + visit time + visit-aligned IOP. A Transformer encoder
    learns point-to-point and visit-to-visit relationships. Region-specific output
    queries then cross-attend to the encoded history tokens and predict the target
    VF values for that anatomical region.
    """
    def __init__(
        self,
        input_dim=59,
        output_dim=59,
        point_indices=None,
        model_dim=64,
        num_heads=4,
        num_layers=2,
        ff_dim=128,
        dropout=0.1,
        pe_input_dim=16,
        meta_hidden_dim=32,
        meta_input_dim=1,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        if point_indices is None:
            point_indices = list(range(output_dim))
        self.register_buffer("point_indices", torch.tensor(point_indices, dtype=torch.long))

        self.value_encoder = nn.Sequential(
            nn.Linear(1, model_dim),
            nn.GELU(),
            nn.Linear(model_dim, model_dim),
        )
        self.point_embed = nn.Embedding(input_dim, model_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=model_dim)
        self.iop_encoder = MetaEncoder(input_dim=meta_input_dim, hidden_dim=meta_hidden_dim, output_dim=model_dim)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.output_point_embed = nn.Embedding(input_dim, model_dim)
        self.output_attn = nn.MultiheadAttention(model_dim, num_heads, batch_first=True, dropout=dropout)
        self.norm1 = nn.LayerNorm(model_dim)
        self.norm2 = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, model_dim),
            nn.Dropout(dropout),
        )
        self.out = nn.Linear(model_dim, 1)

    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        batch_size, seq_len, n_points = padded_seqs.shape
        values = padded_seqs.unsqueeze(-1)
        value_tokens = self.value_encoder(values)

        point_ids = torch.arange(n_points, device=padded_seqs.device)
        point_tokens = self.point_embed(point_ids).view(1, 1, n_points, -1)
        time_tokens = self.time_pe(padded_times).unsqueeze(2)
        iop_tokens = self.iop_encoder(iop_seq).unsqueeze(2)

        tokens = value_tokens + point_tokens + time_tokens + iop_tokens
        tokens = tokens.reshape(batch_size, seq_len * n_points, -1)

        step_ids = torch.arange(seq_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
        visit_mask = step_ids >= lengths.unsqueeze(1)
        token_mask = visit_mask.unsqueeze(-1).expand(batch_size, seq_len, n_points).reshape(batch_size, seq_len * n_points)

        encoded = self.transformer(tokens, src_key_padding_mask=token_mask)

        query_points = self.point_indices.to(padded_seqs.device)
        queries = self.output_point_embed(query_points).unsqueeze(0).expand(batch_size, -1, -1)
        attn_out, _ = self.output_attn(
            query=queries,
            key=encoded,
            value=encoded,
            key_padding_mask=token_mask,
            need_weights=False,
        )
        x = self.norm1(queries + attn_out)
        x = x + self.ffn(self.norm2(x))
        return self.out(x).squeeze(-1)


class PointLevelSpatioTemporalTransformerVFOnlyModel(PointLevelSpatioTemporalTransformerModel):
    """Same point-level Transformer, but the metadata/IOP token is forced to zero."""
    def forward(self, padded_seqs, padded_times, lengths, iop_seq):
        zero_iop = torch.zeros_like(iop_seq)
        return super().forward(padded_seqs, padded_times, lengths, zero_iop)


def _load_bern_point_transformer_weights(model, region_name, checkpoint_prefix="bern_point_transformer"):
    """Load Bern-pretrained point-level Transformer weights if they exist."""
    ckpt = project_root / "models" / f"{checkpoint_prefix}_{region_name}_best.pth"
    if not ckpt.exists():
        print(f"No Bern point-level Transformer checkpoint for {region_name}: {ckpt}")
        return model, []

    source_state = torch.load(ckpt, map_location=device)
    target_state = model.state_dict()
    matched = {
        k: v
        for k, v in source_state.items()
        if k in target_state and tuple(v.shape) == tuple(target_state[k].shape)
    }
    target_state.update(matched)
    model.load_state_dict(target_state)
    return model, sorted(matched.keys())


def _extract_iop_sequence_list(df_part):
    iop_seq_list = []
    for _, row in df_part.iterrows():
        seq_len = int(np.asarray(row["X"]).shape[0])
        hist = np.asarray(row["iop_hist_padded"], dtype=np.float32)
        seq = hist[-seq_len:]
        if seq.shape[0] < seq_len:
            pad = np.full(seq_len - seq.shape[0], np.nan, dtype=np.float32)
            seq = np.concatenate([pad, seq], axis=0)
        iop_seq_list.append(seq.reshape(-1, 1).astype(np.float32))
    return iop_seq_list


def _mask_iop_sequence_list(iop_seq_list, mask):
    idx = np.where(mask)[0]
    return [iop_seq_list[i] for i in idx]


def _scale_iop_sequence_lists(train_seq, val_seq, test_seq):
    train_stack = np.concatenate(train_seq, axis=0).astype(np.float32)
    train_mean = np.nanmean(train_stack, axis=0)
    train_mean = np.where(np.isfinite(train_mean), train_mean, 0.0).astype(np.float32)

    def _fill(seq_list):
        filled = []
        for seq in seq_list:
            seq = np.asarray(seq, dtype=np.float32)
            seq = np.where(np.isfinite(seq), seq, train_mean)
            filled.append(seq.astype(np.float32))
        return filled

    train_filled = _fill(train_seq)
    val_filled = _fill(val_seq)
    test_filled = _fill(test_seq)

    scaler = StandardScaler()
    scaler.fit(np.concatenate(train_filled, axis=0))

    train_s = [scaler.transform(seq).astype(np.float32) for seq in train_filled]
    val_s = [scaler.transform(seq).astype(np.float32) for seq in val_filled]
    test_s = [scaler.transform(seq).astype(np.float32) for seq in test_filled]
    return train_s, val_s, test_s, scaler


def _iop_history_matrix(df_part):
    return np.stack([np.asarray(v, dtype=np.float32) for v in df_part["iop_hist_padded"].values]).astype(np.float32)


def _scale_iop_history_matrices(train_meta, val_meta, test_meta):
    train_mean = np.nanmean(train_meta, axis=0)
    train_mean = np.where(np.isfinite(train_mean), train_mean, 0.0).astype(np.float32)

    def _fill(arr):
        return np.where(np.isfinite(arr), arr, train_mean).astype(np.float32)

    train_filled = _fill(train_meta)
    val_filled = _fill(val_meta)
    test_filled = _fill(test_meta)

    scaler = StandardScaler()
    train_s = scaler.fit_transform(train_filled).astype(np.float32)
    val_s = scaler.transform(val_filled).astype(np.float32)
    test_s = scaler.transform(test_filled).astype(np.float32)
    return train_s, val_s, test_s, scaler


def run_per_visit_iop_early_fusion_screen(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    include_point_level_transformer=False,
    include_compact_point_transformer=False,
    selected_kinds=None,
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)
    train_pooled = _set_model_time_inputs(train_pooled, mode=time_input_mode)
    eval_pooled = _set_model_time_inputs(eval_pooled, mode=time_input_mode)

    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_iop_seq = _extract_iop_sequence_list(train_df_all)
    val_iop_seq = _extract_iop_sequence_list(val_df_all)
    test_iop_seq = _extract_iop_sequence_list(test_df_all)
    train_iop_history = _iop_history_matrix(train_df_all)
    val_iop_history = _iop_history_matrix(val_df_all)
    test_iop_history = _iop_history_matrix(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_iop_seq = _mask_iop_sequence_list(train_iop_seq, train_mask)
    val_iop_seq = _mask_iop_sequence_list(val_iop_seq, val_mask)
    test_iop_seq = _mask_iop_sequence_list(test_iop_seq, test_mask)
    train_iop_history = train_iop_history[train_mask]
    val_iop_history = val_iop_history[val_mask]
    test_iop_history = test_iop_history[test_mask]

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
    train_iop_s, val_iop_s, test_iop_s, iop_seq_scaler = _scale_iop_sequence_lists(train_iop_seq, val_iop_seq, test_iop_seq)
    train_iop_history_s, val_iop_history_s, test_iop_history_s, iop_history_scaler = _scale_iop_history_matrices(
        train_iop_history, val_iop_history, test_iop_history
    )

    variants = [
        {"Model": "VF only", "kind": "vf_only"},
        {"Model": "VF + IOP history late fusion", "kind": "late_iop_history"},
        {"Model": "VF + per-visit IOP pre-encoder fusion", "kind": "per_visit_iop_pre_encoder"},
        {"Model": "VF + per-visit IOP early fusion", "kind": "per_visit_iop"},
        {"Model": "VF + per-visit IOP gated early fusion", "kind": "per_visit_iop_gated"},
        {"Model": "VF + per-visit IOP cross-attention (VF query)", "kind": "per_visit_iop_cross_attention"},
        {"Model": "VF + per-visit IOP cross-attention (metadata query)", "kind": "per_visit_iop_cross_attention_meta_query"},
    ]
    if include_point_level_transformer:
        variants.append({"Model": "VF + per-visit IOP point-level Transformer", "kind": "point_level_transformer"})
    if include_compact_point_transformer:
        variants.append({"Model": "Compact point-level Transformer VF-only", "kind": "point_level_transformer_compact_vf_only"})
        variants.append({"Model": "VF + per-visit IOP compact point-level Transformer", "kind": "point_level_transformer_compact"})

    if selected_kinds is not None:
        selected_kinds = set(selected_kinds)
        variants = [v for v in variants if v["kind"] in selected_kinds]
        if not variants:
            raise ValueError("selected_kinds removed all variants")

    raw_results = []
    run_summaries = []
    print("Per-visit IOP train samples:", len(x_train_s), "| val:", len(x_val_s), "| test:", len(x_test_s))
    print("Per-visit IOP seeds:", seeds)
    print("Training mode:", train_mode, "| eval mode:", eval_mode, "| time input:", time_input_mode)
    print("IOP sequence scaler mean/scale:", iop_seq_scaler.mean_, iop_seq_scaler.scale_)
    print("IOP history late-fusion scaler mean/scale:", iop_history_scaler.mean_, iop_history_scaler.scale_)

    for variant in variants:
        exp_name = variant["Model"]
        exp_kind = variant["kind"]
        print("\n" + "#" * 100)
        print(f"PER-VISIT IOP: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            print(f"-- seed={seed}")
            hist_module._set_seed(int(seed))
            pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                print(f"{exp_name} | seed={seed} | region={region_name}")

                y_train_r = y_train[:, idx]
                y_val_r = y_val[:, idx]
                y_test_r = y_test[:, idx]

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
                y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
                y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

                if exp_kind in {"vf_only", "per_visit_iop_pre_encoder", "per_visit_iop", "per_visit_iop_gated", "per_visit_iop_cross_attention", "per_visit_iop_cross_attention_meta_query", "point_level_transformer", "point_level_transformer_compact", "point_level_transformer_compact_vf_only"}:
                    train_ds = PerVisitIOPHistoryDataset(x_train_s, y_train_s, t_train, train_iop_s)
                    val_ds = PerVisitIOPHistoryDataset(x_val_s, y_val_s, t_val, val_iop_s)
                    test_ds = PerVisitIOPHistoryDataset(x_test_s, y_test_s, t_test, test_iop_s)
                    collate_fn = per_visit_iop_collate
                elif exp_kind == "late_iop_history":
                    train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, train_iop_history_s)
                    val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, val_iop_history_s)
                    test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, test_iop_history_s)
                    collate_fn = region_collate
                else:
                    raise ValueError(f"Unknown variant kind: {exp_kind}")

                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

                if exp_kind == "vf_only":
                    model = VFOnlyModel(output_dim=len(idx))
                elif exp_kind == "per_visit_iop_pre_encoder":
                    model = PerVisitIOPPreEncoderFusionModel(output_dim=len(idx))
                elif exp_kind == "late_iop_history":
                    model = ResidualLateFusionModel(meta_dim=train_iop_history_s.shape[1], output_dim=len(idx))
                elif exp_kind == "per_visit_iop":
                    model = PerVisitIOPEarlyFusionModel(output_dim=len(idx))
                elif exp_kind == "per_visit_iop_gated":
                    model = PerVisitIOPGatedEarlyFusionModel(output_dim=len(idx))
                elif exp_kind == "per_visit_iop_cross_attention":
                    model = PerVisitIOPCrossAttentionFusionModel(output_dim=len(idx))
                elif exp_kind == "per_visit_iop_cross_attention_meta_query":
                    model = PerVisitIOPMetaQueryCrossAttentionFusionModel(output_dim=len(idx))
                elif exp_kind == "point_level_transformer":
                    model = PointLevelSpatioTemporalTransformerModel(output_dim=len(idx), point_indices=idx.tolist())
                elif exp_kind == "point_level_transformer_compact":
                    model = PointLevelSpatioTemporalTransformerModel(
                        output_dim=len(idx),
                        point_indices=idx.tolist(),
                        model_dim=32,
                        num_heads=2,
                        num_layers=1,
                        ff_dim=64,
                        dropout=0.2,
                        meta_hidden_dim=16,
                    )
                elif exp_kind == "point_level_transformer_compact_vf_only":
                    model = PointLevelSpatioTemporalTransformerVFOnlyModel(
                        output_dim=len(idx),
                        point_indices=idx.tolist(),
                        model_dim=32,
                        num_heads=2,
                        num_layers=1,
                        ff_dim=64,
                        dropout=0.2,
                        meta_hidden_dim=16,
                    )
                else:
                    raise ValueError(f"Unknown variant kind: {exp_kind}")

                if exp_kind == "point_level_transformer":
                    model, loaded_keys = _load_bern_point_transformer_weights(model, region_name)
                elif exp_kind in {"point_level_transformer_compact", "point_level_transformer_compact_vf_only"}:
                    model, loaded_keys = _load_bern_point_transformer_weights(
                        model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                    )
                else:
                    model, loaded_keys = _load_bern_weights_into_model(model, region_name)

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"per_visit_iop_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()

    if overall["Model"].eq("VF only").any():
        baseline_label = "VF only"
    elif overall["Model"].eq("Compact point-level Transformer VF-only").any():
        baseline_label = "Compact point-level Transformer VF-only"
    else:
        baseline_label = overall.sort_values("MAE_mean_db")["Model"].iloc[0]

    vf_baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - vf_baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
        "train_mode": train_mode,
        "eval_mode": eval_mode,
        "time_input_mode": time_input_mode,
        "iop_seq_scaler": iop_seq_scaler,
    }







In [ ]:
# Quick run: start with 1 seed to check that the implementation works.
# For reported numbers, set N_PER_VISIT_IOP_SEEDS = 5 or 10.
N_PER_VISIT_IOP_SEEDS = 1
PER_VISIT_IOP_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_PER_VISIT_IOP_SEEDS).tolist())
print("Random per-visit IOP seeds:", PER_VISIT_IOP_SEEDS)

per_visit_iop_outputs = run_per_visit_iop_early_fusion_screen(
    seeds=PER_VISIT_IOP_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
)

display(per_visit_iop_outputs["overall"])



## 14. Multi-Visit Universal Metadata Screen Without Bern Fine-Tuning

This repeats the multi-visit universal metadata-combination screen without loading Bern pretrained weights. The model is trained from scratch on the GRAPE train split.

The goal is to isolate whether metadata helps more when the backbone is not already initialized from the strong Bern VF-only pretrained model. Everything else is kept the same as the previous multi-visit screen: same split, same region-specific setup, same metadata availability plan, same train/validation/test protocol, and same evaluation metric.


In [ ]:
def run_region_aware_metadata_screen_from_scratch(metadata_plan, **kwargs):
    """Run the existing metadata screen without Bern pretrained initialization.

    This keeps the rest of the pipeline unchanged and only disables
    `_load_bern_weights_into_model` during this call.
    """
    original_loader = globals()["_load_bern_weights_into_model"]

    def _skip_bern_weights(model, region_name):
        return model, []

    globals()["_load_bern_weights_into_model"] = _skip_bern_weights
    try:
        outputs = run_region_aware_metadata_screen(metadata_plan, **kwargs)
        outputs["pretraining"] = "none_grape_from_scratch"
        return outputs
    finally:
        globals()["_load_bern_weights_into_model"] = original_loader


if "universal_metadata_plan" not in globals():
    universal_metadata_plan = build_universal_metadata_plan()

scratch_metadata_plan = universal_metadata_plan.copy()
print("GRAPE-from-scratch universal metadata variants:", len(scratch_metadata_plan))
display(scratch_metadata_plan[["variant", "uses_iop", "rnfl_available", "rnfl_mode", "region_rnfl_map"]])

# Use 5 seeds for the reported table. Set to 1 for a smoke test, or 10 for stability checking.
N_SCRATCH_METADATA_SEEDS = 5
SCRATCH_METADATA_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_SCRATCH_METADATA_SEEDS).tolist())
print("Random GRAPE-from-scratch metadata seeds:", SCRATCH_METADATA_SEEDS)

scratch_metadata_outputs = run_region_aware_metadata_screen_from_scratch(
    scratch_metadata_plan,
    seeds=SCRATCH_METADATA_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    iop_mode="history",
    time_input_mode="input_times",
)

scratch_overall = scratch_metadata_outputs["overall"].copy()
scratch_overall = scratch_overall.merge(
    scratch_metadata_plan[["variant", "rnfl_available", "rnfl_mode", "uses_iop"]],
    left_on="Model",
    right_on="variant",
    how="left",
).drop(columns=["variant"])

scratch_display = scratch_overall[
    [
        "Model",
        "uses_iop",
        "rnfl_available",
        "rnfl_mode",
        "MAE ± SD (dB)",
        "MAE_mean_db",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "delta_vs_iop_display",
        "improved_vs_iop",
        "Samples",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True)

display(scratch_display)
display(scratch_display.head(10))

print("GRAPE-from-scratch metadata candidates improving over VF-only:")
display(scratch_display[scratch_display["improved_vs_vf_only"]].reset_index(drop=True))

print("GRAPE-from-scratch metadata candidates improving over IOP-history baseline:")
display(scratch_display[scratch_display["improved_vs_iop"]].reset_index(drop=True))


## 15. Final metadata fusion strategy ablation

Run one clean comparison for the metadata fusion question under the same Bern-finetuned LSTM backbone. This keeps the protocol fixed and only changes where/how IOP metadata enters the LSTM model. The raw VF+IOP before encoder version is the most literal input-level fusion test, but it changes the raw VF input dimension and therefore cannot fully reuse the Bern VF encoder first layer. The two cross-attention rows test both directions: VF as query versus metadata as query.


In [ ]:
# Final clean comparison of the metadata-fusion strategies.
# This is intended as the table for the supervisor discussion.

FUSION_ABLATION_MODEL_ORDER = [
    ("VF only", "1. VF-only baseline"),
    ("VF + IOP history late fusion", "2. Late fusion"),
    ("VF + per-visit IOP pre-encoder fusion", "3. Raw VF+IOP before encoder"),
    ("VF + per-visit IOP early fusion", "4. Per-visit early fusion"),
    ("VF + per-visit IOP gated early fusion", "5. Gated early fusion"),
    ("VF + per-visit IOP cross-attention (VF query)", "6. Cross-attention: VF query"),
    ("VF + per-visit IOP cross-attention (metadata query)", "7. Cross-attention: metadata query"),
]

fusion_ablation_design = pd.DataFrame([
    {
        "display_model": "1. VF-only baseline",
        "Model": "VF only",
        "metadata_alignment": "none",
        "metadata_time_encoding": "none",
        "fusion_position": "none",
        "purpose": "No-metadata reference",
    },
    {
        "display_model": "2. Late fusion",
        "Model": "VF + IOP history late fusion",
        "metadata_alignment": "history vector, not timestep token",
        "metadata_time_encoding": "implicit order only",
        "fusion_position": "after LSTM, prediction correction",
        "purpose": "Original weak correction baseline",
    },
    {
        "display_model": "3. Raw VF+IOP before encoder",
        "Model": "VF + per-visit IOP pre-encoder fusion",
        "metadata_alignment": "per visit",
        "metadata_time_encoding": "time added after encoder",
        "fusion_position": "raw [VF, IOP] before encoder",
        "purpose": "Literal before-encoder test; changes Bern encoder input shape",
    },
    {
        "display_model": "4. Per-visit early fusion",
        "Model": "VF + per-visit IOP early fusion",
        "metadata_alignment": "per visit",
        "metadata_time_encoding": "time in timestep representation",
        "fusion_position": "after VF/time encoding, before LSTM",
        "purpose": "Main practical early-fusion test",
    },
    {
        "display_model": "5. Gated early fusion",
        "Model": "VF + per-visit IOP gated early fusion",
        "metadata_alignment": "per visit",
        "metadata_time_encoding": "time in timestep representation",
        "fusion_position": "after VF/time encoding, before LSTM",
        "purpose": "Learn sigmoid gate for metadata strength",
    },
    {
        "display_model": "6. Cross-attention: VF query",
        "Model": "VF + per-visit IOP cross-attention (VF query)",
        "metadata_alignment": "per visit metadata tokens",
        "metadata_time_encoding": "explicit time encoding on IOP tokens",
        "fusion_position": "VF query attends to IOP key/value before LSTM",
        "purpose": "Transformer-style fusion where VF pulls information from IOP",
    },
    {
        "display_model": "7. Cross-attention: metadata query",
        "Model": "VF + per-visit IOP cross-attention (metadata query)",
        "metadata_alignment": "per visit metadata tokens",
        "metadata_time_encoding": "explicit time encoding on IOP tokens",
        "fusion_position": "IOP query attends to VF key/value before LSTM",
        "purpose": "Supervisor-suggested direction where metadata conditions VF history selection",
    },
])

display(fusion_ablation_design)

# Use 10 seeds for the final table. Set to 1 for a quick smoke test.
N_FINAL_FUSION_SEEDS = 10
FINAL_FUSION_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_FINAL_FUSION_SEEDS).tolist())
print("Random final fusion-ablation seeds:", FINAL_FUSION_SEEDS)

final_fusion_outputs = run_per_visit_iop_early_fusion_screen(
    seeds=FINAL_FUSION_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    include_point_level_transformer=False,
)

order_map = {model: i for i, (model, _) in enumerate(FUSION_ABLATION_MODEL_ORDER)}
label_map = dict(FUSION_ABLATION_MODEL_ORDER)

final_fusion_overall = final_fusion_outputs["overall"].copy()
final_fusion_overall["display_model"] = final_fusion_overall["Model"].map(label_map).fillna(final_fusion_overall["Model"])
final_fusion_overall["concept_order"] = final_fusion_overall["Model"].map(order_map)

final_fusion_ordered = final_fusion_overall.sort_values("concept_order").reset_index(drop=True)
final_fusion_ranked = final_fusion_overall.sort_values("MAE_mean_db").reset_index(drop=True)

final_fusion_ordered_display = final_fusion_ordered[
    [
        "display_model",
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
]

final_fusion_ranked_display = final_fusion_ranked[
    [
        "display_model",
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
]

print("Final fusion ablation table in conceptual order:")
display(final_fusion_ordered_display)

print("Final fusion ablation table ranked by MAE:")
display(final_fusion_ranked_display)


## 16. Point-level spatio-temporal Transformer backbone

This is separated from the LSTM fusion ablation because it is a new backbone. Instead of using one 59-dimensional VF vector per visit, it treats each visit-by-point pair as a token and learns spatial-temporal relationships with a Transformer encoder. It should be interpreted as a from-scratch GRAPE Transformer test, not as a clean Bern-pretrained LSTM fine-tuning experiment.


In [ ]:
# Separate point-level Transformer backbone test.
# Start with 1 seed as a smoke test; increase after confirming runtime.
N_POINT_TRANSFORMER_SEEDS = 1
POINT_TRANSFORMER_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_POINT_TRANSFORMER_SEEDS).tolist())
print("Random point-level Transformer seeds:", POINT_TRANSFORMER_SEEDS)

point_transformer_outputs = run_per_visit_iop_early_fusion_screen(
    seeds=POINT_TRANSFORMER_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    include_point_level_transformer=True,
)

point_transformer_overall = point_transformer_outputs["overall"].copy()
point_transformer_focus = point_transformer_overall[
    point_transformer_overall["Model"].isin([
        "VF only",
        "VF + per-visit IOP early fusion",
        "VF + per-visit IOP cross-attention (VF query)",
        "VF + per-visit IOP point-level Transformer",
    ])
].sort_values("MAE_mean_db").reset_index(drop=True)

display(point_transformer_focus[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])


## 17. Bern pretraining for point-level Transformer

This trains the new point-level Transformer backbone on Bern first. Bern does not have usable numeric IOP values, so the IOP token is set to zero during Bern pretraining. The learned VF value, point, time, Transformer, attention, and decoder weights are then saved region by region and can be reused for GRAPE fine-tuning.


In [ ]:
# Bern point-level Transformer pretraining helper.
# Start with 1 seed and a small epoch count as a smoke test; increase after confirming runtime.

def _build_bern_point_transformer_sequences(k_list=(2, 3, 4), split_seed=42, mode="consecutive"):
    bern_df = pd.read_csv(bern_path, low_memory=False).copy()
    bern_df["Examination"] = pd.to_datetime(bern_df["Examination"], errors="coerce")
    bern_df = bern_df.dropna(subset=["patient_eye", "Examination"]).copy()
    bern_df = bern_df.sort_values(["patient_eye", "Examination"]).reset_index(drop=True)
    bern_df["VisitN"] = bern_df.groupby("patient_eye").cumcount() + 1
    baseline_exam = bern_df.groupby("patient_eye")["Examination"].transform("min")
    bern_df["Time_from_Baseline"] = (bern_df["Examination"] - baseline_exam).dt.days / 365.25

    td_cols = sorted([c for c in bern_df.columns if c.startswith("TD_")], key=lambda x: int(x.split("_")[1]))
    if len(td_cols) != 59:
        raise ValueError(f"Expected 59 Bern TD columns, got {len(td_cols)}")
    for col in td_cols:
        bern_df[col] = pd.to_numeric(bern_df[col], errors="coerce")
    bern_df = bern_df.dropna(subset=td_cols).copy()

    def _row_to_td(row):
        vals = row[td_cols].to_numpy(dtype=np.float32)
        return vals if np.all(np.isfinite(vals)) else None

    x_list, y_list, t_list, eye_list, meta_rows = [], [], [], [], []
    for k_in in k_list:
        need = k_in + 1
        for eye_id, g in bern_df.groupby("patient_eye"):
            g = g.sort_values("VisitN").reset_index(drop=True)
            if len(g) < need:
                continue
            if mode == "consecutive":
                windows = [tuple(range(s, s + need)) for s in range(0, len(g) - need + 1)]
            elif mode == "all":
                windows = combinations(range(len(g)), need)
            else:
                raise ValueError("mode must be 'consecutive' or 'all'")

            for win in windows:
                rows = [g.iloc[w] for w in win]
                times = [float(r["Time_from_Baseline"]) for r in rows]
                if not all(times[i] <= times[i + 1] for i in range(len(times) - 1)):
                    continue
                x = [_row_to_td(r) for r in rows[:k_in]]
                y = _row_to_td(rows[-1])
                if any(v is None for v in x) or y is None:
                    continue
                x_list.append(np.stack(x, axis=0).astype(np.float32))
                y_list.append(y.astype(np.float32))
                t_list.append(np.asarray(times[:k_in], dtype=np.float32))
                eye_list.append(eye_id)
                meta_rows.append({
                    "patient_eye": eye_id,
                    "HistoryLength": k_in,
                    "InputTimes": times[:k_in],
                    "TargetTime": times[-1],
                })

    y_array = np.stack(y_list, axis=0).astype(np.float32)
    eye_array = np.asarray(eye_list)
    unique_eyes = np.unique(eye_array)
    rng = np.random.default_rng(split_seed)
    rng.shuffle(unique_eyes)

    n_trainval_eye = int(0.85 * len(unique_eyes))
    trainval_eyes = unique_eyes[:n_trainval_eye]
    test_eyes = unique_eyes[n_trainval_eye:]
    rng.shuffle(trainval_eyes)
    n_val_eye = int(0.10 * len(trainval_eyes))
    val_eyes = trainval_eyes[:n_val_eye]
    train_eyes = trainval_eyes[n_val_eye:]

    def _take(mask):
        idx = np.where(mask)[0]
        return {
            "x": [x_list[i] for i in idx],
            "y": y_array[idx],
            "t": [t_list[i] for i in idx],
            "meta": [meta_rows[i] for i in idx],
        }

    data = {
        "train": _take(np.isin(eye_array, train_eyes)),
        "val": _take(np.isin(eye_array, val_eyes)),
        "test": _take(np.isin(eye_array, test_eyes)),
        "td_cols": td_cols,
        "split_seed": split_seed,
        "mode": mode,
        "k_list": tuple(k_list),
    }
    print(
        "Bern point-transformer samples:",
        "train", len(data["train"]["x"]),
        "val", len(data["val"]["x"]),
        "test", len(data["test"]["x"]),
    )
    return data


def _zero_iop_sequence_list(time_list):
    return [np.zeros((len(t), 1), dtype=np.float32) for t in time_list]


def run_bern_point_transformer_pretraining(
    *,
    seeds,
    k_list=(2, 3, 4),
    mode="consecutive",
    num_epochs=10,
    batch_size=128,
    lr=1e-3,
    split_seed=42,
    checkpoint_prefix="bern_point_transformer",
    transformer_config=None,
):
    data = _build_bern_point_transformer_sequences(k_list=k_list, split_seed=split_seed, mode=mode)
    x_train_s, x_val_s, x_test_s = _scale_x_list(data["train"]["x"], data["val"]["x"], data["test"]["x"])
    train_iop_zero = _zero_iop_sequence_list(data["train"]["t"])
    val_iop_zero = _zero_iop_sequence_list(data["val"]["t"])

    ckpt_dir = project_root / "models"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    transformer_config = transformer_config or {}
    best_by_region = {region_name: float("inf") for region_name in cluster_mapping}
    summary_rows = []
    history_frames = []

    for seed in seeds:
        hist_module._set_seed(int(seed))
        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            print(f"Bern point-level Transformer | seed={seed} | region={region_name}")

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(data["train"]["y"][:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(data["val"]["y"][:, idx]).astype(np.float32)

            train_ds = PerVisitIOPHistoryDataset(x_train_s, y_train_s, data["train"]["t"], train_iop_zero)
            val_ds = PerVisitIOPHistoryDataset(x_val_s, y_val_s, data["val"]["t"], val_iop_zero)
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=per_visit_iop_collate)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=per_visit_iop_collate)

            model = PointLevelSpatioTemporalTransformerModel(
                output_dim=len(idx),
                point_indices=idx.tolist(),
                **transformer_config,
            )
            model, hist_df, summary = _train_model(
                model=model,
                run_name=f"bern_point_transformer_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader,
                val_loader=val_loader,
                train_ds=train_ds,
                val_ds=val_ds,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs,
                lr=lr,
            )

            seed_ckpt = ckpt_dir / f"{checkpoint_prefix}_{region_name}_seed{int(seed)}_best.pth"
            torch.save(model.state_dict(), seed_ckpt)
            canonical_ckpt = ckpt_dir / f"{checkpoint_prefix}_{region_name}_best.pth"
            if summary["best_val_mae_db"] < best_by_region[region_name]:
                best_by_region[region_name] = summary["best_val_mae_db"]
                torch.save(model.state_dict(), canonical_ckpt)

            summary.update({
                "seed": int(seed),
                "region": region_name,
                "checkpoint": str(seed_ckpt),
                "canonical_checkpoint": str(canonical_ckpt),
                "train_samples": len(train_ds),
                "val_samples": len(val_ds),
            })
            summary_rows.append(summary)
            history_frames.append(hist_df.assign(seed=int(seed), region=region_name))

    summary_df = pd.DataFrame(summary_rows)
    history_df = pd.concat(history_frames, axis=0, ignore_index=True) if history_frames else pd.DataFrame()
    return {"summary": summary_df, "history": history_df, "data": data}


## 18. GRAPE fine-tuning from Bern point-level Transformer

After running the Bern pretraining cell, this fine-tunes the point-level Transformer on GRAPE with real per-visit IOP. The LSTM reference rows still use the existing Bern LSTM weights; the Transformer row now loads the Bern Transformer checkpoints saved above.


In [ ]:
# Run this cell only when you need to create or refresh the Bern Transformer checkpoints.
# If models/bern_point_transformer_{region}_best.pth already exists, you can skip this cell.
N_BERN_POINT_TRANSFORMER_SEEDS = 1
BERN_POINT_TRANSFORMER_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_BERN_POINT_TRANSFORMER_SEEDS).tolist())
print("Random Bern point-level Transformer seeds:", BERN_POINT_TRANSFORMER_SEEDS)

bern_point_transformer_pretrain_outputs = run_bern_point_transformer_pretraining(
    seeds=BERN_POINT_TRANSFORMER_SEEDS,
    k_list=(2, 3, 4),
    mode="consecutive",
    num_epochs=5,
    batch_size=128,
    lr=1e-3,
    split_seed=42,
)

display(bern_point_transformer_pretrain_outputs["summary"])


### GRAPE fine-tuning only

Run this cell after Bern Transformer checkpoints exist.


In [ ]:
# Run this cell to fine-tune on GRAPE using the saved Bern Transformer checkpoints.
# This does not rerun Bern pretraining.
N_BERN_TRANSFORMER_FT_SEEDS = 5
BERN_TRANSFORMER_FT_SEEDS = tuple(np.random.default_rng().integers(0, 1_000_000, size=N_BERN_TRANSFORMER_FT_SEEDS).tolist())
print("Random GRAPE fine-tune seeds:", BERN_TRANSFORMER_FT_SEEDS)

bern_transformer_ft_outputs = run_per_visit_iop_early_fusion_screen(
    seeds=BERN_TRANSFORMER_FT_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    include_point_level_transformer=True,
)

bern_transformer_ft_focus = bern_transformer_ft_outputs["overall"].copy()
display(bern_transformer_ft_focus[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True))


## 19. Compact point-level Transformer rescue test

The full point-level Transformer may be too flexible for the GRAPE fine-tuning size. This section tests a smaller and more regularized Transformer: `model_dim=32`, `num_layers=1`, `num_heads=2`, `ff_dim=64`, and `dropout=0.2`. It gets its own Bern pretraining checkpoint, then the same GRAPE fine-tuning protocol.


In [ ]:
# Run once to create compact Bern Transformer checkpoints.
# Skip this cell later if models/bern_point_transformer_compact_{region}_best.pth already exists.
COMPACT_TRANSFORMER_CONFIG = {
    "model_dim": 32,
    "num_heads": 2,
    "num_layers": 1,
    "ff_dim": 64,
    "dropout": 0.2,
    "meta_hidden_dim": 16,
}

N_BERN_COMPACT_TRANSFORMER_SEEDS = 1
BERN_COMPACT_TRANSFORMER_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_BERN_COMPACT_TRANSFORMER_SEEDS).tolist()
)
print("Random Bern compact Transformer seeds:", BERN_COMPACT_TRANSFORMER_SEEDS)

bern_compact_transformer_pretrain_outputs = run_bern_point_transformer_pretraining(
    seeds=BERN_COMPACT_TRANSFORMER_SEEDS,
    k_list=(2, 3, 4),
    mode="consecutive",
    num_epochs=5,
    batch_size=256,
    lr=1e-3,
    split_seed=42,
    checkpoint_prefix="bern_point_transformer_compact",
    transformer_config=COMPACT_TRANSFORMER_CONFIG,
)

display(bern_compact_transformer_pretrain_outputs["summary"])


### GRAPE fine-tuning for compact Transformer

Run this after the compact Bern checkpoints exist.


In [ ]:
# Fine-tune the compact Transformer on GRAPE and compare against the key references.
N_COMPACT_TRANSFORMER_FT_SEEDS = 5
COMPACT_TRANSFORMER_FT_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_COMPACT_TRANSFORMER_FT_SEEDS).tolist()
)
print("Random compact Transformer GRAPE fine-tune seeds:", COMPACT_TRANSFORMER_FT_SEEDS)

compact_transformer_ft_outputs = run_per_visit_iop_early_fusion_screen(
    seeds=COMPACT_TRANSFORMER_FT_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    include_point_level_transformer=True,
    include_compact_point_transformer=True,
    selected_kinds={
        "vf_only",
        "per_visit_iop",
        "per_visit_iop_cross_attention",
        "point_level_transformer",
        "point_level_transformer_compact",
    },
)

compact_transformer_focus = compact_transformer_ft_outputs["overall"].copy()
display(compact_transformer_focus[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True))


## 20. Transformer-only metadata comparison

This isolates the metadata question under one backbone. Both rows use the same Bern-pretrained compact point-level Transformer architecture and checkpoints. The VF-only row forces the IOP token to zero during GRAPE fine-tuning; the metadata row uses real per-visit IOP.


In [ ]:
# Compare metadata effect within the same compact Transformer backbone.
# Requires compact Bern checkpoints from Cell 52.
N_TRANSFORMER_ONLY_COMPARE_SEEDS = 5
TRANSFORMER_ONLY_COMPARE_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_TRANSFORMER_ONLY_COMPARE_SEEDS).tolist()
)
print("Random Transformer-only comparison seeds:", TRANSFORMER_ONLY_COMPARE_SEEDS)

transformer_only_compare_outputs = run_per_visit_iop_early_fusion_screen(
    seeds=TRANSFORMER_ONLY_COMPARE_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    include_compact_point_transformer=True,
    selected_kinds={
        "point_level_transformer_compact_vf_only",
        "point_level_transformer_compact",
    },
)

transformer_only_overall = transformer_only_compare_outputs["overall"].copy()
display(transformer_only_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
].sort_values("MAE_mean_db").reset_index(drop=True))


## 21. Compact Transformer metadata ablation: IOP vs spatial RNFL

This keeps the backbone fixed: Bern-pretrained compact point-level Transformer. It tests four metadata settings only: VF-only, per-visit IOP only, baseline spatial RNFL only, and IOP + baseline spatial RNFL. IOP is visit-aligned. RNFL is treated as baseline structural metadata and repeated across the input visits for that eye. RNFL uses the region-aware spatial mapping, not RNFL mean.


In [ ]:
# Compact Transformer-only metadata ablation: VF-only / IOP / spatial RNFL / IOP+RNFL.
# Requires compact Bern checkpoints from Cell 52.

COMPACT_TRANSFORMER_CONFIG = {
    "model_dim": 32,
    "num_heads": 2,
    "num_layers": 1,
    "ff_dim": 64,
    "dropout": 0.2,
    "meta_hidden_dim": 16,
}

compact_transformer_spatial_rnfl_map = {
    "Supero_Nasal": ["RNFL_I", "RNFL_N"],
    "Supero_Temporal": ["RNFL_I", "RNFL_T"],
    "Infero_Nasal": ["RNFL_S", "RNFL_N"],
    "Infero_Temporal": ["RNFL_S", "RNFL_T"],
    "Temporal": ["RNFL_T"],
    "Macular": ["RNFL_T"],
}


def _baseline_rnfl_lookup(df_source, rnfl_cols):
    df_meta = df_source.copy()
    if "eye_id" not in df_meta.columns:
        df_meta["eye_id"] = (
            df_meta["Subject Number"].astype(str).str.strip()
            + "_"
            + df_meta["Laterality"].astype(str).str.strip()
        )
    sort_cols = [c for c in ["eye_id", "Time_from_Baseline", "VisitN"] if c in df_meta.columns]
    df_meta = df_meta.sort_values(sort_cols).copy()
    first_rows = df_meta.groupby("eye_id", as_index=False).first()
    out = {}
    for _, row in first_rows.iterrows():
        vals = pd.to_numeric(row[rnfl_cols], errors="coerce").to_numpy(dtype=np.float32)
        out[row["eye_id"]] = vals
    return out


def _rnfl_sequence_list(df_part, rnfl_lookup, rnfl_cols):
    seq_list = []
    for _, row in df_part.iterrows():
        seq_len = int(np.asarray(row["X"]).shape[0])
        eye_id = row["eye_id"]
        rnfl = rnfl_lookup.get(eye_id, np.full(len(rnfl_cols), np.nan, dtype=np.float32))
        rnfl = np.asarray(rnfl, dtype=np.float32).reshape(-1)
        if rnfl.shape[0] != len(rnfl_cols):
            rnfl = rnfl[:len(rnfl_cols)]
        seq = np.repeat(rnfl.reshape(1, -1), seq_len, axis=0)
        seq_list.append(seq.astype(np.float32))
    return seq_list


def _concat_sequence_lists(*seq_groups):
    return [np.concatenate(parts, axis=1).astype(np.float32) for parts in zip(*seq_groups)]


def _scale_meta_sequence_lists(train_seq, val_seq, test_seq):
    train_stack = np.concatenate(train_seq, axis=0).astype(np.float32)
    train_mean = np.nanmean(train_stack, axis=0)
    train_mean = np.where(np.isfinite(train_mean), train_mean, 0.0).astype(np.float32)

    def _fill(seq_list):
        return [np.where(np.isfinite(seq), seq, train_mean).astype(np.float32) for seq in seq_list]

    train_filled = _fill(train_seq)
    val_filled = _fill(val_seq)
    test_filled = _fill(test_seq)

    scaler = StandardScaler()
    scaler.fit(np.concatenate(train_filled, axis=0))
    train_s = [scaler.transform(seq).astype(np.float32) for seq in train_filled]
    val_s = [scaler.transform(seq).astype(np.float32) for seq in val_filled]
    test_s = [scaler.transform(seq).astype(np.float32) for seq in test_filled]
    return train_s, val_s, test_s, scaler


def run_compact_transformer_iop_rnfl_ablation(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)
    train_pooled = _set_model_time_inputs(train_pooled, mode=time_input_mode)
    eval_pooled = _set_model_time_inputs(eval_pooled, mode=time_input_mode)
    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_iop_seq = _extract_iop_sequence_list(train_df_all)
    val_iop_seq = _extract_iop_sequence_list(val_df_all)
    test_iop_seq = _extract_iop_sequence_list(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)
    train_iop_seq = _mask_iop_sequence_list(train_iop_seq, train_mask)
    val_iop_seq = _mask_iop_sequence_list(val_iop_seq, val_mask)
    test_iop_seq = _mask_iop_sequence_list(test_iop_seq, test_mask)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
    train_iop_s, val_iop_s, test_iop_s, iop_seq_scaler = _scale_iop_sequence_lists(train_iop_seq, val_iop_seq, test_iop_seq)

    zero_train = [np.zeros((len(t), 1), dtype=np.float32) for t in t_train]
    zero_val = [np.zeros((len(t), 1), dtype=np.float32) for t in t_val]
    zero_test = [np.zeros((len(t), 1), dtype=np.float32) for t in t_test]

    variants = [
        {"Model": "Compact Transformer VF-only", "kind": "vf_only"},
        {"Model": "Compact Transformer + per-visit IOP", "kind": "iop"},
        {"Model": "Compact Transformer + baseline spatial RNFL", "kind": "rnfl"},
        {"Model": "Compact Transformer + per-visit IOP + baseline spatial RNFL", "kind": "iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    print("Compact Transformer metadata ablation train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for variant in variants:
        exp_name = variant["Model"]
        exp_kind = variant["kind"]
        print("\n" + "#" * 100)
        print(f"COMPACT TRANSFORMER METADATA: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            hist_module._set_seed(int(seed))
            pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                print(f"{exp_name} | seed={seed} | region={region_name} | RNFL={rnfl_cols}")

                region_rnfl_lookup = _baseline_rnfl_lookup(grape_longitudinal_td_aligned_df, rnfl_cols)
                train_rnfl_raw = _rnfl_sequence_list(train_df_all, region_rnfl_lookup, rnfl_cols)
                val_rnfl_raw = _rnfl_sequence_list(val_df_all, region_rnfl_lookup, rnfl_cols)
                test_rnfl_raw = _rnfl_sequence_list(test_df_all, region_rnfl_lookup, rnfl_cols)
                train_rnfl_s, val_rnfl_s, test_rnfl_s, rnfl_scaler = _scale_meta_sequence_lists(
                    train_rnfl_raw, val_rnfl_raw, test_rnfl_raw
                )

                if exp_kind == "vf_only":
                    train_meta_s, val_meta_s, test_meta_s = zero_train, zero_val, zero_test
                    meta_dim = 1
                    model_cls = PointLevelSpatioTemporalTransformerVFOnlyModel
                elif exp_kind == "iop":
                    train_meta_s, val_meta_s, test_meta_s = train_iop_s, val_iop_s, test_iop_s
                    meta_dim = 1
                    model_cls = PointLevelSpatioTemporalTransformerModel
                elif exp_kind == "rnfl":
                    train_meta_s, val_meta_s, test_meta_s = train_rnfl_s, val_rnfl_s, test_rnfl_s
                    meta_dim = len(rnfl_cols)
                    model_cls = PointLevelSpatioTemporalTransformerModel
                elif exp_kind == "iop_rnfl":
                    train_meta_s = _concat_sequence_lists(train_iop_s, train_rnfl_s)
                    val_meta_s = _concat_sequence_lists(val_iop_s, val_rnfl_s)
                    test_meta_s = _concat_sequence_lists(test_iop_s, test_rnfl_s)
                    meta_dim = 1 + len(rnfl_cols)
                    model_cls = PointLevelSpatioTemporalTransformerModel
                else:
                    raise ValueError(exp_kind)

                actual_meta_dim = int(np.asarray(train_meta_s[0]).shape[1])
                if actual_meta_dim != meta_dim:
                    print(f"Adjusting meta_dim for {exp_name} / {region_name}: expected {meta_dim}, got {actual_meta_dim}")
                    meta_dim = actual_meta_dim

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                train_ds = PerVisitIOPHistoryDataset(x_train_s, y_train_s, t_train, train_meta_s)
                val_ds = PerVisitIOPHistoryDataset(x_val_s, y_val_s, t_val, val_meta_s)
                test_ds = PerVisitIOPHistoryDataset(x_test_s, y_test_s, t_test, test_meta_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=per_visit_iop_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=per_visit_iop_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=per_visit_iop_collate)

                model = model_cls(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    meta_input_dim=meta_dim,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )
                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"compact_tf_meta_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs
                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "rnfl_cols": "+".join(rnfl_cols) if rnfl_cols else "none",
                    "meta_dim": meta_dim,
                    "loaded_bern_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_label = "Compact Transformer VF-only"
    baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


In [ ]:
# Run the 4-way compact Transformer metadata comparison.
N_COMPACT_TF_META_SEEDS = 5
COMPACT_TF_META_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_COMPACT_TF_META_SEEDS).tolist()
)
print("Random compact Transformer metadata seeds:", COMPACT_TF_META_SEEDS)

compact_tf_meta_outputs = run_compact_transformer_iop_rnfl_ablation(
    seeds=COMPACT_TF_META_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
)

compact_tf_meta_overall = compact_tf_meta_outputs["overall"].copy()
display(compact_tf_meta_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])


## 22. Safe metadata residual correction

This block tests a conservative metadata strategy. The VF-only Bern-to-GRAPE model is first trained as usual, then frozen. Metadata is only allowed to learn a small additive residual correction on top of the VF-only prediction.

This is useful because if metadata is noisy or unhelpful, the correction can stay near zero and should not damage the strong VF-only backbone.



In [ ]:
# Safe metadata residual correction.
# This block does not modify previous experiments. It trains a VF-only base model, freezes it,
# and then trains only a small metadata correction head.

import copy


class FrozenMetadataResidualCorrectionModel(nn.Module):
    """Frozen VF-only backbone plus a trainable metadata correction head.

    The final correction layer is zero-initialized, so the model starts exactly as the
    trained VF-only base model. Metadata can only improve by learning a residual error
    pattern that remains after VF history has already been used.
    """
    def __init__(
        self,
        base_model,
        meta_dim,
        output_dim,
        hidden_dim=32,
        dropout=0.25,
    ):
        super().__init__()
        self.base_model = base_model
        for p in self.base_model.parameters():
            p.requires_grad = False
        self.base_model.eval()

        self.meta_norm = nn.LayerNorm(meta_dim)
        self.meta_correction = nn.Sequential(
            nn.Linear(meta_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.meta_correction[-1].weight)
        nn.init.zeros_(self.meta_correction[-1].bias)

    def forward(self, padded_seqs, padded_times, lengths, meta_input):
        with torch.no_grad():
            base_pred = self.base_model(padded_seqs, padded_times, lengths, None)
        correction = self.meta_correction(self.meta_norm(meta_input))
        return base_pred + correction


def _iop_exposure_matrix(df_part):
    """Create clinically motivated IOP exposure features from the input history."""
    rows = []
    for _, row in df_part.iterrows():
        k = int(row["k"])
        hist = np.asarray(row["iop_hist_padded"], dtype=np.float32)[-k:]
        times = np.asarray(row["input_times"], dtype=np.float32)
        if len(times) != len(hist):
            times = np.arange(len(hist), dtype=np.float32)

        finite = np.isfinite(hist)
        if finite.sum() == 0:
            rows.append([np.nan] * 7)
            continue

        h = hist[finite]
        t = times[finite]
        last_iop = h[-1]
        mean_iop = np.nanmean(h)
        max_iop = np.nanmax(h)
        std_iop = np.nanstd(h)
        delta_t = float(row.get("delta_t", np.nan))

        if len(h) >= 2 and np.nanmax(t) > np.nanmin(t):
            slope = (h[-1] - h[0]) / max(t[-1] - t[0], 1e-6)
            exposure = float(np.trapz(h, t))
        else:
            slope = 0.0
            exposure = float(mean_iop * max(delta_t, 0.0)) if np.isfinite(delta_t) else np.nan

        future_exposure_proxy = mean_iop * delta_t if np.isfinite(delta_t) else np.nan
        rows.append([
            last_iop,
            mean_iop,
            max_iop,
            std_iop,
            slope,
            exposure,
            future_exposure_proxy,
        ])
    return np.asarray(rows, dtype=np.float32)


def _scale_static_meta_matrices(train_meta, val_meta, test_meta):
    train_meta = np.asarray(train_meta, dtype=np.float32)
    val_meta = np.asarray(val_meta, dtype=np.float32)
    test_meta = np.asarray(test_meta, dtype=np.float32)

    train_mean = np.nanmean(train_meta, axis=0)
    train_mean = np.where(np.isfinite(train_mean), train_mean, 0.0).astype(np.float32)

    def _fill(arr):
        return np.where(np.isfinite(arr), arr, train_mean).astype(np.float32)

    scaler = StandardScaler()
    train_s = scaler.fit_transform(_fill(train_meta)).astype(np.float32)
    val_s = scaler.transform(_fill(val_meta)).astype(np.float32)
    test_s = scaler.transform(_fill(test_meta)).astype(np.float32)
    return train_s, val_s, test_s, scaler


def _region_rnfl_matrix(df_part, rnfl_cols, rnfl_master_cols):
    if len(rnfl_cols) == 0:
        return np.zeros((len(df_part), 0), dtype=np.float32)
    keep = [rnfl_master_cols.index(c) for c in rnfl_cols]
    return np.stack([
        np.asarray(v, dtype=np.float32)[keep]
        for v in df_part["rnfl_static"].values
    ]).astype(np.float32)


def _set_meta_array(df_part, meta_array):
    out = df_part.copy()
    out["meta_sub"] = [np.asarray(v, dtype=np.float32) for v in meta_array]
    return out


def _prepare_safe_residual_data(k_list=(2, 3, 4), train_mode="consecutive", eval_mode="consecutive", time_input_mode="input_times"):
    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    rnfl_master_cols = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]
    max_k = max(k_list)

    train_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=train_mode)
    eval_pooled = hist_module._build_history_pairs(df_exp, td_cols_exp, k_list=k_list, mode=eval_mode)
    train_pooled = _set_model_time_inputs(train_pooled, mode=time_input_mode)
    eval_pooled = _set_model_time_inputs(eval_pooled, mode=time_input_mode)

    train_pooled = _attach_history_metadata(train_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)
    eval_pooled = _attach_history_metadata(eval_pooled, grape_longitudinal_td_aligned_df, rnfl_master_cols, max_k=max_k)

    train_df_all, val_df_all, _ = hist_module._split_by_eye(
        train_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    _, _, test_df_all = hist_module._split_by_eye(
        eval_pooled, meta_train_grape, meta_val_grape, meta_test_grape
    )
    return train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols


def run_safe_metadata_residual_correction(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs_base=20,
    num_epochs_residual=20,
    batch_size=64,
    lr_base=1e-3,
    lr_residual=5e-4,
    time_input_mode="input_times",
    residual_variants=("iop_history", "iop_exposure", "iop_exposure_spatial_rnfl"),
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    raw_results = []
    run_summaries = []

    print("Safe residual train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Safe residual seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"SAFE RESIDUAL SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_predictions = {}
        seed_truth = None

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            print(f"VF-only base | seed={seed} | region={region_name}")

            # VF-only base data.
            empty_train = np.zeros((len(train_df_all), 0), dtype=np.float32)
            empty_val = np.zeros((len(val_df_all), 0), dtype=np.float32)
            empty_test = np.zeros((len(test_df_all), 0), dtype=np.float32)
            train_df_base = _set_meta_array(train_df_all, empty_train)
            val_df_base = _set_meta_array(val_df_all, empty_val)
            test_df_base = _set_meta_array(test_df_all, empty_test)

            x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df_base)
            x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df_base)
            x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df_base)

            train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
            val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
            test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

            x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
            x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
            x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)

            x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
            y_train_r = y_train[:, idx]
            y_val_r = y_val[:, idx]
            y_test_r = y_test[:, idx]
            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
            y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
            y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

            train_ds_base = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train)
            val_ds_base = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val)
            test_ds_base = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test)
            train_loader_base = DataLoader(train_ds_base, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
            val_loader_base = DataLoader(val_ds_base, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
            test_loader_base = DataLoader(test_ds_base, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

            base_model = VFOnlyModel(output_dim=len(idx))
            base_model, loaded_keys = _load_bern_weights_into_model(base_model, region_name)
            base_model, hist_df, base_summary = _train_model(
                model=base_model,
                run_name=f"safe_residual_vf_only_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader_base,
                val_loader=val_loader_base,
                train_ds=train_ds_base,
                val_ds=val_ds_base,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs_base,
                lr=lr_base,
            )

            pred_s, true_s = _predict_model(base_model, test_loader_base, device)
            pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
            if "VF-only base" not in seed_predictions:
                seed_predictions["VF-only base"] = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            if seed_truth is None:
                seed_truth = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            seed_predictions["VF-only base"][:, idx] = pred_abs
            seed_truth[:, idx] = true_abs

            base_summary.update({
                "variant": "VF-only base",
                "kind": "vf_only_base",
                "seed": int(seed),
                "region": region_name,
                "loaded_bern_keys": len(loaded_keys),
                "train_samples": len(train_ds_base),
                "val_samples": len(val_ds_base),
                "test_samples": len(test_ds_base),
            })
            run_summaries.append(base_summary)

            # Metadata residual variants use the same scaled x/y and same frozen base.
            df_train_region = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
            df_val_region = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
            df_test_region = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

            for residual_kind in residual_variants:
                if residual_kind == "iop_history":
                    train_meta_raw = _iop_history_matrix(df_train_region)
                    val_meta_raw = _iop_history_matrix(df_val_region)
                    test_meta_raw = _iop_history_matrix(df_test_region)
                    exp_name = "Frozen residual + IOP history"
                elif residual_kind == "iop_exposure":
                    train_meta_raw = _iop_exposure_matrix(df_train_region)
                    val_meta_raw = _iop_exposure_matrix(df_val_region)
                    test_meta_raw = _iop_exposure_matrix(df_test_region)
                    exp_name = "Frozen residual + IOP exposure"
                elif residual_kind == "iop_exposure_spatial_rnfl":
                    rnfl_cols = compact_transformer_spatial_rnfl_map.get(region_name, [])
                    train_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_train_region),
                        _region_rnfl_matrix(df_train_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    val_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_val_region),
                        _region_rnfl_matrix(df_val_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    test_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_test_region),
                        _region_rnfl_matrix(df_test_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    exp_name = "Frozen residual + IOP exposure + spatial RNFL"
                else:
                    raise ValueError(f"Unknown residual variant: {residual_kind}")

                print(f"{exp_name} | seed={seed} | region={region_name} | meta_dim={train_meta_raw.shape[1]}")
                train_meta_s, val_meta_s, test_meta_s, meta_scaler = _scale_static_meta_matrices(
                    train_meta_raw, val_meta_raw, test_meta_raw
                )

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, train_meta_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, val_meta_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, test_meta_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                residual_model = FrozenMetadataResidualCorrectionModel(
                    base_model=copy.deepcopy(base_model),
                    meta_dim=train_meta_s.shape[1],
                    output_dim=len(idx),
                    hidden_dim=32,
                    dropout=0.25,
                )
                residual_model, hist_df, summary = _train_model(
                    model=residual_model,
                    run_name=f"safe_residual_{residual_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs_residual,
                    lr=lr_residual,
                )

                pred_s, true_s = _predict_model(residual_model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                if exp_name not in seed_predictions:
                    seed_predictions[exp_name] = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                seed_predictions[exp_name][:, idx] = pred_abs

                summary.update({
                    "variant": exp_name,
                    "kind": residual_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "meta_dim": train_meta_s.shape[1],
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        for exp_name, pred_full_abs in seed_predictions.items():
            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, seed_truth, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_mae = float(overall.loc[overall["Model"].eq("VF-only base"), "MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "seeds": seeds,
    }



In [ ]:
# Start small. Increase N_SAFE_RESIDUAL_SEEDS after the smoke test runs cleanly.
N_SAFE_RESIDUAL_SEEDS = 3
SAFE_RESIDUAL_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_SAFE_RESIDUAL_SEEDS).tolist()
)
print("Random safe-residual seeds:", SAFE_RESIDUAL_SEEDS)

safe_residual_outputs = run_safe_metadata_residual_correction(
    seeds=SAFE_RESIDUAL_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs_base=20,
    num_epochs_residual=20,
    batch_size=64,
    lr_base=1e-3,
    lr_residual=5e-4,
    time_input_mode="input_times",
    residual_variants=("iop_history", "iop_exposure", "iop_exposure_spatial_rnfl"),
)

safe_residual_overall = safe_residual_outputs["overall"].copy()
display(safe_residual_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])



## 23. Subgroup analysis for IOP exposure residual

The overall MAE may hide metadata benefit if IOP only helps a subset of eyes. This block reruns the conservative residual setup for the best candidate (`IOP exposure`) and reports the delta versus VF-only within clinically interpretable subgroups.



In [ ]:
# Subgroup analysis for the safe IOP-exposure residual model.
# The goal is to check whether metadata helps specific high-risk subgroups even if overall MAE is flat.


def _last_finite(values):
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]
    return float(values[-1]) if len(values) else np.nan


def _safe_nanmedian(values):
    values = np.asarray(values, dtype=np.float32)
    if np.isfinite(values).sum() == 0:
        return np.nan
    return float(np.nanmedian(values))


def _make_safe_residual_subgroup_frame(test_df):
    rows = []
    for sample_idx, (_, row) in enumerate(test_df.reset_index(drop=True).iterrows()):
        x = np.asarray(row["X"], dtype=np.float32)
        times = np.asarray(row["input_times"], dtype=np.float32)
        iop_hist = np.asarray(row["iop_hist_padded"], dtype=np.float32)
        seq_len = x.shape[0]
        iop_seq = iop_hist[-seq_len:]

        last_iop = _last_finite(iop_seq)
        mean_iop = float(np.nanmean(iop_seq)) if np.isfinite(iop_seq).sum() else np.nan
        delta_t = float(row.get("delta_t", np.nan))
        last_vf_mean = float(np.nanmean(x[-1])) if x.size else np.nan
        rnfl_static = np.asarray(row["rnfl_static"], dtype=np.float32)
        rnfl_mean = float(rnfl_static[0]) if len(rnfl_static) and np.isfinite(rnfl_static[0]) else np.nan

        vf_means = np.nanmean(x, axis=1)
        if len(vf_means) >= 2 and np.isfinite(vf_means).sum() >= 2 and np.nanmax(times) > np.nanmin(times):
            prior_vf_slope = float((vf_means[-1] - vf_means[0]) / max(times[-1] - times[0], 1e-6))
        else:
            prior_vf_slope = np.nan

        exposure_proxy = mean_iop * delta_t if np.isfinite(mean_iop) and np.isfinite(delta_t) else np.nan
        rows.append({
            "sample_idx": sample_idx,
            "last_iop": last_iop,
            "mean_iop": mean_iop,
            "iop_exposure_proxy": exposure_proxy,
            "delta_t": delta_t,
            "last_vf_mean": last_vf_mean,
            "rnfl_mean": rnfl_mean,
            "prior_vf_slope": prior_vf_slope,
        })

    df = pd.DataFrame(rows)
    thresholds = {
        "last_iop_median": _safe_nanmedian(df["last_iop"]),
        "iop_exposure_median": _safe_nanmedian(df["iop_exposure_proxy"]),
        "delta_t_median": _safe_nanmedian(df["delta_t"]),
        "last_vf_mean_median": _safe_nanmedian(df["last_vf_mean"]),
        "rnfl_mean_median": _safe_nanmedian(df["rnfl_mean"]),
        "prior_vf_slope_q33": float(np.nanquantile(df["prior_vf_slope"], 0.33)) if np.isfinite(df["prior_vf_slope"]).sum() else np.nan,
    }
    return df, thresholds


def _subgroup_summary_from_errors(error_df, subgroup_df, thresholds):
    merged = error_df.merge(subgroup_df, on="sample_idx", how="left")
    subgroup_defs = [
        ("All test eyes", np.ones(len(merged), dtype=bool)),
        ("High last IOP", merged["last_iop"] >= thresholds["last_iop_median"]),
        ("High IOP exposure", merged["iop_exposure_proxy"] >= thresholds["iop_exposure_median"]),
        ("Long prediction interval", merged["delta_t"] >= thresholds["delta_t_median"]),
        ("Worse baseline VF", merged["last_vf_mean"] <= thresholds["last_vf_mean_median"]),
        ("Thin baseline RNFL", merged["rnfl_mean"] <= thresholds["rnfl_mean_median"]),
        ("Faster prior VF decline", merged["prior_vf_slope"] <= thresholds["prior_vf_slope_q33"]),
    ]

    rows = []
    for subgroup, mask in subgroup_defs:
        mask = np.asarray(mask, dtype=bool) & np.isfinite(merged["VF-only base"])
        if mask.sum() == 0:
            continue
        base_mae = float(merged.loc[mask, "VF-only base"].mean())
        residual_mae = float(merged.loc[mask, "Frozen residual + IOP exposure"].mean())
        rows.append({
            "Subgroup": subgroup,
            "Samples": int(mask.sum()),
            "VF-only MAE": base_mae,
            "IOP exposure residual MAE": residual_mae,
            "Delta vs VF-only": residual_mae - base_mae,
            "Improved": residual_mae < base_mae,
        })
    out = pd.DataFrame(rows)
    out["VF-only MAE"] = out["VF-only MAE"].map(lambda x: f"{x:.3f}")
    out["IOP exposure residual MAE"] = out["IOP exposure residual MAE"].map(lambda x: f"{x:.3f}")
    out["Delta vs VF-only"] = out["Delta vs VF-only"].map(lambda x: f"{x:+.3f} dB")
    return out


def run_safe_residual_iop_exposure_subgroups(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs_base=20,
    num_epochs_residual=20,
    batch_size=64,
    lr_base=1e-3,
    lr_residual=5e-4,
    time_input_mode="input_times",
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    seed_error_frames = []
    subgroup_df_ref = None
    thresholds_ref = None

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"SAFE RESIDUAL SUBGROUP SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        pred_full_base = None
        pred_full_residual = None
        true_full = None
        test_df_for_subgroups = None

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            print(f"Subgroup run | seed={seed} | region={region_name}")

            empty_train = np.zeros((len(train_df_all), 0), dtype=np.float32)
            empty_val = np.zeros((len(val_df_all), 0), dtype=np.float32)
            empty_test = np.zeros((len(test_df_all), 0), dtype=np.float32)
            train_df_base = _set_meta_array(train_df_all, empty_train)
            val_df_base = _set_meta_array(val_df_all, empty_val)
            test_df_base = _set_meta_array(test_df_all, empty_test)

            x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df_base)
            x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df_base)
            x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df_base)

            train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
            val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
            test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

            x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
            x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
            x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)
            df_train_region = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
            df_val_region = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
            df_test_region = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

            if test_df_for_subgroups is None:
                test_df_for_subgroups = df_test_region.copy()
                subgroup_df_ref, thresholds_ref = _make_safe_residual_subgroup_frame(test_df_for_subgroups)

            x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
            y_train_r = y_train[:, idx]
            y_val_r = y_val[:, idx]
            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
            y_val_s = y_scaler.transform(y_val_r).astype(np.float32)

            train_ds_base = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train)
            val_ds_base = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val)
            test_ds_base = RegionHistoryDataset(x_test_s, y_scaler.transform(y_test[:, idx]).astype(np.float32), t_test, meta_test)
            train_loader_base = DataLoader(train_ds_base, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
            val_loader_base = DataLoader(val_ds_base, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
            test_loader_base = DataLoader(test_ds_base, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

            base_model = VFOnlyModel(output_dim=len(idx))
            base_model, _ = _load_bern_weights_into_model(base_model, region_name)
            base_model, _, _ = _train_model(
                model=base_model,
                run_name=f"safe_subgroup_vf_only_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader_base,
                val_loader=val_loader_base,
                train_ds=train_ds_base,
                val_ds=val_ds_base,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs_base,
                lr=lr_base,
            )

            pred_s, true_s = _predict_model(base_model, test_loader_base, device)
            pred_abs_base, true_abs = _inverse_scale(pred_s, true_s, y_scaler)

            train_meta_raw = _iop_exposure_matrix(df_train_region)
            val_meta_raw = _iop_exposure_matrix(df_val_region)
            test_meta_raw = _iop_exposure_matrix(df_test_region)
            train_meta_s, val_meta_s, test_meta_s, _ = _scale_static_meta_matrices(train_meta_raw, val_meta_raw, test_meta_raw)

            train_ds_res = RegionHistoryDataset(x_train_s, y_train_s, t_train, train_meta_s)
            val_ds_res = RegionHistoryDataset(x_val_s, y_val_s, t_val, val_meta_s)
            test_ds_res = RegionHistoryDataset(x_test_s, y_scaler.transform(y_test[:, idx]).astype(np.float32), t_test, test_meta_s)
            train_loader_res = DataLoader(train_ds_res, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
            val_loader_res = DataLoader(val_ds_res, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
            test_loader_res = DataLoader(test_ds_res, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

            residual_model = FrozenMetadataResidualCorrectionModel(
                base_model=copy.deepcopy(base_model),
                meta_dim=train_meta_s.shape[1],
                output_dim=len(idx),
                hidden_dim=32,
                dropout=0.25,
            )
            residual_model, _, _ = _train_model(
                model=residual_model,
                run_name=f"safe_subgroup_iop_exposure_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader_res,
                val_loader=val_loader_res,
                train_ds=train_ds_res,
                val_ds=val_ds_res,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs_residual,
                lr=lr_residual,
            )
            pred_s, true_s = _predict_model(residual_model, test_loader_res, device)
            pred_abs_res, _ = _inverse_scale(pred_s, true_s, y_scaler)

            if pred_full_base is None:
                pred_full_base = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                pred_full_residual = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                true_full = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            pred_full_base[:, idx] = pred_abs_base
            pred_full_residual[:, idx] = pred_abs_res
            true_full[:, idx] = true_abs

        base_err = np.mean(np.abs(pred_full_base - true_full), axis=1)
        residual_err = np.mean(np.abs(pred_full_residual - true_full), axis=1)
        seed_error_frames.append(pd.DataFrame({
            "seed": int(seed),
            "sample_idx": np.arange(len(base_err)),
            "VF-only base": base_err,
            "Frozen residual + IOP exposure": residual_err,
        }))

    error_df = pd.concat(seed_error_frames, axis=0, ignore_index=True)
    mean_error_df = (
        error_df.groupby("sample_idx", as_index=False)[["VF-only base", "Frozen residual + IOP exposure"]]
        .mean()
    )
    subgroup_summary = _subgroup_summary_from_errors(mean_error_df, subgroup_df_ref, thresholds_ref)
    return {
        "sample_errors": error_df,
        "mean_sample_errors": mean_error_df,
        "subgroup_info": subgroup_df_ref,
        "thresholds": thresholds_ref,
        "subgroup_summary": subgroup_summary,
    }



In [ ]:
# Run subgroup analysis for the best safe residual candidate.
# Start with 3 seeds to match the previous safe-residual smoke test.
N_SAFE_SUBGROUP_SEEDS = 3
SAFE_SUBGROUP_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_SAFE_SUBGROUP_SEEDS).tolist()
)
print("Random safe-residual subgroup seeds:", SAFE_SUBGROUP_SEEDS)

safe_subgroup_outputs = run_safe_residual_iop_exposure_subgroups(
    seeds=SAFE_SUBGROUP_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs_base=20,
    num_epochs_residual=20,
    batch_size=64,
    lr_base=1e-3,
    lr_residual=5e-4,
    time_input_mode="input_times",
)

display(safe_subgroup_outputs["subgroup_summary"])
print("Subgroup thresholds used:")
for k, v in safe_subgroup_outputs["thresholds"].items():
    print(f"{k}: {v:.3f}" if np.isfinite(v) else f"{k}: nan")



## 24. Change-target forecasting

This block changes the target from the absolute future VF value to the VF change from the last input visit.

Instead of learning:

`future VF`

it learns:

`future VF - last input VF`

The final prediction is converted back to absolute dB scale by adding the predicted change to the last input VF. This tests whether IOP/RNFL is more useful for progression/change than for absolute pointwise VF prediction.



In [ ]:
# Change-target forecasting: predict future VF change instead of absolute future VF.
# This keeps the same Bern-to-GRAPE region-specific setup, but the supervised target is delta VF.


def _load_bern_backbone_only(model, region_name, prefixes=("encoder.", "time_pe.", "lstm.")):
    """Load only Bern encoder/time/LSTM weights, leaving the output head random.

    For change-target training, the decoder should not inherit absolute-VF output weights,
    because the target is now future VF minus last input VF.
    """
    ckpt = project_root / "models" / f"bern_seq_lstm_timePE_{region_name}_best.pth"
    if not ckpt.exists():
        raise FileNotFoundError(ckpt)
    source_state = torch.load(ckpt, map_location=device)
    target_state = model.state_dict()
    matched = {
        k: v
        for k, v in source_state.items()
        if k in target_state
        and tuple(v.shape) == tuple(target_state[k].shape)
        and any(k.startswith(prefix) for prefix in prefixes)
    }
    target_state.update(matched)
    model.load_state_dict(target_state)
    return model, sorted(matched.keys())


def run_change_target_metadata_screen(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    variants=("change_vf_only", "change_iop_exposure", "change_iop_exposure_spatial_rnfl"),
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    variant_names = {
        "change_vf_only": "Change target: VF-only",
        "change_iop_exposure": "Change target: VF + IOP exposure",
        "change_iop_exposure_spatial_rnfl": "Change target: VF + IOP exposure + spatial RNFL",
    }

    raw_results = []
    run_summaries = []

    print("Change-target train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Change-target seeds:", seeds)

    for variant in variants:
        exp_name = variant_names[variant]
        print("\n" + "#" * 100)
        print(f"CHANGE TARGET: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            print(f"-- seed={seed}")
            hist_module._set_seed(int(seed))
            pred_full_abs = None
            true_full_abs = None

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                print(f"{exp_name} | seed={seed} | region={region_name}")

                # Build raw arrays first. Meta is filled below per variant.
                empty_train = np.zeros((len(train_df_all), 0), dtype=np.float32)
                empty_val = np.zeros((len(val_df_all), 0), dtype=np.float32)
                empty_test = np.zeros((len(test_df_all), 0), dtype=np.float32)
                train_df_base = _set_meta_array(train_df_all, empty_train)
                val_df_base = _set_meta_array(val_df_all, empty_val)
                test_df_base = _set_meta_array(test_df_all, empty_test)

                x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_base)
                x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_base)
                x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_base)

                train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
                val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
                test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

                x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
                x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
                x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)

                df_train_region = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
                df_val_region = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
                df_test_region = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

                x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
                last_train = np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_train], axis=0)
                last_val = np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_val], axis=0)
                last_test = np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_test], axis=0)

                y_train_change = (y_train[:, idx] - last_train[:, idx]).astype(np.float32)
                y_val_change = (y_val[:, idx] - last_val[:, idx]).astype(np.float32)
                y_test_change = (y_test[:, idx] - last_test[:, idx]).astype(np.float32)

                y_scaler = StandardScaler()
                y_train_change_s = y_scaler.fit_transform(y_train_change).astype(np.float32)
                y_val_change_s = y_scaler.transform(y_val_change).astype(np.float32)
                y_test_change_s = y_scaler.transform(y_test_change).astype(np.float32)

                if variant == "change_vf_only":
                    train_meta = meta_train_empty
                    val_meta = meta_val_empty
                    test_meta = meta_test_empty
                    model = VFOnlyModel(output_dim=len(idx))
                elif variant == "change_iop_exposure":
                    train_meta_raw = _iop_exposure_matrix(df_train_region)
                    val_meta_raw = _iop_exposure_matrix(df_val_region)
                    test_meta_raw = _iop_exposure_matrix(df_test_region)
                    train_meta, val_meta, test_meta, _ = _scale_static_meta_matrices(
                        train_meta_raw, val_meta_raw, test_meta_raw
                    )
                    model = ResidualLateFusionModel(meta_dim=train_meta.shape[1], output_dim=len(idx))
                elif variant == "change_iop_exposure_spatial_rnfl":
                    rnfl_cols = compact_transformer_spatial_rnfl_map.get(region_name, [])
                    train_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_train_region),
                        _region_rnfl_matrix(df_train_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    val_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_val_region),
                        _region_rnfl_matrix(df_val_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    test_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_test_region),
                        _region_rnfl_matrix(df_test_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    train_meta, val_meta, test_meta, _ = _scale_static_meta_matrices(
                        train_meta_raw, val_meta_raw, test_meta_raw
                    )
                    model = ResidualLateFusionModel(meta_dim=train_meta.shape[1], output_dim=len(idx))
                else:
                    raise ValueError(f"Unknown change-target variant: {variant}")

                train_ds = RegionHistoryDataset(x_train_s, y_train_change_s, t_train, train_meta)
                val_ds = RegionHistoryDataset(x_val_s, y_val_change_s, t_val, val_meta)
                test_ds = RegionHistoryDataset(x_test_s, y_test_change_s, t_test, test_meta)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model, loaded_keys = _load_bern_backbone_only(model, region_name)
                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"change_target_{variant}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_change_s, true_change_s = _predict_model(model, test_loader, device)
                pred_change_abs, true_change_abs = _inverse_scale(pred_change_s, true_change_s, y_scaler)
                pred_abs = last_test[:, idx] + pred_change_abs
                true_abs = y_test[:, idx]

                if pred_full_abs is None:
                    pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                    true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": variant,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_backbone_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = variant
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_mae = float(overall.loc[overall["Model"].eq("Change target: VF-only"), "MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "seeds": seeds,
    }



In [ ]:
# Run change-target forecasting test.
# Start with 3 seeds; increase after checking runtime and direction.
N_CHANGE_TARGET_SEEDS = 3
CHANGE_TARGET_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_CHANGE_TARGET_SEEDS).tolist()
)
print("Random change-target seeds:", CHANGE_TARGET_SEEDS)

change_target_outputs = run_change_target_metadata_screen(
    seeds=CHANGE_TARGET_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    time_input_mode="input_times",
    variants=("change_vf_only", "change_iop_exposure", "change_iop_exposure_spatial_rnfl"),
)

change_target_overall = change_target_outputs["overall"].copy()
display(change_target_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])



## 25. Auxiliary progression multitask loss

The previous test showed that making pointwise VF change the main target is unstable. This block keeps the main task as absolute future pointwise VF prediction, but adds auxiliary region-level targets:

- future region mean VF
- region mean change from the last input visit

The idea is that IOP/RNFL may be more useful for global or regional progression signals than for each exact pointwise dB value.



In [ ]:
# Auxiliary progression multitask loss.
# Main task remains future pointwise VF. Auxiliary tasks are region mean and region mean change.


class MultiTaskHistoryDataset(torch.utils.data.Dataset):
    def __init__(self, x_list, y_array, t_list, meta_array, aux_mean, aux_change):
        self.x_list = x_list
        self.y_array = y_array
        self.t_list = t_list
        self.meta_array = meta_array
        self.aux_mean = aux_mean
        self.aux_change = aux_change

    def __len__(self):
        return len(self.x_list)

    def __getitem__(self, idx):
        x = torch.tensor(self.x_list[idx], dtype=torch.float32)
        y = torch.tensor(self.y_array[idx], dtype=torch.float32)
        t = torch.tensor(self.t_list[idx], dtype=torch.float32)
        m = torch.tensor(self.meta_array[idx], dtype=torch.float32)
        aux_mean = torch.tensor(self.aux_mean[idx], dtype=torch.float32)
        aux_change = torch.tensor(self.aux_change[idx], dtype=torch.float32)
        length = torch.tensor(len(self.x_list[idx]), dtype=torch.long)
        return x, t, length, y, m, aux_mean, aux_change


def multitask_collate(batch):
    xs, ts, lens, ys, metas, aux_means, aux_changes = zip(*batch)
    lengths = torch.stack(lens, dim=0)
    order = torch.argsort(lengths, descending=True)

    xs = [xs[i] for i in order]
    ts = [ts[i] for i in order]
    ys = [ys[i] for i in order]
    metas = [metas[i] for i in order]
    aux_means = [aux_means[i] for i in order]
    aux_changes = [aux_changes[i] for i in order]
    lengths = lengths[order]

    padded_x = pad_sequence(xs, batch_first=True)
    padded_t = pad_sequence(ts, batch_first=True)
    targets = torch.stack(ys, dim=0)
    metas = torch.stack(metas, dim=0)
    aux_means = torch.stack(aux_means, dim=0)
    aux_changes = torch.stack(aux_changes, dim=0)
    return padded_x, padded_t, lengths, targets, metas, aux_means, aux_changes


class AuxiliaryProgressionModel(nn.Module):
    """Absolute VF prediction with auxiliary region-level progression heads."""
    def __init__(
        self,
        meta_dim=0,
        input_dim=59,
        latent_dim=64,
        output_dim=59,
        pe_input_dim=16,
        pe_output_dim=16,
        meta_hidden_dim=32,
        meta_emb_dim=16,
        correction_hidden_dim=64,
    ):
        super().__init__()
        self.meta_dim = int(meta_dim)
        self.encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
        self.time_pe = PEBlock(dim_in=pe_input_dim, dim_out=pe_output_dim)
        self.lstm = nn.LSTM(
            input_size=latent_dim + pe_output_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
        )
        self.decoder = Decoder(latent_dim=latent_dim, output_dim=output_dim)

        if self.meta_dim > 0:
            self.meta_encoder = MetaEncoder(input_dim=self.meta_dim, hidden_dim=meta_hidden_dim, output_dim=meta_emb_dim)
            fused_dim = latent_dim + meta_emb_dim
            self.correction_head = nn.Sequential(
                nn.Linear(fused_dim, correction_hidden_dim),
                nn.ReLU(),
                nn.Linear(correction_hidden_dim, output_dim),
            )
            nn.init.zeros_(self.correction_head[-1].weight)
            nn.init.zeros_(self.correction_head[-1].bias)
        else:
            self.meta_encoder = None
            self.correction_head = None
            fused_dim = latent_dim

        self.aux_mean_head = nn.Sequential(
            nn.Linear(fused_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
        self.aux_change_head = nn.Sequential(
            nn.Linear(fused_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, padded_seqs, padded_times, lengths, meta_input):
        enc_out = self.encoder(padded_seqs)
        time_emb = self.time_pe(padded_times)
        lstm_in = torch.cat([enc_out, time_emb], dim=-1)
        packed = pack_padded_sequence(lstm_in, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (h_n, _) = self.lstm(packed)
        h_vf = h_n[-1]

        point_pred = self.decoder(h_vf)
        if self.meta_dim > 0:
            h_meta = self.meta_encoder(meta_input)
            fused = torch.cat([h_vf, h_meta], dim=-1)
            point_pred = point_pred + self.correction_head(fused)
        else:
            fused = h_vf

        aux_mean = self.aux_mean_head(fused)
        aux_change = self.aux_change_head(fused)
        return point_pred, aux_mean, aux_change


def _train_multitask_model(
    model,
    run_name,
    train_loader,
    val_loader,
    train_ds,
    val_ds,
    y_scaler,
    device,
    num_epochs=20,
    lr=1e-3,
    aux_weight=0.15,
):
    criterion = nn.MSELoss()
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    all_stds = torch.tensor(y_scaler.scale_, dtype=torch.float32).to(device)

    best_val_mae_db = float("inf")
    best_epoch = -1
    best_state_dict = None
    history_rows = []

    for epoch in range(1, num_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        for padded_seqs, padded_times, lengths, targets, metas, aux_mean, aux_change in train_loader:
            padded_seqs = padded_seqs.to(device)
            padded_times = padded_times.to(device)
            lengths = lengths.to(device)
            targets = targets.to(device)
            metas = metas.to(device)
            aux_mean = aux_mean.to(device)
            aux_change = aux_change.to(device)

            optimizer.zero_grad()
            pred_point, pred_mean, pred_change = model(padded_seqs, padded_times, lengths, metas)
            loss_point = criterion(pred_point, targets)
            loss_mean = criterion(pred_mean, aux_mean)
            loss_change = criterion(pred_change, aux_change)
            loss = loss_point + aux_weight * (loss_mean + loss_change)
            loss.backward()
            optimizer.step()
            train_loss_sum += loss.item() * padded_seqs.size(0)

        train_loss = train_loss_sum / len(train_ds)
        model.eval()
        val_loss_sum = 0.0
        val_mae_db_sum = 0.0
        with torch.no_grad():
            for padded_seqs, padded_times, lengths, targets, metas, aux_mean, aux_change in val_loader:
                padded_seqs = padded_seqs.to(device)
                padded_times = padded_times.to(device)
                lengths = lengths.to(device)
                targets = targets.to(device)
                metas = metas.to(device)
                aux_mean = aux_mean.to(device)
                aux_change = aux_change.to(device)

                pred_point, pred_mean, pred_change = model(padded_seqs, padded_times, lengths, metas)
                loss_point = criterion(pred_point, targets)
                loss_mean = criterion(pred_mean, aux_mean)
                loss_change = criterion(pred_change, aux_change)
                val_loss = loss_point + aux_weight * (loss_mean + loss_change)
                val_loss_sum += val_loss.item() * padded_seqs.size(0)
                abs_err = torch.abs(pred_point - targets)
                batch_mae_db = (abs_err * all_stds).mean().item()
                val_mae_db_sum += batch_mae_db * padded_seqs.size(0)

        val_loss_avg = val_loss_sum / len(val_ds)
        val_mae_db = val_mae_db_sum / len(val_ds)
        if val_mae_db < best_val_mae_db:
            best_val_mae_db = val_mae_db
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())

        history_rows.append({
            "run_name": run_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss_avg,
            "val_mae_db": val_mae_db,
            "is_best": epoch == best_epoch,
        })

    model.load_state_dict(best_state_dict)
    return model, pd.DataFrame(history_rows), {
        "run_name": run_name,
        "best_val_mae_db": best_val_mae_db,
        "best_epoch": best_epoch,
    }


def _predict_multitask_point(model, data_loader, device):
    model.eval()
    preds_all = []
    trues_all = []
    with torch.no_grad():
        for padded_seqs, padded_times, lengths, targets, metas, aux_mean, aux_change in data_loader:
            padded_seqs = padded_seqs.to(device)
            padded_times = padded_times.to(device)
            lengths = lengths.to(device)
            targets = targets.to(device)
            metas = metas.to(device)
            pred_point, _, _ = model(padded_seqs, padded_times, lengths, metas)
            preds_all.append(pred_point.detach().cpu().numpy())
            trues_all.append(targets.detach().cpu().numpy())
    return np.vstack(preds_all), np.vstack(trues_all)


auxiliary_progression_rnfl_map = {
    "Supero_Nasal": ["RNFL_I", "RNFL_N"],
    "Supero_Temporal": ["RNFL_I", "RNFL_T"],
    "Infero_Nasal": ["RNFL_S", "RNFL_N"],
    "Infero_Temporal": ["RNFL_S", "RNFL_T"],
    "Temporal": ["RNFL_T"],
    "Macular": ["RNFL_T"],
}


def run_auxiliary_progression_multitask_screen(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    aux_weight=0.15,
    time_input_mode="input_times",
    variants=("aux_vf_only", "aux_iop_exposure", "aux_iop_exposure_spatial_rnfl"),
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    variant_names = {
        "aux_vf_only": "Auxiliary task: VF-only",
        "aux_iop_exposure": "Auxiliary task: VF + IOP exposure",
        "aux_iop_exposure_spatial_rnfl": "Auxiliary task: VF + IOP exposure + spatial RNFL",
    }

    raw_results = []
    run_summaries = []

    print("Auxiliary multitask train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Auxiliary multitask seeds:", seeds)
    print("Auxiliary loss weight:", aux_weight)

    for variant in variants:
        exp_name = variant_names[variant]
        print("\n" + "#" * 100)
        print(f"AUXILIARY MULTITASK: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            print(f"-- seed={seed}")
            hist_module._set_seed(int(seed))
            pred_full_abs = None
            true_full_abs = None

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                print(f"{exp_name} | seed={seed} | region={region_name}")

                empty_train = np.zeros((len(train_df_all), 0), dtype=np.float32)
                empty_val = np.zeros((len(val_df_all), 0), dtype=np.float32)
                empty_test = np.zeros((len(test_df_all), 0), dtype=np.float32)
                train_df_base = _set_meta_array(train_df_all, empty_train)
                val_df_base = _set_meta_array(val_df_all, empty_val)
                test_df_base = _set_meta_array(test_df_all, empty_test)

                x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_base)
                x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_base)
                x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_base)

                train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
                val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
                test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

                x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
                x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
                x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)

                df_train_region = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
                df_val_region = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
                df_test_region = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

                x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
                y_train_r = y_train[:, idx]
                y_val_r = y_val[:, idx]
                y_test_r = y_test[:, idx]

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
                y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
                y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

                last_train = np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_train], axis=0)[:, idx]
                last_val = np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_val], axis=0)[:, idx]
                last_test = np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_test], axis=0)[:, idx]

                aux_mean_train_raw = np.mean(y_train_r, axis=1, keepdims=True).astype(np.float32)
                aux_mean_val_raw = np.mean(y_val_r, axis=1, keepdims=True).astype(np.float32)
                aux_mean_test_raw = np.mean(y_test_r, axis=1, keepdims=True).astype(np.float32)
                aux_change_train_raw = (np.mean(y_train_r, axis=1, keepdims=True) - np.mean(last_train, axis=1, keepdims=True)).astype(np.float32)
                aux_change_val_raw = (np.mean(y_val_r, axis=1, keepdims=True) - np.mean(last_val, axis=1, keepdims=True)).astype(np.float32)
                aux_change_test_raw = (np.mean(y_test_r, axis=1, keepdims=True) - np.mean(last_test, axis=1, keepdims=True)).astype(np.float32)

                aux_mean_scaler = StandardScaler()
                aux_change_scaler = StandardScaler()
                aux_mean_train = aux_mean_scaler.fit_transform(aux_mean_train_raw).astype(np.float32)
                aux_mean_val = aux_mean_scaler.transform(aux_mean_val_raw).astype(np.float32)
                aux_mean_test = aux_mean_scaler.transform(aux_mean_test_raw).astype(np.float32)
                aux_change_train = aux_change_scaler.fit_transform(aux_change_train_raw).astype(np.float32)
                aux_change_val = aux_change_scaler.transform(aux_change_val_raw).astype(np.float32)
                aux_change_test = aux_change_scaler.transform(aux_change_test_raw).astype(np.float32)

                if variant == "aux_vf_only":
                    train_meta = meta_train_empty
                    val_meta = meta_val_empty
                    test_meta = meta_test_empty
                    meta_dim = 0
                elif variant == "aux_iop_exposure":
                    train_meta_raw = _iop_exposure_matrix(df_train_region)
                    val_meta_raw = _iop_exposure_matrix(df_val_region)
                    test_meta_raw = _iop_exposure_matrix(df_test_region)
                    train_meta, val_meta, test_meta, _ = _scale_static_meta_matrices(
                        train_meta_raw, val_meta_raw, test_meta_raw
                    )
                    meta_dim = train_meta.shape[1]
                elif variant == "aux_iop_exposure_spatial_rnfl":
                    rnfl_cols = auxiliary_progression_rnfl_map.get(region_name, [])
                    train_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_train_region),
                        _region_rnfl_matrix(df_train_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    val_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_val_region),
                        _region_rnfl_matrix(df_val_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    test_meta_raw = np.concatenate([
                        _iop_exposure_matrix(df_test_region),
                        _region_rnfl_matrix(df_test_region, rnfl_cols, rnfl_master_cols),
                    ], axis=1)
                    train_meta, val_meta, test_meta, _ = _scale_static_meta_matrices(
                        train_meta_raw, val_meta_raw, test_meta_raw
                    )
                    meta_dim = train_meta.shape[1]
                else:
                    raise ValueError(f"Unknown auxiliary variant: {variant}")

                train_ds = MultiTaskHistoryDataset(x_train_s, y_train_s, t_train, train_meta, aux_mean_train, aux_change_train)
                val_ds = MultiTaskHistoryDataset(x_val_s, y_val_s, t_val, val_meta, aux_mean_val, aux_change_val)
                test_ds = MultiTaskHistoryDataset(x_test_s, y_test_s, t_test, test_meta, aux_mean_test, aux_change_test)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=multitask_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=multitask_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=multitask_collate)

                model = AuxiliaryProgressionModel(meta_dim=meta_dim, output_dim=len(idx))
                model, loaded_keys = _load_bern_weights_into_model(model, region_name)
                model, hist_df, summary = _train_multitask_model(
                    model=model,
                    run_name=f"aux_multitask_{variant}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                    aux_weight=aux_weight,
                )

                pred_s, true_s = _predict_multitask_point(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                if pred_full_abs is None:
                    pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                    true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": variant,
                    "seed": int(seed),
                    "region": region_name,
                    "meta_dim": meta_dim,
                    "aux_weight": aux_weight,
                    "loaded_bern_keys": len(loaded_keys),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = variant
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_mae = float(overall.loc[overall["Model"].eq("Auxiliary task: VF-only"), "MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "seeds": seeds,
        "aux_weight": aux_weight,
    }



In [ ]:
# Run auxiliary progression multitask test.
# Start with 3 seeds. If the direction is promising, repeat with 5-10 seeds.
N_AUX_MULTITASK_SEEDS = 3
AUX_MULTITASK_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_AUX_MULTITASK_SEEDS).tolist()
)
print("Random auxiliary multitask seeds:", AUX_MULTITASK_SEEDS)

aux_multitask_outputs = run_auxiliary_progression_multitask_screen(
    seeds=AUX_MULTITASK_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=1e-3,
    aux_weight=0.15,
    time_input_mode="input_times",
    variants=("aux_vf_only", "aux_iop_exposure", "aux_iop_exposure_spatial_rnfl"),
)

aux_multitask_overall = aux_multitask_outputs["overall"].copy()
display(aux_multitask_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])



## 26. Metadata signal diagnostic: can metadata predict VF-only residuals?

This block directly tests whether metadata contains signal after the VF-only model has already made its prediction.

Procedure:

1. Train the usual Bern-initialized VF-only model.
2. Compute test residuals: `true VF - VF-only prediction`.
3. Use simple non-deep-learning models to predict the residual from metadata only.
4. Add the predicted residual back to the VF-only prediction and check whether MAE improves.

If even simple residual models cannot improve MAE, then the current metadata likely has very limited additional signal for pointwise VF forecasting.



In [ ]:
# Metadata signal diagnostic: can simple metadata-only models predict VF-only residuals?
# This is meant as a data-signal check, not a new neural architecture.

from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor


def _fit_residual_model(model_kind, x_train, y_train):
    if model_kind == "ridge":
        return RidgeCV(alphas=np.logspace(-4, 4, 17)).fit(x_train, y_train)
    if model_kind == "random_forest":
        base = RandomForestRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=8,
            random_state=42,
            n_jobs=-1,
        )
        return MultiOutputRegressor(base).fit(x_train, y_train)
    if model_kind == "extra_trees":
        base = ExtraTreesRegressor(
            n_estimators=300,
            max_depth=3,
            min_samples_leaf=8,
            random_state=42,
            n_jobs=-1,
        )
        return MultiOutputRegressor(base).fit(x_train, y_train)
    raise ValueError(f"Unknown residual model kind: {model_kind}")


def _predict_residual_model(model, x):
    return np.asarray(model.predict(x), dtype=np.float32)


def _safe_mae_matrix(pred, true):
    return float(np.mean(np.abs(np.asarray(pred, dtype=np.float32) - np.asarray(true, dtype=np.float32))))


def _metadata_matrix_for_diagnostic(df_part, kind, region_name, rnfl_master_cols):
    if kind == "iop_history":
        return _iop_history_matrix(df_part)
    if kind == "iop_exposure":
        return _iop_exposure_matrix(df_part)
    if kind == "spatial_rnfl":
        rnfl_cols = auxiliary_progression_rnfl_map.get(region_name, [])
        return _region_rnfl_matrix(df_part, rnfl_cols, rnfl_master_cols)
    if kind == "iop_exposure_spatial_rnfl":
        rnfl_cols = auxiliary_progression_rnfl_map.get(region_name, [])
        return np.concatenate([
            _iop_exposure_matrix(df_part),
            _region_rnfl_matrix(df_part, rnfl_cols, rnfl_master_cols),
        ], axis=1)
    raise ValueError(f"Unknown metadata diagnostic kind: {kind}")


def run_metadata_residual_signal_diagnostic(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs_base=20,
    batch_size=64,
    lr_base=1e-3,
    time_input_mode="input_times",
    metadata_kinds=("iop_history", "iop_exposure", "spatial_rnfl", "iop_exposure_spatial_rnfl"),
    residual_model_kinds=("ridge", "random_forest", "extra_trees"),
    correction_scales=(0.25, 0.5, 1.0),
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    raw_rows = []
    model_rows = []

    print("Residual diagnostic train samples:", len(train_df_all), "| val:", len(val_df_all), "| test:", len(test_df_all))
    print("Residual diagnostic seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"METADATA RESIDUAL DIAGNOSTIC SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_base_pred = None
        seed_true = None
        seed_corrected = None

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            print(f"Diagnostic base VF-only | seed={seed} | region={region_name}")

            empty_train = np.zeros((len(train_df_all), 0), dtype=np.float32)
            empty_val = np.zeros((len(val_df_all), 0), dtype=np.float32)
            empty_test = np.zeros((len(test_df_all), 0), dtype=np.float32)
            train_df_base = _set_meta_array(train_df_all, empty_train)
            val_df_base = _set_meta_array(val_df_all, empty_val)
            test_df_base = _set_meta_array(test_df_all, empty_test)

            x_train, y_train, t_train, meta_train = _df_to_arrays_with_history(train_df_base)
            x_val, y_val, t_val, meta_val = _df_to_arrays_with_history(val_df_base)
            x_test, y_test, t_test, meta_test = _df_to_arrays_with_history(test_df_base)

            train_mask = _finite_mask(x_train, y_train, t_train, meta_train)
            val_mask = _finite_mask(x_val, y_val, t_val, meta_val)
            test_mask = _finite_mask(x_test, y_test, t_test, meta_test)

            x_train, y_train, t_train, meta_train = _apply_mask(x_train, y_train, t_train, meta_train, train_mask)
            x_val, y_val, t_val, meta_val = _apply_mask(x_val, y_val, t_val, meta_val, val_mask)
            x_test, y_test, t_test, meta_test = _apply_mask(x_test, y_test, t_test, meta_test, test_mask)

            df_train_region = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
            df_val_region = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
            df_test_region = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

            x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
            y_train_r = y_train[:, idx]
            y_val_r = y_val[:, idx]
            y_test_r = y_test[:, idx]

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train_r).astype(np.float32)
            y_val_s = y_scaler.transform(y_val_r).astype(np.float32)
            y_test_s = y_scaler.transform(y_test_r).astype(np.float32)

            train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train)
            val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val)
            test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test)
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
            test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

            base_model = VFOnlyModel(output_dim=len(idx))
            base_model, _ = _load_bern_weights_into_model(base_model, region_name)
            base_model, _, _ = _train_model(
                model=base_model,
                run_name=f"metadata_signal_vf_only_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader,
                val_loader=val_loader,
                train_ds=train_ds,
                val_ds=val_ds,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs_base,
                lr=lr_base,
            )

            train_pred_s, train_true_s = _predict_model(base_model, train_loader, device)
            val_pred_s, val_true_s = _predict_model(base_model, val_loader, device)
            test_pred_s, test_true_s = _predict_model(base_model, test_loader, device)
            train_pred_abs, train_true_abs = _inverse_scale(train_pred_s, train_true_s, y_scaler)
            val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)
            test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

            train_residual = (train_true_abs - train_pred_abs).astype(np.float32)
            val_residual = (val_true_abs - val_pred_abs).astype(np.float32)
            test_residual = (test_true_abs - test_pred_abs).astype(np.float32)

            if seed_base_pred is None:
                n_test_clean = len(y_test)
                seed_base_pred = np.zeros((n_test_clean, len(td_cols_exp)), dtype=np.float32)
                seed_true = np.zeros((n_test_clean, len(td_cols_exp)), dtype=np.float32)
                seed_corrected = {
                    (meta_kind, residual_kind, scale): np.zeros((n_test_clean, len(td_cols_exp)), dtype=np.float32)
                    for meta_kind in metadata_kinds
                    for residual_kind in residual_model_kinds
                    for scale in correction_scales
                }

            seed_base_pred[:, idx] = test_pred_abs
            seed_true[:, idx] = test_true_abs

            for meta_kind in metadata_kinds:
                train_meta_raw = _metadata_matrix_for_diagnostic(df_train_region, meta_kind, region_name, rnfl_master_cols)
                val_meta_raw = _metadata_matrix_for_diagnostic(df_val_region, meta_kind, region_name, rnfl_master_cols)
                test_meta_raw = _metadata_matrix_for_diagnostic(df_test_region, meta_kind, region_name, rnfl_master_cols)
                train_meta_s, val_meta_s, test_meta_s, _ = _scale_static_meta_matrices(
                    train_meta_raw, val_meta_raw, test_meta_raw
                )

                for residual_kind in residual_model_kinds:
                    residual_model = _fit_residual_model(residual_kind, train_meta_s, train_residual)
                    val_residual_hat = _predict_residual_model(residual_model, val_meta_s)
                    test_residual_hat = _predict_residual_model(residual_model, test_meta_s)

                    val_scale_rows = []
                    for scale in correction_scales:
                        val_corrected = val_pred_abs + scale * val_residual_hat
                        val_mae = _safe_mae_matrix(val_corrected, val_true_abs)
                        val_scale_rows.append((scale, val_mae))
                    best_scale, best_val_mae = sorted(val_scale_rows, key=lambda x: x[1])[0]

                    for scale in correction_scales:
                        test_corrected = test_pred_abs + scale * test_residual_hat
                        seed_corrected[(meta_kind, residual_kind, scale)][:, idx] = test_corrected

                    model_rows.append({
                        "seed": int(seed),
                        "region": region_name,
                        "metadata": meta_kind,
                        "residual_model": residual_kind,
                        "best_val_scale": best_scale,
                        "best_val_region_mae": best_val_mae,
                        "base_val_region_mae": _safe_mae_matrix(val_pred_abs, val_true_abs),
                        "base_test_region_mae": _safe_mae_matrix(test_pred_abs, test_true_abs),
                    })

        base_seed_summary = _summarize_abs_prediction_matrix(seed_base_pred, seed_true, cluster_mapping, "VF-only base")
        base_seed_summary["seed"] = int(seed)
        raw_rows.append(base_seed_summary)

        for (meta_kind, residual_kind, scale), corrected_pred in seed_corrected.items():
            model_name = f"Residual diagnostic: {meta_kind} + {residual_kind} x{scale:g}"
            seed_summary = _summarize_abs_prediction_matrix(corrected_pred, seed_true, cluster_mapping, model_name)
            seed_summary["seed"] = int(seed)
            seed_summary["metadata"] = meta_kind
            seed_summary["residual_model"] = residual_kind
            seed_summary["scale"] = scale
            raw_rows.append(seed_summary)

    raw_long = pd.concat(raw_rows, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_mae = float(overall.loc[overall["Model"].eq("VF-only base"), "MAE_mean_db"].iloc[0])
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "residual_model_diagnostics": pd.DataFrame(model_rows),
        "seeds": seeds,
    }




In [ ]:
# Run metadata residual-signal diagnostic.
# This is cheaper than deep Transformer experiments but still trains VF-only bases per region/seed.
N_METADATA_SIGNAL_SEEDS = 3
METADATA_SIGNAL_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_METADATA_SIGNAL_SEEDS).tolist()
)
print("Random metadata-signal seeds:", METADATA_SIGNAL_SEEDS)

metadata_signal_outputs = run_metadata_residual_signal_diagnostic(
    seeds=METADATA_SIGNAL_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs_base=20,
    batch_size=64,
    lr_base=1e-3,
    time_input_mode="input_times",
    metadata_kinds=("iop_history", "iop_exposure", "spatial_rnfl", "iop_exposure_spatial_rnfl"),
    residual_model_kinds=("ridge", "random_forest", "extra_trees"),
    correction_scales=(0.25, 0.5, 1.0),
)

metadata_signal_overall = metadata_signal_outputs["overall"].copy()
display(metadata_signal_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
].head(20))



## 27. Compact Transformer FiLM metadata adapter

First metadata rescue attempt for the compact Bern-pretrained Transformer.

Instead of concatenating metadata directly into the token stream, this version keeps the VF-only compact Transformer path intact and lets metadata apply a small FiLM-style modulation to the encoded history features. The adapter is zero-initialized and scale-limited, so the model starts from the VF-only Transformer behavior and metadata can only gradually adjust it during GRAPE fine-tuning.


In [ ]:
# Compact Transformer + FiLM / conditional adapter.
# This block appends a new experiment and does not change any previous cells.

class CompactPointTransformerFiLMAdapterModel(PointLevelSpatioTemporalTransformerModel):
    """Compact point-level Transformer with safe metadata FiLM modulation.

    The base token path is the Bern-pretrained VF-only compact Transformer path:
    VF value + point embedding + time encoding + zero-IOP token embedding.

    Metadata is not concatenated as an extra token. Instead, a small adapter predicts
    gamma/beta values that gently modulate the encoded Transformer features:
        encoded = encoded * (1 + gamma) + beta

    The final adapter layer is zero-initialized, so the initial model is exactly the
    same as the VF-only compact Transformer. film_scale limits how strongly metadata
    can perturb the backbone during fine-tuning.
    """
    def __init__(
        self,
        *args,
        film_meta_dim=0,
        film_hidden_dim=None,
        film_scale=0.05,
        meta_hidden_dim=16,
        **kwargs,
    ):
        super().__init__(*args, meta_hidden_dim=meta_hidden_dim, **kwargs)
        self.film_meta_dim = int(film_meta_dim)
        self.film_scale = float(film_scale)
        model_dim = self.value_encoder[0].out_features
        if film_hidden_dim is None:
            film_hidden_dim = max(16, meta_hidden_dim)

        if self.film_meta_dim > 0:
            self.film_net = nn.Sequential(
                nn.LayerNorm(self.film_meta_dim),
                nn.Linear(self.film_meta_dim, film_hidden_dim),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(film_hidden_dim, 2 * model_dim),
            )
            nn.init.zeros_(self.film_net[-1].weight)
            nn.init.zeros_(self.film_net[-1].bias)
        else:
            self.film_net = None

    def forward(self, padded_seqs, padded_times, lengths, meta_input):
        batch_size, seq_len, n_points = padded_seqs.shape
        values = padded_seqs.unsqueeze(-1)
        value_tokens = self.value_encoder(values)

        point_ids = torch.arange(n_points, device=padded_seqs.device)
        point_tokens = self.point_embed(point_ids).view(1, 1, n_points, -1)
        time_tokens = self.time_pe(padded_times).unsqueeze(2)

        # Match Bern VF-only pretraining: pass a zero metadata channel through the
        # original metadata encoder instead of removing that branch completely.
        zero_iop = torch.zeros(batch_size, seq_len, 1, device=padded_seqs.device, dtype=padded_seqs.dtype)
        zero_iop_tokens = self.iop_encoder(zero_iop).unsqueeze(2)

        tokens = value_tokens + point_tokens + time_tokens + zero_iop_tokens
        tokens = tokens.reshape(batch_size, seq_len * n_points, -1)

        step_ids = torch.arange(seq_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
        visit_mask = step_ids >= lengths.unsqueeze(1)
        token_mask = visit_mask.unsqueeze(-1).expand(batch_size, seq_len, n_points).reshape(batch_size, seq_len * n_points)

        encoded = self.transformer(tokens, src_key_padding_mask=token_mask)

        if self.film_net is not None:
            film = self.film_net(meta_input)
            gamma, beta = torch.chunk(film, 2, dim=-1)
            gamma = self.film_scale * torch.tanh(gamma).unsqueeze(1)
            beta = self.film_scale * torch.tanh(beta).unsqueeze(1)
            encoded = encoded * (1.0 + gamma) + beta

        query_points = self.point_indices.to(padded_seqs.device)
        queries = self.output_point_embed(query_points).unsqueeze(0).expand(batch_size, -1, -1)
        attn_out, _ = self.output_attn(
            query=queries,
            key=encoded,
            value=encoded,
            key_padding_mask=token_mask,
            need_weights=False,
        )
        x = self.norm1(queries + attn_out)
        x = x + self.ffn(self.norm2(x))
        return self.out(x).squeeze(-1)


def run_compact_transformer_film_metadata_adapter(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    # Build the common VF-only arrays first. Static metadata is attached per variant below.
    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Compact Transformer VF-only (FiLM run)", "kind": "vf_only"},
        {"Model": "Compact Transformer + FiLM IOP exposure", "kind": "film_iop_exposure"},
        {"Model": "Compact Transformer + FiLM spatial RNFL", "kind": "film_spatial_rnfl"},
        {"Model": "Compact Transformer + FiLM IOP exposure + spatial RNFL", "kind": "film_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    print("Compact Transformer FiLM train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)
    print("FiLM scale:", film_scale, "| lr:", lr)

    for variant in variants:
        exp_name = variant["Model"]
        exp_kind = variant["kind"]
        print("\n" + "#" * 100)
        print(f"COMPACT TRANSFORMER FILM: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            hist_module._set_seed(int(seed))
            pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                print(f"{exp_name} | seed={seed} | region={region_name} | RNFL={rnfl_cols}")

                rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                    rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
                )

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "film_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "film_spatial_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif exp_kind == "film_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerFiLMAdapterModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    film_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"compact_transformer_film_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "film_scale": film_scale,
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_label = "Compact Transformer VF-only (FiLM run)"
    baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
        "iop_scaler": iop_scaler,
    }


In [ ]:
# Run the first rescue experiment with 3 random seeds.
# If this is still slow, reduce num_epochs to 10 for a smoke test.
N_FILM_ADAPTER_SEEDS = 3
FILM_ADAPTER_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_FILM_ADAPTER_SEEDS).tolist()
)
print("Random FiLM adapter seeds:", FILM_ADAPTER_SEEDS)

film_adapter_outputs = run_compact_transformer_film_metadata_adapter(
    seeds=FILM_ADAPTER_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
)

film_adapter_overall = film_adapter_outputs["overall"].copy()
display(film_adapter_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])


## 28. Compact Transformer policy-gated metadata fusion

Second metadata rescue attempt.

This is more conservative than FiLM. The model first keeps a VF-only prediction, then creates a metadata-adjusted prediction, and finally learns a small gate deciding how much of the metadata-adjusted prediction should be used.

The gate is initialized almost closed and capped by `gate_max`, so the model starts very close to the VF-only compact Transformer and can only use metadata if validation training finds a consistent benefit.


In [ ]:
# Compact Transformer + policy/reliability gate.
# The model explicitly mixes VF-only prediction and metadata-adjusted prediction.

class CompactPointTransformerPolicyGateModel(PointLevelSpatioTemporalTransformerModel):
    """Compact Transformer with a conservative metadata policy gate.

    base_pred: prediction from the Bern-pretrained VF-only token path.
    meta_pred: prediction after FiLM modulation from metadata.
    gate: learned metadata reliability weight, capped by gate_max.

    final = base_pred + gate * (meta_pred - base_pred)

    The FiLM adapter is zero-initialized, and the gate bias is strongly negative, so
    the initial model is effectively the VF-only compact Transformer.
    """
    def __init__(
        self,
        *args,
        policy_meta_dim=0,
        policy_hidden_dim=None,
        gate_max=0.25,
        film_scale=0.05,
        gate_init_bias=-6.0,
        meta_hidden_dim=16,
        **kwargs,
    ):
        super().__init__(*args, meta_hidden_dim=meta_hidden_dim, **kwargs)
        self.policy_meta_dim = int(policy_meta_dim)
        self.gate_max = float(gate_max)
        self.film_scale = float(film_scale)
        model_dim = self.value_encoder[0].out_features
        if policy_hidden_dim is None:
            policy_hidden_dim = max(16, meta_hidden_dim)

        if self.policy_meta_dim > 0:
            self.policy_norm = nn.LayerNorm(self.policy_meta_dim)
            self.film_net = nn.Sequential(
                nn.Linear(self.policy_meta_dim, policy_hidden_dim),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(policy_hidden_dim, 2 * model_dim),
            )
            self.gate_net = nn.Sequential(
                nn.Linear(self.policy_meta_dim, policy_hidden_dim),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(policy_hidden_dim, 1),
            )
            nn.init.zeros_(self.film_net[-1].weight)
            nn.init.zeros_(self.film_net[-1].bias)
            nn.init.zeros_(self.gate_net[-1].weight)
            nn.init.constant_(self.gate_net[-1].bias, gate_init_bias)
        else:
            self.policy_norm = None
            self.film_net = None
            self.gate_net = None

    def _encode_vf_only_tokens(self, padded_seqs, padded_times, lengths):
        batch_size, seq_len, n_points = padded_seqs.shape
        values = padded_seqs.unsqueeze(-1)
        value_tokens = self.value_encoder(values)
        point_ids = torch.arange(n_points, device=padded_seqs.device)
        point_tokens = self.point_embed(point_ids).view(1, 1, n_points, -1)
        time_tokens = self.time_pe(padded_times).unsqueeze(2)
        zero_iop = torch.zeros(batch_size, seq_len, 1, device=padded_seqs.device, dtype=padded_seqs.dtype)
        zero_iop_tokens = self.iop_encoder(zero_iop).unsqueeze(2)

        tokens = value_tokens + point_tokens + time_tokens + zero_iop_tokens
        tokens = tokens.reshape(batch_size, seq_len * n_points, -1)

        step_ids = torch.arange(seq_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
        visit_mask = step_ids >= lengths.unsqueeze(1)
        token_mask = visit_mask.unsqueeze(-1).expand(batch_size, seq_len, n_points).reshape(batch_size, seq_len * n_points)
        encoded = self.transformer(tokens, src_key_padding_mask=token_mask)
        return encoded, token_mask

    def _decode_encoded_tokens(self, encoded, token_mask, device):
        batch_size = encoded.shape[0]
        query_points = self.point_indices.to(device)
        queries = self.output_point_embed(query_points).unsqueeze(0).expand(batch_size, -1, -1)
        attn_out, _ = self.output_attn(
            query=queries,
            key=encoded,
            value=encoded,
            key_padding_mask=token_mask,
            need_weights=False,
        )
        x = self.norm1(queries + attn_out)
        x = x + self.ffn(self.norm2(x))
        return self.out(x).squeeze(-1)

    def forward(self, padded_seqs, padded_times, lengths, meta_input):
        encoded, token_mask = self._encode_vf_only_tokens(padded_seqs, padded_times, lengths)
        base_pred = self._decode_encoded_tokens(encoded, token_mask, padded_seqs.device)

        if self.policy_meta_dim == 0 or self.film_net is None:
            return base_pred

        meta = self.policy_norm(meta_input)
        film = self.film_net(meta)
        gamma, beta = torch.chunk(film, 2, dim=-1)
        gamma = self.film_scale * torch.tanh(gamma).unsqueeze(1)
        beta = self.film_scale * torch.tanh(beta).unsqueeze(1)
        meta_encoded = encoded * (1.0 + gamma) + beta
        meta_pred = self._decode_encoded_tokens(meta_encoded, token_mask, padded_seqs.device)

        gate = self.gate_max * torch.sigmoid(self.gate_net(meta))
        return base_pred + gate * (meta_pred - base_pred)


def run_compact_transformer_policy_gate_metadata(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Compact Transformer VF-only (policy run)", "kind": "vf_only"},
        {"Model": "Compact Transformer + policy gate IOP exposure", "kind": "policy_iop_exposure"},
        {"Model": "Compact Transformer + policy gate spatial RNFL", "kind": "policy_spatial_rnfl"},
        {"Model": "Compact Transformer + policy gate IOP exposure + spatial RNFL", "kind": "policy_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    print("Compact Transformer policy gate train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)
    print("FiLM scale:", film_scale, "| gate max:", gate_max, "| lr:", lr)

    for variant in variants:
        exp_name = variant["Model"]
        exp_kind = variant["kind"]
        print("\n" + "#" * 100)
        print(f"COMPACT TRANSFORMER POLICY GATE: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            hist_module._set_seed(int(seed))
            pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                print(f"{exp_name} | seed={seed} | region={region_name} | RNFL={rnfl_cols}")

                rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                    rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
                )

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_spatial_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif exp_kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"compact_transformer_policy_gate_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "film_scale": film_scale,
                    "gate_max": gate_max,
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_label = "Compact Transformer VF-only (policy run)"
    baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
        "iop_scaler": iop_scaler,
    }


In [ ]:
# Run the second rescue experiment with 3 random seeds.
N_POLICY_GATE_SEEDS = 3
POLICY_GATE_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_POLICY_GATE_SEEDS).tolist()
)
print("Random policy-gate seeds:", POLICY_GATE_SEEDS)

policy_gate_outputs = run_compact_transformer_policy_gate_metadata(
    seeds=POLICY_GATE_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
)

policy_gate_overall = policy_gate_outputs["overall"].copy()
display(policy_gate_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])


## 29. Structure-function discordance metadata

Third metadata rescue attempt.

Raw RNFL may not help because much of it is already reflected in the current VF. This block converts baseline RNFL into a structure-function discordance feature:

`RNFL discordance = observed baseline RNFL - RNFL expected from the last input VF`

The idea is to give the model only the structural information that is not already explained by VF history. This is tested with the conservative policy-gate compact Transformer from the previous block.


In [ ]:
# Structure-function discordance metadata.
# Use train-only Ridge models to estimate expected RNFL from the last input VF,
# then use observed RNFL - expected RNFL as the metadata signal.
from sklearn.linear_model import Ridge


def _last_vf_matrix_from_x_list(x_list):
    return np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_list], axis=0).astype(np.float32)


def _fit_transform_rnfl_discordance(train_x, val_x, test_x, train_rnfl, val_rnfl, test_rnfl, alpha=10.0):
    train_x = np.asarray(train_x, dtype=np.float32)
    val_x = np.asarray(val_x, dtype=np.float32)
    test_x = np.asarray(test_x, dtype=np.float32)
    train_rnfl = np.asarray(train_rnfl, dtype=np.float32)
    val_rnfl = np.asarray(val_rnfl, dtype=np.float32)
    test_rnfl = np.asarray(test_rnfl, dtype=np.float32)

    x_mean = np.nanmean(train_x, axis=0)
    x_mean = np.where(np.isfinite(x_mean), x_mean, 0.0).astype(np.float32)
    y_mean = np.nanmean(train_rnfl, axis=0)
    y_mean = np.where(np.isfinite(y_mean), y_mean, 0.0).astype(np.float32)

    def _fill_x(x):
        return np.where(np.isfinite(x), x, x_mean).astype(np.float32)

    def _fill_y(y):
        return np.where(np.isfinite(y), y, y_mean).astype(np.float32)

    x_train_f = _fill_x(train_x)
    x_val_f = _fill_x(val_x)
    x_test_f = _fill_x(test_x)
    y_train_f = _fill_y(train_rnfl)
    y_val_f = _fill_y(val_rnfl)
    y_test_f = _fill_y(test_rnfl)

    model = Ridge(alpha=alpha)
    model.fit(x_train_f, y_train_f)

    train_disc = y_train_f - model.predict(x_train_f)
    val_disc = y_val_f - model.predict(x_val_f)
    test_disc = y_test_f - model.predict(x_test_f)
    return train_disc.astype(np.float32), val_disc.astype(np.float32), test_disc.astype(np.float32), model


def run_compact_transformer_policy_gate_discordance_metadata(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    discordance_alpha=10.0,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)
    last_vf_train = _last_vf_matrix_from_x_list(x_train)
    last_vf_val = _last_vf_matrix_from_x_list(x_val)
    last_vf_test = _last_vf_matrix_from_x_list(x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Compact Transformer VF-only (discordance run)", "kind": "vf_only"},
        {"Model": "Compact Transformer + policy gate IOP exposure", "kind": "policy_iop_exposure"},
        {"Model": "Compact Transformer + policy gate RNFL discordance", "kind": "policy_rnfl_discordance"},
        {"Model": "Compact Transformer + policy gate IOP exposure + RNFL discordance", "kind": "policy_iop_discordance"},
    ]

    raw_results = []
    run_summaries = []
    print("Compact Transformer discordance train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)
    print("FiLM scale:", film_scale, "| gate max:", gate_max, "| discordance alpha:", discordance_alpha)

    for variant in variants:
        exp_name = variant["Model"]
        exp_kind = variant["kind"]
        print("\n" + "#" * 100)
        print(f"COMPACT TRANSFORMER DISCORDANCE: {exp_name}")
        print("#" * 100)

        for seed in seeds:
            hist_module._set_seed(int(seed))
            pred_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            true_full_abs = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                print(f"{exp_name} | seed={seed} | region={region_name} | RNFL={rnfl_cols}")

                rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_disc_train, rnfl_disc_val, rnfl_disc_test, rnfl_disc_model = _fit_transform_rnfl_discordance(
                    last_vf_train,
                    last_vf_val,
                    last_vf_test,
                    rnfl_train_raw,
                    rnfl_val_raw,
                    rnfl_test_raw,
                    alpha=discordance_alpha,
                )
                rnfl_disc_train_s, rnfl_disc_val_s, rnfl_disc_test_s, rnfl_disc_scaler = _scale_static_meta_matrices(
                    rnfl_disc_train,
                    rnfl_disc_val,
                    rnfl_disc_test,
                )

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_rnfl_discordance":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_disc_train_s, rnfl_disc_val_s, rnfl_disc_test_s
                elif exp_kind == "policy_iop_discordance":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_disc_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_disc_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_disc_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"compact_transformer_discordance_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                pred_full_abs[:, idx] = pred_abs
                true_full_abs[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "film_scale": film_scale,
                    "gate_max": gate_max,
                    "discordance_alpha": discordance_alpha,
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            seed_result = _summarize_abs_prediction_matrix(pred_full_abs, true_full_abs, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_label = "Compact Transformer VF-only (discordance run)"
    baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
        "iop_scaler": iop_scaler,
    }


In [ ]:
# Run the third rescue experiment with 3 random seeds.
N_DISCORDANCE_SEEDS = 3
DISCORDANCE_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_DISCORDANCE_SEEDS).tolist()
)
print("Random discordance seeds:", DISCORDANCE_SEEDS)

discordance_outputs = run_compact_transformer_policy_gate_discordance_metadata(
    seeds=DISCORDANCE_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    discordance_alpha=10.0,
)

discordance_overall = discordance_outputs["overall"].copy()
display(discordance_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])


## 30. Soft ensemble: choose metadata only when validation supports it

Fourth metadata rescue attempt.

The previous experiments show that metadata sometimes helps but is unstable. This block does not force a metadata model to replace the VF-only model. Instead, it learns a single validation-selected ensemble weight:

`final prediction = (1 - w) * VF-only prediction + w * metadata prediction`

The weight `w` is selected on the validation set from a small grid. If metadata is not useful, the best validation weight should move toward 0. This is a practical check of whether metadata contains enough signal to improve prediction without relying on end-to-end training to discover it.


In [ ]:
# Soft ensemble between VF-only compact Transformer and policy-gated metadata models.
# The ensemble weight is chosen on validation MAE in dB, then applied to test.


def _mae_abs(pred, true):
    return float(np.nanmean(np.abs(np.asarray(pred, dtype=np.float32) - np.asarray(true, dtype=np.float32))))


def _select_ensemble_weight(pred_base_val, pred_meta_val, true_val, weight_grid=None):
    if weight_grid is None:
        weight_grid = np.linspace(0.0, 0.5, 11, dtype=np.float32)
    rows = []
    best = None
    for w in weight_grid:
        pred = (1.0 - float(w)) * pred_base_val + float(w) * pred_meta_val
        mae = _mae_abs(pred, true_val)
        rows.append({"w": float(w), "val_mae": mae})
        if best is None or mae < best["val_mae"]:
            best = {"w": float(w), "val_mae": mae}
    return best, pd.DataFrame(rows)


def run_compact_transformer_policy_gate_soft_ensemble(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=None,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Soft ensemble: VF-only", "kind": "vf_only"},
        {"Model": "Soft ensemble: policy IOP exposure", "kind": "policy_iop_exposure"},
        {"Model": "Soft ensemble: policy IOP exposure + spatial RNFL", "kind": "policy_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    ensemble_rows = []
    print("Soft ensemble train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"SOFT ENSEMBLE SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_val_pred = {v["kind"]: np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_test_pred = {v["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_val_true = np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32)
        seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            for variant in variants:
                exp_name = variant["Model"]
                exp_kind = variant["kind"]

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"soft_ensemble_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                val_pred_s, val_true_s = _predict_model(model, val_loader, device)
                val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)
                test_pred_s, test_true_s = _predict_model(model, test_loader, device)
                test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

                seed_val_pred[exp_kind][:, idx] = val_pred_abs
                seed_test_pred[exp_kind][:, idx] = test_pred_abs
                seed_val_true[:, idx] = val_true_abs
                seed_test_true[:, idx] = test_true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        # Store base and raw metadata models.
        for variant in variants:
            exp_name = variant["Model"]
            exp_kind = variant["kind"]
            seed_result = _summarize_abs_prediction_matrix(seed_test_pred[exp_kind], seed_test_true, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

        # Add validation-selected soft ensembles.
        for meta_kind, meta_label in [
            ("policy_iop_exposure", "Soft ensemble: VF-only + policy IOP exposure"),
            ("policy_iop_rnfl", "Soft ensemble: VF-only + policy IOP exposure + spatial RNFL"),
        ]:
            best, grid_df = _select_ensemble_weight(
                seed_val_pred["vf_only"],
                seed_val_pred[meta_kind],
                seed_val_true,
                weight_grid=weight_grid,
            )
            w = float(best["w"])
            ens_test_pred = (1.0 - w) * seed_test_pred["vf_only"] + w * seed_test_pred[meta_kind]
            seed_result = _summarize_abs_prediction_matrix(ens_test_pred, seed_test_true, cluster_mapping, meta_label)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = f"ensemble_{meta_kind}"
            raw_results.append(seed_result)
            ensemble_rows.append({
                "seed": int(seed),
                "meta_kind": meta_kind,
                "selected_w": w,
                "best_val_mae": float(best["val_mae"]),
            })

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_label = "Soft ensemble: VF-only"
    baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "ensemble_weights": pd.DataFrame(ensemble_rows),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


In [ ]:
# Run the soft-ensemble experiment with 3 random seeds.
N_SOFT_ENSEMBLE_SEEDS = 3
SOFT_ENSEMBLE_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_SOFT_ENSEMBLE_SEEDS).tolist()
)
print("Random soft-ensemble seeds:", SOFT_ENSEMBLE_SEEDS)

soft_ensemble_outputs = run_compact_transformer_policy_gate_soft_ensemble(
    seeds=SOFT_ENSEMBLE_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
)

soft_ensemble_overall = soft_ensemble_outputs["overall"].copy()
display(soft_ensemble_overall[
    [
        "Model",
        "Region",
        "MAE_mean_db",
        "MAE_seed_std_db",
        "MAE_sample_std_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]
])

display(soft_ensemble_outputs["ensemble_weights"])


## 31. Regional metadata benefit diagnostic

This block does not train new models. It checks whether the metadata benefit is hidden in specific anatomical regions.

For each recent metadata experiment, it computes per-region delta against the matching VF-only baseline from the same run. This helps separate two possibilities:

1. metadata is globally weak, or
2. metadata helps only certain regions but gets diluted in the overall MAE.


In [ ]:
# Regional diagnostic for recent metadata rescue experiments.
# Run after the FiLM / policy-gate / discordance / soft-ensemble outputs exist.

def _regional_delta_table(outputs, experiment_name, baseline_label):
    if outputs is None or "summary_long" not in outputs:
        return pd.DataFrame()
    df = outputs["summary_long"].copy()
    if df.empty:
        return pd.DataFrame()
    base = df[df["Model"].eq(baseline_label)][["Region", "MAE_mean_db"]].rename(
        columns={"MAE_mean_db": "baseline_mae_db"}
    )
    out = df.merge(base, on="Region", how="left")
    out = out[~out["Model"].eq(baseline_label)].copy()
    out["experiment"] = experiment_name
    out["delta_vs_region_baseline"] = out["MAE_mean_db"] - out["baseline_mae_db"]
    out["delta_display"] = out["delta_vs_region_baseline"].map(lambda x: f"{x:+.3f} dB")
    out["improved_region"] = out["delta_vs_region_baseline"] < 0
    return out

regional_frames = []
if "film_adapter_outputs" in globals():
    regional_frames.append(_regional_delta_table(
        film_adapter_outputs,
        "FiLM adapter",
        "Compact Transformer VF-only (FiLM run)",
    ))
if "policy_gate_outputs" in globals():
    regional_frames.append(_regional_delta_table(
        policy_gate_outputs,
        "Policy gate",
        "Compact Transformer VF-only (policy run)",
    ))
if "discordance_outputs" in globals():
    regional_frames.append(_regional_delta_table(
        discordance_outputs,
        "RNFL discordance",
        "Compact Transformer VF-only (discordance run)",
    ))
if "soft_ensemble_outputs" in globals():
    regional_frames.append(_regional_delta_table(
        soft_ensemble_outputs,
        "Soft ensemble",
        "Soft ensemble: VF-only",
    ))

regional_delta_all = pd.concat([x for x in regional_frames if not x.empty], axis=0, ignore_index=True)

if regional_delta_all.empty:
    print("No recent experiment outputs found. Run the FiLM / policy / discordance / soft-ensemble cells first.")
else:
    regional_best = (
        regional_delta_all[regional_delta_all["Region"].ne("Overall")]
        .sort_values(["Region", "delta_vs_region_baseline"])
        .groupby("Region", as_index=False)
        .first()
        .sort_values("delta_vs_region_baseline")
        .reset_index(drop=True)
    )

    regional_model_counts = (
        regional_delta_all[regional_delta_all["Region"].ne("Overall")]
        .groupby(["experiment", "Model"], as_index=False)
        .agg(
            improved_regions=("improved_region", "sum"),
            mean_delta=("delta_vs_region_baseline", "mean"),
            best_region_delta=("delta_vs_region_baseline", "min"),
            worst_region_delta=("delta_vs_region_baseline", "max"),
        )
        .sort_values(["improved_regions", "mean_delta"], ascending=[False, True])
        .reset_index(drop=True)
    )
    regional_model_counts["mean_delta_display"] = regional_model_counts["mean_delta"].map(lambda x: f"{x:+.3f} dB")
    regional_model_counts["best_region_delta_display"] = regional_model_counts["best_region_delta"].map(lambda x: f"{x:+.3f} dB")
    regional_model_counts["worst_region_delta_display"] = regional_model_counts["worst_region_delta"].map(lambda x: f"{x:+.3f} dB")

    print("Best metadata method per anatomical region:")
    display(regional_best[[
        "Region",
        "experiment",
        "Model",
        "MAE_mean_db",
        "baseline_mae_db",
        "delta_display",
        "improved_region",
    ]])

    print("Metadata model summary across regions:")
    display(regional_model_counts[[
        "experiment",
        "Model",
        "improved_regions",
        "mean_delta_display",
        "best_region_delta_display",
        "worst_region_delta_display",
    ]])

    print("Full regional delta table:")
    regional_delta_display = (
        regional_delta_all
        .sort_values(["Region", "delta_vs_region_baseline"])
        .reset_index(drop=True)
    )
    display(regional_delta_display[[
        "experiment",
        "Model",
        "Region",
        "MAE_mean_db",
        "baseline_mae_db",
        "delta_display",
        "improved_region",
    ]])


## 32. Final region-specific soft-ensemble comparison

Clean comparison for reporting.

Backbone: Bern-pretrained compact point-level Transformer, fine-tuned on GRAPE with the same region-specific setup as before.

Comparison:

1. VF-only compact Transformer baseline
2. Soft ensemble: VF-only + policy-gated IOP exposure + baseline spatial RNFL

The output table reports Overall and every anatomical region, so we can see whether metadata helps only in specific regions instead of relying only on Overall MAE.


In [ ]:
# Final region-specific comparison table for PPT/reporting.
# This runs a clean 3-seed soft-ensemble experiment and then displays Overall + all regions.

FINAL_REGION_BASELINE_LABEL = "Soft ensemble: VF-only"
FINAL_REGION_META_LABEL = "Soft ensemble: VF-only + policy IOP exposure + spatial RNFL"


def make_region_specific_soft_ensemble_table(outputs, baseline_label=FINAL_REGION_BASELINE_LABEL, meta_label=FINAL_REGION_META_LABEL):
    summary = outputs["summary_long"].copy()
    base = summary[summary["Model"].eq(baseline_label)][[
        "Region",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "Samples",
        "#Pts",
    ]].rename(columns={
        "MAE_mean_db": "VF-only MAE_mean_db",
        "MAE ± SD (dB)": "VF-only MAE ± SD (dB)",
        "Samples": "VF-only Samples",
        "#Pts": "#Pts",
    })
    meta = summary[summary["Model"].eq(meta_label)][[
        "Region",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "Samples",
        "#Pts",
    ]].rename(columns={
        "MAE_mean_db": "Soft ensemble MAE_mean_db",
        "MAE ± SD (dB)": "Soft ensemble MAE ± SD (dB)",
        "Samples": "Soft ensemble Samples",
        "#Pts": "meta_#Pts",
    })

    table = base.merge(meta, on="Region", how="inner")
    table["Delta vs VF-only"] = table["Soft ensemble MAE_mean_db"] - table["VF-only MAE_mean_db"]
    table["Delta vs VF-only display"] = table["Delta vs VF-only"].map(lambda x: f"{x:+.3f} dB")
    table["Improved"] = table["Delta vs VF-only"] < 0
    table["Samples"] = table["Soft ensemble Samples"]

    region_order = [
        "Overall",
        "Supero_Nasal",
        "Supero_Temporal",
        "Infero_Nasal",
        "Infero_Temporal",
        "Temporal",
        "Macular",
    ]
    table["Region"] = pd.Categorical(table["Region"], categories=region_order, ordered=True)
    table = table.sort_values("Region").reset_index(drop=True)
    table["Region"] = table["Region"].astype(str)

    display_table = table[[
        "Region",
        "VF-only MAE ± SD (dB)",
        "Soft ensemble MAE ± SD (dB)",
        "Delta vs VF-only display",
        "Improved",
        "Samples",
        "#Pts",
    ]].copy()
    return table, display_table


N_FINAL_REGION_SOFT_ENSEMBLE_SEEDS = 3
FINAL_REGION_SOFT_ENSEMBLE_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_FINAL_REGION_SOFT_ENSEMBLE_SEEDS).tolist()
)
print("Random final region-specific soft-ensemble seeds:", FINAL_REGION_SOFT_ENSEMBLE_SEEDS)

final_region_soft_ensemble_outputs = run_compact_transformer_policy_gate_soft_ensemble(
    seeds=FINAL_REGION_SOFT_ENSEMBLE_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
)

final_region_soft_ensemble_table, final_region_soft_ensemble_display = make_region_specific_soft_ensemble_table(
    final_region_soft_ensemble_outputs
)

print("Final PPT-style region-specific comparison:")
display(final_region_soft_ensemble_display)

print("Selected validation ensemble weights:")
display(final_region_soft_ensemble_outputs["ensemble_weights"])

print("Numeric table for export if needed:")
display(final_region_soft_ensemble_table[[
    "Region",
    "VF-only MAE_mean_db",
    "Soft ensemble MAE_mean_db",
    "Delta vs VF-only",
    "Delta vs VF-only display",
    "Improved",
    "Samples",
    "#Pts",
]])


## 33. Region-specific metadata selector

This experiment tests whether metadata should be used only for selected anatomical regions.

For each region and each seed, the selector uses validation MAE to choose one of:

1. VF-only compact Transformer
2. VF-only + soft ensemble with policy-gated IOP exposure
3. VF-only + soft ensemble with policy-gated IOP exposure + baseline spatial RNFL

The chosen strategy is then evaluated on the test set. This directly tests whether region-specific metadata usage is better than applying one global metadata strategy to every region.


In [ ]:
# Region-specific validation selector for metadata usage.
# This is the direct test of: should some regions ignore metadata?


def _select_region_ensemble_weight(pred_base_val, pred_meta_val, true_val, weight_grid=None):
    if weight_grid is None:
        weight_grid = np.linspace(0.0, 0.5, 11, dtype=np.float32)
    best = None
    rows = []
    for w in weight_grid:
        pred = (1.0 - float(w)) * pred_base_val + float(w) * pred_meta_val
        mae = _mae_abs(pred, true_val)
        row = {"w": float(w), "val_mae": mae}
        rows.append(row)
        if best is None or mae < best["val_mae"]:
            best = row
    return best, pd.DataFrame(rows)


def run_region_specific_metadata_selector(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=None,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"label": "VF-only", "kind": "vf_only"},
        {"label": "Soft IOP exposure", "kind": "policy_iop_exposure"},
        {"label": "Soft IOP exposure + spatial RNFL", "kind": "policy_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    selector_rows = []
    print("Region selector train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"REGION-SPECIFIC SELECTOR SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_base_test = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
        seed_selected_test = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
        seed_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            variant_pred = {}
            for variant in variants:
                exp_kind = variant["kind"]
                exp_label = variant["label"]

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"region_selector_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                val_pred_s, val_true_s = _predict_model(model, val_loader, device)
                val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)
                test_pred_s, test_true_s = _predict_model(model, test_loader, device)
                test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

                variant_pred[exp_kind] = {
                    "label": exp_label,
                    "val_pred": val_pred_abs,
                    "val_true": val_true_abs,
                    "test_pred": test_pred_abs,
                    "test_true": test_true_abs,
                }

                summary.update({
                    "variant": exp_label,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

            base = variant_pred["vf_only"]
            candidates = [{
                "selected_strategy": "VF-only",
                "selected_weight": 0.0,
                "val_mae": _mae_abs(base["val_pred"], base["val_true"]),
                "test_pred": base["test_pred"],
            }]

            for meta_kind, label in [
                ("policy_iop_exposure", "Soft IOP exposure"),
                ("policy_iop_rnfl", "Soft IOP exposure + spatial RNFL"),
            ]:
                meta = variant_pred[meta_kind]
                best, grid_df = _select_region_ensemble_weight(
                    base["val_pred"],
                    meta["val_pred"],
                    base["val_true"],
                    weight_grid=weight_grid,
                )
                w = float(best["w"])
                test_pred = (1.0 - w) * base["test_pred"] + w * meta["test_pred"]
                candidates.append({
                    "selected_strategy": label,
                    "selected_weight": w,
                    "val_mae": float(best["val_mae"]),
                    "test_pred": test_pred,
                })

            chosen = min(candidates, key=lambda x: x["val_mae"])
            seed_base_test[:, idx] = base["test_pred"]
            seed_selected_test[:, idx] = chosen["test_pred"]
            seed_true[:, idx] = base["test_true"]

            selector_rows.append({
                "seed": int(seed),
                "Region": region_name,
                "selected_strategy": chosen["selected_strategy"],
                "selected_weight": chosen["selected_weight"],
                "selected_val_mae": chosen["val_mae"],
                "vf_only_val_mae": candidates[0]["val_mae"],
            })

        base_result = _summarize_abs_prediction_matrix(seed_base_test, seed_true, cluster_mapping, "Region selector: VF-only")
        base_result["seed"] = int(seed)
        base_result["kind"] = "vf_only"
        raw_results.append(base_result)

        selected_result = _summarize_abs_prediction_matrix(seed_selected_test, seed_true, cluster_mapping, "Region selector: validation-selected metadata")
        selected_result["seed"] = int(seed)
        selected_result["kind"] = "region_selected"
        raw_results.append(selected_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    overall = summary_long[summary_long["Region"].eq("Overall")].copy()
    baseline_label = "Region selector: VF-only"
    baseline_mae = float(overall.loc[overall["Model"].eq(baseline_label), "MAE_mean_db"].iloc[0])
    overall["baseline_model"] = baseline_label
    overall["delta_vs_vf_only"] = overall["MAE_mean_db"] - baseline_mae
    overall["improved_vs_vf_only"] = overall["delta_vs_vf_only"] < 0
    overall["delta_vs_vf_only_display"] = overall["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    selector_df = pd.DataFrame(selector_rows)
    selection_summary = (
        selector_df
        .groupby(["Region", "selected_strategy"], as_index=False)
        .agg(count=("selected_strategy", "size"), mean_weight=("selected_weight", "mean"))
        .sort_values(["Region", "count"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "selector_df": selector_df,
        "selection_summary": selection_summary,
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


def make_region_selector_display(outputs):
    summary = outputs["summary_long"].copy()
    base_label = "Region selector: VF-only"
    selected_label = "Region selector: validation-selected metadata"
    base = summary[summary["Model"].eq(base_label)][[
        "Region", "MAE_mean_db", "MAE ± SD (dB)", "Samples", "#Pts"
    ]].rename(columns={
        "MAE_mean_db": "VF-only MAE_mean_db",
        "MAE ± SD (dB)": "VF-only MAE ± SD (dB)",
    })
    selected = summary[summary["Model"].eq(selected_label)][[
        "Region", "MAE_mean_db", "MAE ± SD (dB)", "Samples", "#Pts"
    ]].rename(columns={
        "MAE_mean_db": "Selected MAE_mean_db",
        "MAE ± SD (dB)": "Selected MAE ± SD (dB)",
        "Samples": "selected_samples",
        "#Pts": "selected_pts",
    })
    table = base.merge(selected, on="Region", how="inner")
    table["Delta vs VF-only"] = table["Selected MAE_mean_db"] - table["VF-only MAE_mean_db"]
    table["Delta vs VF-only display"] = table["Delta vs VF-only"].map(lambda x: f"{x:+.3f} dB")
    table["Improved"] = table["Delta vs VF-only"] < 0
    order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Infero_Nasal", "Infero_Temporal", "Temporal", "Macular"]
    table["Region"] = pd.Categorical(table["Region"], categories=order, ordered=True)
    table = table.sort_values("Region").reset_index(drop=True)
    table["Region"] = table["Region"].astype(str)
    display_table = table[[
        "Region",
        "VF-only MAE ± SD (dB)",
        "Selected MAE ± SD (dB)",
        "Delta vs VF-only display",
        "Improved",
        "Samples",
        "#Pts",
    ]]
    return table, display_table


In [ ]:
# Run the region-specific selector with 3 seeds.
N_REGION_SELECTOR_SEEDS = 3
REGION_SELECTOR_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_REGION_SELECTOR_SEEDS).tolist()
)
print("Random region-selector seeds:", REGION_SELECTOR_SEEDS)

region_selector_outputs = run_region_specific_metadata_selector(
    seeds=REGION_SELECTOR_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
)

region_selector_table, region_selector_display = make_region_selector_display(region_selector_outputs)

print("Region-specific validation selector result:")
display(region_selector_display)

print("Selection summary by region:")
display(region_selector_outputs["selection_summary"])

print("Raw selected strategy per seed and region:")
display(region_selector_outputs["selector_df"])


## 34. Recover/display region-selector results

Display-only helper for the region-specific selector.

Run this after cell 33 if the training finished but no tables were shown. It does not retrain models.


In [ ]:
# Display-only recovery cell for region selector outputs.
# This does not retrain anything.

if "region_selector_outputs" not in globals():
    print("region_selector_outputs does not exist in this kernel. Cell 33 likely did not finish assigning the output object.")
    print("If the previous cell is still running, wait. If it finished without assigning, rerun cell 33 or use a faster seed/epoch setting.")
else:
    print("region_selector_outputs found.")
    print("Available keys:", list(region_selector_outputs.keys()))

    try:
        region_selector_table, region_selector_display = make_region_selector_display(region_selector_outputs)
        print("Region-specific validation selector result:")
        display(region_selector_display)
    except Exception as e:
        print("Could not build region_selector_display:", repr(e))

    if "overall" in region_selector_outputs:
        print("Overall summary:")
        display(region_selector_outputs["overall"])

    if "selection_summary" in region_selector_outputs:
        print("Selection summary by region:")
        display(region_selector_outputs["selection_summary"])

    if "selector_df" in region_selector_outputs:
        print("Raw selected strategy per seed and region:")
        display(region_selector_outputs["selector_df"])

    if "summary_long" in region_selector_outputs:
        print("Summary long head:")
        display(region_selector_outputs["summary_long"].head(20))


## 35. Final soft-ensemble metadata ablation

Clean final ablation under the same compact Transformer soft-ensemble framework.

This compares:

1. VF-only
2. IOP only
3. RNFL only
4. IOP + RNFL

The metadata-aware predictions are softly ensembled with the VF-only prediction using a validation-selected weight. Results are reported Overall and by anatomical region.



In [ ]:
# Final clean soft-ensemble ablation: VF-only vs IOP-only vs RNFL-only vs IOP+RNFL.
# This cell intentionally reuses the same compact Transformer + policy gate + validation-selected ensemble setup.


def run_compact_transformer_soft_ensemble_final_ablation(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=None,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Final ablation: VF-only", "kind": "vf_only"},
        {"Model": "Final ablation: policy IOP exposure", "kind": "policy_iop_exposure"},
        {"Model": "Final ablation: policy spatial RNFL", "kind": "policy_rnfl"},
        {"Model": "Final ablation: policy IOP exposure + spatial RNFL", "kind": "policy_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    ensemble_rows = []
    print("Final ablation train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"FINAL SOFT-ENSEMBLE ABLATION SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_val_pred = {v["kind"]: np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_test_pred = {v["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_val_true = np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32)
        seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            for variant in variants:
                exp_name = variant["Model"]
                exp_kind = variant["kind"]

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif exp_kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"final_ablation_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                val_pred_s, val_true_s = _predict_model(model, val_loader, device)
                val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)
                test_pred_s, test_true_s = _predict_model(model, test_loader, device)
                test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

                seed_val_pred[exp_kind][:, idx] = val_pred_abs
                seed_test_pred[exp_kind][:, idx] = test_pred_abs
                seed_val_true[:, idx] = val_true_abs
                seed_test_true[:, idx] = test_true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        for variant in variants:
            exp_name = variant["Model"]
            exp_kind = variant["kind"]
            seed_result = _summarize_abs_prediction_matrix(seed_test_pred[exp_kind], seed_test_true, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

        for meta_kind, meta_label in [
            ("policy_iop_exposure", "Final ablation: soft ensemble + IOP only"),
            ("policy_rnfl", "Final ablation: soft ensemble + RNFL only"),
            ("policy_iop_rnfl", "Final ablation: soft ensemble + IOP + RNFL"),
        ]:
            best, grid_df = _select_ensemble_weight(
                seed_val_pred["vf_only"],
                seed_val_pred[meta_kind],
                seed_val_true,
                weight_grid=weight_grid,
            )
            w = float(best["w"])
            ens_test_pred = (1.0 - w) * seed_test_pred["vf_only"] + w * seed_test_pred[meta_kind]
            seed_result = _summarize_abs_prediction_matrix(ens_test_pred, seed_test_true, cluster_mapping, meta_label)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = f"ensemble_{meta_kind}"
            raw_results.append(seed_result)
            ensemble_rows.append({
                "seed": int(seed),
                "meta_kind": meta_kind,
                "selected_w": w,
                "best_val_mae": float(best["val_mae"]),
            })

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    baseline_label = "Final ablation: VF-only"

    def _add_delta(df):
        df = df.copy()
        baseline_by_region = df[df["Model"].eq(baseline_label)][["Region", "MAE_mean_db"]].rename(
            columns={"MAE_mean_db": "baseline_mae_db"}
        )
        out = df.merge(baseline_by_region, on="Region", how="left")
        out["delta_vs_vf_only"] = out["MAE_mean_db"] - out["baseline_mae_db"]
        out["improved_vs_vf_only"] = out["delta_vs_vf_only"] < 0
        out["delta_vs_vf_only_display"] = out["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
        return out

    summary_with_delta = _add_delta(summary_long)
    overall = summary_with_delta[summary_with_delta["Region"].eq("Overall")].copy()
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "summary_with_delta": summary_with_delta,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "ensemble_weights": pd.DataFrame(ensemble_rows),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


def make_final_metadata_ablation_region_table(outputs):
    df = outputs["summary_with_delta"].copy()
    keep_models = [
        "Final ablation: VF-only",
        "Final ablation: soft ensemble + IOP only",
        "Final ablation: soft ensemble + RNFL only",
        "Final ablation: soft ensemble + IOP + RNFL",
    ]
    region_order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Infero_Nasal", "Infero_Temporal", "Temporal", "Macular"]
    model_order = {m: i for i, m in enumerate(keep_models)}
    out = df[df["Model"].isin(keep_models)].copy()
    out["Region"] = pd.Categorical(out["Region"], categories=region_order, ordered=True)
    out["model_order"] = out["Model"].map(model_order)
    out = out.sort_values(["Region", "model_order"]).reset_index(drop=True)
    out["Region"] = out["Region"].astype(str)
    return out[[
        "Region",
        "Model",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]]


N_FINAL_METADATA_ABLATION_SEEDS = 3
FINAL_METADATA_ABLATION_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_FINAL_METADATA_ABLATION_SEEDS).tolist()
)
print("Random final metadata ablation seeds:", FINAL_METADATA_ABLATION_SEEDS)

final_metadata_ablation_outputs = run_compact_transformer_soft_ensemble_final_ablation(
    seeds=FINAL_METADATA_ABLATION_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
)

print("Final metadata ablation - Overall:")
display(final_metadata_ablation_outputs["overall"][[
    "Model",
    "Region",
    "MAE_mean_db",
    "MAE_seed_std_db",
    "MAE_sample_std_db",
    "MAE ± SD (dB)",
    "delta_vs_vf_only_display",
    "improved_vs_vf_only",
    "Samples",
    "#Pts",
]])

print("Final metadata ablation - Overall + region details:")
final_metadata_ablation_region_table = make_final_metadata_ablation_region_table(final_metadata_ablation_outputs)
display(final_metadata_ablation_region_table)

print("Validation-selected soft-ensemble weights:")
display(final_metadata_ablation_outputs["ensemble_weights"])



## 36. Multi-branch soft ensemble for IOP/RNFL complementarity

This experiment tests whether IOP and RNFL can complement each other if we do not force them into one metadata branch.

Instead of selecting one metadata model, it trains the same four branches:

1. VF-only
2. IOP-only
3. RNFL-only
4. IOP + RNFL

Then validation selects non-negative ensemble weights across all branches, with total metadata weight capped to avoid overfitting.



In [ ]:
# Multi-branch soft ensemble: jointly weight VF-only, IOP-only, RNFL-only, and IOP+RNFL branches.
# This tests whether IOP and RNFL can complement each other without being forced into one metadata branch.


def _select_multibranch_ensemble_weights(seed_val_pred, true_val, weight_grid=None, max_total_meta_weight=0.5):
    if weight_grid is None:
        weight_grid = np.linspace(0.0, 0.5, 11, dtype=np.float32)
    meta_kinds = ["policy_iop_exposure", "policy_rnfl", "policy_iop_rnfl"]
    best = None
    rows = []
    for w_iop in weight_grid:
        for w_rnfl in weight_grid:
            for w_both in weight_grid:
                total_meta = float(w_iop + w_rnfl + w_both)
                if total_meta > max_total_meta_weight + 1e-8:
                    continue
                w_base = 1.0 - total_meta
                pred = (
                    w_base * seed_val_pred["vf_only"]
                    + float(w_iop) * seed_val_pred["policy_iop_exposure"]
                    + float(w_rnfl) * seed_val_pred["policy_rnfl"]
                    + float(w_both) * seed_val_pred["policy_iop_rnfl"]
                )
                mae = _mae_abs(pred, true_val)
                row = {
                    "w_vf_only": float(w_base),
                    "w_iop": float(w_iop),
                    "w_rnfl": float(w_rnfl),
                    "w_iop_rnfl": float(w_both),
                    "total_meta_weight": total_meta,
                    "val_mae": mae,
                }
                rows.append(row)
                if best is None or mae < best["val_mae"]:
                    best = row
    return best, pd.DataFrame(rows)


def run_compact_transformer_multibranch_soft_ensemble(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    single_weight_grid=None,
    multibranch_weight_grid=None,
    max_total_meta_weight=0.5,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Multibranch: VF-only", "kind": "vf_only"},
        {"Model": "Multibranch: policy IOP exposure", "kind": "policy_iop_exposure"},
        {"Model": "Multibranch: policy spatial RNFL", "kind": "policy_rnfl"},
        {"Model": "Multibranch: policy IOP exposure + spatial RNFL", "kind": "policy_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    single_weight_rows = []
    multibranch_weight_rows = []
    print("Multibranch soft ensemble train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"MULTIBRANCH SOFT ENSEMBLE SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_val_pred = {v["kind"]: np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_test_pred = {v["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_val_true = np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32)
        seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            for variant in variants:
                exp_name = variant["Model"]
                exp_kind = variant["kind"]

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif exp_kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"multibranch_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                val_pred_s, val_true_s = _predict_model(model, val_loader, device)
                val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)
                test_pred_s, test_true_s = _predict_model(model, test_loader, device)
                test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

                seed_val_pred[exp_kind][:, idx] = val_pred_abs
                seed_test_pred[exp_kind][:, idx] = test_pred_abs
                seed_val_true[:, idx] = val_true_abs
                seed_test_true[:, idx] = test_true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        for variant in variants:
            exp_name = variant["Model"]
            exp_kind = variant["kind"]
            seed_result = _summarize_abs_prediction_matrix(seed_test_pred[exp_kind], seed_test_true, cluster_mapping, exp_name)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

        for meta_kind, meta_label in [
            ("policy_iop_exposure", "Multibranch: soft ensemble + IOP only"),
            ("policy_rnfl", "Multibranch: soft ensemble + RNFL only"),
            ("policy_iop_rnfl", "Multibranch: soft ensemble + IOP + RNFL"),
        ]:
            best, grid_df = _select_ensemble_weight(
                seed_val_pred["vf_only"],
                seed_val_pred[meta_kind],
                seed_val_true,
                weight_grid=single_weight_grid,
            )
            w = float(best["w"])
            ens_test_pred = (1.0 - w) * seed_test_pred["vf_only"] + w * seed_test_pred[meta_kind]
            seed_result = _summarize_abs_prediction_matrix(ens_test_pred, seed_test_true, cluster_mapping, meta_label)
            seed_result["seed"] = int(seed)
            seed_result["kind"] = f"single_ensemble_{meta_kind}"
            raw_results.append(seed_result)
            single_weight_rows.append({
                "seed": int(seed),
                "meta_kind": meta_kind,
                "selected_w": w,
                "best_val_mae": float(best["val_mae"]),
            })

        best_multi, multi_grid_df = _select_multibranch_ensemble_weights(
            seed_val_pred,
            seed_val_true,
            weight_grid=multibranch_weight_grid,
            max_total_meta_weight=max_total_meta_weight,
        )
        multi_test_pred = (
            best_multi["w_vf_only"] * seed_test_pred["vf_only"]
            + best_multi["w_iop"] * seed_test_pred["policy_iop_exposure"]
            + best_multi["w_rnfl"] * seed_test_pred["policy_rnfl"]
            + best_multi["w_iop_rnfl"] * seed_test_pred["policy_iop_rnfl"]
        )
        seed_result = _summarize_abs_prediction_matrix(
            multi_test_pred,
            seed_test_true,
            cluster_mapping,
            "Multibranch: validation-weighted IOP/RNFL ensemble",
        )
        seed_result["seed"] = int(seed)
        seed_result["kind"] = "multibranch_ensemble"
        raw_results.append(seed_result)
        multibranch_weight_rows.append({"seed": int(seed), **best_multi})

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    baseline_label = "Multibranch: VF-only"

    baseline_by_region = summary_long[summary_long["Model"].eq(baseline_label)][["Region", "MAE_mean_db"]].rename(
        columns={"MAE_mean_db": "baseline_mae_db"}
    )
    summary_with_delta = summary_long.merge(baseline_by_region, on="Region", how="left")
    summary_with_delta["delta_vs_vf_only"] = summary_with_delta["MAE_mean_db"] - summary_with_delta["baseline_mae_db"]
    summary_with_delta["improved_vs_vf_only"] = summary_with_delta["delta_vs_vf_only"] < 0
    summary_with_delta["delta_vs_vf_only_display"] = summary_with_delta["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = summary_with_delta[summary_with_delta["Region"].eq("Overall")].copy()
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "summary_with_delta": summary_with_delta,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "single_ensemble_weights": pd.DataFrame(single_weight_rows),
        "multibranch_weights": pd.DataFrame(multibranch_weight_rows),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


def make_multibranch_region_table(outputs):
    df = outputs["summary_with_delta"].copy()
    keep_models = [
        "Multibranch: VF-only",
        "Multibranch: soft ensemble + IOP only",
        "Multibranch: soft ensemble + RNFL only",
        "Multibranch: soft ensemble + IOP + RNFL",
        "Multibranch: validation-weighted IOP/RNFL ensemble",
    ]
    region_order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Infero_Nasal", "Infero_Temporal", "Temporal", "Macular"]
    model_order = {m: i for i, m in enumerate(keep_models)}
    out = df[df["Model"].isin(keep_models)].copy()
    out["Region"] = pd.Categorical(out["Region"], categories=region_order, ordered=True)
    out["model_order"] = out["Model"].map(model_order)
    out = out.sort_values(["Region", "model_order"]).reset_index(drop=True)
    out["Region"] = out["Region"].astype(str)
    return out[[
        "Region",
        "Model",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]]


N_MULTIBRANCH_SOFT_ENSEMBLE_SEEDS = 5
MULTIBRANCH_SOFT_ENSEMBLE_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_MULTIBRANCH_SOFT_ENSEMBLE_SEEDS).tolist()
)
print("Random multibranch soft ensemble seeds:", MULTIBRANCH_SOFT_ENSEMBLE_SEEDS)

multibranch_soft_ensemble_outputs = run_compact_transformer_multibranch_soft_ensemble(
    seeds=MULTIBRANCH_SOFT_ENSEMBLE_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    single_weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
    multibranch_weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
    max_total_meta_weight=0.5,
)

print("Multibranch soft ensemble - Overall:")
display(multibranch_soft_ensemble_outputs["overall"][[
    "Model",
    "Region",
    "MAE_mean_db",
    "MAE_seed_std_db",
    "MAE_sample_std_db",
    "MAE ± SD (dB)",
    "delta_vs_vf_only_display",
    "improved_vs_vf_only",
    "Samples",
    "#Pts",
]])

print("Multibranch soft ensemble - Overall + region details:")
multibranch_region_table = make_multibranch_region_table(multibranch_soft_ensemble_outputs)
display(multibranch_region_table)

print("Single-branch validation weights:")
display(multibranch_soft_ensemble_outputs["single_ensemble_weights"])

print("Multi-branch validation weights:")
display(multibranch_soft_ensemble_outputs["multibranch_weights"])



## 37. Region-specific multi-branch soft ensemble

This experiment extends the multi-branch ensemble by selecting separate validation weights for each anatomical region.

The same four branches are trained:

1. VF-only
2. IOP-only
3. RNFL-only
4. IOP + RNFL

Then we compare:

- Global multi-branch weights selected on all 59 points
- Region-specific multi-branch weights selected separately for each anatomical region

This tests whether metadata should contribute differently across regions.



In [ ]:
# Region-specific multi-branch soft ensemble.
# Same branches as cell 36, but validation weights are selected separately per anatomical region.


def run_compact_transformer_region_multibranch_soft_ensemble(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    single_weight_grid=None,
    multibranch_weight_grid=None,
    max_total_meta_weight=0.5,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Multibranch: VF-only", "kind": "vf_only"},
        {"Model": "Policy branch: IOP exposure", "kind": "policy_iop_exposure"},
        {"Model": "Policy branch: spatial RNFL", "kind": "policy_rnfl"},
        {"Model": "Policy branch: IOP exposure + spatial RNFL", "kind": "policy_iop_rnfl"},
    ]

    raw_results = []
    run_summaries = []
    global_weight_rows = []
    region_weight_rows = []
    print("Region-specific multibranch train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"REGION MULTIBRANCH SOFT ENSEMBLE SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_val_pred = {v["kind"]: np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_test_pred = {v["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32) for v in variants}
        seed_val_true = np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32)
        seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            for variant in variants:
                exp_name = variant["Model"]
                exp_kind = variant["kind"]

                if exp_kind == "vf_only":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif exp_kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif exp_kind == "policy_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif exp_kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(exp_kind)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"region_multibranch_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                val_pred_s, val_true_s = _predict_model(model, val_loader, device)
                val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)
                test_pred_s, test_true_s = _predict_model(model, test_loader, device)
                test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

                seed_val_pred[exp_kind][:, idx] = val_pred_abs
                seed_test_pred[exp_kind][:, idx] = test_pred_abs
                seed_val_true[:, idx] = val_true_abs
                seed_test_true[:, idx] = test_true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        # Store only the clean VF-only baseline, not the raw policy branches.
        seed_result = _summarize_abs_prediction_matrix(
            seed_test_pred["vf_only"],
            seed_test_true,
            cluster_mapping,
            "Multibranch: VF-only",
        )
        seed_result["seed"] = int(seed)
        seed_result["kind"] = "vf_only"
        raw_results.append(seed_result)

        # Add validation-selected two-branch soft ensembles for each metadata source.
        for meta_kind, meta_label in [
            ("policy_iop_exposure", "Multibranch: soft ensemble + IOP only"),
            ("policy_rnfl", "Multibranch: soft ensemble + RNFL only"),
            ("policy_iop_rnfl", "Multibranch: soft ensemble + IOP + RNFL"),
        ]:
            best_single, _ = _select_ensemble_weight(
                seed_val_pred["vf_only"],
                seed_val_pred[meta_kind],
                seed_val_true,
                weight_grid=single_weight_grid,
            )
            single_test_pred = (
                (1.0 - best_single["w"]) * seed_test_pred["vf_only"]
                + best_single["w"] * seed_test_pred[meta_kind]
            )
            seed_result = _summarize_abs_prediction_matrix(
                single_test_pred,
                seed_test_true,
                cluster_mapping,
                meta_label,
            )
            seed_result["seed"] = int(seed)
            seed_result["kind"] = f"soft_{meta_kind}"
            raw_results.append(seed_result)

        best_global, _ = _select_multibranch_ensemble_weights(
            seed_val_pred,
            seed_val_true,
            weight_grid=multibranch_weight_grid,
            max_total_meta_weight=max_total_meta_weight,
        )
        global_test_pred = (
            best_global["w_vf_only"] * seed_test_pred["vf_only"]
            + best_global["w_iop"] * seed_test_pred["policy_iop_exposure"]
            + best_global["w_rnfl"] * seed_test_pred["policy_rnfl"]
            + best_global["w_iop_rnfl"] * seed_test_pred["policy_iop_rnfl"]
        )
        seed_result = _summarize_abs_prediction_matrix(
            global_test_pred,
            seed_test_true,
            cluster_mapping,
            "Multibranch: validation-weighted IOP/RNFL ensemble",
        )
        seed_result["seed"] = int(seed)
        seed_result["kind"] = "global_multibranch_ensemble"
        raw_results.append(seed_result)
        global_weight_rows.append({"seed": int(seed), **best_global})

        region_test_pred = np.zeros_like(seed_test_true, dtype=np.float32)
        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            val_pred_region = {k: v[:, idx] for k, v in seed_val_pred.items()}
            test_pred_region = {k: v[:, idx] for k, v in seed_test_pred.items()}
            best_region, _ = _select_multibranch_ensemble_weights(
                val_pred_region,
                seed_val_true[:, idx],
                weight_grid=multibranch_weight_grid,
                max_total_meta_weight=max_total_meta_weight,
            )
            region_test_pred[:, idx] = (
                best_region["w_vf_only"] * test_pred_region["vf_only"]
                + best_region["w_iop"] * test_pred_region["policy_iop_exposure"]
                + best_region["w_rnfl"] * test_pred_region["policy_rnfl"]
                + best_region["w_iop_rnfl"] * test_pred_region["policy_iop_rnfl"]
            )
            region_weight_rows.append({"seed": int(seed), "Region": region_name, **best_region})

        seed_result = _summarize_abs_prediction_matrix(
            region_test_pred,
            seed_test_true,
            cluster_mapping,
            "Region multibranch: region-specific validation-weighted ensemble",
        )
        seed_result["seed"] = int(seed)
        seed_result["kind"] = "region_specific_multibranch_ensemble"
        raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    baseline_label = "Multibranch: VF-only"

    baseline_by_region = summary_long[summary_long["Model"].eq(baseline_label)][["Region", "MAE_mean_db"]].rename(
        columns={"MAE_mean_db": "baseline_mae_db"}
    )
    summary_with_delta = summary_long.merge(baseline_by_region, on="Region", how="left")
    summary_with_delta["delta_vs_vf_only"] = summary_with_delta["MAE_mean_db"] - summary_with_delta["baseline_mae_db"]
    summary_with_delta["improved_vs_vf_only"] = summary_with_delta["delta_vs_vf_only"] < 0
    summary_with_delta["delta_vs_vf_only_display"] = summary_with_delta["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = summary_with_delta[summary_with_delta["Region"].eq("Overall")].copy()
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "summary_with_delta": summary_with_delta,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "global_weights": pd.DataFrame(global_weight_rows),
        "region_weights": pd.DataFrame(region_weight_rows),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


def make_region_multibranch_table(outputs):
    df = outputs["summary_with_delta"].copy()
    keep_models = [
        "Region multibranch: VF-only",
        "Multibranch: validation-weighted IOP/RNFL ensemble",
        "Region multibranch: region-specific validation-weighted ensemble",
    ]
    region_order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Infero_Nasal", "Infero_Temporal", "Temporal", "Macular"]
    model_order = {m: i for i, m in enumerate(keep_models)}
    out = df[df["Model"].isin(keep_models)].copy()
    out["Region"] = pd.Categorical(out["Region"], categories=region_order, ordered=True)
    out["model_order"] = out["Model"].map(model_order)
    out = out.sort_values(["Region", "model_order"]).reset_index(drop=True)
    out["Region"] = out["Region"].astype(str)
    return out[[
        "Region",
        "Model",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]]


N_REGION_MULTIBRANCH_SEEDS = 10
REGION_MULTIBRANCH_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_REGION_MULTIBRANCH_SEEDS).tolist()
)
print("Random region multibranch seeds:", REGION_MULTIBRANCH_SEEDS)

region_multibranch_outputs = run_compact_transformer_region_multibranch_soft_ensemble(
    seeds=REGION_MULTIBRANCH_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    single_weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
    multibranch_weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
    max_total_meta_weight=0.5,
)

print("Region-specific multibranch ensemble - Overall:")
display(region_multibranch_outputs["overall"][[
    "Model",
    "Region",
    "MAE_mean_db",
    "MAE_seed_std_db",
    "MAE_sample_std_db",
    "MAE ± SD (dB)",
    "delta_vs_vf_only_display",
    "improved_vs_vf_only",
    "Samples",
    "#Pts",
]])

print("Region-specific multibranch ensemble - comparison table:")
region_multibranch_table = make_region_multibranch_table(region_multibranch_outputs)
display(region_multibranch_table)

print("Global multi-branch validation weights:")
display(region_multibranch_outputs["global_weights"])

print("Region-specific multi-branch validation weights:")
display(region_multibranch_outputs["region_weights"])



## 38. Separate IOP/RNFL gates inside the compact Transformer

This experiment tests whether IOP and RNFL should be modeled as two separate internal correction sources instead of being concatenated into one metadata branch.

Model idea:

```text
base_pred = VF-only Transformer prediction
final = base_pred + gate_iop * delta_iop + gate_rnfl * delta_rnfl
```

This is more expressive than a single metadata gate, but less conservative than the validation-weighted multi-branch ensemble.



In [ ]:
# Separate internal gates for IOP and RNFL corrections.
# This tests model-internal complementarity instead of prediction-level ensembling.


class CompactPointTransformerSeparateIOPRNFLGateModel(PointLevelSpatioTemporalTransformerModel):
    """Compact Transformer with separate IOP and RNFL metadata gates.

    final = base_pred + gate_iop * (iop_pred - base_pred) + gate_rnfl * (rnfl_pred - base_pred)

    Each metadata source has its own FiLM adapter and reliability gate. The adapters
    are zero-initialized and gates start near zero, so the initial model behaves like
    the VF-only compact Transformer.
    """

    def __init__(
        self,
        *args,
        iop_meta_dim=0,
        rnfl_meta_dim=0,
        policy_hidden_dim=None,
        gate_max=0.25,
        film_scale=0.05,
        gate_init_bias=-6.0,
        meta_hidden_dim=16,
        **kwargs,
    ):
        super().__init__(*args, meta_hidden_dim=meta_hidden_dim, **kwargs)
        self.iop_meta_dim = int(iop_meta_dim)
        self.rnfl_meta_dim = int(rnfl_meta_dim)
        self.gate_max = float(gate_max)
        self.film_scale = float(film_scale)
        model_dim = self.value_encoder[0].out_features
        if policy_hidden_dim is None:
            policy_hidden_dim = max(16, meta_hidden_dim)

        def _make_branch(meta_dim):
            if meta_dim <= 0:
                return None, None, None
            norm = nn.LayerNorm(meta_dim)
            film = nn.Sequential(
                nn.Linear(meta_dim, policy_hidden_dim),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(policy_hidden_dim, 2 * model_dim),
            )
            gate = nn.Sequential(
                nn.Linear(meta_dim, policy_hidden_dim),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(policy_hidden_dim, 1),
            )
            nn.init.zeros_(film[-1].weight)
            nn.init.zeros_(film[-1].bias)
            nn.init.zeros_(gate[-1].weight)
            nn.init.constant_(gate[-1].bias, gate_init_bias)
            return norm, film, gate

        self.iop_norm, self.iop_film_net, self.iop_gate_net = _make_branch(self.iop_meta_dim)
        self.rnfl_norm, self.rnfl_film_net, self.rnfl_gate_net = _make_branch(self.rnfl_meta_dim)

    def _encode_vf_only_tokens(self, padded_seqs, padded_times, lengths):
        batch_size, seq_len, n_points = padded_seqs.shape
        values = padded_seqs.unsqueeze(-1)
        value_tokens = self.value_encoder(values)
        point_ids = torch.arange(n_points, device=padded_seqs.device)
        point_tokens = self.point_embed(point_ids).view(1, 1, n_points, -1)
        time_tokens = self.time_pe(padded_times).unsqueeze(2)
        zero_iop = torch.zeros(batch_size, seq_len, 1, device=padded_seqs.device, dtype=padded_seqs.dtype)
        zero_iop_tokens = self.iop_encoder(zero_iop).unsqueeze(2)

        tokens = value_tokens + point_tokens + time_tokens + zero_iop_tokens
        tokens = tokens.reshape(batch_size, seq_len * n_points, -1)

        step_ids = torch.arange(seq_len, device=lengths.device).unsqueeze(0).expand(batch_size, -1)
        visit_mask = step_ids >= lengths.unsqueeze(1)
        token_mask = visit_mask.unsqueeze(-1).expand(batch_size, seq_len, n_points).reshape(batch_size, seq_len * n_points)
        encoded = self.transformer(tokens, src_key_padding_mask=token_mask)
        return encoded, token_mask

    def _decode_encoded_tokens(self, encoded, token_mask, device):
        batch_size = encoded.shape[0]
        query_points = self.point_indices.to(device)
        queries = self.output_point_embed(query_points).unsqueeze(0).expand(batch_size, -1, -1)
        attn_out, _ = self.output_attn(
            query=queries,
            key=encoded,
            value=encoded,
            key_padding_mask=token_mask,
            need_weights=False,
        )
        x = self.norm1(queries + attn_out)
        x = x + self.ffn(self.norm2(x))
        return self.out(x).squeeze(-1)

    def _branch_delta(self, encoded, token_mask, meta, norm, film_net, gate_net, device):
        if meta is None or norm is None or film_net is None or gate_net is None:
            return None, None
        meta = norm(meta)
        film = film_net(meta)
        gamma, beta = torch.chunk(film, 2, dim=-1)
        gamma = self.film_scale * torch.tanh(gamma).unsqueeze(1)
        beta = self.film_scale * torch.tanh(beta).unsqueeze(1)
        meta_encoded = encoded * (1.0 + gamma) + beta
        meta_pred = self._decode_encoded_tokens(meta_encoded, token_mask, device)
        gate = self.gate_max * torch.sigmoid(gate_net(meta))
        return meta_pred, gate

    def forward(self, padded_seqs, padded_times, lengths, meta_input):
        encoded, token_mask = self._encode_vf_only_tokens(padded_seqs, padded_times, lengths)
        base_pred = self._decode_encoded_tokens(encoded, token_mask, padded_seqs.device)

        if self.iop_meta_dim + self.rnfl_meta_dim == 0:
            return base_pred

        cursor = 0
        iop_meta = None
        rnfl_meta = None
        if self.iop_meta_dim > 0:
            iop_meta = meta_input[:, cursor:cursor + self.iop_meta_dim]
            cursor += self.iop_meta_dim
        if self.rnfl_meta_dim > 0:
            rnfl_meta = meta_input[:, cursor:cursor + self.rnfl_meta_dim]

        out = base_pred
        iop_pred, iop_gate = self._branch_delta(
            encoded, token_mask, iop_meta, self.iop_norm, self.iop_film_net, self.iop_gate_net, padded_seqs.device
        )
        if iop_pred is not None:
            out = out + iop_gate * (iop_pred - base_pred)

        rnfl_pred, rnfl_gate = self._branch_delta(
            encoded, token_mask, rnfl_meta, self.rnfl_norm, self.rnfl_film_net, self.rnfl_gate_net, padded_seqs.device
        )
        if rnfl_pred is not None:
            out = out + rnfl_gate * (rnfl_pred - base_pred)

        return out


def run_compact_transformer_separate_iop_rnfl_gates(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"Model": "Separate gates: VF-only", "kind": "vf_only", "use_iop": False, "use_rnfl": False},
        {"Model": "Separate gates: IOP gate only", "kind": "iop_gate", "use_iop": True, "use_rnfl": False},
        {"Model": "Separate gates: RNFL gate only", "kind": "rnfl_gate", "use_iop": False, "use_rnfl": True},
        {"Model": "Separate gates: IOP gate + RNFL gate", "kind": "separate_iop_rnfl_gates", "use_iop": True, "use_rnfl": True},
    ]

    raw_results = []
    run_summaries = []
    print("Separate gates train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"SEPARATE IOP/RNFL GATES SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_test_pred = {
            v["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            for v in variants
        }
        seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            for variant in variants:
                exp_name = variant["Model"]
                exp_kind = variant["kind"]
                use_iop = bool(variant["use_iop"])
                use_rnfl = bool(variant["use_rnfl"])

                meta_parts_train = []
                meta_parts_val = []
                meta_parts_test = []
                iop_dim = 0
                rnfl_dim = 0
                if use_iop:
                    meta_parts_train.append(iop_train_s)
                    meta_parts_val.append(iop_val_s)
                    meta_parts_test.append(iop_test_s)
                    iop_dim = iop_train_s.shape[1]
                if use_rnfl:
                    meta_parts_train.append(rnfl_train_s)
                    meta_parts_val.append(rnfl_val_s)
                    meta_parts_test.append(rnfl_test_s)
                    rnfl_dim = rnfl_train_s.shape[1]

                if meta_parts_train:
                    meta_train_s = np.concatenate(meta_parts_train, axis=1).astype(np.float32)
                    meta_val_s = np.concatenate(meta_parts_val, axis=1).astype(np.float32)
                    meta_test_s = np.concatenate(meta_parts_test, axis=1).astype(np.float32)
                else:
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerSeparateIOPRNFLGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    iop_meta_dim=iop_dim,
                    rnfl_meta_dim=rnfl_dim,
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"separate_gates_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                seed_test_pred[exp_kind][:, idx] = pred_abs
                seed_test_true[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "iop_dim": int(iop_dim),
                    "rnfl_dim": int(rnfl_dim),
                    "meta_dim": int(meta_train_s.shape[1]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        for variant in variants:
            exp_name = variant["Model"]
            exp_kind = variant["kind"]
            seed_result = _summarize_abs_prediction_matrix(
                seed_test_pred[exp_kind],
                seed_test_true,
                cluster_mapping,
                exp_name,
            )
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    baseline_label = "Separate gates: VF-only"
    baseline_by_region = summary_long[summary_long["Model"].eq(baseline_label)][["Region", "MAE_mean_db"]].rename(
        columns={"MAE_mean_db": "baseline_mae_db"}
    )
    summary_with_delta = summary_long.merge(baseline_by_region, on="Region", how="left")
    summary_with_delta["delta_vs_vf_only"] = summary_with_delta["MAE_mean_db"] - summary_with_delta["baseline_mae_db"]
    summary_with_delta["improved_vs_vf_only"] = summary_with_delta["delta_vs_vf_only"] < 0
    summary_with_delta["delta_vs_vf_only_display"] = summary_with_delta["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = summary_with_delta[summary_with_delta["Region"].eq("Overall")].copy()
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "summary_with_delta": summary_with_delta,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


def make_separate_gates_region_table(outputs):
    df = outputs["summary_with_delta"].copy()
    keep_models = [
        "Separate gates: VF-only",
        "Separate gates: IOP gate only",
        "Separate gates: RNFL gate only",
        "Separate gates: IOP gate + RNFL gate",
    ]
    region_order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Infero_Nasal", "Infero_Temporal", "Temporal", "Macular"]
    model_order = {m: i for i, m in enumerate(keep_models)}
    out = df[df["Model"].isin(keep_models)].copy()
    out["Region"] = pd.Categorical(out["Region"], categories=region_order, ordered=True)
    out["model_order"] = out["Model"].map(model_order)
    out = out.sort_values(["Region", "model_order"]).reset_index(drop=True)
    out["Region"] = out["Region"].astype(str)
    return out[[
        "Region",
        "Model",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]]


N_SEPARATE_GATES_SEEDS = 5
SEPARATE_GATES_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_SEPARATE_GATES_SEEDS).tolist()
)
print("Random separate-gates seeds:", SEPARATE_GATES_SEEDS)

separate_gates_outputs = run_compact_transformer_separate_iop_rnfl_gates(
    seeds=SEPARATE_GATES_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
)

print("Separate IOP/RNFL gates - Overall:")
display(separate_gates_outputs["overall"][[
    "Model",
    "Region",
    "MAE_mean_db",
    "MAE_seed_std_db",
    "MAE_sample_std_db",
    "MAE ± SD (dB)",
    "delta_vs_vf_only_display",
    "improved_vs_vf_only",
    "Samples",
    "#Pts",
]])

print("Separate IOP/RNFL gates - Overall + region details:")
separate_gates_region_table = make_separate_gates_region_table(separate_gates_outputs)
display(separate_gates_region_table)




## 39. Focused policy-gate optimization

This block focuses on the best-looking raw policy-gate direction.

It tests whether the strong `Policy IOP exposure` result can be reproduced or improved by tuning the gate behavior:

1. Current IOP policy gate
2. Stronger IOP gate
3. Easier-to-open IOP gate
4. Current IOP+RNFL policy gate
5. Stronger IOP+RNFL policy gate

This is intentionally separate from the soft-ensemble experiments. The goal is to see whether raw policy gate can beat the multi-branch ensemble consistently.



In [ ]:
# Focused policy-gate optimization.
# Goal: verify whether the strong raw Policy IOP exposure result is stable and tunable.


def run_policy_gate_optimization_screen(
    *,
    seeds,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    configs=None,
):
    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, iop_scaler = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    if configs is None:
        configs = [
            {
                "Model": "Policy opt: VF-only",
                "kind": "vf_only",
                "meta_kind": "none",
                "gate_max": 0.25,
                "film_scale": 0.05,
                "gate_init_bias": -6.0,
            },
            {
                "Model": "Policy opt: IOP current gate",
                "kind": "iop_current",
                "meta_kind": "iop",
                "gate_max": 0.25,
                "film_scale": 0.05,
                "gate_init_bias": -6.0,
            },
            {
                "Model": "Policy opt: IOP stronger gate",
                "kind": "iop_gate_max_050",
                "meta_kind": "iop",
                "gate_max": 0.50,
                "film_scale": 0.05,
                "gate_init_bias": -6.0,
            },
            {
                "Model": "Policy opt: IOP easier gate init",
                "kind": "iop_gate_bias_m4",
                "meta_kind": "iop",
                "gate_max": 0.25,
                "film_scale": 0.05,
                "gate_init_bias": -4.0,
            },
            {
                "Model": "Policy opt: IOP+RNFL current gate",
                "kind": "iop_rnfl_current",
                "meta_kind": "iop_rnfl",
                "gate_max": 0.25,
                "film_scale": 0.05,
                "gate_init_bias": -6.0,
            },
            {
                "Model": "Policy opt: IOP+RNFL stronger gate",
                "kind": "iop_rnfl_gate_max_050",
                "meta_kind": "iop_rnfl",
                "gate_max": 0.50,
                "film_scale": 0.05,
                "gate_init_bias": -6.0,
            },
        ]

    raw_results = []
    run_summaries = []
    print("Policy gate optimization train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds)

    for seed in seeds:
        print("\n" + "#" * 100)
        print(f"POLICY GATE OPTIMIZATION SEED {seed}")
        print("#" * 100)
        hist_module._set_seed(int(seed))

        seed_test_pred = {
            cfg["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)
            for cfg in configs
        }
        seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

        for region_name, idx in cluster_mapping.items():
            idx = np.asarray(idx, dtype=int)
            rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
            print(f"seed={seed} | region={region_name} | RNFL={rnfl_cols}")

            rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
            rnfl_train_s, rnfl_val_s, rnfl_test_s, rnfl_scaler = _scale_static_meta_matrices(
                rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
            )

            y_scaler = StandardScaler()
            y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
            y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
            y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

            for cfg in configs:
                exp_name = cfg["Model"]
                exp_kind = cfg["kind"]
                meta_kind = cfg["meta_kind"]

                if meta_kind == "none":
                    meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                    meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                    meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
                elif meta_kind == "iop":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif meta_kind == "rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif meta_kind == "iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(meta_kind)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=float(cfg["film_scale"]),
                    gate_max=float(cfg["gate_max"]),
                    gate_init_bias=float(cfg["gate_init_bias"]),
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )

                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"policy_opt_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                pred_s, true_s = _predict_model(model, test_loader, device)
                pred_abs, true_abs = _inverse_scale(pred_s, true_s, y_scaler)
                seed_test_pred[exp_kind][:, idx] = pred_abs
                seed_test_true[:, idx] = true_abs

                summary.update({
                    "variant": exp_name,
                    "kind": exp_kind,
                    "seed": int(seed),
                    "region": region_name,
                    "loaded_bern_keys": len(loaded_keys),
                    "meta_kind": meta_kind,
                    "meta_dim": int(meta_train_s.shape[1]),
                    "gate_max": float(cfg["gate_max"]),
                    "film_scale": float(cfg["film_scale"]),
                    "gate_init_bias": float(cfg["gate_init_bias"]),
                    "train_samples": len(train_ds),
                    "val_samples": len(val_ds),
                    "test_samples": len(test_ds),
                })
                run_summaries.append(summary)

        for cfg in configs:
            exp_name = cfg["Model"]
            exp_kind = cfg["kind"]
            seed_result = _summarize_abs_prediction_matrix(
                seed_test_pred[exp_kind],
                seed_test_true,
                cluster_mapping,
                exp_name,
            )
            seed_result["seed"] = int(seed)
            seed_result["kind"] = exp_kind
            raw_results.append(seed_result)

    raw_long = pd.concat(raw_results, axis=0, ignore_index=True)
    summary_long = _metadata_summary_agg(raw_long)
    baseline_label = "Policy opt: VF-only"
    baseline_by_region = summary_long[summary_long["Model"].eq(baseline_label)][["Region", "MAE_mean_db"]].rename(
        columns={"MAE_mean_db": "baseline_mae_db"}
    )
    summary_with_delta = summary_long.merge(baseline_by_region, on="Region", how="left")
    summary_with_delta["delta_vs_vf_only"] = summary_with_delta["MAE_mean_db"] - summary_with_delta["baseline_mae_db"]
    summary_with_delta["improved_vs_vf_only"] = summary_with_delta["delta_vs_vf_only"] < 0
    summary_with_delta["delta_vs_vf_only_display"] = summary_with_delta["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")
    overall = summary_with_delta[summary_with_delta["Region"].eq("Overall")].copy()
    overall = overall.sort_values("MAE_mean_db").reset_index(drop=True)

    return {
        "raw_long": raw_long,
        "summary_long": summary_long,
        "summary_with_delta": summary_with_delta,
        "overall": overall,
        "runs_df": pd.DataFrame(run_summaries),
        "configs": configs,
        "train_samples": len(x_train_s),
        "val_samples": len(x_val_s),
        "test_samples": len(x_test_s),
        "seeds": seeds,
    }


def make_policy_gate_optimization_region_table(outputs):
    df = outputs["summary_with_delta"].copy()
    region_order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Infero_Nasal", "Infero_Temporal", "Temporal", "Macular"]
    model_order = {cfg["Model"]: i for i, cfg in enumerate(outputs["configs"])}
    out = df.copy()
    out["Region"] = pd.Categorical(out["Region"], categories=region_order, ordered=True)
    out["model_order"] = out["Model"].map(model_order)
    out = out.sort_values(["Region", "model_order"]).reset_index(drop=True)
    out["Region"] = out["Region"].astype(str)
    return out[[
        "Region",
        "Model",
        "MAE_mean_db",
        "MAE ± SD (dB)",
        "delta_vs_vf_only_display",
        "improved_vs_vf_only",
        "Samples",
        "#Pts",
    ]]


N_POLICY_GATE_OPT_SEEDS = 3
POLICY_GATE_OPT_SEEDS = tuple(
    np.random.default_rng().integers(0, 1_000_000, size=N_POLICY_GATE_OPT_SEEDS).tolist()
)
print("Random policy gate optimization seeds:", POLICY_GATE_OPT_SEEDS)

policy_gate_optimization_outputs = run_policy_gate_optimization_screen(
    seeds=POLICY_GATE_OPT_SEEDS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
)

print("Policy gate optimization - Overall:")
display(policy_gate_optimization_outputs["overall"][[
    "Model",
    "Region",
    "MAE_mean_db",
    "MAE_seed_std_db",
    "MAE_sample_std_db",
    "MAE ± SD (dB)",
    "delta_vs_vf_only_display",
    "improved_vs_vf_only",
    "Samples",
    "#Pts",
]])

print("Policy gate optimization - Overall + region details:")
policy_gate_optimization_region_table = make_policy_gate_optimization_region_table(policy_gate_optimization_outputs)
display(policy_gate_optimization_region_table)

print("Policy gate optimization run summaries:")
display(policy_gate_optimization_outputs["runs_df"][[
    "variant",
    "kind",
    "seed",
    "region",
    "best_val_mae_db",
    "best_epoch",
    "meta_kind",
    "meta_dim",
    "gate_max",
    "film_scale",
    "gate_init_bias",
]].sort_values(["variant", "region", "seed"]).reset_index(drop=True))



In [ ]:
import sys
import time
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import patches
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

project_root = Path(".")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import notebooks2.grape_region_full_experiment as grape_region_full_module
grape_region_full_module = importlib.reload(grape_region_full_module)

generate_voronoi_images_given_image_size = grape_region_full_module.generate_voronoi_images_given_image_size


def plot_compact_transformer_region_multibranch_stage_examples_separate_figures(
    *,
    seeds=(0,),
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    single_weight_grid=None,
    multibranch_weight_grid=None,
    max_total_meta_weight=0.5,
    coord_path=project_root / "GRAPE" / "GRAPE coordinate.xlsx",
    seed=0,
    stage_specs=None,
    prefer_region_specific_improved=True,
    image_size=(61, 61),
    n_rows=2,
):
    """
    Presentation-style qualitative plots for the latest compact Transformer
    region-specific multi-branch soft ensemble.

    Panels:
    recent history, ground truth, no-change, VF-only, soft IOP, soft RNFL,
    soft IOP+RNFL, global multibranch, region-specific multibranch.
    """
    if single_weight_grid is None:
        single_weight_grid = np.linspace(0.0, 0.5, 11, dtype=np.float32)
    if multibranch_weight_grid is None:
        multibranch_weight_grid = np.linspace(0.0, 0.5, 11, dtype=np.float32)

    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)

    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, _ = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variants = [
        {"label": "Transformer VF-only", "kind": "vf_only"},
        {"label": "IOP branch", "kind": "policy_iop_exposure"},
        {"label": "RNFL branch", "kind": "policy_rnfl"},
        {"label": "IOP+RNFL branch", "kind": "policy_iop_rnfl"},
    ]

    hist_module._set_seed(int(seed))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    seed_val_pred = {v["kind"]: np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32) for v in variants}
    seed_test_pred = {v["kind"]: np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32) for v in variants}
    seed_val_true = np.zeros((len(y_val), len(td_cols_exp)), dtype=np.float32)
    seed_test_true = np.zeros((len(y_test), len(td_cols_exp)), dtype=np.float32)

    print("Qualitative multibranch train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Using seed:", seed)

    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
        print(f"region={region_name} | RNFL={rnfl_cols}")

        rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
        rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
        rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
        rnfl_train_s, rnfl_val_s, rnfl_test_s, _ = _scale_static_meta_matrices(
            rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
        )

        y_scaler = StandardScaler()
        y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
        y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
        y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

        for variant in variants:
            exp_kind = variant["kind"]

            if exp_kind == "vf_only":
                meta_train_s = np.zeros((len(x_train_s), 0), dtype=np.float32)
                meta_val_s = np.zeros((len(x_val_s), 0), dtype=np.float32)
                meta_test_s = np.zeros((len(x_test_s), 0), dtype=np.float32)
            elif exp_kind == "policy_iop_exposure":
                meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
            elif exp_kind == "policy_rnfl":
                meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
            elif exp_kind == "policy_iop_rnfl":
                meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
            else:
                raise ValueError(exp_kind)

            train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
            val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
            test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)

            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
            test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

            model = CompactPointTransformerPolicyGateModel(
                output_dim=len(idx),
                point_indices=idx.tolist(),
                policy_meta_dim=meta_train_s.shape[1],
                film_scale=film_scale,
                gate_max=gate_max,
                **COMPACT_TRANSFORMER_CONFIG,
            )
            model, loaded_keys = _load_bern_point_transformer_weights(
                model, region_name, checkpoint_prefix="bern_point_transformer_compact"
            )

            model, hist_df, summary = _train_model(
                model=model,
                run_name=f"qual_region_multibranch_{exp_kind}_{region_name}_seed{seed}_{time.time_ns()}",
                train_loader=train_loader,
                val_loader=val_loader,
                train_ds=train_ds,
                val_ds=val_ds,
                y_scaler=y_scaler,
                device=device,
                num_epochs=num_epochs,
                lr=lr,
            )

            val_pred_s, val_true_s = _predict_model(model, val_loader, device)
            val_pred_abs, val_true_abs = _inverse_scale(val_pred_s, val_true_s, y_scaler)

            test_pred_s, test_true_s = _predict_model(model, test_loader, device)
            test_pred_abs, test_true_abs = _inverse_scale(test_pred_s, test_true_s, y_scaler)

            seed_val_pred[exp_kind][:, idx] = val_pred_abs
            seed_test_pred[exp_kind][:, idx] = test_pred_abs
            seed_val_true[:, idx] = val_true_abs
            seed_test_true[:, idx] = test_true_abs

    predictions_db = {
        "Transformer VF-only": seed_test_pred["vf_only"],
    }

    # Two-branch soft ensembles.
    single_rows = []
    for meta_kind, label in [
        ("policy_iop_exposure", "Soft ensemble + IOP"),
        ("policy_rnfl", "Soft ensemble + RNFL"),
        ("policy_iop_rnfl", "Soft ensemble + IOP + RNFL"),
    ]:
        best_single, _ = _select_ensemble_weight(
            seed_val_pred["vf_only"],
            seed_val_pred[meta_kind],
            seed_val_true,
            weight_grid=single_weight_grid,
        )
        predictions_db[label] = (
            (1.0 - best_single["w"]) * seed_test_pred["vf_only"]
            + best_single["w"] * seed_test_pred[meta_kind]
        )
        single_rows.append({"model": label, **best_single})

    # Global multibranch ensemble.
    best_global, _ = _select_multibranch_ensemble_weights(
        seed_val_pred,
        seed_val_true,
        weight_grid=multibranch_weight_grid,
        max_total_meta_weight=max_total_meta_weight,
    )
    predictions_db["Global multibranch"] = (
        best_global["w_vf_only"] * seed_test_pred["vf_only"]
        + best_global["w_iop"] * seed_test_pred["policy_iop_exposure"]
        + best_global["w_rnfl"] * seed_test_pred["policy_rnfl"]
        + best_global["w_iop_rnfl"] * seed_test_pred["policy_iop_rnfl"]
    )

    # Region-specific multibranch ensemble.
    region_specific_pred = np.zeros_like(seed_test_true, dtype=np.float32)
    region_weight_rows = []
    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        val_pred_region = {k: v[:, idx] for k, v in seed_val_pred.items()}
        test_pred_region = {k: v[:, idx] for k, v in seed_test_pred.items()}

        best_region, _ = _select_multibranch_ensemble_weights(
            val_pred_region,
            seed_val_true[:, idx],
            weight_grid=multibranch_weight_grid,
            max_total_meta_weight=max_total_meta_weight,
        )

        region_specific_pred[:, idx] = (
            best_region["w_vf_only"] * test_pred_region["vf_only"]
            + best_region["w_iop"] * test_pred_region["policy_iop_exposure"]
            + best_region["w_rnfl"] * test_pred_region["policy_rnfl"]
            + best_region["w_iop_rnfl"] * test_pred_region["policy_iop_rnfl"]
        )
        region_weight_rows.append({"Region": region_name, **best_region})

    predictions_db["Region-specific multibranch"] = region_specific_pred

    target_db = seed_test_true
    last_visit_db = np.stack([np.asarray(x[-1], dtype=np.float32) for x in x_test], axis=0)
    baseline_mae = np.mean(np.abs(last_visit_db - target_db), axis=1)
    mae_by_model = {
        name: np.mean(np.abs(pred - target_db), axis=1)
        for name, pred in predictions_db.items()
    }

    coord_df = pd.read_excel(Path(coord_path))
    coord_df.columns = [str(c).strip() for c in coord_df.columns]
    coord_df = coord_df[~coord_df["ids Grape"].isin([21, 32])].copy()
    coord_df["bern_idx"] = coord_df["Ids G"] + 1
    coord_df = coord_df.sort_values("bern_idx").reset_index(drop=True)
    xy59 = coord_df[["V1", "V2"]].to_numpy(dtype=float)

    approx_md = np.nanmean(target_db, axis=1)
    candidate_idx = np.arange(len(target_db))

    # Prefer examples where the final model beats VF-only and no-change, if available.
    if prefer_region_specific_improved:
        final_name = "Region-specific multibranch"
        mask = (mae_by_model[final_name] <= mae_by_model["Transformer VF-only"]) & (
            mae_by_model[final_name] <= baseline_mae
        )
        filtered = candidate_idx[mask]
        if len(filtered) > 0:
            candidate_idx = filtered

    if stage_specs is None:
        stage_specs = [
            ("Mild", lambda md: md >= -3.0),
            ("Moderate", lambda md: (md < -3.0) & (md >= -6.0)),
            ("Advanced", lambda md: md < -6.0),
        ]

    final_name = "Region-specific multibranch"
    vf_name = "Transformer VF-only"

    def _pattern_score(i):
        target = np.asarray(target_db[i], dtype=np.float32)
        last = np.asarray(last_visit_db[i], dtype=np.float32)

        # Higher score means a clearer spatial defect/progression pattern for presentation.
        spatial_std = float(np.nanstd(target))
        defect_depth = float(-np.nanmin(target))
        vf_change = float(np.nanmean(np.abs(target - last)))

        final_gain_vs_vf = float(mae_by_model[vf_name][i] - mae_by_model[final_name][i])
        final_gain_vs_nochange = float(baseline_mae[i] - mae_by_model[final_name][i])

        return (
            0.45 * spatial_std
            + 0.25 * defect_depth
            + 0.20 * vf_change
            + 3.00 * max(final_gain_vs_vf, 0.0)
            + 1.50 * max(final_gain_vs_nochange, 0.0)
        )

    selected = []
    used = set()

    for stage_name, stage_fn in stage_specs:
        stage_mask = np.asarray(stage_fn(approx_md), dtype=bool)
        stage_candidates = [i for i in candidate_idx if stage_mask[i] and i not in used]

        # Prefer stage examples where the final model improves both VF-only and no-change.
        improved_candidates = [
            i for i in stage_candidates
            if (mae_by_model[final_name][i] <= mae_by_model[vf_name][i])
            and (mae_by_model[final_name][i] <= baseline_mae[i])
        ]

        if len(improved_candidates) > 0:
            pool = improved_candidates
        elif len(stage_candidates) > 0:
            pool = stage_candidates
        else:
            pool = [i for i in range(len(target_db)) if i not in used]

        if len(pool) == 0:
            continue

        # Deterministically choose the clearest pattern in each disease stage.
        chosen = int(max(pool, key=_pattern_score))
        selected.append((stage_name, chosen))
        used.add(chosen)

    if len(selected) == 0:
        raise ValueError("No examples available for stage-specific plotting.")

    display_name_map = {
        "Transformer VF-only": "Transformer\nVF-only",
        "Soft ensemble + IOP": "Soft ensemble\n+ IOP",
        "Soft ensemble + RNFL": "Soft ensemble\n+ RNFL",
        "Soft ensemble + IOP + RNFL": "Soft ensemble\n+ IOP + RNFL",
        "Global multibranch": "Global\nmultibranch",
        "Region-specific multibranch": "Region-specific\nmultibranch",
    }

    figures = []
    selected_rows = []

    for stage_name, sample_idx in selected:
        history_db = np.asarray(x_test[sample_idx], dtype=np.float32)
        history_times = np.asarray(t_test[sample_idx], dtype=np.float32)

        # Same style as previous helper: show the two most recent history visits.
        if history_db.shape[0] > 2:
            history_db = history_db[-2:]
            history_times = history_times[-2:]

        sample_target = np.asarray(target_db[sample_idx], dtype=np.float32)
        sample_last = np.asarray(last_visit_db[sample_idx], dtype=np.float32)
        sample_md = float(approx_md[sample_idx])
        sample_baseline_mae = float(np.mean(np.abs(sample_last - sample_target)))

        maps = [history_db[i] for i in range(history_db.shape[0])]
        titles = [f"Recent VF {i+1}\nTime: {float(history_times[i]):.2f} y" for i in range(history_db.shape[0])]

        maps.extend([sample_target, sample_last])
        titles.extend([
            f"Ground Truth\nApprox MD: {sample_md:.2f} dB",
            f"No-change baseline\nMAE: {sample_baseline_mae:.2f} dB",
        ])

        # For PPT, keep only the five history/baseline panels plus the final model.
        selected_prediction_panels = [
            "Transformer VF-only",
            "Region-specific multibranch",
        ]
        for name in selected_prediction_panels:
            pred = predictions_db[name]
            mae_val = float(mae_by_model[name][sample_idx])
            display_name = display_name_map.get(name, name)
            maps.append(np.asarray(pred[sample_idx], dtype=np.float32))
            titles.append(f"{display_name}\nMAE: {mae_val:.2f} dB")

        maps = np.stack(maps, axis=0)
        vor_imgs = generate_voronoi_images_given_image_size(maps, xy59, image_size=image_size)

        n_panels = len(titles)
        n_cols = 3
        n_rows = int(np.ceil(n_panels / n_cols))

        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(5.9 * n_cols + 1.0, 6.2 * n_rows),
            constrained_layout=False,
        )
        axes = np.atleast_1d(axes).ravel()

        im = None
        h, w = vor_imgs[0].shape
        center = ((w - 1) / 2, (h - 1) / 2)
        radius = min(center) - 0.8

        for ax, img, title_str in zip(axes, vor_imgs, titles):
            im = ax.imshow(img, vmin=-30, vmax=5, cmap="viridis", interpolation="nearest")
            circle_clip = patches.Circle(center, radius, transform=ax.transData)
            im.set_clip_path(circle_clip)
            ax.axis("off")
            ax.set_title(title_str, fontsize=22, pad=16, fontweight="bold")

        for ax in axes[len(titles):]:
            ax.axis("off")

        fig.subplots_adjust(left=0.03, right=0.93, top=0.80, bottom=0.05, wspace=0.12, hspace=0.32)
        cax = fig.add_axes([0.945, 0.16, 0.012, 0.56])
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("TD (dB)", fontsize=18)
        cbar.ax.tick_params(labelsize=16)

        fig.suptitle(
            "Representative Example from the GRAPE Dataset History + Final Transformer Models",
            fontsize=34,
            fontweight="bold",
            y=0.97,
        )

        figures.append({
            "stage": stage_name,
            "sample_index": int(sample_idx),
            "approx_md": sample_md,
            "figure": fig,
            "axes": axes,
            "titles": titles,
        })
        selected_rows.append({
            "Stage": stage_name,
            "sample_index": int(sample_idx),
            "approx_md": sample_md,
            "no_change_mae": sample_baseline_mae,
            "vf_only_mae": float(mae_by_model["Transformer VF-only"][sample_idx]),
            "region_specific_mae": float(mae_by_model["Region-specific multibranch"][sample_idx]),
        })

    return {
        "figures": figures,
        "selected_examples": pd.DataFrame(selected_rows),
        "predictions_db": predictions_db,
        "mae_by_model": mae_by_model,
        "target_db": target_db,
        "last_visit_db": last_visit_db,
        "global_weights": pd.DataFrame([best_global]),
        "single_weights": pd.DataFrame(single_rows),
        "region_weights": pd.DataFrame(region_weight_rows),
    }


# Run qualitative example plotting with a random seed so each run can produce different PPT figures.
QUAL_EXAMPLE_SEED = int(np.random.default_rng().integers(0, 100000))
print("Using qualitative example seed:", QUAL_EXAMPLE_SEED)

tf_multi_sep_out = plot_compact_transformer_region_multibranch_stage_examples_separate_figures(
    seeds=(QUAL_EXAMPLE_SEED,),
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=20,
    batch_size=64,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
    single_weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
    multibranch_weight_grid=np.linspace(0.0, 0.5, 11, dtype=np.float32),
    max_total_meta_weight=0.5,
    coord_path=project_root / "GRAPE" / "GRAPE coordinate.xlsx",
    seed=QUAL_EXAMPLE_SEED,
    prefer_region_specific_improved=True,
    image_size=(61, 61),
    n_rows=2,
)

print("Selected examples:")
display(tf_multi_sep_out["selected_examples"])

print("Global weights:")
display(tf_multi_sep_out["global_weights"])

print("Single soft-ensemble weights:")
display(tf_multi_sep_out["single_weights"])

print("Region-specific weights:")
display(tf_multi_sep_out["region_weights"])

for item in tf_multi_sep_out["figures"]:
    print(
        f"Stage: {item['stage']} | "
        f"sample_index: {item['sample_index']} | "
        f"approx_MD: {item['approx_md']:.2f} dB"
    )
    plt.figure(item["figure"].number)
    plt.show()


In [ ]:
# PPT figure: prediction challenge with known VF history and unknown future VF.
# Synthetic illustration only. No real patient data and no model training.

import sys
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.patches import FancyArrowPatch

project_root = Path(".")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import notebooks2.grape_region_full_experiment as grape_region_full_module
grape_region_full_module = importlib.reload(grape_region_full_module)

generate_voronoi_images_given_image_size = (
    grape_region_full_module.generate_voronoi_images_given_image_size
)

coord_path = project_root / "GRAPE" / "GRAPE coordinate.xlsx"
coord_df = pd.read_excel(coord_path)
coord_df.columns = [str(c).strip() for c in coord_df.columns]
coord_df = coord_df[~coord_df["ids Grape"].isin([21, 32])].copy()
coord_df["bern_idx"] = coord_df["Ids G"] + 1
coord_df = coord_df.sort_values("bern_idx").reset_index(drop=True)
xy59 = coord_df[["V1", "V2"]].to_numpy(dtype=float)


def make_synthetic_progressive_vf(xy, severity=0.0, noise_seed=0):
    """Create a clean synthetic TD map for PPT illustration."""
    rng = np.random.default_rng(noise_seed)
    x = xy[:, 0].astype(float)
    y = xy[:, 1].astype(float)
    x = (x - np.nanmean(x)) / (np.nanstd(x) + 1e-6)
    y = (y - np.nanmean(y)) / (np.nanstd(y) + 1e-6)

    td = -3.0 - 0.55 * np.sin(1.5 * x) - 0.45 * np.cos(1.1 * y)
    arcuate_defect = np.exp(-((y + 0.55 + 0.25 * x) ** 2 / 0.20 + (x + 0.08) ** 2 / 1.2))
    nasal_step = np.exp(-((x - 1.02) ** 2 / 0.15 + (y + 0.02) ** 2 / 1.6))
    central_defect = np.exp(-((x + 0.22) ** 2 / 0.16 + (y + 0.04) ** 2 / 0.18))
    td -= severity * (8.0 * arcuate_defect + 4.6 * nasal_step + 3.7 * central_defect)
    td += rng.normal(0, 0.35, size=td.shape)
    return np.clip(td, -30, 3).astype(np.float32)


def draw_circular_vf(ax, img, *, title, vmin=-30, vmax=5):
    im = ax.imshow(img, vmin=vmin, vmax=vmax, cmap="viridis", interpolation="nearest")
    h, w = img.shape
    center = ((w - 1) / 2, (h - 1) / 2)
    radius = min(center) - 0.8
    im.set_clip_path(patches.Circle(center, radius, transform=ax.transData))
    ax.axis("off")
    ax.set_title(title, fontsize=18, fontweight="bold", pad=12)
    return im


def draw_future_unknown(ax, *, title):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.axis("off")
    circle = patches.Circle(
        (0.5, 0.5),
        0.42,
        facecolor="none",
        edgecolor="#2E7D67",
        linewidth=3.0,
        linestyle=(0, (6, 5)),
    )
    ax.add_patch(circle)
    ax.text(0.5, 0.5, "?", ha="center", va="center", fontsize=42, fontweight="bold", color="#17382F")
    ax.set_title(title, fontsize=18, fontweight="bold", pad=12)


times = [0.00, 0.55, 1.20, 2.10]
severities = [0.20, 0.58, 1.03]
known_vfs = np.stack([
    make_synthetic_progressive_vf(xy59, severity=s, noise_seed=300 + i)
    for i, s in enumerate(severities)
], axis=0)

vor_imgs = generate_voronoi_images_given_image_size(known_vfs, xy59, image_size=(61, 61))

fig = plt.figure(figsize=(16, 9), dpi=180, facecolor="#F5F1E8")
fig.suptitle(
    "Prediction Challenge: Forecasting Future Visual Fields",
    fontsize=30,
    fontweight="bold",
    color="#17382F",
    y=0.93,
)

fig.text(
    0.5,
    0.855,
    "Known longitudinal VF history with irregular follow-up intervals",
    ha="center",
    va="center",
    fontsize=17,
    color="#6C746F",
)

# Uneven panel positions make irregular timing visually clear.
panel_y = 0.36
panel_w = 0.155
panel_h = 0.275
panel_xs = [0.09, 0.30, 0.54, 0.79]

axes = [fig.add_axes([x, panel_y, panel_w, panel_h]) for x in panel_xs]
draw_circular_vf(axes[0], vor_imgs[0], title=f"VF 1\nTime: {times[0]:.2f} y")
draw_circular_vf(axes[1], vor_imgs[1], title=f"VF 2\nTime: {times[1]:.2f} y")
draw_circular_vf(axes[2], vor_imgs[2], title=f"VF 3\nTime: {times[2]:.2f} y")
draw_future_unknown(axes[3], title=f"Future VF ?\nTime: {times[3]:.2f} y")

# Irregular arrows between visits.
arrow_y = 0.70
gap_labels = ["0.55 y", "0.65 y", "1.10 y"]
for i in range(3):
    x_start = panel_xs[i] + panel_w + 0.01
    x_end = panel_xs[i + 1] - 0.015
    arrow = FancyArrowPatch(
        (x_start, arrow_y),
        (x_end, arrow_y),
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=2.3,
        color="#2E7D67",
        transform=fig.transFigure,
    )
    fig.patches.append(arrow)
    fig.text(
        (x_start + x_end) / 2,
        arrow_y + 0.03,
        gap_labels[i],
        ha="center",
        va="bottom",
        fontsize=13,
        color="#6C746F",
    )

fig.text(
    0.5,
    0.18,
    "Future deterioration must be inferred from sparse and irregular historical examinations.",
    ha="center",
    va="center",
    fontsize=21,
    fontweight="bold",
    color="#17382F",
)

save_dir = project_root / "ppt_assets"
save_dir.mkdir(exist_ok=True)
save_path = save_dir / "vf_prediction_challenge_known_history_future_unknown.png"
fig.savefig(save_path, dpi=220, facecolor="#F5F1E8", bbox_inches="tight", pad_inches=0.12)
print("Saved:", save_path)
plt.show()


In [ ]:
# PPT figure: spatially heterogeneous VF progression.
# Synthetic illustration only. No real patient data and no model training.

import sys
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches

project_root = Path(".")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import notebooks2.grape_region_full_experiment as grape_region_full_module
grape_region_full_module = importlib.reload(grape_region_full_module)

generate_voronoi_images_given_image_size = (
    grape_region_full_module.generate_voronoi_images_given_image_size
)

coord_path = project_root / "GRAPE" / "GRAPE coordinate.xlsx"
coord_df = pd.read_excel(coord_path)
coord_df.columns = [str(c).strip() for c in coord_df.columns]
coord_df = coord_df[~coord_df["ids Grape"].isin([21, 32])].copy()
coord_df["bern_idx"] = coord_df["Ids G"] + 1
coord_df = coord_df.sort_values("bern_idx").reset_index(drop=True)
xy59 = coord_df[["V1", "V2"]].to_numpy(dtype=float)


def normalize_xy_for_pattern(xy):
    x = xy[:, 0].astype(float)
    y = xy[:, 1].astype(float)
    x = (x - np.nanmean(x)) / (np.nanstd(x) + 1e-6)
    y = (y - np.nanmean(y)) / (np.nanstd(y) + 1e-6)
    return x, y


def make_region_heterogeneous_vf(xy, stage=0.0, noise_seed=0):
    """Synthetic TD map where different regions worsen at different rates."""
    rng = np.random.default_rng(noise_seed)
    x, y = normalize_xy_for_pattern(xy)

    td = -3.1 - 0.45 * np.sin(1.2 * x) - 0.35 * np.cos(1.0 * y)

    # Region-like defects with intentionally different progression speeds.
    macular = np.exp(-((x - 0.45) ** 2 / 0.20 + (y + 0.02) ** 2 / 0.24))
    infero_nasal = np.exp(-((x - 0.35) ** 2 / 0.42 + (y + 0.85) ** 2 / 0.22))
    supero_temporal = np.exp(-((x + 0.85) ** 2 / 0.32 + (y - 0.75) ** 2 / 0.25))
    temporal = np.exp(-((x + 1.05) ** 2 / 0.16 + (y + 0.02) ** 2 / 0.85))

    td -= stage * (9.0 * macular + 6.0 * infero_nasal + 2.8 * supero_temporal + 1.2 * temporal)
    td += rng.normal(0, 0.28, size=td.shape)
    return np.clip(td, -30, 3).astype(np.float32)


def draw_circular_vf_no_colorbar(ax, img, *, title):
    im = ax.imshow(img, vmin=-30, vmax=5, cmap="viridis", interpolation="nearest")
    h, w = img.shape
    center = ((w - 1) / 2, (h - 1) / 2)
    radius = min(center) - 0.8
    im.set_clip_path(patches.Circle(center, radius, transform=ax.transData))
    ax.axis("off")
    ax.set_title(title, fontsize=18, fontweight="bold", pad=12)


stages = [0.20, 0.65, 1.05]
times = [0.00, 0.75, 1.80]
vf_maps = np.stack([
    make_region_heterogeneous_vf(xy59, stage=s, noise_seed=500 + i)
    for i, s in enumerate(stages)
], axis=0)
vor_imgs = generate_voronoi_images_given_image_size(vf_maps, xy59, image_size=(61, 61))

fig = plt.figure(figsize=(16, 9), dpi=180, facecolor="#F5F1E8")
fig.suptitle(
    "Example of Localized Visual Field Progression",
    fontsize=30,
    fontweight="bold",
    color="#17382F",
    y=0.93,
)
fig.text(
    0.5,
    0.855,
    "A localized defect becomes clearer across visits.",
    ha="center",
    va="center",
    fontsize=17,
    color="#6C746F",
)

panel_y = 0.38
panel_w = 0.18
panel_h = 0.31
panel_xs = [0.17, 0.39, 0.61]
axes = [fig.add_axes([x, panel_y, panel_w, panel_h]) for x in panel_xs]

for i, ax in enumerate(axes):
    draw_circular_vf_no_colorbar(ax, vor_imgs[i], title=f"Visit {i+1}\nTime: {times[i]:.2f} y")

save_dir = project_root / "ppt_assets"
save_dir.mkdir(exist_ok=True)
save_path = save_dir / "vf_localized_progression_example.png"
fig.savefig(save_path, dpi=220, facecolor="#F5F1E8", bbox_inches="tight", pad_inches=0.12)
print("Saved:", save_path)
plt.show()


## 40. Marta feedback response: objective and evaluation audit

This section does **not** retrain models. It documents the updated thesis story and the evaluation rules we should follow before running new experiments.

Main correction from the feedback: the project should be framed as building a VF forecasting model that can use **incomplete longitudinal clinical data**, rather than only trying many model variants.

In [ ]:
import pandas as pd

marta_feedback_action_plan = pd.DataFrame([
    {
        "Priority": 1,
        "Feedback issue": "Overall goal was not explicit enough",
        "Notebook action": "Reframe the project around incomplete longitudinal clinical data",
        "Status": "documented below",
    },
    {
        "Priority": 2,
        "Feedback issue": "RNFL usage was unclear",
        "Notebook action": "Create an IOP/RNFL availability and fusion table",
        "Status": "documented below",
    },
    {
        "Priority": 3,
        "Feedback issue": "Validation/evaluation fitting risk",
        "Notebook action": "State train/validation/test roles explicitly; no test-set weight fitting",
        "Status": "documented below",
    },
    {
        "Priority": 4,
        "Feedback issue": "Random seed averaging is not ideal",
        "Notebook action": "Replace seed-focused reporting with patient-eye-level cross-validation",
        "Status": "next experiment",
    },
    {
        "Priority": 5,
        "Feedback issue": "Policy gate may not actually use metadata",
        "Notebook action": "Add gate/correction diagnostics: gate size, metadata correction size, final correction size",
        "Status": "next diagnostic",
    },
])

display(marta_feedback_action_plan)


### Updated project objective

**Working objective:** build a visual field forecasting model that can use incomplete longitudinal clinical data.

In this project, VF history is the strongest signal, but real clinical data are incomplete and heterogeneous: visits are irregular, IOP is available longitudinally, and RNFL is only available as baseline structural information. The model should therefore predict future pointwise TD values while using extra clinical information only when it improves generalization.

**Presentation wording:**

> We aim to forecast future pointwise visual field TD values from irregular VF histories, while safely incorporating incomplete clinical metadata such as longitudinal IOP and baseline RNFL.

In [ ]:
metadata_usage_table = pd.DataFrame([
    {
        "Signal": "VF history",
        "Availability": "Every historical VF visit",
        "Information type": "Longitudinal functional signal",
        "How it enters the best model": "Main Transformer input / VF-only branch",
        "Important note": "Strongest baseline signal",
    },
    {
        "Signal": "IOP",
        "Availability": "Per visit",
        "Information type": "Longitudinal clinical metadata",
        "How it enters the best model": "IOP metadata branch, then validation-weighted ensemble",
        "Important note": "Can align with each VF visit",
    },
    {
        "Signal": "RNFL",
        "Availability": "Baseline only in GRAPE",
        "Information type": "Static structural metadata",
        "How it enters the best model": "RNFL metadata branch, then validation-weighted ensemble",
        "Important note": "Cannot be treated like a per-visit time series",
    },
    {
        "Signal": "IOP + RNFL",
        "Availability": "Mixed: IOP longitudinal, RNFL baseline",
        "Information type": "Dynamic + static metadata",
        "How it enters the best model": "Separate candidate branch, not forced to dominate",
        "Important note": "Directly combining them was often unstable",
    },
])

display(metadata_usage_table)


In [ ]:
evaluation_protocol_table = pd.DataFrame([
    {
        "Dataset split": "Train",
        "Allowed use": "Fit model parameters for each branch",
        "Not allowed": "Use test examples or test errors during training",
    },
    {
        "Dataset split": "Validation",
        "Allowed use": "Select ensemble weights / choose hyperparameters",
        "Not allowed": "Report final performance as if it were untouched evaluation",
    },
    {
        "Dataset split": "Test",
        "Allowed use": "Final evaluation only, after weights are fixed",
        "Not allowed": "Select ensemble weights or tune the model",
    },
])

print("Key rule: validation selects weights; test reports final MAE only once.")
display(evaluation_protocol_table)


### Weight-selection leakage audit

Manual code audit result: the soft-ensemble and multi-branch ensemble weights are selected from **validation predictions** (`seed_val_pred`, `seed_val_true`) and then applied once to **test predictions** (`seed_test_pred`).

This means the main reported ensemble MAE is not directly choosing weights from the test set.

Remaining caution: qualitative PPT example selection can use test-set MAE to choose visually clear examples. That is acceptable for illustration only, but it should not be described as model selection or final evaluation.


In [ ]:
weight_selection_audit = pd.DataFrame([
    {
        "Component": "Two-branch soft ensemble",
        "Weight selection input": "seed_val_pred + seed_val_true",
        "Test-set role": "Apply fixed selected weight and report MAE",
        "Leakage risk": "Low if test is not used for choosing w",
    },
    {
        "Component": "Global multi-branch ensemble",
        "Weight selection input": "seed_val_pred + seed_val_true",
        "Test-set role": "Apply fixed selected weights and report MAE",
        "Leakage risk": "Low if test is not used for choosing branch weights",
    },
    {
        "Component": "Region-specific multi-branch ensemble",
        "Weight selection input": "validation predictions restricted to each region",
        "Test-set role": "Apply fixed region weights and report region/test MAE",
        "Leakage risk": "Moderate: more flexible, so confirm with patient-eye CV",
    },
    {
        "Component": "Qualitative PPT example selection",
        "Weight selection input": "not used for training or model weights",
        "Test-set role": "Can choose clearer examples for visualization",
        "Leakage risk": "Do not treat selected examples as evidence of final performance",
    },
])

display(weight_selection_audit)


### Replacement for random-seed-focused reporting

The earlier 5/10 random-seed averages were useful while exploring, but the next rigorous version should use **patient-eye-level cross-validation**.

Recommended next protocol:

1. Split patient-eyes into K folds.
2. For each fold, train branches on training patient-eyes.
3. Select ensemble weights only on validation patient-eyes inside that fold.
4. Evaluate once on the held-out fold.
5. Report mean MAE across folds.

This directly addresses the concern that repeated random seeds are less interpretable than a proper held-out cross-validation protocol.

### Lightweight patient-eye CV pilot (no training)

This cell only validates the cross-validation split logic. It does **not** train the Transformer.

Goal: before launching an expensive cross-validation experiment, verify that each fold separates train / validation / held-out test at the patient-eye level, so no eye appears in more than one split.

In [ ]:
# Lightweight patient-eye-level CV split simulation. No model training.
import numpy as np
import pandas as pd

PILOT_CV_K = 3
PILOT_VAL_FRACTION = 0.15
PILOT_SPLIT_SEED = 20260624

def _meta_sequence_frame(meta_obj, original_split):
    """Create a minimal sequence dataframe from existing GRAPE sequence metadata."""
    df = meta_obj.copy() if isinstance(meta_obj, pd.DataFrame) else pd.DataFrame(list(meta_obj))
    if "eye_id" not in df.columns and "patient_eye" in df.columns:
        df["eye_id"] = df["patient_eye"]
    if "eye_id" not in df.columns:
        raise KeyError(f"{original_split} metadata does not contain eye_id or patient_eye.")
    if "patient_uid" not in df.columns:
        for candidate_col in ["subject_id", "patient_id", "Subject Number", "subject"]:
            if candidate_col in df.columns:
                df["patient_uid"] = df[candidate_col]
                break
    if "patient_uid" not in df.columns:
        df["patient_uid"] = np.nan
    return df[["eye_id", "patient_uid"]].assign(original_split=original_split)

if "_prepare_safe_residual_data" in globals():
    train_df_fixed, val_df_fixed, test_df_fixed, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=(2, 3, 4),
        train_mode="consecutive",
        eval_mode="consecutive",
        time_input_mode="input_times",
    )
    all_seq_df = pd.concat(
        [
            train_df_fixed.assign(original_split="old_train"),
            val_df_fixed.assign(original_split="old_val"),
            test_df_fixed.assign(original_split="old_test"),
        ],
        axis=0,
        ignore_index=True,
    )
    split_source = "prepared Transformer sequence dataframe"
elif all(name in globals() for name in ["meta_train_grape", "meta_val_grape", "meta_test_grape"]):
    all_seq_df = pd.concat(
        [
            _meta_sequence_frame(meta_train_grape, "old_train"),
            _meta_sequence_frame(meta_val_grape, "old_val"),
            _meta_sequence_frame(meta_test_grape, "old_test"),
        ],
        axis=0,
        ignore_index=True,
    )
    split_source = "existing GRAPE sequence metadata"
else:
    # Fully standalone fallback: simulate GRAPE-like patient-eye/sample structure.
    # This validates the split algorithm only; it does not use real VF data.
    mock_rng = np.random.default_rng(PILOT_SPLIT_SEED)
    n_mock_eyes = 190  # roughly GRAPE train/val/test patient-eyes combined
    rows = []
    for eye_idx in range(n_mock_eyes):
        n_samples = int(mock_rng.integers(3, 7))
        for sample_idx in range(n_samples):
            rows.append({
                "eye_id": f"mock_eye_{eye_idx:03d}",
                "patient_uid": f"mock_patient_{eye_idx:03d}",
                "original_split": "synthetic_mock",
            })
    all_seq_df = pd.DataFrame(rows)
    split_source = "synthetic mock sequence metadata"

if "eye_id" not in all_seq_df.columns:
    raise KeyError("Expected eye_id in prepared sequence dataframe.")

unique_eyes = np.asarray(sorted(all_seq_df["eye_id"].dropna().unique()))
rng = np.random.default_rng(PILOT_SPLIT_SEED)
shuffled_eyes = unique_eyes.copy()
rng.shuffle(shuffled_eyes)
eye_folds = np.array_split(shuffled_eyes, PILOT_CV_K)

summary_rows = []
leakage_rows = []

for fold_idx, heldout_eyes in enumerate(eye_folds, start=1):
    heldout_eyes = np.asarray(heldout_eyes)
    trainval_eyes = np.setdiff1d(unique_eyes, heldout_eyes)

    fold_rng = np.random.default_rng(PILOT_SPLIT_SEED + fold_idx)
    trainval_shuffled = trainval_eyes.copy()
    fold_rng.shuffle(trainval_shuffled)

    n_val = max(1, int(round(PILOT_VAL_FRACTION * len(trainval_shuffled))))
    val_eyes = trainval_shuffled[:n_val]
    train_eyes = trainval_shuffled[n_val:]

    split_eye_sets = {
        "train": set(train_eyes.tolist()),
        "validation": set(val_eyes.tolist()),
        "heldout_test": set(heldout_eyes.tolist()),
    }

    leakage_rows.append({
        "fold": fold_idx,
        "train_val_overlap": len(split_eye_sets["train"] & split_eye_sets["validation"]),
        "train_test_overlap": len(split_eye_sets["train"] & split_eye_sets["heldout_test"]),
        "val_test_overlap": len(split_eye_sets["validation"] & split_eye_sets["heldout_test"]),
    })

    for split_name, eye_set in split_eye_sets.items():
        part = all_seq_df[all_seq_df["eye_id"].isin(eye_set)]
        summary_rows.append({
            "fold": fold_idx,
            "split": split_name,
            "samples": len(part),
            "patient_eyes": part["eye_id"].nunique(),
            "patients": part["patient_uid"].nunique() if "patient_uid" in part.columns else np.nan,
            "old_train_samples": int((part["original_split"] == "old_train").sum()),
            "old_val_samples": int((part["original_split"] == "old_val").sum()),
            "old_test_samples": int((part["original_split"] == "old_test").sum()),
        })

cv_split_summary = pd.DataFrame(summary_rows)
cv_leakage_check = pd.DataFrame(leakage_rows)

print("Patient-eye CV pilot only: no model training was run.")
print(f"K={PILOT_CV_K}, validation fraction inside non-test eyes={PILOT_VAL_FRACTION:.2f}")
display(cv_split_summary)
display(cv_leakage_check)

overlap_cols = ["train_val_overlap", "train_test_overlap", "val_test_overlap"]
assert (cv_leakage_check[overlap_cols] == 0).all().all()
print("Leakage check passed: no patient-eye overlap between train / validation / held-out test within each fold.")


### Tiny CV training pilot for split sensitivity

This is the first **actual model-training** check after the split-only pilot.

Purpose: test whether the current best idea still points in the same direction when the patient-eye split changes.

To keep computation manageable, this is intentionally small: 2-fold, 1 seed, few epochs, and coarse ensemble weights. It is not the final thesis-level cross-validation result.

In [ ]:
# Tiny patient-eye-level CV training pilot.
# Run this only after the Transformer + multibranch model cells have been defined.
import time
import numpy as np
import pandas as pd

TINY_CV_K = 2
TINY_CV_SEED = 0
TINY_CV_SPLIT_SEED = 20260624
TINY_CV_VAL_FRACTION = 0.15
TINY_CV_EPOCHS = 5
TINY_CV_BATCH_SIZE = 128
TINY_CV_WEIGHT_GRID = np.asarray([0.0, 0.25, 0.5], dtype=np.float32)

required_names = [
    "_prepare_safe_residual_data",
    "run_compact_transformer_region_multibranch_soft_ensemble",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(
        "Run the Transformer / multibranch definition cells first. Missing: "
        + ", ".join(missing)
    )

# Build one pooled sequence table from the existing fixed split, then repartition by eye_id.
base_train_df, base_val_df, base_test_df, base_td_cols, base_rnfl_cols = _prepare_safe_residual_data(
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    time_input_mode="input_times",
)
base_all_df = pd.concat(
    [
        base_train_df.assign(original_split="old_train"),
        base_val_df.assign(original_split="old_val"),
        base_test_df.assign(original_split="old_test"),
    ],
    axis=0,
    ignore_index=True,
)

unique_eyes = np.asarray(sorted(base_all_df["eye_id"].dropna().unique()))
split_rng = np.random.default_rng(TINY_CV_SPLIT_SEED)
shuffled_eyes = unique_eyes.copy()
split_rng.shuffle(shuffled_eyes)
eye_folds = np.array_split(shuffled_eyes, TINY_CV_K)

original_prepare_safe_residual_data = _prepare_safe_residual_data
tiny_cv_rows = []
tiny_cv_weight_rows = []

try:
    for fold_idx, heldout_eyes in enumerate(eye_folds, start=1):
        heldout_eyes = np.asarray(heldout_eyes)
        trainval_eyes = np.setdiff1d(unique_eyes, heldout_eyes)

        fold_rng = np.random.default_rng(TINY_CV_SPLIT_SEED + fold_idx)
        trainval_shuffled = trainval_eyes.copy()
        fold_rng.shuffle(trainval_shuffled)
        n_val = max(1, int(round(TINY_CV_VAL_FRACTION * len(trainval_shuffled))))
        val_eyes = trainval_shuffled[:n_val]
        train_eyes = trainval_shuffled[n_val:]

        fold_train_df = base_all_df[base_all_df["eye_id"].isin(train_eyes)].reset_index(drop=True)
        fold_val_df = base_all_df[base_all_df["eye_id"].isin(val_eyes)].reset_index(drop=True)
        fold_test_df = base_all_df[base_all_df["eye_id"].isin(heldout_eyes)].reset_index(drop=True)

        print("\n" + "=" * 90)
        print(
            f"Tiny CV fold {fold_idx}/{TINY_CV_K}: "
            f"train={len(fold_train_df)}, val={len(fold_val_df)}, test={len(fold_test_df)} samples"
        )

        def _prepare_fold_safe_residual_data(*args, **kwargs):
            return (
                fold_train_df.copy(),
                fold_val_df.copy(),
                fold_test_df.copy(),
                base_td_cols,
                base_rnfl_cols,
            )

        globals()["_prepare_safe_residual_data"] = _prepare_fold_safe_residual_data
        fold_start = time.time()
        fold_outputs = run_compact_transformer_region_multibranch_soft_ensemble(
            seeds=(TINY_CV_SEED,),
            k_list=(2, 3, 4),
            train_mode="consecutive",
            eval_mode="consecutive",
            num_epochs=TINY_CV_EPOCHS,
            batch_size=TINY_CV_BATCH_SIZE,
            lr=5e-4,
            time_input_mode="input_times",
            film_scale=0.05,
            gate_max=0.25,
            single_weight_grid=TINY_CV_WEIGHT_GRID,
            multibranch_weight_grid=TINY_CV_WEIGHT_GRID,
            max_total_meta_weight=0.5,
        )
        fold_seconds = time.time() - fold_start

        keep_models = [
            "Multibranch: VF-only",
            "Region multibranch: VF-only",
            "Multibranch: validation-weighted IOP/RNFL ensemble",
            "Region multibranch: global validation-weighted IOP/RNFL ensemble",
            "Region multibranch: region-specific validation-weighted ensemble",
        ]
        fold_overall = fold_outputs["overall"].copy()
        fold_overall = fold_overall[fold_overall["Model"].isin(keep_models)].copy()
        fold_overall["fold"] = fold_idx
        fold_overall["fold_seconds"] = fold_seconds
        fold_overall["train_samples"] = len(fold_train_df)
        fold_overall["val_samples"] = len(fold_val_df)
        fold_overall["test_samples"] = len(fold_test_df)
        tiny_cv_rows.append(fold_overall)

        rw = fold_outputs["region_weights"].copy()
        rw["fold"] = fold_idx
        tiny_cv_weight_rows.append(rw)

finally:
    globals()["_prepare_safe_residual_data"] = original_prepare_safe_residual_data

tiny_cv_overall = pd.concat(tiny_cv_rows, ignore_index=True)
tiny_cv_region_weights = pd.concat(tiny_cv_weight_rows, ignore_index=True)

tiny_cv_summary = (
    tiny_cv_overall
    .groupby("Model", as_index=False)
    .agg(
        folds=("fold", "nunique"),
        mean_MAE_db=("MAE_mean_db", "mean"),
        std_MAE_db=("MAE_mean_db", "std"),
        mean_sample_sd_db=("MAE_sample_std_db", "mean"),
    )
    .sort_values("mean_MAE_db")
    .reset_index(drop=True)
)
vf_model_candidates = tiny_cv_summary.loc[
    tiny_cv_summary["Model"].astype(str).str.contains("VF-only", case=False, regex=False),
    "Model",
].tolist()
if not vf_model_candidates:
    print("Available models in tiny_cv_summary:")
    display(tiny_cv_summary[["Model"]])
    raise ValueError("Could not identify the VF-only baseline model in tiny_cv_summary.")
vf_model_name = vf_model_candidates[0]
vf_baseline_cv = float(
    tiny_cv_summary.loc[tiny_cv_summary["Model"].eq(vf_model_name), "mean_MAE_db"].iloc[0]
)
print("Using VF-only CV baseline:", vf_model_name)
tiny_cv_summary["delta_vs_vf_only"] = tiny_cv_summary["mean_MAE_db"] - vf_baseline_cv
tiny_cv_summary["delta_display"] = tiny_cv_summary["delta_vs_vf_only"].map(lambda x: f"{x:+.3f} dB")

print("\nTiny CV overall by fold:")
display(tiny_cv_overall[[
    "fold", "Model", "MAE_mean_db", "MAE ± SD (dB)", "delta_vs_vf_only_display",
    "train_samples", "val_samples", "test_samples", "fold_seconds",
]])

print("Tiny CV mean summary:")
display(tiny_cv_summary)

print("Region-specific weights selected inside each fold:")
display(tiny_cv_region_weights)


In [ ]:
candidate_output_names = [
    "policy_gate_outputs",
    "soft_ensemble_outputs",
    "final_region_soft_ensemble_outputs",
    "final_metadata_ablation_outputs",
    "multibranch_soft_ensemble_outputs",
    "region_multibranch_soft_ensemble_outputs",
    "policy_gate_optimization_outputs",
    "separate_gates_outputs",
]

existing_output_inventory = pd.DataFrame([
    {
        "Object": name,
        "Available in current kernel": name in globals(),
        "Type": type(globals()[name]).__name__ if name in globals() else None,
        "Keys": list(globals()[name].keys()) if name in globals() and isinstance(globals()[name], dict) else None,
    }
    for name in candidate_output_names
])

display(existing_output_inventory)


### Next diagnostic: does the policy gate actually use metadata?

Marta's question can be answered by checking three quantities:

- **Gate size:** how large is the learned metadata reliability weight?
- **Raw metadata correction:** how different is the metadata-adjusted prediction from the VF-only prediction?
- **Final correction:** how much does the final prediction actually move after applying the gate?

Conceptually:

`final = VF-only prediction + gate * (metadata-adjusted prediction - VF-only prediction)`

If the gate and final correction are near zero, metadata is not being used much. If the correction is large but MAE worsens, metadata is being used but is noisy.

In [ ]:
# Policy gate usage diagnostic: does metadata actually change the prediction?
# This is a small diagnostic run, not a final performance experiment.
POLICY_DIAG_SEEDS = (0,)
POLICY_DIAG_EPOCHS = 5
POLICY_DIAG_BATCH_SIZE = 128
POLICY_DIAG_VARIANTS = (
    "policy_iop_exposure",
    "policy_spatial_rnfl",
    "policy_iop_rnfl",
)


def _predict_policy_gate_components(model, loader, device, y_scaler):
    """Return base/meta/final predictions and gate diagnostics in original dB scale."""
    model.eval()
    base_chunks, meta_chunks, final_chunks, true_chunks, gate_chunks = [], [], [], [], []

    with torch.no_grad():
        for padded_x, padded_t, lengths, targets, meta_input in loader:
            padded_x = padded_x.to(device)
            padded_t = padded_t.to(device)
            lengths = lengths.to(device)
            targets = targets.to(device)
            meta_input = meta_input.to(device)

            encoded, token_mask = model._encode_vf_only_tokens(padded_x, padded_t, lengths)
            base_pred = model._decode_encoded_tokens(encoded, token_mask, padded_x.device)

            if model.policy_meta_dim == 0 or model.film_net is None:
                meta_pred = base_pred
                gate = torch.zeros((padded_x.shape[0], 1), device=device, dtype=padded_x.dtype)
                final_pred = base_pred
            else:
                meta = model.policy_norm(meta_input)
                film = model.film_net(meta)
                gamma, beta = torch.chunk(film, 2, dim=-1)
                gamma = model.film_scale * torch.tanh(gamma).unsqueeze(1)
                beta = model.film_scale * torch.tanh(beta).unsqueeze(1)
                meta_encoded = encoded * (1.0 + gamma) + beta
                meta_pred = model._decode_encoded_tokens(meta_encoded, token_mask, padded_x.device)

                # gate is the fraction of the raw metadata correction that is applied.
                gate = model.gate_max * torch.sigmoid(model.gate_net(meta))
                final_pred = base_pred + gate * (meta_pred - base_pred)

            base_chunks.append(base_pred.cpu().numpy())
            meta_chunks.append(meta_pred.cpu().numpy())
            final_chunks.append(final_pred.cpu().numpy())
            true_chunks.append(targets.cpu().numpy())
            gate_chunks.append(gate.cpu().numpy())

    base_s = np.concatenate(base_chunks, axis=0)
    meta_s = np.concatenate(meta_chunks, axis=0)
    final_s = np.concatenate(final_chunks, axis=0)
    true_s = np.concatenate(true_chunks, axis=0)
    gate = np.concatenate(gate_chunks, axis=0)

    return {
        "base_abs": y_scaler.inverse_transform(base_s),
        "meta_abs": y_scaler.inverse_transform(meta_s),
        "final_abs": y_scaler.inverse_transform(final_s),
        "true_abs": y_scaler.inverse_transform(true_s),
        "gate": gate,
    }


def _component_metric_row(*, model_name, kind, seed, region, components):
    base = components["base_abs"]
    meta = components["meta_abs"]
    final = components["final_abs"]
    true = components["true_abs"]
    gate = components["gate"]

    raw_corr = np.abs(meta - base)
    final_corr = np.abs(final - base)
    raw_corr_mean = float(np.mean(raw_corr))
    final_corr_mean = float(np.mean(final_corr))
    base_mae = float(np.mean(np.abs(base - true)))
    meta_mae = float(np.mean(np.abs(meta - true)))
    final_mae = float(np.mean(np.abs(final - true)))

    return {
        "Model": model_name,
        "kind": kind,
        "seed": int(seed),
        "Region": region,
        "gate_mean": float(np.mean(gate)),
        "gate_std": float(np.std(gate)),
        "gate_min": float(np.min(gate)),
        "gate_max": float(np.max(gate)),
        "raw_metadata_correction_mae_db": raw_corr_mean,
        "final_applied_correction_mae_db": final_corr_mean,
        "applied_over_raw_correction": float(final_corr_mean / raw_corr_mean) if raw_corr_mean > 0 else 0.0,
        "base_mae_db": base_mae,
        "metadata_adjusted_mae_db": meta_mae,
        "final_mae_db": final_mae,
        "delta_final_vs_base_db": final_mae - base_mae,
        "samples": int(final.shape[0]),
        "points": int(final.shape[1]),
    }


def run_policy_gate_usage_diagnostics(
    *,
    seeds=POLICY_DIAG_SEEDS,
    variants=POLICY_DIAG_VARIANTS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=POLICY_DIAG_EPOCHS,
    batch_size=POLICY_DIAG_BATCH_SIZE,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
):
    required_names = [
        "torch",
        "StandardScaler",
        "DataLoader",
        "RegionHistoryDataset",
        "region_collate",
        "CompactPointTransformerPolicyGateModel",
        "COMPACT_TRANSFORMER_CONFIG",
        "cluster_mapping",
        "compact_transformer_spatial_rnfl_map",
        "hist_module",
        "_prepare_safe_residual_data",
        "_df_to_arrays_with_history",
        "_finite_mask",
        "_apply_mask",
        "_scale_x_list",
        "_iop_exposure_matrix",
        "_scale_static_meta_matrices",
        "_region_rnfl_matrix",
        "_load_bern_point_transformer_weights",
        "_train_model",
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise NameError(
            "Run the Transformer / policy-gate prerequisite cells first. Missing: "
            + ", ".join(missing)
        )

    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, _ = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variant_table = {
        "policy_iop_exposure": "Policy diagnostic: IOP exposure",
        "policy_spatial_rnfl": "Policy diagnostic: spatial RNFL",
        "policy_iop_rnfl": "Policy diagnostic: IOP exposure + spatial RNFL",
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []

    print("Policy gate diagnostic train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds, "| epochs:", num_epochs)

    for seed in seeds:
        hist_module._set_seed(int(seed))
        for kind in variants:
            model_name = variant_table[kind]
            print("\n" + "#" * 90)
            print(f"POLICY GATE DIAGNOSTIC | seed={seed} | {model_name}")
            print("#" * 90)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                print(f"seed={seed} | region={region_name} | kind={kind} | RNFL={rnfl_cols}")

                rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_train_s, rnfl_val_s, rnfl_test_s, _ = _scale_static_meta_matrices(
                    rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
                )

                if kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif kind == "policy_spatial_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(kind)

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, meta_test_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )
                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"policy_gate_usage_diag_{kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                components = _predict_policy_gate_components(model, test_loader, device, y_scaler)
                rows.append(_component_metric_row(
                    model_name=model_name,
                    kind=kind,
                    seed=seed,
                    region=region_name,
                    components=components,
                ))

    region_df = pd.DataFrame(rows)
    overall_df = (
        region_df
        .groupby(["Model", "kind"], as_index=False)
        .agg(
            gate_mean=("gate_mean", "mean"),
            gate_std=("gate_mean", "std"),
            gate_min=("gate_min", "min"),
            gate_max=("gate_max", "max"),
            raw_metadata_correction_mae_db=("raw_metadata_correction_mae_db", "mean"),
            final_applied_correction_mae_db=("final_applied_correction_mae_db", "mean"),
            applied_over_raw_correction=("applied_over_raw_correction", "mean"),
            base_mae_db=("base_mae_db", "mean"),
            metadata_adjusted_mae_db=("metadata_adjusted_mae_db", "mean"),
            final_mae_db=("final_mae_db", "mean"),
            delta_final_vs_base_db=("delta_final_vs_base_db", "mean"),
        )
        .sort_values("final_mae_db")
        .reset_index(drop=True)
    )
    overall_df["delta_display"] = overall_df["delta_final_vs_base_db"].map(lambda x: f"{x:+.4f} dB")

    return {
        "overall": overall_df,
        "by_region": region_df,
        "settings": {
            "seeds": seeds,
            "epochs": num_epochs,
            "batch_size": batch_size,
            "film_scale": film_scale,
            "gate_max": gate_max,
        },
    }


policy_gate_usage_diagnostics = run_policy_gate_usage_diagnostics()

print("Policy gate usage diagnostic - overall:")
display(policy_gate_usage_diagnostics["overall"])

print("Policy gate usage diagnostic - by region:")
display(policy_gate_usage_diagnostics["by_region"])



In [ ]:
# Metadata perturbation diagnostic: do metadata-aware branches depend on metadata?
# This checks whether predictions change when metadata is zeroed or shuffled.
PERTURB_DIAG_SEEDS = (0,)
PERTURB_DIAG_EPOCHS = 5
PERTURB_DIAG_BATCH_SIZE = 128
PERTURB_DIAG_VARIANTS = (
    "policy_iop_exposure",
    "policy_spatial_rnfl",
    "policy_iop_rnfl",
)
PERTURB_SHUFFLE_SEED = 20260625


def _make_perturbed_meta(meta_array, mode, seed=PERTURB_SHUFFLE_SEED):
    meta_array = np.asarray(meta_array, dtype=np.float32)
    if mode == "normal":
        return meta_array.copy()
    if mode == "zero":
        return np.zeros_like(meta_array, dtype=np.float32)
    if mode == "shuffle":
        rng = np.random.default_rng(seed)
        order = rng.permutation(len(meta_array))
        return meta_array[order].astype(np.float32)
    raise ValueError(mode)


def _policy_perturbation_metric_row(*, model_name, kind, seed, region, normal, zeroed, shuffled):
    true = normal["true_abs"]

    normal_final_mae = float(np.mean(np.abs(normal["final_abs"] - true)))
    zero_final_mae = float(np.mean(np.abs(zeroed["final_abs"] - true)))
    shuffle_final_mae = float(np.mean(np.abs(shuffled["final_abs"] - true)))

    raw_corr = float(np.mean(np.abs(normal["meta_abs"] - normal["base_abs"])))
    applied_corr = float(np.mean(np.abs(normal["final_abs"] - normal["base_abs"])))

    return {
        "Model": model_name,
        "kind": kind,
        "seed": int(seed),
        "Region": region,
        "gate_mean_normal": float(np.mean(normal["gate"])),
        "gate_mean_zero": float(np.mean(zeroed["gate"])),
        "gate_mean_shuffle": float(np.mean(shuffled["gate"])),
        "raw_metadata_correction_mae_db": raw_corr,
        "final_applied_correction_mae_db": applied_corr,
        "raw_meta_pred_change_zero_db": float(np.mean(np.abs(normal["meta_abs"] - zeroed["meta_abs"]))),
        "raw_meta_pred_change_shuffle_db": float(np.mean(np.abs(normal["meta_abs"] - shuffled["meta_abs"]))),
        "final_pred_change_zero_db": float(np.mean(np.abs(normal["final_abs"] - zeroed["final_abs"]))),
        "final_pred_change_shuffle_db": float(np.mean(np.abs(normal["final_abs"] - shuffled["final_abs"]))),
        "gate_change_zero": float(np.mean(np.abs(normal["gate"] - zeroed["gate"]))),
        "gate_change_shuffle": float(np.mean(np.abs(normal["gate"] - shuffled["gate"]))),
        "normal_final_mae_db": normal_final_mae,
        "zero_final_mae_db": zero_final_mae,
        "shuffle_final_mae_db": shuffle_final_mae,
        "zero_minus_normal_mae_db": zero_final_mae - normal_final_mae,
        "shuffle_minus_normal_mae_db": shuffle_final_mae - normal_final_mae,
        "samples": int(normal["final_abs"].shape[0]),
        "points": int(normal["final_abs"].shape[1]),
    }


def run_metadata_perturbation_diagnostics(
    *,
    seeds=PERTURB_DIAG_SEEDS,
    variants=PERTURB_DIAG_VARIANTS,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=PERTURB_DIAG_EPOCHS,
    batch_size=PERTURB_DIAG_BATCH_SIZE,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
):
    required_names = [
        "torch",
        "StandardScaler",
        "DataLoader",
        "RegionHistoryDataset",
        "region_collate",
        "CompactPointTransformerPolicyGateModel",
        "COMPACT_TRANSFORMER_CONFIG",
        "cluster_mapping",
        "compact_transformer_spatial_rnfl_map",
        "hist_module",
        "_prepare_safe_residual_data",
        "_df_to_arrays_with_history",
        "_finite_mask",
        "_apply_mask",
        "_scale_x_list",
        "_iop_exposure_matrix",
        "_scale_static_meta_matrices",
        "_region_rnfl_matrix",
        "_load_bern_point_transformer_weights",
        "_train_model",
        "_predict_policy_gate_components",
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise NameError(
            "Run the Transformer / policy-gate diagnostic prerequisite cells first. Missing: "
            + ", ".join(missing)
        )

    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, meta_val_empty = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, meta_test_empty = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)
    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, _ = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variant_table = {
        "policy_iop_exposure": "Perturb diagnostic: IOP exposure",
        "policy_spatial_rnfl": "Perturb diagnostic: spatial RNFL",
        "policy_iop_rnfl": "Perturb diagnostic: IOP exposure + spatial RNFL",
    }

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []

    print("Metadata perturbation diagnostic train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds, "| epochs:", num_epochs)

    for seed in seeds:
        hist_module._set_seed(int(seed))
        for kind in variants:
            model_name = variant_table[kind]
            print("\n" + "#" * 90)
            print(f"METADATA PERTURBATION DIAGNOSTIC | seed={seed} | {model_name}")
            print("#" * 90)

            for region_name, idx in cluster_mapping.items():
                idx = np.asarray(idx, dtype=int)
                rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                print(f"seed={seed} | region={region_name} | kind={kind} | RNFL={rnfl_cols}")

                rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                rnfl_train_s, rnfl_val_s, rnfl_test_s, _ = _scale_static_meta_matrices(
                    rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
                )

                if kind == "policy_iop_exposure":
                    meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                elif kind == "policy_spatial_rnfl":
                    meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                elif kind == "policy_iop_rnfl":
                    meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                    meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                    meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                else:
                    raise ValueError(kind)

                y_scaler = StandardScaler()
                y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=region_collate)
                val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                model = CompactPointTransformerPolicyGateModel(
                    output_dim=len(idx),
                    point_indices=idx.tolist(),
                    policy_meta_dim=meta_train_s.shape[1],
                    film_scale=film_scale,
                    gate_max=gate_max,
                    **COMPACT_TRANSFORMER_CONFIG,
                )
                model, loaded_keys = _load_bern_point_transformer_weights(
                    model, region_name, checkpoint_prefix="bern_point_transformer_compact"
                )
                model, hist_df, summary = _train_model(
                    model=model,
                    run_name=f"metadata_perturb_diag_{kind}_{region_name}_seed{seed}_{time.time_ns()}",
                    train_loader=train_loader,
                    val_loader=val_loader,
                    train_ds=train_ds,
                    val_ds=val_ds,
                    y_scaler=y_scaler,
                    device=device,
                    num_epochs=num_epochs,
                    lr=lr,
                )

                normal_test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, _make_perturbed_meta(meta_test_s, "normal"))
                zero_test_ds = RegionHistoryDataset(x_test_s, y_test_s, t_test, _make_perturbed_meta(meta_test_s, "zero"))
                shuffle_test_ds = RegionHistoryDataset(
                    x_test_s,
                    y_test_s,
                    t_test,
                    _make_perturbed_meta(meta_test_s, "shuffle", seed=PERTURB_SHUFFLE_SEED + int(seed)),
                )

                normal_loader = DataLoader(normal_test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                zero_loader = DataLoader(zero_test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)
                shuffle_loader = DataLoader(shuffle_test_ds, batch_size=batch_size, shuffle=False, collate_fn=region_collate)

                normal = _predict_policy_gate_components(model, normal_loader, device, y_scaler)
                zeroed = _predict_policy_gate_components(model, zero_loader, device, y_scaler)
                shuffled = _predict_policy_gate_components(model, shuffle_loader, device, y_scaler)

                rows.append(_policy_perturbation_metric_row(
                    model_name=model_name,
                    kind=kind,
                    seed=seed,
                    region=region_name,
                    normal=normal,
                    zeroed=zeroed,
                    shuffled=shuffled,
                ))

    region_df = pd.DataFrame(rows)
    overall_df = (
        region_df
        .groupby(["Model", "kind"], as_index=False)
        .agg(
            gate_mean_normal=("gate_mean_normal", "mean"),
            raw_metadata_correction_mae_db=("raw_metadata_correction_mae_db", "mean"),
            final_applied_correction_mae_db=("final_applied_correction_mae_db", "mean"),
            raw_meta_pred_change_zero_db=("raw_meta_pred_change_zero_db", "mean"),
            raw_meta_pred_change_shuffle_db=("raw_meta_pred_change_shuffle_db", "mean"),
            final_pred_change_zero_db=("final_pred_change_zero_db", "mean"),
            final_pred_change_shuffle_db=("final_pred_change_shuffle_db", "mean"),
            normal_final_mae_db=("normal_final_mae_db", "mean"),
            zero_final_mae_db=("zero_final_mae_db", "mean"),
            shuffle_final_mae_db=("shuffle_final_mae_db", "mean"),
            zero_minus_normal_mae_db=("zero_minus_normal_mae_db", "mean"),
            shuffle_minus_normal_mae_db=("shuffle_minus_normal_mae_db", "mean"),
        )
        .sort_values("normal_final_mae_db")
        .reset_index(drop=True)
    )
    overall_df["zero_delta_display"] = overall_df["zero_minus_normal_mae_db"].map(lambda x: f"{x:+.4f} dB")
    overall_df["shuffle_delta_display"] = overall_df["shuffle_minus_normal_mae_db"].map(lambda x: f"{x:+.4f} dB")

    return {
        "overall": overall_df,
        "by_region": region_df,
        "settings": {
            "seeds": seeds,
            "epochs": num_epochs,
            "batch_size": batch_size,
        },
    }


metadata_perturbation_diagnostics = run_metadata_perturbation_diagnostics()

print("Metadata perturbation diagnostic - overall:")
display(metadata_perturbation_diagnostics["overall"])

print("Metadata perturbation diagnostic - by region:")
display(metadata_perturbation_diagnostics["by_region"])



In [ ]:
# Attribution audit: what can we conclude from the policy-gate diagnostics?
# This cell does not train a model. It merges the gate and perturbation diagnostics into a report-ready table.
required_objects = [
    "policy_gate_usage_diagnostics",
    "metadata_perturbation_diagnostics",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise NameError(
        "Run the policy-gate usage and metadata perturbation diagnostic cells first. Missing: "
        + ", ".join(missing)
    )

_gate_overall = policy_gate_usage_diagnostics["overall"].copy()
_perturb_overall = metadata_perturbation_diagnostics["overall"].copy()

_gate_keep = _gate_overall[[
    "kind",
    "gate_mean",
    "raw_metadata_correction_mae_db",
    "final_applied_correction_mae_db",
    "delta_final_vs_base_db",
]].rename(columns={
    "gate_mean": "learned_gate_mean",
    "raw_metadata_correction_mae_db": "raw_metadata_correction_db",
    "final_applied_correction_mae_db": "applied_metadata_correction_db",
    "delta_final_vs_base_db": "policy_delta_vs_vf_only_db",
})

_perturb_keep = _perturb_overall[[
    "kind",
    "raw_meta_pred_change_zero_db",
    "raw_meta_pred_change_shuffle_db",
    "final_pred_change_zero_db",
    "final_pred_change_shuffle_db",
    "zero_minus_normal_mae_db",
    "shuffle_minus_normal_mae_db",
]].rename(columns={
    "raw_meta_pred_change_zero_db": "raw_branch_change_when_zeroed_db",
    "raw_meta_pred_change_shuffle_db": "raw_branch_change_when_shuffled_db",
    "final_pred_change_zero_db": "final_pred_change_when_zeroed_db",
    "final_pred_change_shuffle_db": "final_pred_change_when_shuffled_db",
})

metadata_attribution_audit = _gate_keep.merge(_perturb_keep, on="kind", how="inner")
metadata_attribution_audit["evidence_summary"] = np.where(
    metadata_attribution_audit["final_pred_change_when_zeroed_db"].abs() < 1e-4,
    "Final prediction is almost unchanged when metadata is zeroed/shuffled",
    "Final prediction changes when metadata is perturbed",
)
metadata_attribution_audit["interpretation"] = np.where(
    metadata_attribution_audit["learned_gate_mean"] < 1e-3,
    "Metadata branch exists, but gate is almost closed",
    "Metadata gate is non-negligible",
)
metadata_attribution_audit["safe_claim"] = np.where(
    metadata_attribution_audit["learned_gate_mean"] < 1e-3,
    "Do not claim strong effective metadata use for this policy-gate branch",
    "Metadata may be contributing; verify with stronger perturbation/CV",
)

print("Metadata attribution audit:")
display(metadata_attribution_audit)

print("Report-ready conclusion:")
print(
    "Policy-gate branches technically include IOP/RNFL metadata, but the learned gates are nearly closed. "
    "Zeroing or shuffling metadata changes the raw metadata-adjusted branch slightly, but the final prediction and MAE are almost unchanged. "
    "Therefore, the policy-gate results should not be interpreted as strong evidence that metadata was effectively used. "
    "Any small gain from soft/multibranch ensembles may partly reflect conservative prediction averaging rather than true metadata contribution."
)



In [ ]:
# Without-pretraining diagnostic: does metadata matter more when the VF backbone is weaker?
# This directly addresses Marta's question:
# "Without fine-tuning, could you check if metadata has a bigger effect?"

NO_PRETRAIN_DIAG_SEEDS = (0,)
NO_PRETRAIN_DIAG_EPOCHS = 5
NO_PRETRAIN_DIAG_BATCH_SIZE = 128
NO_PRETRAIN_DIAG_VARIANTS = (
    "policy_iop_exposure",
    "policy_spatial_rnfl",
    "policy_iop_rnfl",
)
NO_PRETRAIN_DIAG_INIT_MODES = (
    "bern_pretrained",
    "scratch_no_pretraining",
)
NO_PRETRAIN_SHUFFLE_SEED = 20260625


def run_without_pretraining_metadata_sensitivity_diagnostic(
    *,
    seeds=NO_PRETRAIN_DIAG_SEEDS,
    variants=NO_PRETRAIN_DIAG_VARIANTS,
    init_modes=NO_PRETRAIN_DIAG_INIT_MODES,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=NO_PRETRAIN_DIAG_EPOCHS,
    batch_size=NO_PRETRAIN_DIAG_BATCH_SIZE,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
):
    required_names = [
        "torch",
        "StandardScaler",
        "DataLoader",
        "RegionHistoryDataset",
        "region_collate",
        "CompactPointTransformerPolicyGateModel",
        "COMPACT_TRANSFORMER_CONFIG",
        "cluster_mapping",
        "compact_transformer_spatial_rnfl_map",
        "hist_module",
        "_prepare_safe_residual_data",
        "_df_to_arrays_with_history",
        "_finite_mask",
        "_apply_mask",
        "_scale_x_list",
        "_iop_exposure_matrix",
        "_scale_static_meta_matrices",
        "_region_rnfl_matrix",
        "_load_bern_point_transformer_weights",
        "_train_model",
        "_predict_policy_gate_components",
        "_make_perturbed_meta",
        "_policy_perturbation_metric_row",
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise NameError(
            "Run the Transformer / metadata diagnostic prerequisite cells first. Missing: "
            + ", ".join(missing)
        )

    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(
        x_train, y_train, t_train, meta_train_empty, train_mask
    )
    x_val, y_val, t_val, meta_val_empty = _apply_mask(
        x_val, y_val, t_val, meta_val_empty, val_mask
    )
    x_test, y_test, t_test, meta_test_empty = _apply_mask(
        x_test, y_test, t_test, meta_test_empty, test_mask
    )

    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, _ = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variant_table = {
        "policy_iop_exposure": "IOP exposure",
        "policy_spatial_rnfl": "spatial RNFL",
        "policy_iop_rnfl": "IOP exposure + spatial RNFL",
    }
    init_table = {
        "bern_pretrained": True,
        "scratch_no_pretraining": False,
    }

    unknown_init_modes = [mode for mode in init_modes if mode not in init_table]
    if unknown_init_modes:
        raise ValueError(f"Unknown init_modes: {unknown_init_modes}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []

    print("Without-pretraining metadata sensitivity diagnostic")
    print("train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds, "| epochs:", num_epochs, "| init_modes:", init_modes)

    for init_mode in init_modes:
        use_bern_pretraining = init_table[init_mode]
        for seed in seeds:
            hist_module._set_seed(int(seed))
            for kind in variants:
                model_name = f"{init_mode}: {variant_table[kind]}"
                print("\n" + "#" * 96)
                print(f"NO-PRETRAIN DIAGNOSTIC | init={init_mode} | seed={seed} | {variant_table[kind]}")
                print("#" * 96)

                for region_name, idx in cluster_mapping.items():
                    idx = np.asarray(idx, dtype=int)
                    rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                    print(
                        f"init={init_mode} | seed={seed} | region={region_name} | "
                        f"kind={kind} | RNFL={rnfl_cols}"
                    )

                    rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                    rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                    rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                    rnfl_train_s, rnfl_val_s, rnfl_test_s, _ = _scale_static_meta_matrices(
                        rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
                    )

                    if kind == "policy_iop_exposure":
                        meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                    elif kind == "policy_spatial_rnfl":
                        meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                    elif kind == "policy_iop_rnfl":
                        meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                        meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                        meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                    else:
                        raise ValueError(kind)

                    y_scaler = StandardScaler()
                    y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                    y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                    y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                    train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                    val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                    train_loader = DataLoader(
                        train_ds,
                        batch_size=batch_size,
                        shuffle=True,
                        collate_fn=region_collate,
                    )
                    val_loader = DataLoader(
                        val_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )

                    model = CompactPointTransformerPolicyGateModel(
                        output_dim=len(idx),
                        point_indices=idx.tolist(),
                        policy_meta_dim=meta_train_s.shape[1],
                        film_scale=film_scale,
                        gate_max=gate_max,
                        **COMPACT_TRANSFORMER_CONFIG,
                    )

                    loaded_key_count = 0
                    if use_bern_pretraining:
                        model, loaded_keys = _load_bern_point_transformer_weights(
                            model,
                            region_name,
                            checkpoint_prefix="bern_point_transformer_compact",
                        )
                        loaded_key_count = len(loaded_keys)

                    model, hist_df, summary = _train_model(
                        model=model,
                        run_name=(
                            f"no_pretrain_diag_{init_mode}_{kind}_{region_name}_"
                            f"seed{seed}_{time.time_ns()}"
                        ),
                        train_loader=train_loader,
                        val_loader=val_loader,
                        train_ds=train_ds,
                        val_ds=val_ds,
                        y_scaler=y_scaler,
                        device=device,
                        num_epochs=num_epochs,
                        lr=lr,
                    )

                    normal_test_ds = RegionHistoryDataset(
                        x_test_s,
                        y_test_s,
                        t_test,
                        _make_perturbed_meta(meta_test_s, "normal"),
                    )
                    zero_test_ds = RegionHistoryDataset(
                        x_test_s,
                        y_test_s,
                        t_test,
                        _make_perturbed_meta(meta_test_s, "zero"),
                    )
                    shuffle_test_ds = RegionHistoryDataset(
                        x_test_s,
                        y_test_s,
                        t_test,
                        _make_perturbed_meta(
                            meta_test_s,
                            "shuffle",
                            seed=NO_PRETRAIN_SHUFFLE_SEED + int(seed),
                        ),
                    )

                    normal_loader = DataLoader(
                        normal_test_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )
                    zero_loader = DataLoader(
                        zero_test_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )
                    shuffle_loader = DataLoader(
                        shuffle_test_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )

                    normal = _predict_policy_gate_components(model, normal_loader, device, y_scaler)
                    zeroed = _predict_policy_gate_components(model, zero_loader, device, y_scaler)
                    shuffled = _predict_policy_gate_components(model, shuffle_loader, device, y_scaler)

                    row = _policy_perturbation_metric_row(
                        model_name=model_name,
                        kind=kind,
                        seed=seed,
                        region=region_name,
                        normal=normal,
                        zeroed=zeroed,
                        shuffled=shuffled,
                    )
                    row["init_mode"] = init_mode
                    row["bern_pretrained"] = bool(use_bern_pretraining)
                    row["loaded_bern_key_count"] = int(loaded_key_count)
                    rows.append(row)

    by_region_df = pd.DataFrame(rows)
    overall_df = (
        by_region_df
        .groupby(["init_mode", "bern_pretrained", "Model", "kind"], as_index=False)
        .agg(
            gate_mean_normal=("gate_mean_normal", "mean"),
            raw_metadata_correction_mae_db=("raw_metadata_correction_mae_db", "mean"),
            final_applied_correction_mae_db=("final_applied_correction_mae_db", "mean"),
            raw_meta_pred_change_zero_db=("raw_meta_pred_change_zero_db", "mean"),
            raw_meta_pred_change_shuffle_db=("raw_meta_pred_change_shuffle_db", "mean"),
            final_pred_change_zero_db=("final_pred_change_zero_db", "mean"),
            final_pred_change_shuffle_db=("final_pred_change_shuffle_db", "mean"),
            normal_final_mae_db=("normal_final_mae_db", "mean"),
            zero_final_mae_db=("zero_final_mae_db", "mean"),
            shuffle_final_mae_db=("shuffle_final_mae_db", "mean"),
            zero_minus_normal_mae_db=("zero_minus_normal_mae_db", "mean"),
            shuffle_minus_normal_mae_db=("shuffle_minus_normal_mae_db", "mean"),
            loaded_bern_key_count=("loaded_bern_key_count", "mean"),
        )
        .sort_values(["kind", "init_mode"])
        .reset_index(drop=True)
    )
    overall_df["zero_delta_display"] = overall_df["zero_minus_normal_mae_db"].map(lambda x: f"{x:+.4f} dB")
    overall_df["shuffle_delta_display"] = overall_df["shuffle_minus_normal_mae_db"].map(lambda x: f"{x:+.4f} dB")

    compact_cols = [
        "kind",
        "init_mode",
        "gate_mean_normal",
        "raw_metadata_correction_mae_db",
        "final_applied_correction_mae_db",
        "final_pred_change_zero_db",
        "final_pred_change_shuffle_db",
        "normal_final_mae_db",
        "zero_delta_display",
        "shuffle_delta_display",
    ]
    compact_df = overall_df[compact_cols].copy()

    return {
        "overall": overall_df,
        "compact": compact_df,
        "by_region": by_region_df,
        "settings": {
            "seeds": seeds,
            "epochs": num_epochs,
            "batch_size": batch_size,
            "init_modes": init_modes,
        },
    }


without_pretraining_metadata_outputs = run_without_pretraining_metadata_sensitivity_diagnostic()

print("Without-pretraining metadata sensitivity diagnostic - compact summary:")
display(without_pretraining_metadata_outputs["compact"])

print("Full overall diagnostic table:")
display(without_pretraining_metadata_outputs["overall"])

print("By-region diagnostic table:")
display(without_pretraining_metadata_outputs["by_region"])



In [ ]:
# No-fine-tuning diagnostic: does metadata matter more before adapting the Bern backbone to GRAPE?
# This is closer to Marta's wording than the scratch diagnostic.
# It compares:
# 1) zero-shot Bern backbone: no GRAPE training at all; metadata adapter is untrained, so this is only a sanity check.
# 2) frozen Bern backbone: keep the VF Transformer fixed and train only the metadata adapter/gate on GRAPE.

NO_FINETUNE_DIAG_SEEDS = (0,)
NO_FINETUNE_DIAG_EPOCHS = 10
NO_FINETUNE_DIAG_BATCH_SIZE = 128
NO_FINETUNE_DIAG_VARIANTS = (
    "policy_iop_exposure",
    "policy_spatial_rnfl",
    "policy_iop_rnfl",
)
NO_FINETUNE_DIAG_MODES = (
    "zero_shot_bern_no_grape_training",
    "frozen_bern_backbone_train_metadata_only",
)
NO_FINETUNE_SHUFFLE_SEED = 20260627


def _freeze_non_metadata_policy_params(model):
    metadata_param_keywords = ("policy_norm", "film_net", "gate_net")
    for name, param in model.named_parameters():
        param.requires_grad = any(key in name for key in metadata_param_keywords)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, frozen


def run_no_finetuning_metadata_sensitivity_diagnostic(
    *,
    seeds=NO_FINETUNE_DIAG_SEEDS,
    variants=NO_FINETUNE_DIAG_VARIANTS,
    modes=NO_FINETUNE_DIAG_MODES,
    k_list=(2, 3, 4),
    train_mode="consecutive",
    eval_mode="consecutive",
    num_epochs=NO_FINETUNE_DIAG_EPOCHS,
    batch_size=NO_FINETUNE_DIAG_BATCH_SIZE,
    lr=5e-4,
    time_input_mode="input_times",
    film_scale=0.05,
    gate_max=0.25,
):
    required_names = [
        "torch",
        "StandardScaler",
        "DataLoader",
        "RegionHistoryDataset",
        "region_collate",
        "CompactPointTransformerPolicyGateModel",
        "COMPACT_TRANSFORMER_CONFIG",
        "cluster_mapping",
        "compact_transformer_spatial_rnfl_map",
        "hist_module",
        "_prepare_safe_residual_data",
        "_df_to_arrays_with_history",
        "_finite_mask",
        "_apply_mask",
        "_scale_x_list",
        "_iop_exposure_matrix",
        "_scale_static_meta_matrices",
        "_region_rnfl_matrix",
        "_load_bern_point_transformer_weights",
        "_train_model",
        "_predict_policy_gate_components",
        "_make_perturbed_meta",
        "_policy_perturbation_metric_row",
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise NameError(
            "Run the Transformer / metadata diagnostic prerequisite cells first. Missing: "
            + ", ".join(missing)
        )

    train_df_all, val_df_all, test_df_all, td_cols_exp, rnfl_master_cols = _prepare_safe_residual_data(
        k_list=k_list,
        train_mode=train_mode,
        eval_mode=eval_mode,
        time_input_mode=time_input_mode,
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part["meta_sub"] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, meta_train_empty = _apply_mask(
        x_train, y_train, t_train, meta_train_empty, train_mask
    )
    x_val, y_val, t_val, meta_val_empty = _apply_mask(
        x_val, y_val, t_val, meta_val_empty, val_mask
    )
    x_test, y_test, t_test, meta_test_empty = _apply_mask(
        x_test, y_test, t_test, meta_test_empty, test_mask
    )

    train_df_all = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df_all = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df_all = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    x_train_s, x_val_s, x_test_s = _scale_x_list(x_train, x_val, x_test)

    iop_train_raw = _iop_exposure_matrix(train_df_all)
    iop_val_raw = _iop_exposure_matrix(val_df_all)
    iop_test_raw = _iop_exposure_matrix(test_df_all)
    iop_train_s, iop_val_s, iop_test_s, _ = _scale_static_meta_matrices(
        iop_train_raw, iop_val_raw, iop_test_raw
    )

    variant_table = {
        "policy_iop_exposure": "IOP exposure",
        "policy_spatial_rnfl": "spatial RNFL",
        "policy_iop_rnfl": "IOP exposure + spatial RNFL",
    }
    mode_table = {
        "zero_shot_bern_no_grape_training": {
            "load_bern": True,
            "train_on_grape": False,
            "freeze_non_metadata": False,
            "description": "Bern weights loaded; no GRAPE training. Metadata adapter remains untrained.",
        },
        "frozen_bern_backbone_train_metadata_only": {
            "load_bern": True,
            "train_on_grape": True,
            "freeze_non_metadata": True,
            "description": "Bern weights loaded; only metadata adapter/gate trained on GRAPE.",
        },
    }

    unknown_modes = [mode for mode in modes if mode not in mode_table]
    if unknown_modes:
        raise ValueError(f"Unknown modes: {unknown_modes}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []

    print("No-fine-tuning metadata sensitivity diagnostic")
    print("train/val/test:", len(x_train_s), len(x_val_s), len(x_test_s))
    print("Seeds:", seeds, "| epochs for frozen-metadata mode:", num_epochs, "| modes:", modes)

    for mode in modes:
        mode_cfg = mode_table[mode]
        for seed in seeds:
            hist_module._set_seed(int(seed))
            for kind in variants:
                model_name = f"{mode}: {variant_table[kind]}"
                print("\n" + "#" * 96)
                print(f"NO-FINETUNE DIAGNOSTIC | mode={mode} | seed={seed} | {variant_table[kind]}")
                print(mode_cfg["description"])
                print("#" * 96)

                for region_name, idx in cluster_mapping.items():
                    idx = np.asarray(idx, dtype=int)
                    rnfl_cols = compact_transformer_spatial_rnfl_map[region_name]
                    print(
                        f"mode={mode} | seed={seed} | region={region_name} | "
                        f"kind={kind} | RNFL={rnfl_cols}"
                    )

                    rnfl_train_raw = _region_rnfl_matrix(train_df_all, rnfl_cols, rnfl_master_cols)
                    rnfl_val_raw = _region_rnfl_matrix(val_df_all, rnfl_cols, rnfl_master_cols)
                    rnfl_test_raw = _region_rnfl_matrix(test_df_all, rnfl_cols, rnfl_master_cols)
                    rnfl_train_s, rnfl_val_s, rnfl_test_s, _ = _scale_static_meta_matrices(
                        rnfl_train_raw, rnfl_val_raw, rnfl_test_raw
                    )

                    if kind == "policy_iop_exposure":
                        meta_train_s, meta_val_s, meta_test_s = iop_train_s, iop_val_s, iop_test_s
                    elif kind == "policy_spatial_rnfl":
                        meta_train_s, meta_val_s, meta_test_s = rnfl_train_s, rnfl_val_s, rnfl_test_s
                    elif kind == "policy_iop_rnfl":
                        meta_train_s = np.concatenate([iop_train_s, rnfl_train_s], axis=1).astype(np.float32)
                        meta_val_s = np.concatenate([iop_val_s, rnfl_val_s], axis=1).astype(np.float32)
                        meta_test_s = np.concatenate([iop_test_s, rnfl_test_s], axis=1).astype(np.float32)
                    else:
                        raise ValueError(kind)

                    y_scaler = StandardScaler()
                    y_train_s = y_scaler.fit_transform(y_train[:, idx]).astype(np.float32)
                    y_val_s = y_scaler.transform(y_val[:, idx]).astype(np.float32)
                    y_test_s = y_scaler.transform(y_test[:, idx]).astype(np.float32)

                    train_ds = RegionHistoryDataset(x_train_s, y_train_s, t_train, meta_train_s)
                    val_ds = RegionHistoryDataset(x_val_s, y_val_s, t_val, meta_val_s)
                    train_loader = DataLoader(
                        train_ds,
                        batch_size=batch_size,
                        shuffle=True,
                        collate_fn=region_collate,
                    )
                    val_loader = DataLoader(
                        val_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )

                    model = CompactPointTransformerPolicyGateModel(
                        output_dim=len(idx),
                        point_indices=idx.tolist(),
                        policy_meta_dim=meta_train_s.shape[1],
                        film_scale=film_scale,
                        gate_max=gate_max,
                        **COMPACT_TRANSFORMER_CONFIG,
                    )

                    loaded_key_count = 0
                    if mode_cfg["load_bern"]:
                        model, loaded_keys = _load_bern_point_transformer_weights(
                            model,
                            region_name,
                            checkpoint_prefix="bern_point_transformer_compact",
                        )
                        loaded_key_count = len(loaded_keys)

                    trainable_params_before = sum(p.numel() for p in model.parameters() if p.requires_grad)
                    frozen_params_before = sum(p.numel() for p in model.parameters() if not p.requires_grad)
                    trainable_params_after = trainable_params_before
                    frozen_params_after = frozen_params_before

                    if mode_cfg["freeze_non_metadata"]:
                        trainable_params_after, frozen_params_after = _freeze_non_metadata_policy_params(model)

                    if mode_cfg["train_on_grape"]:
                        model, hist_df, summary = _train_model(
                            model=model,
                            run_name=(
                                f"no_finetune_diag_{mode}_{kind}_{region_name}_"
                                f"seed{seed}_{time.time_ns()}"
                            ),
                            train_loader=train_loader,
                            val_loader=val_loader,
                            train_ds=train_ds,
                            val_ds=val_ds,
                            y_scaler=y_scaler,
                            device=device,
                            num_epochs=num_epochs,
                            lr=lr,
                        )
                        best_val_mae_db = float(summary.get("best_val_mae_db", np.nan))
                    else:
                        model = model.to(device)
                        best_val_mae_db = np.nan

                    normal_test_ds = RegionHistoryDataset(
                        x_test_s,
                        y_test_s,
                        t_test,
                        _make_perturbed_meta(meta_test_s, "normal"),
                    )
                    zero_test_ds = RegionHistoryDataset(
                        x_test_s,
                        y_test_s,
                        t_test,
                        _make_perturbed_meta(meta_test_s, "zero"),
                    )
                    shuffle_test_ds = RegionHistoryDataset(
                        x_test_s,
                        y_test_s,
                        t_test,
                        _make_perturbed_meta(
                            meta_test_s,
                            "shuffle",
                            seed=NO_FINETUNE_SHUFFLE_SEED + int(seed),
                        ),
                    )

                    normal_loader = DataLoader(
                        normal_test_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )
                    zero_loader = DataLoader(
                        zero_test_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )
                    shuffle_loader = DataLoader(
                        shuffle_test_ds,
                        batch_size=batch_size,
                        shuffle=False,
                        collate_fn=region_collate,
                    )

                    normal = _predict_policy_gate_components(model, normal_loader, device, y_scaler)
                    zeroed = _predict_policy_gate_components(model, zero_loader, device, y_scaler)
                    shuffled = _predict_policy_gate_components(model, shuffle_loader, device, y_scaler)

                    row = _policy_perturbation_metric_row(
                        model_name=model_name,
                        kind=kind,
                        seed=seed,
                        region=region_name,
                        normal=normal,
                        zeroed=zeroed,
                        shuffled=shuffled,
                    )
                    row["mode"] = mode
                    row["bern_pretrained"] = bool(mode_cfg["load_bern"])
                    row["trained_on_grape"] = bool(mode_cfg["train_on_grape"])
                    row["frozen_non_metadata"] = bool(mode_cfg["freeze_non_metadata"])
                    row["loaded_bern_key_count"] = int(loaded_key_count)
                    row["trainable_params_before_freeze"] = int(trainable_params_before)
                    row["frozen_params_before_freeze"] = int(frozen_params_before)
                    row["trainable_params_after_freeze"] = int(trainable_params_after)
                    row["frozen_params_after_freeze"] = int(frozen_params_after)
                    row["best_val_mae_db"] = best_val_mae_db
                    rows.append(row)

    by_region_df = pd.DataFrame(rows)
    overall_df = (
        by_region_df
        .groupby([
            "mode",
            "bern_pretrained",
            "trained_on_grape",
            "frozen_non_metadata",
            "Model",
            "kind",
        ], as_index=False)
        .agg(
            gate_mean_normal=("gate_mean_normal", "mean"),
            raw_metadata_correction_mae_db=("raw_metadata_correction_mae_db", "mean"),
            final_applied_correction_mae_db=("final_applied_correction_mae_db", "mean"),
            raw_meta_pred_change_zero_db=("raw_meta_pred_change_zero_db", "mean"),
            raw_meta_pred_change_shuffle_db=("raw_meta_pred_change_shuffle_db", "mean"),
            final_pred_change_zero_db=("final_pred_change_zero_db", "mean"),
            final_pred_change_shuffle_db=("final_pred_change_shuffle_db", "mean"),
            normal_final_mae_db=("normal_final_mae_db", "mean"),
            zero_final_mae_db=("zero_final_mae_db", "mean"),
            shuffle_final_mae_db=("shuffle_final_mae_db", "mean"),
            zero_minus_normal_mae_db=("zero_minus_normal_mae_db", "mean"),
            shuffle_minus_normal_mae_db=("shuffle_minus_normal_mae_db", "mean"),
            loaded_bern_key_count=("loaded_bern_key_count", "mean"),
            trainable_params_after_freeze=("trainable_params_after_freeze", "mean"),
            best_val_mae_db=("best_val_mae_db", "mean"),
        )
        .sort_values(["kind", "mode"])
        .reset_index(drop=True)
    )
    overall_df["zero_delta_display"] = overall_df["zero_minus_normal_mae_db"].map(lambda x: f"{x:+.4f} dB")
    overall_df["shuffle_delta_display"] = overall_df["shuffle_minus_normal_mae_db"].map(lambda x: f"{x:+.4f} dB")

    compact_cols = [
        "kind",
        "mode",
        "trained_on_grape",
        "frozen_non_metadata",
        "gate_mean_normal",
        "raw_metadata_correction_mae_db",
        "final_applied_correction_mae_db",
        "final_pred_change_zero_db",
        "final_pred_change_shuffle_db",
        "normal_final_mae_db",
        "zero_delta_display",
        "shuffle_delta_display",
    ]
    compact_df = overall_df[compact_cols].copy()

    return {
        "overall": overall_df,
        "compact": compact_df,
        "by_region": by_region_df,
        "settings": {
            "seeds": seeds,
            "epochs_for_frozen_metadata": num_epochs,
            "batch_size": batch_size,
            "modes": modes,
        },
    }


no_finetune_metadata_outputs = run_no_finetuning_metadata_sensitivity_diagnostic()

print("No-fine-tuning metadata sensitivity diagnostic - compact summary:")
display(no_finetune_metadata_outputs["compact"])

print("Full overall diagnostic table:")
display(no_finetune_metadata_outputs["overall"])

print("By-region diagnostic table:")
display(no_finetune_metadata_outputs["by_region"])



In [ ]:
# Flooring and 1VF-to-1VF RNFL diagnostic
# Marta asked whether RNFL might have a clearer effect in a simpler 1VF-to-1VF setting,
# and whether floor effects may hide progression. This cell is data-only: no model training.

import numpy as np
import pandas as pd

FLOOR_DIAG_THRESHOLD_DB = -25.0
FLOOR_DIAG_NEXT_TARGET_ONLY = True
RNFL_DIAG_COLS = ["RNFL_Mean", "RNFL_S", "RNFL_N", "RNFL_I", "RNFL_T"]


def _build_one_vf_next_pairs_for_diagnostic(df, td_cols, *, target_mode="next"):
    """One historical VF -> future VF pairs, using baseline as the single input."""
    if target_mode not in {"next", "all_future"}:
        raise ValueError("target_mode must be 'next' or 'all_future'")

    rows = []
    for eye_id, g in df.groupby("eye_id"):
        g = g.sort_values(["Time_from_Baseline", "VisitN"]).reset_index(drop=True)
        if len(g) < 2:
            continue
        baseline = g.iloc[0]
        target_positions = [1] if target_mode == "next" else range(1, len(g))
        for target_pos in target_positions:
            target = g.iloc[target_pos]
            delta_t = float(target["Time_from_Baseline"] - baseline["Time_from_Baseline"])
            rows.append({
                "eye_id": eye_id,
                "k": 1,
                "visit_in_last": int(baseline["VisitN"]),
                "visit_out": int(target["VisitN"]),
                "time_in_last": float(baseline["Time_from_Baseline"]),
                "time_out": float(target["Time_from_Baseline"]),
                "delta_t": delta_t,
                "input_times": np.array([delta_t], dtype=np.float32),
                "X": baseline[td_cols].to_numpy(dtype=np.float32)[None, :],
                "Y": target[td_cols].to_numpy(dtype=np.float32),
                "meta_sub": np.zeros((0,), dtype=np.float32),
            })
    return pd.DataFrame(rows)


def _rnfl_dataframe_from_static(seq_df, rnfl_cols=RNFL_DIAG_COLS):
    values = np.stack([np.asarray(v, dtype=np.float32) for v in seq_df["rnfl_static"].values])
    return pd.DataFrame(values, columns=rnfl_cols, index=seq_df.index)


def _add_one_vf_outcomes(seq_df, td_cols, floor_threshold=FLOOR_DIAG_THRESHOLD_DB):
    out = seq_df.copy()
    x = np.stack([np.asarray(v, dtype=np.float32)[-1] for v in out["X"].values], axis=0)
    y = np.stack([np.asarray(v, dtype=np.float32) for v in out["Y"].values], axis=0)
    change = y - x

    out["input_mean_td_db"] = np.nanmean(x, axis=1)
    out["target_mean_td_db"] = np.nanmean(y, axis=1)
    out["mean_td_change_db"] = np.nanmean(change, axis=1)
    out["mean_deterioration_db"] = np.nanmean(np.clip(-change, 0, None), axis=1)
    out["input_floor_fraction"] = np.nanmean(x <= floor_threshold, axis=1)
    out["target_floor_fraction"] = np.nanmean(y <= floor_threshold, axis=1)
    out["new_floor_fraction"] = np.nanmean((x > floor_threshold) & (y <= floor_threshold), axis=1)
    out["worsened_point_fraction_1db"] = np.nanmean(change <= -1.0, axis=1)

    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        out[f"{region_name}_target_mean_td_db"] = np.nanmean(y[:, idx], axis=1)
        out[f"{region_name}_mean_td_change_db"] = np.nanmean(change[:, idx], axis=1)
        out[f"{region_name}_mean_deterioration_db"] = np.nanmean(np.clip(-change[:, idx], 0, None), axis=1)
        out[f"{region_name}_target_floor_fraction"] = np.nanmean(y[:, idx] <= floor_threshold, axis=1)
    return out


def _split_one_vf_table(one_vf_df):
    train_df, val_df, test_df = hist_module._split_by_eye(
        one_vf_df,
        meta_train_grape,
        meta_val_grape,
        meta_test_grape,
    )
    parts = []
    for split_name, df_part in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df),
    ]:
        part = df_part.copy()
        part["split"] = split_name
        parts.append(part)
    return pd.concat(parts, ignore_index=True)


def _safe_corr(df, x_col, y_col, method="spearman"):
    sub = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(sub) < 5 or sub[x_col].nunique() < 2 or sub[y_col].nunique() < 2:
        return np.nan
    return float(sub[x_col].corr(sub[y_col], method=method))


def run_flooring_and_one_vf_rnfl_diagnostic(target_mode="next"):
    required = [
        "hist_module",
        "grape_longitudinal_td_aligned_df",
        "meta_train_grape",
        "meta_val_grape",
        "meta_test_grape",
        "_attach_history_metadata",
        "cluster_mapping",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise NameError("Run prerequisite data-preparation cells first. Missing: " + ", ".join(missing))

    df_exp, td_cols_exp = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    one_vf = _build_one_vf_next_pairs_for_diagnostic(df_exp, td_cols_exp, target_mode=target_mode)
    one_vf = _attach_history_metadata(one_vf, grape_longitudinal_td_aligned_df, RNFL_DIAG_COLS, max_k=1)
    one_vf = _add_one_vf_outcomes(one_vf, td_cols_exp)
    one_vf = _split_one_vf_table(one_vf)

    rnfl_df = _rnfl_dataframe_from_static(one_vf, RNFL_DIAG_COLS)
    one_vf = pd.concat([one_vf.reset_index(drop=True), rnfl_df.reset_index(drop=True)], axis=1)

    floor_rows = []
    for split_name, df_part in list(one_vf.groupby("split")) + [("all", one_vf)]:
        floor_rows.append({
            "split": split_name,
            "samples": len(df_part),
            "patient_eyes": int(df_part["eye_id"].nunique()),
            "mean_delta_t_years": float(df_part["delta_t"].mean()),
            "input_mean_td_db": float(df_part["input_mean_td_db"].mean()),
            "target_mean_td_db": float(df_part["target_mean_td_db"].mean()),
            "mean_td_change_db": float(df_part["mean_td_change_db"].mean()),
            "input_floor_fraction": float(df_part["input_floor_fraction"].mean()),
            "target_floor_fraction": float(df_part["target_floor_fraction"].mean()),
            "new_floor_fraction": float(df_part["new_floor_fraction"].mean()),
            "worsened_point_fraction_1db": float(df_part["worsened_point_fraction_1db"].mean()),
        })
    floor_summary = pd.DataFrame(floor_rows)

    overall_corr_rows = []
    outcomes = [
        "target_mean_td_db",
        "mean_td_change_db",
        "mean_deterioration_db",
        "target_floor_fraction",
        "worsened_point_fraction_1db",
    ]
    for split_name, df_part in list(one_vf.groupby("split")) + [("all", one_vf)]:
        for rnfl_col in RNFL_DIAG_COLS:
            for outcome in outcomes:
                overall_corr_rows.append({
                    "split": split_name,
                    "rnfl_feature": rnfl_col,
                    "outcome": outcome,
                    "n_finite": int(df_part[[rnfl_col, outcome]].dropna().shape[0]),
                    "spearman_corr": _safe_corr(df_part, rnfl_col, outcome, method="spearman"),
                    "pearson_corr": _safe_corr(df_part, rnfl_col, outcome, method="pearson"),
                })
    rnfl_signal_overall = pd.DataFrame(overall_corr_rows)
    rnfl_signal_overall["abs_spearman_corr"] = rnfl_signal_overall["spearman_corr"].abs()

    region_corr_rows = []
    for split_name, df_part in list(one_vf.groupby("split")) + [("all", one_vf)]:
        for region_name in cluster_mapping.keys():
            for rnfl_col in RNFL_DIAG_COLS:
                for outcome_suffix in ["target_mean_td_db", "mean_td_change_db", "mean_deterioration_db", "target_floor_fraction"]:
                    outcome = f"{region_name}_{outcome_suffix}"
                    region_corr_rows.append({
                        "split": split_name,
                        "Region": region_name,
                        "rnfl_feature": rnfl_col,
                        "outcome": outcome_suffix,
                        "n_finite": int(df_part[[rnfl_col, outcome]].dropna().shape[0]),
                        "spearman_corr": _safe_corr(df_part, rnfl_col, outcome, method="spearman"),
                        "pearson_corr": _safe_corr(df_part, rnfl_col, outcome, method="pearson"),
                    })
    rnfl_signal_by_region = pd.DataFrame(region_corr_rows)
    rnfl_signal_by_region["abs_spearman_corr"] = rnfl_signal_by_region["spearman_corr"].abs()

    strongest_overall = (
        rnfl_signal_overall[rnfl_signal_overall["split"].eq("all")]
        .sort_values("abs_spearman_corr", ascending=False)
        .head(12)
        .reset_index(drop=True)
    )
    strongest_region = (
        rnfl_signal_by_region[rnfl_signal_by_region["split"].eq("all")]
        .sort_values("abs_spearman_corr", ascending=False)
        .head(18)
        .reset_index(drop=True)
    )

    interpretation = pd.DataFrame([
        {
            "Question": "Does 1VF-to-1VF give a cleaner RNFL setting?",
            "What this checks": "Baseline VF is the only VF input; RNFL is measured at the same baseline visit; target is the next VF.",
            "Readout": "Look for RNFL correlations with future TD or deterioration in strongest_overall / strongest_region.",
        },
        {
            "Question": "Could floor effects hide progression?",
            "What this checks": f"Fraction of points already <= {FLOOR_DIAG_THRESHOLD_DB:.0f} dB at input/target.",
            "Readout": "High floor_fraction means some locations have little room to worsen, so RNFL/progression signal can be censored.",
        },
        {
            "Question": "Is this a model result?",
            "What this checks": "No. This is a data diagnostic before training a new 1VF-to-1VF RNFL model.",
            "Readout": "Use it to decide whether a full RNFL-focused experiment is worth running.",
        },
    ])

    return {
        "one_vf_table": one_vf,
        "floor_summary": floor_summary,
        "rnfl_signal_overall": rnfl_signal_overall,
        "rnfl_signal_by_region": rnfl_signal_by_region,
        "strongest_overall": strongest_overall,
        "strongest_region": strongest_region,
        "interpretation": interpretation,
        "target_mode": target_mode,
        "floor_threshold_db": FLOOR_DIAG_THRESHOLD_DB,
    }


one_vf_rnfl_floor_outputs = run_flooring_and_one_vf_rnfl_diagnostic(
    target_mode="next" if FLOOR_DIAG_NEXT_TARGET_ONLY else "all_future"
)

print("1VF-to-1VF / flooring diagnostic summary:")
display(one_vf_rnfl_floor_outputs["floor_summary"])

print("Strongest overall RNFL associations:")
display(one_vf_rnfl_floor_outputs["strongest_overall"][[
    "rnfl_feature",
    "outcome",
    "n_finite",
    "spearman_corr",
    "pearson_corr",
]])

print("Strongest region-level RNFL associations:")
display(one_vf_rnfl_floor_outputs["strongest_region"][[
    "Region",
    "rnfl_feature",
    "outcome",
    "n_finite",
    "spearman_corr",
    "pearson_corr",
]])

print("How to interpret this cell:")
display(one_vf_rnfl_floor_outputs["interpretation"])


In [ ]:
# 1VF-to-1VF simple RNFL model check
# This follows the flooring/RNFL diagnostic with a small, clean model experiment.
# Goal: test whether baseline RNFL adds predictive value when the input is only one VF.
# No Transformer, no gate, no ensemble. Validation selects Ridge alpha; test remains held out.

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

required_objects = ["one_vf_rnfl_floor_outputs", "cluster_mapping"]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise NameError(
        "Run the flooring / 1VF-to-1VF RNFL diagnostic cell first. Missing: "
        + ", ".join(missing)
    )

ONE_VF_RNFL_ALPHAS = np.logspace(-4, 4, 17)
ONE_VF_RNFL_COLS = [c for c in RNFL_DIAG_COLS if c in one_vf_rnfl_floor_outputs["one_vf_table"].columns]


def _stack_one_vf_arrays(df):
    x_vf = np.stack([np.asarray(v, dtype=np.float32)[-1] for v in df["X"].values], axis=0)
    y = np.stack([np.asarray(v, dtype=np.float32) for v in df["Y"].values], axis=0)
    rnfl = df[ONE_VF_RNFL_COLS].to_numpy(dtype=np.float32)
    delta_t = df[["delta_t"]].to_numpy(dtype=np.float32)
    return x_vf, y, rnfl, delta_t


def _finite_rows(*arrays):
    mask = np.ones(len(arrays[0]), dtype=bool)
    for arr in arrays:
        arr = np.asarray(arr)
        if arr.ndim == 1:
            mask &= np.isfinite(arr)
        else:
            mask &= np.isfinite(arr).all(axis=tuple(range(1, arr.ndim)))
    return mask


def _make_feature_matrix(x_vf, rnfl, delta_t, feature_mode):
    if feature_mode == "vf_only":
        return np.concatenate([x_vf, delta_t], axis=1)
    if feature_mode == "rnfl_only":
        return np.concatenate([rnfl, delta_t], axis=1)
    if feature_mode == "vf_plus_rnfl":
        return np.concatenate([x_vf, rnfl, delta_t], axis=1)
    raise ValueError(feature_mode)


def _mae_db(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def _sample_sd_db(y_true, y_pred):
    sample_mae = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), axis=1)
    return float(np.std(sample_mae, ddof=1)) if len(sample_mae) > 1 else 0.0


def _fit_ridge_with_val_selection(train_df, val_df, test_df, *, feature_mode, target_mode):
    x_train_vf, y_train_abs, rnfl_train, dt_train = _stack_one_vf_arrays(train_df)
    x_val_vf, y_val_abs, rnfl_val, dt_val = _stack_one_vf_arrays(val_df)
    x_test_vf, y_test_abs, rnfl_test, dt_test = _stack_one_vf_arrays(test_df)

    if target_mode == "absolute_td":
        y_train = y_train_abs
        y_val = y_val_abs
        y_test = y_test_abs
        to_abs_pred = lambda pred, x_ref: pred
    elif target_mode == "td_change":
        y_train = y_train_abs - x_train_vf
        y_val = y_val_abs - x_val_vf
        y_test = y_test_abs - x_test_vf
        to_abs_pred = lambda pred, x_ref: x_ref + pred
    else:
        raise ValueError(target_mode)

    x_train = _make_feature_matrix(x_train_vf, rnfl_train, dt_train, feature_mode)
    x_val = _make_feature_matrix(x_val_vf, rnfl_val, dt_val, feature_mode)
    x_test = _make_feature_matrix(x_test_vf, rnfl_test, dt_test, feature_mode)

    train_mask = _finite_rows(x_train, y_train)
    val_mask = _finite_rows(x_val, y_val)
    test_mask = _finite_rows(x_test, y_test)

    x_train, y_train = x_train[train_mask], y_train[train_mask]
    x_val, y_val = x_val[val_mask], y_val[val_mask]
    x_test, y_test = x_test[test_mask], y_test[test_mask]
    x_test_vf_kept = x_test_vf[test_mask]
    y_test_abs_kept = y_test_abs[test_mask]

    val_rows = []
    for alpha in ONE_VF_RNFL_ALPHAS:
        model = make_pipeline(StandardScaler(), Ridge(alpha=float(alpha)))
        model.fit(x_train, y_train)
        val_pred = model.predict(x_val)
        val_rows.append({
            "alpha": float(alpha),
            "val_mae_db": _mae_db(y_val, val_pred),
        })

    val_df_alpha = pd.DataFrame(val_rows).sort_values("val_mae_db", ascending=True).reset_index(drop=True)
    best_alpha = float(val_df_alpha.loc[0, "alpha"])

    # Refit on train + validation after selecting alpha. Test remains untouched.
    x_fit = np.concatenate([x_train, x_val], axis=0)
    y_fit = np.concatenate([y_train, y_val], axis=0)
    final_model = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
    final_model.fit(x_fit, y_fit)

    pred_target_space = final_model.predict(x_test)
    pred_abs = to_abs_pred(pred_target_space, x_test_vf_kept)

    return {
        "best_alpha": best_alpha,
        "best_val_mae_db": float(val_df_alpha.loc[0, "val_mae_db"]),
        "test_mae_db": _mae_db(y_test_abs_kept, pred_abs),
        "test_sample_sd_db": _sample_sd_db(y_test_abs_kept, pred_abs),
        "test_samples": int(len(y_test_abs_kept)),
        "test_patient_eyes": int(test_df.loc[test_mask, "eye_id"].nunique()),
        "pred_abs": pred_abs,
        "true_abs": y_test_abs_kept,
    }


def run_one_vf_rnfl_simple_model_check():
    one_vf = one_vf_rnfl_floor_outputs["one_vf_table"].copy()
    # Compare all models on the same RNFL-available rows; otherwise VF-only and VF+RNFL
    # may evaluate on different test eyes.
    rnfl_available_mask = np.isfinite(one_vf[ONE_VF_RNFL_COLS].to_numpy(dtype=np.float32)).all(axis=1)
    one_vf = one_vf.loc[rnfl_available_mask].reset_index(drop=True)
    split_dfs = {split: df.reset_index(drop=True) for split, df in one_vf.groupby("split")}
    required_splits = {"train", "validation", "test"}
    missing_splits = required_splits.difference(split_dfs)
    if missing_splits:
        raise ValueError(f"Missing required splits: {sorted(missing_splits)}")

    train_df = split_dfs["train"]
    val_df = split_dfs["validation"]
    test_df = split_dfs["test"]
    x_test_vf, y_test_abs, _, _ = _stack_one_vf_arrays(test_df)

    rows = []
    predictions = {}

    no_change_pred = x_test_vf.copy()
    predictions["No-change baseline"] = no_change_pred
    rows.append({
        "Model": "No-change baseline",
        "feature_mode": "last_vf_only",
        "target_mode": "copy_last_vf",
        "best_alpha": np.nan,
        "best_val_mae_db": np.nan,
        "test_mae_db": _mae_db(y_test_abs, no_change_pred),
        "test_sample_sd_db": _sample_sd_db(y_test_abs, no_change_pred),
        "test_samples": int(len(test_df)),
        "test_patient_eyes": int(test_df["eye_id"].nunique()),
    })

    feature_labels = {
        "vf_only": "Ridge: baseline VF only",
        "rnfl_only": "Ridge: RNFL only",
        "vf_plus_rnfl": "Ridge: baseline VF + RNFL",
    }
    target_labels = {
        "absolute_td": "predict future TD",
        "td_change": "predict TD change",
    }

    for target_mode in ["absolute_td", "td_change"]:
        for feature_mode in ["vf_only", "rnfl_only", "vf_plus_rnfl"]:
            result = _fit_ridge_with_val_selection(
                train_df,
                val_df,
                test_df,
                feature_mode=feature_mode,
                target_mode=target_mode,
            )
            model_name = f"{feature_labels[feature_mode]} ({target_labels[target_mode]})"
            predictions[model_name] = result.pop("pred_abs")
            true_abs = result.pop("true_abs")
            rows.append({
                "Model": model_name,
                "feature_mode": feature_mode,
                "target_mode": target_mode,
                **result,
            })

    results = pd.DataFrame(rows).sort_values("test_mae_db", ascending=True).reset_index(drop=True)
    no_change_mae = float(results.loc[results["Model"].eq("No-change baseline"), "test_mae_db"].iloc[0])
    vf_abs_mae = float(results.loc[
        (results["feature_mode"].eq("vf_only")) & (results["target_mode"].eq("absolute_td")),
        "test_mae_db",
    ].iloc[0])
    vf_rnfl_abs_mae = float(results.loc[
        (results["feature_mode"].eq("vf_plus_rnfl")) & (results["target_mode"].eq("absolute_td")),
        "test_mae_db",
    ].iloc[0])
    vf_change_mae = float(results.loc[
        (results["feature_mode"].eq("vf_only")) & (results["target_mode"].eq("td_change")),
        "test_mae_db",
    ].iloc[0])
    vf_rnfl_change_mae = float(results.loc[
        (results["feature_mode"].eq("vf_plus_rnfl")) & (results["target_mode"].eq("td_change")),
        "test_mae_db",
    ].iloc[0])

    comparisons = pd.DataFrame([
        {
            "Comparison": "Absolute target: add RNFL to baseline VF",
            "base_model": "Ridge: baseline VF only",
            "meta_model": "Ridge: baseline VF + RNFL",
            "base_test_mae_db": vf_abs_mae,
            "meta_test_mae_db": vf_rnfl_abs_mae,
            "delta_meta_minus_base_db": vf_rnfl_abs_mae - vf_abs_mae,
        },
        {
            "Comparison": "Change target: add RNFL to baseline VF",
            "base_model": "Ridge: baseline VF only",
            "meta_model": "Ridge: baseline VF + RNFL",
            "base_test_mae_db": vf_change_mae,
            "meta_test_mae_db": vf_rnfl_change_mae,
            "delta_meta_minus_base_db": vf_rnfl_change_mae - vf_change_mae,
        },
        {
            "Comparison": "No-change vs best simple model",
            "base_model": "No-change baseline",
            "meta_model": results.loc[0, "Model"],
            "base_test_mae_db": no_change_mae,
            "meta_test_mae_db": float(results.loc[0, "test_mae_db"]),
            "delta_meta_minus_base_db": float(results.loc[0, "test_mae_db"] - no_change_mae),
        },
    ])
    comparisons["delta_display"] = comparisons["delta_meta_minus_base_db"].map(lambda x: f"{x:+.4f} dB")

    region_rows = []
    best_name = str(results.loc[0, "Model"])
    best_pred = predictions[best_name]
    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        region_rows.append({
            "Region": region_name,
            "best_model": best_name,
            "best_model_mae_db": _mae_db(y_test_abs[:, idx], best_pred[:, idx]),
            "no_change_mae_db": _mae_db(y_test_abs[:, idx], no_change_pred[:, idx]),
            "delta_best_vs_no_change_db": _mae_db(y_test_abs[:, idx], best_pred[:, idx]) - _mae_db(y_test_abs[:, idx], no_change_pred[:, idx]),
            "points": int(len(idx)),
        })
    region_results = pd.DataFrame(region_rows)
    region_results["delta_display"] = region_results["delta_best_vs_no_change_db"].map(lambda x: f"{x:+.4f} dB")

    interpretation = pd.DataFrame([
        {
            "Question": "Does RNFL add incremental value in clean 1VF-to-1VF prediction?",
            "Readout": "Check delta_meta_minus_base_db. Negative means RNFL improved test MAE beyond baseline VF.",
        },
        {
            "Question": "Is RNFL mainly severity information or progression information?",
            "Readout": "Compare absolute-target and change-target rows. If RNFL helps absolute more than change, it mostly tracks disease severity.",
        },
        {
            "Question": "Is this final evidence?",
            "Readout": "No. This is a lightweight check. If useful, repeat with patient-eye cross-validation before making a thesis-level claim.",
        },
    ])

    return {
        "results": results,
        "comparisons": comparisons,
        "region_results": region_results,
        "interpretation": interpretation,
        "predictions": predictions,
        "true_abs": y_test_abs,
    }


one_vf_rnfl_simple_outputs = run_one_vf_rnfl_simple_model_check()

print("1VF-to-1VF simple model check - test MAE:")
display(one_vf_rnfl_simple_outputs["results"][[
    "Model",
    "feature_mode",
    "target_mode",
    "best_alpha",
    "best_val_mae_db",
    "test_mae_db",
    "test_sample_sd_db",
    "test_samples",
    "test_patient_eyes",
]])

print("Incremental RNFL comparisons:")
display(one_vf_rnfl_simple_outputs["comparisons"])

print("Best simple model by region:")
display(one_vf_rnfl_simple_outputs["region_results"])

print("How to read this diagnostic:")
display(one_vf_rnfl_simple_outputs["interpretation"])


## Macular Fine-tuning Diagnostic

Marta asked why the Macular region became worse after Bern -> GRAPE fine-tuning. This diagnostic does not train a model. It checks whether Macular has a different distribution or progression pattern compared with other regions, and whether Bern and GRAPE differ strongly for this region.


In [ ]:
# ---------------------------------------------------------------------------
# Macular fine-tuning diagnostic: data distribution and Bern -> GRAPE shift
# ---------------------------------------------------------------------------
# Purpose:
# Marta asked why Macular worsened during Bern -> GRAPE fine-tuning.
# This cell does not train a model. It checks whether Macular is special in the data:
# 1) baseline TD distribution,
# 2) one-step progression magnitude,
# 3) floor / near-normal fractions,
# 4) Bern vs GRAPE domain shift by anatomical region.

import numpy as np
import pandas as pd
from pathlib import Path

MACULA_FLOOR_THRESHOLD_DB = -25.0
MACULA_NEAR_NORMAL_THRESHOLD_DB = -2.0
MACULA_SEVERE_THRESHOLD_DB = -12.0


def _sorted_td_cols(df):
    cols = [c for c in df.columns if str(c).startswith("TD_")]
    return sorted(cols, key=lambda x: int(str(x).split("_")[1]))


def _prepare_grape_visit_table():
    if "grape_longitudinal_td_aligned_df" not in globals():
        raise NameError("Missing grape_longitudinal_td_aligned_df. Run the GRAPE data preparation cells first.")

    if "hist_module" in globals() and hasattr(hist_module, "_prepare_df"):
        df, td_cols = hist_module._prepare_df(grape_longitudinal_td_aligned_df)
    else:
        df = grape_longitudinal_td_aligned_df.copy()
        td_cols = _sorted_td_cols(df)
        if "eye_id" not in df.columns:
            df["eye_id"] = (
                df["Subject Number"].astype(str).str.strip()
                + "_"
                + df["Laterality"].astype(str).str.strip()
            )
        if "VisitN" not in df.columns:
            df["VisitN"] = pd.to_numeric(df["Visit Number"], errors="coerce")
        if "Time_from_Baseline" not in df.columns:
            df["Time_from_Baseline"] = pd.to_numeric(df["Interval Years"], errors="coerce")

    df = df.copy()
    for col in td_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["eye_id", "VisitN", "Time_from_Baseline"] + td_cols).copy()
    df = df.sort_values(["eye_id", "Time_from_Baseline", "VisitN"]).reset_index(drop=True)
    return df, td_cols


def _prepare_bern_visit_table():
    # Bern CSV is the source used for Bern pretraining. It already contains TD_1 ... TD_59.
    if "bern_path" in globals():
        path = Path(bern_path)
    else:
        path = Path("./data/Bern/bern_59_master_dataset_splitIDs.csv")
    if not path.exists():
        raise FileNotFoundError(f"Could not find Bern CSV: {path}")

    df = pd.read_csv(path, low_memory=False).copy()
    td_cols = _sorted_td_cols(df)
    if len(td_cols) != 59:
        raise ValueError(f"Expected 59 Bern TD columns, got {len(td_cols)}")

    if "patient_eye" in df.columns:
        df["eye_id"] = df["patient_eye"].astype(str)
    elif {"patient_id", "Eye"}.issubset(df.columns):
        df["eye_id"] = df["patient_id"].astype(str) + "_" + df["Eye"].astype(str)
    else:
        raise KeyError("Bern table needs patient_eye or patient_id/Eye columns.")

    if "Examination" in df.columns:
        df["Examination"] = pd.to_datetime(df["Examination"], errors="coerce")
        df = df.dropna(subset=["eye_id", "Examination"]).copy()
        df = df.sort_values(["eye_id", "Examination"]).reset_index(drop=True)
        df["VisitN"] = df.groupby("eye_id").cumcount() + 1
        baseline_exam = df.groupby("eye_id")["Examination"].transform("min")
        df["Time_from_Baseline"] = (df["Examination"] - baseline_exam).dt.days / 365.25
    elif "VisitN" in df.columns and "Time_from_Baseline" in df.columns:
        df["VisitN"] = pd.to_numeric(df["VisitN"], errors="coerce")
        df["Time_from_Baseline"] = pd.to_numeric(df["Time_from_Baseline"], errors="coerce")
    else:
        raise KeyError("Bern table needs Examination or VisitN/Time_from_Baseline columns.")

    for col in td_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["eye_id", "VisitN", "Time_from_Baseline"] + td_cols).copy()
    df = df.sort_values(["eye_id", "Time_from_Baseline", "VisitN"]).reset_index(drop=True)
    return df, td_cols


def _region_visit_summary(df, td_cols, dataset_name):
    rows = []
    for region_name, idx in cluster_mapping.items():
        region_cols = [td_cols[i] for i in idx]
        vals = df[region_cols].to_numpy(dtype=np.float32)
        rows.append({
            "dataset": dataset_name,
            "Region": region_name,
            "visits": int(len(df)),
            "patient_eyes": int(df["eye_id"].nunique()),
            "mean_td_db": float(np.nanmean(vals)),
            "sd_td_db": float(np.nanstd(vals)),
            "point_variability_db": float(np.nanmean(np.nanstd(vals, axis=1))),
            "floor_fraction_leq_-25": float(np.nanmean(vals <= MACULA_FLOOR_THRESHOLD_DB)),
            "near_normal_fraction_geq_-2": float(np.nanmean(vals >= MACULA_NEAR_NORMAL_THRESHOLD_DB)),
            "severe_fraction_leq_-12": float(np.nanmean(vals <= MACULA_SEVERE_THRESHOLD_DB)),
            "points": int(len(idx)),
        })
    return pd.DataFrame(rows)


def _baseline_next_pairs(df, td_cols):
    rows = []
    for eye_id, g in df.groupby("eye_id"):
        g = g.sort_values(["Time_from_Baseline", "VisitN"]).reset_index(drop=True)
        if len(g) < 2:
            continue
        b = g.iloc[0]
        n = g.iloc[1]
        rows.append({
            "eye_id": eye_id,
            "delta_t_years": float(n["Time_from_Baseline"] - b["Time_from_Baseline"]),
            "baseline": b[td_cols].to_numpy(dtype=np.float32),
            "next": n[td_cols].to_numpy(dtype=np.float32),
        })
    return pd.DataFrame(rows)


def _region_pair_summary(pair_df, td_cols, dataset_name):
    rows = []
    if len(pair_df) == 0:
        return pd.DataFrame(rows)
    baseline_all = np.stack(pair_df["baseline"].to_numpy())
    next_all = np.stack(pair_df["next"].to_numpy())
    change_all = next_all - baseline_all

    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        base = baseline_all[:, idx]
        nxt = next_all[:, idx]
        change = change_all[:, idx]
        rows.append({
            "dataset": dataset_name,
            "Region": region_name,
            "pairs": int(len(pair_df)),
            "patient_eyes": int(pair_df["eye_id"].nunique()),
            "mean_delta_t_years": float(pair_df["delta_t_years"].mean()),
            "baseline_mean_td_db": float(np.nanmean(base)),
            "next_mean_td_db": float(np.nanmean(nxt)),
            "one_step_change_db": float(np.nanmean(change)),
            "one_step_abs_change_db": float(np.nanmean(np.abs(change))),
            "one_step_deterioration_db": float(np.nanmean(np.clip(-change, 0, None))),
            "baseline_floor_fraction": float(np.nanmean(base <= MACULA_FLOOR_THRESHOLD_DB)),
            "next_floor_fraction": float(np.nanmean(nxt <= MACULA_FLOOR_THRESHOLD_DB)),
            "baseline_near_normal_fraction": float(np.nanmean(base >= MACULA_NEAR_NORMAL_THRESHOLD_DB)),
            "next_near_normal_fraction": float(np.nanmean(nxt >= MACULA_NEAR_NORMAL_THRESHOLD_DB)),
            "baseline_point_variability_db": float(np.nanmean(np.nanstd(base, axis=1))),
            "points": int(len(idx)),
        })
    return pd.DataFrame(rows)


def _make_domain_shift_table(bern_region_pairs, grape_region_pairs):
    cols = [
        "Region",
        "baseline_mean_td_db",
        "one_step_change_db",
        "one_step_abs_change_db",
        "one_step_deterioration_db",
        "baseline_floor_fraction",
        "baseline_near_normal_fraction",
        "baseline_point_variability_db",
    ]
    b = bern_region_pairs[cols].copy().add_prefix("bern_").rename(columns={"bern_Region": "Region"})
    g = grape_region_pairs[cols].copy().add_prefix("grape_").rename(columns={"grape_Region": "Region"})
    out = b.merge(g, on="Region", how="inner")
    for metric in cols[1:]:
        out[f"grape_minus_bern_{metric}"] = out[f"grape_{metric}"] - out[f"bern_{metric}"]
    out["abs_baseline_shift_db"] = out["grape_minus_bern_baseline_mean_td_db"].abs()
    out["abs_progression_shift_db"] = out["grape_minus_bern_one_step_change_db"].abs()
    out = out.sort_values(["Region"]).reset_index(drop=True)
    return out


def _rank_within_dataset(pair_stats):
    out = pair_stats.copy()
    # Rank 1 means strongest / largest value for that metric.
    out["rank_most_negative_baseline"] = out.groupby("dataset")["baseline_mean_td_db"].rank(method="min", ascending=True).astype(int)
    out["rank_largest_abs_change"] = out.groupby("dataset")["one_step_abs_change_db"].rank(method="min", ascending=False).astype(int)
    out["rank_largest_near_normal"] = out.groupby("dataset")["baseline_near_normal_fraction"].rank(method="min", ascending=False).astype(int)
    out["rank_largest_floor"] = out.groupby("dataset")["baseline_floor_fraction"].rank(method="min", ascending=False).astype(int)
    return out


grape_visit_df, grape_td_cols = _prepare_grape_visit_table()
bern_visit_df, bern_td_cols = _prepare_bern_visit_table()

if len(grape_td_cols) != len(bern_td_cols):
    raise ValueError(f"TD column mismatch: GRAPE={len(grape_td_cols)}, Bern={len(bern_td_cols)}")

# Visit-level distribution: all available visits.
grape_visit_stats = _region_visit_summary(grape_visit_df, grape_td_cols, "GRAPE")
bern_visit_stats = _region_visit_summary(bern_visit_df, bern_td_cols, "Bern")
visit_region_stats = pd.concat([bern_visit_stats, grape_visit_stats], ignore_index=True)

# Baseline -> next visit diagnostic: cleanest setting for transfer/data shift.
grape_pairs = _baseline_next_pairs(grape_visit_df, grape_td_cols)
bern_pairs = _baseline_next_pairs(bern_visit_df, bern_td_cols)
grape_pair_stats = _region_pair_summary(grape_pairs, grape_td_cols, "GRAPE")
bern_pair_stats = _region_pair_summary(bern_pairs, bern_td_cols, "Bern")
pair_region_stats = pd.concat([bern_pair_stats, grape_pair_stats], ignore_index=True)
pair_region_stats_ranked = _rank_within_dataset(pair_region_stats)

domain_shift = _make_domain_shift_table(bern_pair_stats, grape_pair_stats)

macular_focus = pair_region_stats_ranked[pair_region_stats_ranked["Region"].eq("Macular")].copy()
macular_shift = domain_shift[domain_shift["Region"].eq("Macular")].copy()

# Compact interpretation table for thesis / Marta discussion.
if len(macular_shift):
    ms = macular_shift.iloc[0]
    baseline_shift = float(ms["grape_minus_bern_baseline_mean_td_db"])
    progression_shift = float(ms["grape_minus_bern_one_step_change_db"])
    near_normal_shift = float(ms["grape_minus_bern_baseline_near_normal_fraction"])
    floor_shift = float(ms["grape_minus_bern_baseline_floor_fraction"])
else:
    baseline_shift = progression_shift = near_normal_shift = floor_shift = np.nan

macula_interpretation = pd.DataFrame([
    {
        "Question": "Is Macular distribution different between Bern and GRAPE?",
        "Readout": f"GRAPE - Bern baseline mean TD shift = {baseline_shift:+.3f} dB",
        "Interpretation": "Large shift would suggest domain mismatch; small shift suggests macula worsening is less likely from baseline severity alone.",
    },
    {
        "Question": "Is Macular short-term progression different?",
        "Readout": f"GRAPE - Bern one-step change shift = {progression_shift:+.3f} dB",
        "Interpretation": "If GRAPE has less or noisier macular progression, fine-tuning can over-adapt and hurt transfer performance.",
    },
    {
        "Question": "Could ceiling / floor effects explain it?",
        "Readout": f"Near-normal shift = {near_normal_shift:+.3f}; floor shift = {floor_shift:+.3f}",
        "Interpretation": "Near-normal means little remaining signal to predict; floor means severe points cannot worsen much. Either can reduce useful training signal.",
    },
    {
        "Question": "Is this a final causal proof?",
        "Readout": "No, this is a data diagnostic, not a trained ablation.",
        "Interpretation": "Use it to decide whether to run a Macular-specific fine-tuning ablation, e.g. lower LR, freeze more layers, or skip Macular fine-tuning.",
    },
])

macula_finetune_diag_outputs = {
    "visit_region_stats": visit_region_stats,
    "pair_region_stats": pair_region_stats_ranked,
    "domain_shift": domain_shift,
    "macular_focus": macular_focus,
    "macular_shift": macular_shift,
    "interpretation": macula_interpretation,
    "grape_pairs": grape_pairs,
    "bern_pairs": bern_pairs,
}

print("Macular fine-tuning diagnostic: visit-level region stats")
display(visit_region_stats[[
    "dataset", "Region", "visits", "patient_eyes", "mean_td_db", "sd_td_db",
    "near_normal_fraction_geq_-2", "floor_fraction_leq_-25", "point_variability_db", "points"
]])

print("Macular fine-tuning diagnostic: baseline -> next-visit region stats")
display(pair_region_stats_ranked[[
    "dataset", "Region", "pairs", "patient_eyes", "mean_delta_t_years",
    "baseline_mean_td_db", "one_step_change_db", "one_step_abs_change_db",
    "one_step_deterioration_db", "baseline_near_normal_fraction", "baseline_floor_fraction",
    "rank_most_negative_baseline", "rank_largest_abs_change", "rank_largest_near_normal", "rank_largest_floor",
    "points"
]])

print("Bern -> GRAPE domain shift by region")
display(domain_shift[[
    "Region",
    "bern_baseline_mean_td_db", "grape_baseline_mean_td_db", "grape_minus_bern_baseline_mean_td_db",
    "bern_one_step_change_db", "grape_one_step_change_db", "grape_minus_bern_one_step_change_db",
    "bern_baseline_near_normal_fraction", "grape_baseline_near_normal_fraction", "grape_minus_bern_baseline_near_normal_fraction",
    "bern_baseline_floor_fraction", "grape_baseline_floor_fraction", "grape_minus_bern_baseline_floor_fraction",
    "abs_baseline_shift_db", "abs_progression_shift_db"
]].sort_values("abs_baseline_shift_db", ascending=False).reset_index(drop=True))

print("Macular focus")
display(macular_focus)

print("Interpretation")
display(macula_interpretation)


## Macular Horizon-matched / Annualized Progression Diagnostic

This follow-up diagnostic addresses one limitation of the previous Macular check: Bern baseline-to-next intervals were much longer than GRAPE intervals. Here we compare Bern and GRAPE using a similar follow-up window and annualized TD change, so Macular progression is not dominated by different time spans.


In [ ]:
# ---------------------------------------------------------------------------
# Macular horizon-matched / annualized progression diagnostic
# ---------------------------------------------------------------------------
# Purpose:
# The previous Macular diagnostic showed Bern and GRAPE have different baseline-to-next
# time spans. This cell checks whether the Macular interpretation still holds after:
# 1) matching Bern/GRAPE pairs to a similar follow-up interval window, and
# 2) converting TD change into annualized change per year.
#
# This is a data diagnostic only. No model training is run.

import numpy as np
import pandas as pd

if "macula_finetune_diag_outputs" not in globals():
    raise NameError("Run the previous Macular fine-tuning diagnostic cell first.")
if "cluster_mapping" not in globals():
    raise NameError("Missing cluster_mapping. Run the anatomical region definition cell first.")

MACULA_FLOOR_THRESHOLD_DB = globals().get("MACULA_FLOOR_THRESHOLD_DB", -25.0)
MACULA_NEAR_NORMAL_THRESHOLD_DB = globals().get("MACULA_NEAR_NORMAL_THRESHOLD_DB", -2.0)

bern_pairs_all = macula_finetune_diag_outputs["bern_pairs"].copy()
grape_pairs_all = macula_finetune_diag_outputs["grape_pairs"].copy()

# Use the central GRAPE follow-up range, clipped to a clinically reasonable short-term window.
grape_dt = grape_pairs_all["delta_t_years"].astype(float)
MATCH_MIN_YEARS = max(0.25, float(grape_dt.quantile(0.05)))
MATCH_MAX_YEARS = min(1.50, float(grape_dt.quantile(0.95)))


def _filter_dt_window(pair_df, min_years, max_years):
    out = pair_df.copy()
    out["delta_t_years"] = out["delta_t_years"].astype(float)
    out = out[(out["delta_t_years"] >= min_years) & (out["delta_t_years"] <= max_years)].copy()
    return out.reset_index(drop=True)


bern_pairs_matched = _filter_dt_window(bern_pairs_all, MATCH_MIN_YEARS, MATCH_MAX_YEARS)
grape_pairs_matched = _filter_dt_window(grape_pairs_all, MATCH_MIN_YEARS, MATCH_MAX_YEARS)

# If the quantile window becomes too narrow for Bern, fall back to a fixed short-term window.
if len(bern_pairs_matched) < 100 or len(grape_pairs_matched) < 20:
    MATCH_MIN_YEARS = 0.25
    MATCH_MAX_YEARS = 1.50
    bern_pairs_matched = _filter_dt_window(bern_pairs_all, MATCH_MIN_YEARS, MATCH_MAX_YEARS)
    grape_pairs_matched = _filter_dt_window(grape_pairs_all, MATCH_MIN_YEARS, MATCH_MAX_YEARS)


match_window_summary = pd.DataFrame([
    {
        "dataset": "Bern_all_pairs",
        "pairs": int(len(bern_pairs_all)),
        "patient_eyes": int(bern_pairs_all["eye_id"].nunique()),
        "mean_delta_t_years": float(bern_pairs_all["delta_t_years"].mean()),
        "median_delta_t_years": float(bern_pairs_all["delta_t_years"].median()),
        "used_for_matched_analysis": False,
    },
    {
        "dataset": "GRAPE_all_pairs",
        "pairs": int(len(grape_pairs_all)),
        "patient_eyes": int(grape_pairs_all["eye_id"].nunique()),
        "mean_delta_t_years": float(grape_pairs_all["delta_t_years"].mean()),
        "median_delta_t_years": float(grape_pairs_all["delta_t_years"].median()),
        "used_for_matched_analysis": False,
    },
    {
        "dataset": "Bern_matched_window",
        "pairs": int(len(bern_pairs_matched)),
        "patient_eyes": int(bern_pairs_matched["eye_id"].nunique()),
        "mean_delta_t_years": float(bern_pairs_matched["delta_t_years"].mean()),
        "median_delta_t_years": float(bern_pairs_matched["delta_t_years"].median()),
        "used_for_matched_analysis": True,
    },
    {
        "dataset": "GRAPE_matched_window",
        "pairs": int(len(grape_pairs_matched)),
        "patient_eyes": int(grape_pairs_matched["eye_id"].nunique()),
        "mean_delta_t_years": float(grape_pairs_matched["delta_t_years"].mean()),
        "median_delta_t_years": float(grape_pairs_matched["delta_t_years"].median()),
        "used_for_matched_analysis": True,
    },
])
match_window_summary["matching_window_years"] = f"{MATCH_MIN_YEARS:.2f}-{MATCH_MAX_YEARS:.2f}"


def _annualized_region_summary(pair_df, dataset_name):
    rows = []
    if len(pair_df) == 0:
        return pd.DataFrame(rows)

    baseline_all = np.stack(pair_df["baseline"].to_numpy()).astype(np.float32)
    next_all = np.stack(pair_df["next"].to_numpy()).astype(np.float32)
    dt = pair_df["delta_t_years"].to_numpy(dtype=np.float32)[:, None]
    change_all = next_all - baseline_all
    annual_change_all = change_all / np.maximum(dt, 1e-6)

    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        base = baseline_all[:, idx]
        change = change_all[:, idx]
        annual = annual_change_all[:, idx]

        rows.append({
            "dataset": dataset_name,
            "Region": region_name,
            "pairs": int(len(pair_df)),
            "patient_eyes": int(pair_df["eye_id"].nunique()),
            "mean_delta_t_years": float(pair_df["delta_t_years"].mean()),
            "baseline_mean_td_db": float(np.nanmean(base)),
            "one_step_change_db": float(np.nanmean(change)),
            "annualized_change_db_per_year": float(np.nanmean(annual)),
            "annualized_abs_change_db_per_year": float(np.nanmean(np.abs(annual))),
            "annualized_deterioration_db_per_year": float(np.nanmean(np.clip(-annual, 0, None))),
            "baseline_near_normal_fraction": float(np.nanmean(base >= MACULA_NEAR_NORMAL_THRESHOLD_DB)),
            "baseline_floor_fraction": float(np.nanmean(base <= MACULA_FLOOR_THRESHOLD_DB)),
            "points": int(len(idx)),
        })
    return pd.DataFrame(rows)


bern_ann = _annualized_region_summary(bern_pairs_matched, "Bern_matched")
grape_ann = _annualized_region_summary(grape_pairs_matched, "GRAPE_matched")
annualized_region_stats = pd.concat([bern_ann, grape_ann], ignore_index=True)

annualized_region_stats["rank_largest_abs_annual_change"] = (
    annualized_region_stats.groupby("dataset")["annualized_abs_change_db_per_year"]
    .rank(method="min", ascending=False)
    .astype(int)
)
annualized_region_stats["rank_largest_annual_deterioration"] = (
    annualized_region_stats.groupby("dataset")["annualized_deterioration_db_per_year"]
    .rank(method="min", ascending=False)
    .astype(int)
)
annualized_region_stats["rank_largest_near_normal"] = (
    annualized_region_stats.groupby("dataset")["baseline_near_normal_fraction"]
    .rank(method="min", ascending=False)
    .astype(int)
)


def _annualized_shift_table(bern_stats, grape_stats):
    cols = [
        "Region",
        "baseline_mean_td_db",
        "annualized_change_db_per_year",
        "annualized_abs_change_db_per_year",
        "annualized_deterioration_db_per_year",
        "baseline_near_normal_fraction",
        "baseline_floor_fraction",
    ]
    b = bern_stats[cols].copy().add_prefix("bern_").rename(columns={"bern_Region": "Region"})
    g = grape_stats[cols].copy().add_prefix("grape_").rename(columns={"grape_Region": "Region"})
    out = b.merge(g, on="Region", how="inner")
    for col in cols[1:]:
        out[f"grape_minus_bern_{col}"] = out[f"grape_{col}"] - out[f"bern_{col}"]
    out["abs_baseline_shift_db"] = out["grape_minus_bern_baseline_mean_td_db"].abs()
    out["abs_annualized_progression_shift_db_per_year"] = out["grape_minus_bern_annualized_change_db_per_year"].abs()
    return out.sort_values("abs_annualized_progression_shift_db_per_year", ascending=False).reset_index(drop=True)


annualized_shift = _annualized_shift_table(bern_ann, grape_ann)
macular_annualized_focus = annualized_region_stats[
    annualized_region_stats["Region"].eq("Macular")
].reset_index(drop=True)
macular_annualized_shift = annualized_shift[
    annualized_shift["Region"].eq("Macular")
].reset_index(drop=True)

if len(macular_annualized_shift) > 0:
    m = macular_annualized_shift.iloc[0]
    macular_annualized_interpretation = pd.DataFrame([
        {
            "Question": "After matching follow-up time, is Macular still different between Bern and GRAPE?",
            "Readout": (
                f"GRAPE - Bern annualized Macular change = "
                f"{m['grape_minus_bern_annualized_change_db_per_year']:+.3f} dB/year"
            ),
            "Interpretation": "This removes the main confound that Bern baseline-to-next intervals were much longer than GRAPE intervals.",
        },
        {
            "Question": "Does this prove why fine-tuning worsened Macular?",
            "Readout": "No. This is still a data diagnostic, not a model ablation.",
            "Interpretation": "It tells us whether Macular has a weak/noisy or shifted progression signal that could make fine-tuning less reliable.",
        },
        {
            "Question": "Safe conclusion for Marta",
            "Readout": "Use the matched annualized table together with the previous distribution table.",
            "Interpretation": "If Macular has low change or high near-normal fraction, the safest claim is limited GRAPE fine-tuning signal rather than a causal proof.",
        },
    ])
else:
    macular_annualized_interpretation = pd.DataFrame([
        {
            "Question": "Could Macular be evaluated after matching follow-up time?",
            "Readout": "No Macular row was available after filtering.",
            "Interpretation": "The matched window is too restrictive; widen the time window before interpreting.",
        }
    ])

macular_horizon_matched_outputs = {
    "match_window_summary": match_window_summary,
    "annualized_region_stats": annualized_region_stats,
    "annualized_shift": annualized_shift,
    "macular_annualized_focus": macular_annualized_focus,
    "macular_annualized_shift": macular_annualized_shift,
    "interpretation": macular_annualized_interpretation,
    "bern_pairs_matched": bern_pairs_matched,
    "grape_pairs_matched": grape_pairs_matched,
}

print("Horizon-matched follow-up window")
display(match_window_summary)

print("Annualized region progression after follow-up matching")
display(annualized_region_stats[[
    "dataset", "Region", "pairs", "patient_eyes", "mean_delta_t_years",
    "baseline_mean_td_db", "annualized_change_db_per_year",
    "annualized_abs_change_db_per_year", "annualized_deterioration_db_per_year",
    "baseline_near_normal_fraction", "baseline_floor_fraction",
    "rank_largest_abs_annual_change", "rank_largest_annual_deterioration",
    "rank_largest_near_normal", "points"
]])

print("Bern -> GRAPE annualized shift by region")
display(annualized_shift[[
    "Region",
    "bern_baseline_mean_td_db", "grape_baseline_mean_td_db", "grape_minus_bern_baseline_mean_td_db",
    "bern_annualized_change_db_per_year", "grape_annualized_change_db_per_year", "grape_minus_bern_annualized_change_db_per_year",
    "bern_annualized_abs_change_db_per_year", "grape_annualized_abs_change_db_per_year", "grape_minus_bern_annualized_abs_change_db_per_year",
    "bern_baseline_near_normal_fraction", "grape_baseline_near_normal_fraction", "grape_minus_bern_baseline_near_normal_fraction",
    "bern_baseline_floor_fraction", "grape_baseline_floor_fraction", "grape_minus_bern_baseline_floor_fraction",
]])

print("Macular focus after horizon matching")
display(macular_annualized_focus)

print("Interpretation")
display(macular_annualized_interpretation)


## CFP / ROI Image Integration Pilot

This section starts the image branch work without changing the current Transformer results.

Goal:
- Verify that `Corresponding CFP` maps cleanly to the image folders.
- Build a baseline image table for each patient-eye.
- Prepare a lightweight frozen-image-feature pipeline before replacing it with RetFound embeddings.

Main modeling idea for the next experiment:

```text
VF history -> current best Transformer -> VF-only prediction A
baseline CFP / ROI -> frozen image encoder -> image-aware prediction or residual B
validation / CV selects how much B should contribute
```

This keeps the strong VF-only Transformer as the baseline and uses images conservatively.


In [ ]:
# ---------------------------------------------------------------------------
# CFP / ROI mapping audit
# ---------------------------------------------------------------------------
from pathlib import Path
import re
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path('.')
GRAPE_DIR = PROJECT_ROOT / 'GRAPE'
CFP_DIR = GRAPE_DIR / 'CFPs'
ROI_DIR = GRAPE_DIR / 'ROI images'
CLINICAL_XLSX = GRAPE_DIR / 'VF and clinical information.xlsx'


def _clean_cfp_name(value):
    if pd.isna(value):
        return None
    name = str(value).strip()
    if name in {'', '/', 'nan', 'None'}:
        return None
    return Path(name).name


def _infer_eye_id(row, cfp_name=None):
    subject = row.get('Subject Number', np.nan)
    laterality = row.get('Laterality', np.nan)

    if pd.isna(subject) and cfp_name:
        match = re.match(r'^(\d+)_(OD|OS)_', cfp_name)
        if match:
            subject = match.group(1)

    if pd.isna(laterality) and cfp_name:
        match = re.match(r'^\d+_(OD|OS)_', cfp_name)
        if match:
            laterality = match.group(1)

    if pd.isna(subject) or pd.isna(laterality):
        return None

    try:
        subject_text = str(int(float(subject)))
    except Exception:
        subject_text = str(subject).strip()
    laterality_text = str(laterality).strip()
    return f'{subject_text}_{laterality_text}'


def _image_size(path):
    try:
        with Image.open(path) as img:
            return img.size
    except Exception:
        return None

baseline_sheet = pd.read_excel(CLINICAL_XLSX, sheet_name='Baseline')
followup_sheet = pd.read_excel(CLINICAL_XLSX, sheet_name='Follow-up')

cfp_files = {p.name: p for p in sorted(CFP_DIR.glob('*.jpg'))}
roi_files = {p.name: p for p in sorted(ROI_DIR.glob('*.jpg'))}


def _build_mapping(sheet_df, sheet_name):
    rows = []
    for row_idx, row in sheet_df.iterrows():
        cfp_name = _clean_cfp_name(row.get('Corresponding CFP'))
        eye_id = _infer_eye_id(row, cfp_name)
        visit_number = row.get('Visit Number', 1 if sheet_name == 'Baseline' else np.nan)
        interval_years = row.get('Interval Years', 0.0 if sheet_name == 'Baseline' else np.nan)

        rows.append({
            'source_sheet': sheet_name,
            'source_row': int(row_idx),
            'Subject Number': row.get('Subject Number', np.nan),
            'Laterality': row.get('Laterality', np.nan),
            'eye_id': eye_id,
            'Visit Number': visit_number,
            'Interval Years': interval_years,
            'Corresponding CFP': row.get('Corresponding CFP', np.nan),
            'cfp_name': cfp_name,
            'has_valid_cfp_ref': cfp_name is not None,
            'cfp_path': str(cfp_files[cfp_name]) if cfp_name in cfp_files else None,
            'roi_path': str(roi_files[cfp_name]) if cfp_name in roi_files else None,
            'has_cfp_file': cfp_name in cfp_files if cfp_name is not None else False,
            'has_roi_file': cfp_name in roi_files if cfp_name is not None else False,
        })
    return pd.DataFrame(rows)

cfp_mapping_df = pd.concat([
    _build_mapping(baseline_sheet, 'Baseline'),
    _build_mapping(followup_sheet, 'Follow-up'),
], ignore_index=True)

valid_cfp_mapping_df = cfp_mapping_df[cfp_mapping_df['has_valid_cfp_ref']].copy()
baseline_cfp_df = valid_cfp_mapping_df[valid_cfp_mapping_df['source_sheet'].eq('Baseline')].copy()
followup_cfp_df = valid_cfp_mapping_df[valid_cfp_mapping_df['source_sheet'].eq('Follow-up')].copy()

mapping_summary = pd.DataFrame([
    {
        'source': 'CFP folder',
        'rows': len(cfp_files),
        'valid_cfp_refs': np.nan,
        'unique_cfp_refs': np.nan,
        'matched_cfp_files': np.nan,
        'matched_roi_files': np.nan,
        'missing_or_slash_refs': np.nan,
    },
    {
        'source': 'ROI folder',
        'rows': len(roi_files),
        'valid_cfp_refs': np.nan,
        'unique_cfp_refs': np.nan,
        'matched_cfp_files': np.nan,
        'matched_roi_files': np.nan,
        'missing_or_slash_refs': np.nan,
    },
    {
        'source': 'Baseline sheet',
        'rows': len(baseline_sheet),
        'valid_cfp_refs': int(baseline_cfp_df['has_valid_cfp_ref'].sum()),
        'unique_cfp_refs': int(baseline_cfp_df['cfp_name'].nunique()),
        'matched_cfp_files': int(baseline_cfp_df['has_cfp_file'].sum()),
        'matched_roi_files': int(baseline_cfp_df['has_roi_file'].sum()),
        'missing_or_slash_refs': int(len(baseline_sheet) - baseline_cfp_df['has_valid_cfp_ref'].sum()),
    },
    {
        'source': 'Follow-up sheet',
        'rows': len(followup_sheet),
        'valid_cfp_refs': int(followup_cfp_df['has_valid_cfp_ref'].sum()),
        'unique_cfp_refs': int(followup_cfp_df['cfp_name'].nunique()),
        'matched_cfp_files': int(followup_cfp_df['has_cfp_file'].sum()),
        'matched_roi_files': int(followup_cfp_df['has_roi_file'].sum()),
        'missing_or_slash_refs': int(len(followup_sheet) - followup_cfp_df['has_valid_cfp_ref'].sum()),
    },
])

visit_availability = (
    cfp_mapping_df[cfp_mapping_df['source_sheet'].eq('Follow-up')]
    .assign(has_image=lambda d: d['has_cfp_file'] & d['has_roi_file'])
    .groupby('Visit Number', dropna=False)
    .agg(
        rows=('cfp_name', 'size'),
        rows_with_cfp_ref=('has_valid_cfp_ref', 'sum'),
        rows_with_matched_image=('has_image', 'sum'),
        patient_eyes=('eye_id', 'nunique'),
    )
    .reset_index()
)

all_image_names = sorted(set(cfp_files) | set(roi_files))
size_rows = []
for name in all_image_names[:]:
    cfp_path = cfp_files.get(name)
    roi_path = roi_files.get(name)
    size_rows.append({
        'cfp_name': name,
        'cfp_size': _image_size(cfp_path) if cfp_path else None,
        'roi_size': _image_size(roi_path) if roi_path else None,
        'has_cfp_file': cfp_path is not None,
        'has_roi_file': roi_path is not None,
    })
image_file_audit_df = pd.DataFrame(size_rows)

unmatched_refs = valid_cfp_mapping_df[
    ~(valid_cfp_mapping_df['has_cfp_file'] & valid_cfp_mapping_df['has_roi_file'])
].copy()
unreferenced_cfp_files = sorted(set(cfp_files) - set(valid_cfp_mapping_df['cfp_name'].dropna()))

print('CFP / ROI mapping summary')
display(mapping_summary)

print('Follow-up image availability by visit')
display(visit_availability)

print('Image size summary')
display(
    image_file_audit_df
    .groupby(['cfp_size', 'roi_size'], dropna=False)
    .size()
    .reset_index(name='n_images')
    .sort_values('n_images', ascending=False)
)

print('Unmatched valid CFP references:', len(unmatched_refs))
print('Unreferenced CFP image files:', len(unreferenced_cfp_files))

if len(unmatched_refs):
    display(unmatched_refs[['source_sheet', 'source_row', 'eye_id', 'Visit Number', 'Corresponding CFP', 'cfp_name', 'has_cfp_file', 'has_roi_file']].head(20))


In [ ]:
# ---------------------------------------------------------------------------
# Lightweight frozen image feature extraction smoke test
# ---------------------------------------------------------------------------
# These are NOT final RetFound features. They are deterministic CFP/ROI image
# descriptors used to validate the data pipeline before plugging in RetFound.
import time
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

IMAGE_FEATURE_CACHE = PROJECT_ROOT / 'notebooks2' / 'cache' / 'grape_cfp_roi_handcrafted_features.csv'
IMAGE_FEATURE_CACHE.parent.mkdir(parents=True, exist_ok=True)


def _safe_open_rgb(path, resize=224):
    img = Image.open(path).convert('RGB')
    img = img.resize((resize, resize), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32) / 255.0


def _masked_pixels(arr):
    # Exclude black camera background from full CFP images.
    gray = arr.mean(axis=2)
    mask = gray > 0.08
    if int(mask.sum()) < 500:
        mask = np.ones(gray.shape, dtype=bool)
    return arr[mask], gray, mask


def _extract_basic_image_features(path, prefix, resize=224):
    arr = _safe_open_rgb(path, resize=resize)
    pix, gray, mask = _masked_pixels(arr)
    out = {}

    for channel_idx, channel_name in enumerate(['r', 'g', 'b']):
        vals = pix[:, channel_idx]
        out[f'{prefix}_{channel_name}_mean'] = float(np.mean(vals))
        out[f'{prefix}_{channel_name}_std'] = float(np.std(vals))
        out[f'{prefix}_{channel_name}_p10'] = float(np.percentile(vals, 10))
        out[f'{prefix}_{channel_name}_p50'] = float(np.percentile(vals, 50))
        out[f'{prefix}_{channel_name}_p90'] = float(np.percentile(vals, 90))

    gray_vals = gray[mask]
    out[f'{prefix}_brightness_mean'] = float(np.mean(gray_vals))
    out[f'{prefix}_brightness_std'] = float(np.std(gray_vals))
    out[f'{prefix}_dark_fraction'] = float(np.mean(gray_vals < 0.25))
    out[f'{prefix}_bright_fraction'] = float(np.mean(gray_vals > 0.75))
    out[f'{prefix}_valid_mask_fraction'] = float(np.mean(mask))

    rg = pix[:, 0] - pix[:, 1]
    gb = pix[:, 1] - pix[:, 2]
    out[f'{prefix}_red_green_mean_diff'] = float(np.mean(rg))
    out[f'{prefix}_green_blue_mean_diff'] = float(np.mean(gb))

    gy, gx = np.gradient(gray)
    edge = np.sqrt(gx ** 2 + gy ** 2)
    out[f'{prefix}_edge_mean'] = float(np.mean(edge[mask]))
    out[f'{prefix}_edge_p90'] = float(np.percentile(edge[mask], 90))

    h, w = gray.shape
    y0, y1 = h // 4, 3 * h // 4
    x0, x1 = w // 4, 3 * w // 4
    center = arr[y0:y1, x0:x1]
    center_pix, center_gray, center_mask = _masked_pixels(center)
    out[f'{prefix}_center_brightness_mean'] = float(np.mean(center_gray[center_mask]))
    out[f'{prefix}_center_dark_fraction'] = float(np.mean(center_gray[center_mask] < 0.25))

    return out


def _extract_one_image_row(cfp_name):
    row = {'cfp_name': cfp_name}
    cfp_path = cfp_files.get(cfp_name)
    roi_path = roi_files.get(cfp_name)

    if cfp_path is not None:
        row.update(_extract_basic_image_features(cfp_path, 'cfp'))
    if roi_path is not None:
        row.update(_extract_basic_image_features(roi_path, 'roi'))
    return row

if IMAGE_FEATURE_CACHE.exists():
    image_feature_df = pd.read_csv(IMAGE_FEATURE_CACHE)
    print(f'Loaded cached image features: {IMAGE_FEATURE_CACHE}')
else:
    start = time.time()
    rows = []
    for i, cfp_name in enumerate(sorted(cfp_files)):
        rows.append(_extract_one_image_row(cfp_name))
        if (i + 1) % 100 == 0:
            print(f'Extracted {i + 1}/{len(cfp_files)} image feature rows')
    image_feature_df = pd.DataFrame(rows)
    image_feature_df.to_csv(IMAGE_FEATURE_CACHE, index=False)
    print(f'Saved image features to {IMAGE_FEATURE_CACHE} in {time.time() - start:.1f}s')

baseline_image_feature_df = (
    baseline_cfp_df[['eye_id', 'Subject Number', 'Laterality', 'Visit Number', 'cfp_name', 'cfp_path', 'roi_path']]
    .dropna(subset=['eye_id', 'cfp_name'])
    .merge(image_feature_df, on='cfp_name', how='left')
)

followup_image_feature_df = (
    followup_cfp_df[['eye_id', 'Subject Number', 'Laterality', 'Visit Number', 'Interval Years', 'cfp_name', 'cfp_path', 'roi_path']]
    .dropna(subset=['eye_id', 'cfp_name'])
    .merge(image_feature_df, on='cfp_name', how='left')
)

feature_columns = [c for c in image_feature_df.columns if c != 'cfp_name']
image_feature_summary = pd.DataFrame([
    {
        'table': 'image_feature_df',
        'rows': len(image_feature_df),
        'feature_columns': len(feature_columns),
        'missing_feature_values': int(image_feature_df[feature_columns].isna().sum().sum()),
    },
    {
        'table': 'baseline_image_feature_df',
        'rows': len(baseline_image_feature_df),
        'patient_eyes': baseline_image_feature_df['eye_id'].nunique(),
        'feature_columns': len(feature_columns),
        'missing_feature_values': int(baseline_image_feature_df[feature_columns].isna().sum().sum()),
    },
    {
        'table': 'followup_image_feature_df',
        'rows': len(followup_image_feature_df),
        'patient_eyes': followup_image_feature_df['eye_id'].nunique(),
        'feature_columns': len(feature_columns),
        'missing_feature_values': int(followup_image_feature_df[feature_columns].isna().sum().sum()),
    },
])

print('Image feature tables prepared')
display(image_feature_summary)

print('Baseline image feature preview')
display(
    baseline_image_feature_df[
        ['eye_id', 'cfp_name', 'cfp_brightness_mean', 'cfp_dark_fraction', 'roi_brightness_mean', 'roi_dark_fraction']
    ].head()
)

print('Next step: replace these handcrafted features with RetFound CFP embeddings, then test image-aware residual / soft-ensemble branches against the VF-only Transformer baseline.')


### Planned model integration after the audit

Recommended first image experiment:

```text
1. Current best VF-only Transformer remains the main baseline.
2. Baseline CFP / ROI image features are treated as static structural context for the eye.
3. Train a small image-aware residual branch, not a full end-to-end image model.
4. Combine with VF-only prediction using validation-selected weights inside patient-eye CV.
```

Why baseline image first:
- Baseline images are available for almost every patient-eye.
- Follow-up CFP images are missing for many visits, so per-visit image sequences would immediately reduce the dataset or require masking.
- This mirrors RNFL conceptually: structural information measured at baseline, used to help forecast future functional TD.

RetFound replacement point:
- The handcrafted features in `image_feature_df` are only a pipeline check.
- For the real experiment, replace them with frozen RetFound CFP embeddings from full CFP and/or ROI images.


## First Image-Aware Signal Check

This is a lightweight pilot before using RetFound.

Question:
- Do baseline CFP / ROI image features contain incremental information beyond the VF history?

Design:
- Use the existing GRAPE train / validation / test sequence split.
- Use the last historical VF as the main VF baseline.
- Train ridge residual models on training data.
- Select regularization strength on validation data.
- Report only held-out test MAE.

This is not the final Transformer image branch yet. It is a fast check that the image mapping and image features can support prediction before we spend time on RetFound embeddings.


In [ ]:
# ---------------------------------------------------------------------------
# First image-aware signal check: VF history + baseline CFP/ROI features
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

required_names = [
    '_prepare_safe_residual_data',
    '_df_to_arrays_with_history',
    '_finite_mask',
    '_apply_mask',
    'cluster_mapping',
    'baseline_image_feature_df',
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise NameError(
        'Run the data-preparation cells and the CFP / ROI image audit cells first. Missing: '
        + ', '.join(missing)
    )

IMAGE_RIDGE_ALPHAS = np.logspace(-3, 4, 16)


def _last_vf_from_history(x_list):
    return np.stack([np.asarray(x, dtype=np.float32)[-1] for x in x_list], axis=0).astype(np.float32)


def _history_summary_features(x_list, t_list):
    rows = []
    for x, t in zip(x_list, t_list):
        x = np.asarray(x, dtype=np.float32)
        t = np.asarray(t, dtype=np.float32)
        if len(t) == 0:
            t = np.zeros((len(x),), dtype=np.float32)
        rows.append([
            float(len(x)),
            float(np.nanmin(t)),
            float(np.nanmax(t)),
            float(np.nanmax(t) - np.nanmin(t)),
            float(np.nanmean(np.abs(x[-1] - x[0]))) if len(x) > 1 else 0.0,
            float(np.nanmean(x[-1])),
        ])
    return np.asarray(rows, dtype=np.float32)


def _mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def _sample_sd(y_true, y_pred):
    return float(np.std(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)), axis=1)))


def _prepare_sequence_arrays_for_image_check():
    train_df_all, val_df_all, test_df_all, td_cols_img, rnfl_cols_img = _prepare_safe_residual_data(
        k_list=(2, 3, 4),
        train_mode='consecutive',
        eval_mode='consecutive',
        time_input_mode='input_times',
    )

    for df_part in (train_df_all, val_df_all, test_df_all):
        df_part['meta_sub'] = [np.zeros((0,), dtype=np.float32) for _ in range(len(df_part))]

    x_train, y_train, t_train, meta_train_empty = _df_to_arrays_with_history(train_df_all)
    x_val, y_val, t_val, meta_val_empty = _df_to_arrays_with_history(val_df_all)
    x_test, y_test, t_test, meta_test_empty = _df_to_arrays_with_history(test_df_all)

    train_mask = _finite_mask(x_train, y_train, t_train, meta_train_empty)
    val_mask = _finite_mask(x_val, y_val, t_val, meta_val_empty)
    test_mask = _finite_mask(x_test, y_test, t_test, meta_test_empty)

    x_train, y_train, t_train, _ = _apply_mask(x_train, y_train, t_train, meta_train_empty, train_mask)
    x_val, y_val, t_val, _ = _apply_mask(x_val, y_val, t_val, meta_val_empty, val_mask)
    x_test, y_test, t_test, _ = _apply_mask(x_test, y_test, t_test, meta_test_empty, test_mask)

    train_df = train_df_all.iloc[np.where(train_mask)[0]].reset_index(drop=True)
    val_df = val_df_all.iloc[np.where(val_mask)[0]].reset_index(drop=True)
    test_df = test_df_all.iloc[np.where(test_mask)[0]].reset_index(drop=True)

    return {
        'train_df': train_df,
        'val_df': val_df,
        'test_df': test_df,
        'x_train': x_train,
        'x_val': x_val,
        'x_test': x_test,
        'y_train': y_train.astype(np.float32),
        'y_val': y_val.astype(np.float32),
        'y_test': y_test.astype(np.float32),
        't_train': t_train,
        't_val': t_val,
        't_test': t_test,
        'td_cols': td_cols_img,
    }


def _feature_columns_for_mode(mode):
    numeric_cols = set(baseline_image_feature_df.select_dtypes(include=[np.number]).columns)
    if mode == 'cfp':
        return [c for c in baseline_image_feature_df.columns if c.startswith('cfp_') and c in numeric_cols]
    if mode == 'roi':
        return [c for c in baseline_image_feature_df.columns if c.startswith('roi_') and c in numeric_cols]
    if mode == 'cfp_roi':
        return [c for c in baseline_image_feature_df.columns if c.startswith(('cfp_', 'roi_')) and c in numeric_cols]
    if mode == 'none':
        return []
    raise ValueError(mode)


def _attach_baseline_image_features(seq_df, mode):
    feature_cols = _feature_columns_for_mode(mode)
    if not feature_cols:
        return np.zeros((len(seq_df), 0), dtype=np.float32), feature_cols

    image_lookup = baseline_image_feature_df[['eye_id'] + feature_cols].drop_duplicates('eye_id')
    merged = seq_df[['eye_id']].merge(image_lookup, on='eye_id', how='left')
    missing_rows = int(merged[feature_cols].isna().any(axis=1).sum())
    if missing_rows:
        raise ValueError(f'{missing_rows} sequence rows are missing baseline image features for mode={mode}')
    return merged[feature_cols].to_numpy(dtype=np.float32), feature_cols


def _build_design_matrix(x_list, t_list, image_matrix):
    last_vf = _last_vf_from_history(x_list)
    hist_summary = _history_summary_features(x_list, t_list)
    return np.concatenate([last_vf, hist_summary, image_matrix], axis=1).astype(np.float32), last_vf


def _fit_best_ridge_residual(X_train, y_train, last_train, X_val, y_val, last_val, alphas=IMAGE_RIDGE_ALPHAS):
    y_res_train = y_train - last_train
    best = None
    for alpha in alphas:
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=float(alpha), random_state=0),
        )
        model.fit(X_train, y_res_train)
        val_pred = last_val + model.predict(X_val)
        val_mae = _mae(y_val, val_pred)
        row = {'alpha': float(alpha), 'val_mae': val_mae, 'model': model}
        if best is None or row['val_mae'] < best['val_mae']:
            best = row
    return best


def run_handcrafted_image_feature_signal_check():
    seq = _prepare_sequence_arrays_for_image_check()

    rows = []
    pred_store = {}
    val_pred_store = {}

    nochange_test_pred = _last_vf_from_history(seq['x_test'])
    nochange_val_pred = _last_vf_from_history(seq['x_val'])
    rows.append({
        'Model': 'No-change baseline',
        'feature_mode': 'last_vf_only',
        'target_mode': 'copy_last_vf',
        'best_alpha': np.nan,
        'val_mae_db': _mae(seq['y_val'], nochange_val_pred),
        'test_mae_db': _mae(seq['y_test'], nochange_test_pred),
        'test_sample_sd_db': _sample_sd(seq['y_test'], nochange_test_pred),
        'image_feature_count': 0,
    })
    pred_store['No-change baseline'] = nochange_test_pred
    val_pred_store['No-change baseline'] = nochange_val_pred

    mode_labels = {
        'none': 'Ridge residual: VF history only',
        'cfp': 'Ridge residual: VF history + baseline CFP features',
        'roi': 'Ridge residual: VF history + baseline ROI features',
        'cfp_roi': 'Ridge residual: VF history + baseline CFP + ROI features',
    }

    for mode, label in mode_labels.items():
        img_train, img_cols = _attach_baseline_image_features(seq['train_df'], mode)
        img_val, _ = _attach_baseline_image_features(seq['val_df'], mode)
        img_test, _ = _attach_baseline_image_features(seq['test_df'], mode)

        X_train, last_train = _build_design_matrix(seq['x_train'], seq['t_train'], img_train)
        X_val, last_val = _build_design_matrix(seq['x_val'], seq['t_val'], img_val)
        X_test, last_test = _build_design_matrix(seq['x_test'], seq['t_test'], img_test)

        best = _fit_best_ridge_residual(X_train, seq['y_train'], last_train, X_val, seq['y_val'], last_val)
        val_pred = last_val + best['model'].predict(X_val)
        test_pred = last_test + best['model'].predict(X_test)

        rows.append({
            'Model': label,
            'feature_mode': mode,
            'target_mode': 'residual_from_last_vf',
            'best_alpha': best['alpha'],
            'val_mae_db': _mae(seq['y_val'], val_pred),
            'test_mae_db': _mae(seq['y_test'], test_pred),
            'test_sample_sd_db': _sample_sd(seq['y_test'], test_pred),
            'image_feature_count': len(img_cols),
        })
        pred_store[label] = test_pred.astype(np.float32)
        val_pred_store[label] = val_pred.astype(np.float32)

    results = pd.DataFrame(rows).sort_values('test_mae_db').reset_index(drop=True)
    vf_only_name = 'Ridge residual: VF history only'
    vf_only_mae = float(results.loc[results['Model'].eq(vf_only_name), 'test_mae_db'].iloc[0])
    results['delta_vs_vf_history_only_db'] = results['test_mae_db'] - vf_only_mae
    results['delta_display'] = results['delta_vs_vf_history_only_db'].map(lambda x: f'{x:+.4f} dB')

    comparisons = []
    for label in mode_labels.values():
        if label == vf_only_name:
            continue
        row = results[results['Model'].eq(label)].iloc[0]
        comparisons.append({
            'Comparison': f'{label} vs VF history only',
            'base_test_mae_db': vf_only_mae,
            'image_test_mae_db': float(row['test_mae_db']),
            'delta_image_minus_base_db': float(row['test_mae_db'] - vf_only_mae),
            'delta_display': f'{float(row["test_mae_db"] - vf_only_mae):+.4f} dB',
        })
    comparisons = pd.DataFrame(comparisons).sort_values('delta_image_minus_base_db').reset_index(drop=True)

    best_image_label = comparisons.iloc[0]['Comparison'].replace(' vs VF history only', '') if len(comparisons) else vf_only_name
    region_rows = []
    for region_name, idx in cluster_mapping.items():
        idx = np.asarray(idx, dtype=int)
        for label in [vf_only_name, best_image_label]:
            pred = pred_store[label]
            region_rows.append({
                'Region': region_name,
                'Model': label,
                'test_mae_db': _mae(seq['y_test'][:, idx], pred[:, idx]),
                'points': int(len(idx)),
            })
    region_results = pd.DataFrame(region_rows)
    region_base = region_results[region_results['Model'].eq(vf_only_name)][['Region', 'test_mae_db']].rename(
        columns={'test_mae_db': 'vf_history_only_mae_db'}
    )
    region_results = region_results.merge(region_base, on='Region', how='left')
    region_results['delta_vs_vf_history_only_db'] = region_results['test_mae_db'] - region_results['vf_history_only_mae_db']
    region_results['delta_display'] = region_results['delta_vs_vf_history_only_db'].map(lambda x: f'{x:+.4f} dB')

    interpretation = pd.DataFrame([
        {
            'Question': 'Do handcrafted baseline CFP/ROI features add signal beyond VF history?',
            'Readout': comparisons.iloc[0]['delta_display'] if len(comparisons) else 'n/a',
            'Interpretation': 'Negative delta means image features improved the simple residual model; positive delta means no useful image signal in this lightweight check.',
        },
        {
            'Question': 'Is this the final image model?',
            'Readout': 'No',
            'Interpretation': 'This only validates mapping and incremental image signal. The next real model should replace handcrafted features with frozen RetFound CFP embeddings.',
        },
        {
            'Question': 'How should this connect to the Transformer?',
            'Readout': 'Use as image-aware residual / prediction-level branch',
            'Interpretation': 'Keep the current Transformer as VF-only baseline, then add image branch conservatively with validation/CV-selected weights.',
        },
    ])

    return {
        'sequence': seq,
        'results': results,
        'comparisons': comparisons,
        'region_results': region_results,
        'interpretation': interpretation,
        'predictions': pred_store,
        'validation_predictions': val_pred_store,
    }

handcrafted_image_signal_outputs = run_handcrafted_image_feature_signal_check()

print('Handcrafted CFP/ROI image feature signal check - overall test MAE')
display(handcrafted_image_signal_outputs['results'])

print('Incremental image comparisons')
display(handcrafted_image_signal_outputs['comparisons'])

print('Region-wise comparison for the best image mode')
display(handcrafted_image_signal_outputs['region_results'])

print('Interpretation')
display(handcrafted_image_signal_outputs['interpretation'])


## Frozen Pretrained Image Embedding Signal Check

This repeats the image pilot with frozen ImageNet ResNet18 embeddings from baseline CFP / ROI images. It is still not RetFound, but it is a closer dry run for the final image branch than handcrafted color/texture features.

Run this after the CFP / ROI mapping audit and the first handcrafted image signal check.

In [ ]:
# Frozen pretrained image embedding pilot: VF history + baseline CFP/ROI embeddings
# This script creates/uses cached ResNet18 embeddings, then compares image-aware
# residual models against the VF-history-only residual baseline.
%run -i ./notebooks2/image_embedding_pilot.py


## Transformer + Image Prediction-Level Soft Ensemble Pilot

This tests the conservative image-integration route: keep the Bern-pretrained VF-only Transformer as the stable backbone, train a separate image-aware residual branch, and let validation choose how much image prediction to trust.


In [ ]:
# Conservative image integration: prediction-level soft ensemble
# The image branch is allowed to help only if validation selects a non-zero image weight.
%run -i ./notebooks2/transformer_image_soft_ensemble_pilot.py


## Transformer Residual Image Correction Pilot

This is a stricter image test: instead of asking images to predict the full future VF, the image branch only tries to explain the residual error left by the Bern-pretrained VF-only Transformer. Validation then chooses the correction strength globally and by anatomical region.


In [ ]:
# Image branch predicts residual error left by the VF-only Transformer.
# Final prediction = Transformer prediction + validation-selected image correction.
%run -i ./notebooks2/transformer_image_residual_correction_pilot.py


## Local VF-estimation image embeddings

Extract 128-dimensional CFP/ROI embeddings from the local ResNet50 checkpoints trained for baseline VF estimation. This is not RETFound, but it is a domain-specific image-representation pilot before the RetFound experiment.

In [ ]:
# Local ophthalmic image embeddings from existing VF-estimation checkpoints
%run -i ./notebooks2/vf_estimation_image_embedding_pilot.py


## RETFound image embeddings

Extract 1024-dimensional CFP/ROI embeddings from the official RETFound MAE CFP foundation model. This requires HuggingFace access to `YukunZhou/RETFound_mae_natureCFP` and local login with `huggingface-cli login --token ...`.

In [ ]:
# RETFound retinal foundation model embeddings.
# If this raises a gated-repo error, request access on HuggingFace and login first.
%run -i ./notebooks2/retfound_image_embedding_pilot.py


## Transformer residual correction with RETFound embeddings

Use RETFound image features to predict the residual error left by the VF-only Transformer. Validation controls the correction strength, so the image branch can be rejected if it is noisy.

In [ ]:
# Test RETFound ROI embeddings as image-derived residual correction.
# Alternatives: "retfound_cfp_embedding" or "retfound_cfp_roi_embedding".
IMAGE_RESIDUAL_FEATURE_MODE = "retfound_roi_embedding"
%run -i ./notebooks2/transformer_image_residual_correction_pilot.py


## Transformer residual correction with VF-estimation image embeddings

Use the current Transformer as the stable VF-history baseline. The image branch only predicts the residual error, and validation decides whether the image correction should be applied.

In [ ]:
# Test local VF-estimation ROI embeddings as the image branch.
# Alternatives to try later: "vf_estimation_cfp_embedding" or "vf_estimation_cfp_roi_embedding".
IMAGE_RESIDUAL_FEATURE_MODE = "vf_estimation_roi_embedding"
%run -i ./notebooks2/transformer_image_residual_correction_pilot.py


In [ ]:
%run -i ./notebooks2/retfound_residual_mode_comparison.py

In [ ]:
%run -i ./notebooks2/retfound_pca_residual_correction_pilot.py

In [ ]:
%run -i ./notebooks2/retfound_image_adapter_fusion_pilot.py

In [ ]:
%run -i ./notebooks2/retfound_image_adapter_sweep_pilot.py

In [ ]:
RETFOUND_ADAPTER_SEEDS = (0, 1, 2)

RETFOUND_ADAPTER_VARIANTS = (
    {
        "image_feature_mode": "retfound_cfp_roi_embedding",
        "pca_components": 8,
        "label": "CFP+ROI PCA8 all-finetune",
        "film_scale": 0.10,
        "gate_max": 0.50,
        "gate_init_bias": -1.5,
        "freeze_non_adapter": False,
    },
)

%run -i ./notebooks2/retfound_image_adapter_fusion_pilot.py

In [ ]:
RETFOUND_ADAPTER_SEEDS = (0, 1, 2)
%run -i ./notebooks2/retfound_image_adapter_pca_sweep_pilot.py

In [ ]:
RETFOUND_ADAPTER_SEEDS = (0, 1, 2)
%run -i ./notebooks2/retfound_image_adapter_final_confirmation.py

In [ ]:
%run -i ./notebooks2/retfound_current_vf_sanity_check.py

In [ ]:
%run -i ./notebooks2/retfound_current_proxy_adapter_pilot.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_adapter_pilot.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_final_confirmation.py

In [ ]:
RETF_REPEATED_SPLIT_SEEDS = (0,)
RETF_REPEATED_MODEL_SEEDS = (0,)
%run -i ./notebooks2/retfound_severity_proxy_repeated_split_check.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_repeated_split_check.py

In [ ]:
RETF_REPEATED_SPLIT_SEEDS = (0, 1, 2)
RETF_REPEATED_MODEL_SEEDS = (0,)
%run -i ./notebooks2/retfound_severity_proxy_repeated_split_check.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_stratified_split_check.py

In [ ]:
RETF_SEVERITY_PROXY_SEEDS = (0, 1, 2, 3, 4)

RETF_SEVERITY_PROXY_VARIANTS = (
    {
        "image_feature_mode": "retfound_roi_embedding",
        "source_pca_components": 8,
        "include_source_pca": False,
        "use_time_features": False,
        "label": "ROI PCA8 -> current severity",
    },
    {
        "image_feature_mode": "retfound_roi_embedding",
        "source_pca_components": 8,
        "include_source_pca": True,
        "use_time_features": False,
        "label": "ROI PCA8 -> current severity + source",
    },
    {
        "image_feature_mode": "retfound_cfp_roi_embedding",
        "source_pca_components": 4,
        "include_source_pca": False,
        "use_time_features": False,
        "label": "CFP+ROI PCA4 -> current severity",
    },
    {
        "image_feature_mode": "retfound_cfp_roi_embedding",
        "source_pca_components": 4,
        "include_source_pca": True,
        "use_time_features": False,
        "label": "CFP+ROI PCA4 -> current severity + source",
    },
)

%run -i ./notebooks2/retfound_severity_proxy_final_confirmation.py

In [ ]:
%run -i ./notebooks2/vf_only_transformer_patient_eye_cv.py

In [ ]:
%run -i ./notebooks2/policy_gate_no_gate_ablation.py

In [ ]:
%run -i ./notebooks2/one_vf_rnfl_transformer_check.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_adapter_pilot.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_effect_audit.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_subgroup_benefit.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_selective_rules.py

In [ ]:
RETF_SUBGROUP_SEEDS = (0, 1, 2)

RETF_SUBGROUP_VARIANT = {
    "image_feature_mode": "retfound_roi_embedding",
    "source_pca_components": 8,
    "include_source_pca": False,
    "use_time_features": False,
    "label": "ROI PCA8 -> current severity",
}

%run -i ./notebooks2/retfound_severity_proxy_subgroup_benefit.py

In [ ]:
%run -i ./notebooks2/retfound_severity_proxy_selective_rules.py

In [ ]:
RETF_SEED_ENSEMBLE_SEEDS = (0, 1, 2)

RETF_SEED_ENSEMBLE_VARIANT = {
    "image_feature_mode": "retfound_roi_embedding",
    "source_pca_components": 8,
    "include_source_pca": False,
    "use_time_features": False,
    "label": "ROI PCA8 -> current severity",
}

%run -i ./notebooks2/retfound_severity_proxy_seed_ensemble.py

In [ ]:
RETF_REGION_CONSENSUS_SEEDS = (0, 1, 2)

RETF_REGION_CONSENSUS_VARIANT = {
    "image_feature_mode": "retfound_roi_embedding",
    "source_pca_components": 8,
    "include_source_pca": False,
    "use_time_features": False,
    "label": "ROI PCA8 -> current severity",
}

%run -i ./notebooks2/retfound_severity_proxy_region_consensus.py

In [ ]:
RETF_GAMMA_REG_SEEDS = (0, 1, 2)

RETF_GAMMA_REG_VARIANT = {
    "image_feature_mode": "retfound_roi_embedding",
    "source_pca_components": 8,
    "include_source_pca": False,
    "use_time_features": False,
    "label": "ROI PCA8 -> current severity",
}

%run -i ./notebooks2/retfound_severity_proxy_gamma_regularization.py

In [ ]:
RETF_PROXY_TARGET_SWEEP_SEEDS = (0, 1, 2)

%run -i ./notebooks2/retfound_proxy_target_sweep.py

## Optional structure-function discordance modules (unchanged VF backbone)

This experiment keeps the validated six-region VF Transformer unchanged and asks whether optional structural information explains its remaining regional errors. It compares raw baseline RNFL, observed RNFL minus VF-expected RNFL, and ROI-image-predicted RNFL minus VF-expected RNFL.

Safeguards: patient-eye cross-fitted training residuals, validation-only alpha/gamma selection, six regional corrections rather than a new 59-point model, and exact VF-only fallback when the selected gamma is zero or a modality is missing. This is a fixed-clinical-split pilot; promising modules must still proceed to outer patient-eye CV.

In [ ]:
# Structure-function discordance pilot on the same six-region VF backbone.
# Prerequisite: run the RETFound embedding cell in this notebook first.
SF_DISCORDANCE_SEEDS = (0,)
SF_DISCORDANCE_CROSSFIT_FOLDS = 3
SF_DISCORDANCE_IMAGE_MODE = 'retfound_roi_embedding'
SF_DISCORDANCE_PCA_COMPONENTS = 8

%run -i ./notebooks2/retfound_structure_function_discordance_pilot.py

## Image-only optional module: supervised no-PCA linear progression route

This experiment isolates image information. Frozen RETFound ROI embeddings are mapped directly from 1024 dimensions to six regional annualized VF-change estimates using strongly regularized multi-output Ridge regression, with no PCA and no RNFL. The validated VF-only Transformer remains unchanged.

Ridge alpha is chosen by patient-eye GroupKFold within training data. Validation selects one global late-fusion gamma, and the image module is activated only when eye-balanced validation MAE improves by at least 0.001 dB. Missing or rejected images give an exact VF-only fallback.

In [ ]:
# Image-only supervised linear route: RETFound ROI 1024 -> Ridge -> six progression rates.
IMAGE_LINEAR_SEEDS = (0,)
IMAGE_LINEAR_FEATURE_MODE = 'retfound_roi_embedding'
IMAGE_LINEAR_INNER_FOLDS = 3
IMAGE_LINEAR_MIN_VAL_GAIN_DB = 0.001
IMAGE_LINEAR_MIN_DELTA_T = 0.25

%run -i ./notebooks2/retfound_image_only_linear_progression_module.py

## Image-only optional module: robust eye-level longitudinal slopes

The previous one-step annualized-change target was dominated by short-term VF fluctuation. This final image-only diagnostic instead uses every available VF visit from each patient-eye to estimate six robust long-term regional slopes (median pairwise/Theil-Sen slopes). Baseline ROI RETFound embeddings are mapped directly from 1024 dimensions to those six slopes using strongly regularized multi-output Ridge regression, without PCA, RNFL, FiLM, or a new Transformer.

The validated six-region VF Transformer remains unchanged. The image module only proposes regional mean corrections; one global gamma is selected on validation, and a missing or rejected image gives exact VF-only fallback.

In [ ]:
# Final image-only diagnostic: baseline ROI -> no-PCA Ridge -> six robust long-term slopes.
# Prerequisite: run the RETFound embedding cell in this notebook first.
IMAGE_SLOPE_SEEDS = (0,)
IMAGE_SLOPE_FEATURE_MODE = 'retfound_roi_embedding'
IMAGE_SLOPE_INNER_FOLDS = 3
IMAGE_SLOPE_MIN_VISITS = 3
IMAGE_SLOPE_MIN_SPAN_YEARS = 1.0
IMAGE_SLOPE_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_image_longitudinal_slope_module.py

## Optional image module: no-PCA, train-only PCA, and supervised projection

This experiment keeps the validated six-region VF Transformer unchanged and compares three shallow ROI image modules on the same robust eye-level longitudinal-slope target: standardized no-PCA Ridge, train-only PCA followed by Ridge, and supervised Partial Least Squares (PLS) projection.

PCA dimensions, PLS dimensions, and Ridge strengths are selected by patient-eye GroupKFold using training eyes only. Projection family and late-fusion gamma are selected on validation only. Exactly one selected route is applied to test; if validation rejects every image route, the prediction falls back exactly to VF-only.

In [ ]:
# Fair optional-module comparison on the unchanged VF backbone.
# Prerequisite: run the RETFound embedding cell in this notebook first.
IMAGE_PROJECTION_SEEDS = (0,)
IMAGE_PROJECTION_FEATURE_MODE = 'retfound_roi_embedding'
IMAGE_PROJECTION_INNER_FOLDS = 3
IMAGE_PROJECTION_PCA_COMPONENTS = (4, 8, 12, 16, 24, 32)
IMAGE_PROJECTION_PLS_COMPONENTS = (1, 2, 4, 6, 8, 12)
IMAGE_PROJECTION_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_image_projection_comparison.py

## Image-only optional module: current-severity discordance

This experiment stops asking a baseline image to directly predict VF progression. Instead, no-PCA Ridge, train-only PCA + Ridge, and supervised PLS first translate the ROI RETFound representation into six baseline regional VF-severity estimates. The image-predicted severity minus the observed baseline VF severity forms a six-dimensional structure-function discordance signal.

An independent eye-balanced Ridge module tests whether this discordance explains patient-eye cross-fitted residuals from the unchanged six-region VF Transformer. Image proxy hyperparameters use training-eye GroupKFold; projection family, residual regularization and gamma use validation only; test is evaluated once. Rejected or missing image information gives exact VF-only fallback.

In [ ]:
# Image current-severity proxy -> image/VF discordance -> optional six-region residual module.
# This cell trains three cross-fitted VF backbones and can take longer than prior pilots.
IMAGE_SEVERITY_SEEDS = (0,)
IMAGE_SEVERITY_FEATURE_MODE = 'retfound_roi_embedding'
IMAGE_SEVERITY_CROSSFIT_FOLDS = 3
IMAGE_SEVERITY_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_image_severity_discordance_module.py

## Teacher-aligned internal optional image token

This experiment implements image as an internal optional modality module rather than an output correction. For each of the six regional compact Transformer models, one ROI image token is appended to the longitudinal VF point tokens before encoder self-attention. The image model starts from the corresponding GRAPE-fine-tuned VF-only regional state; the VF encoders, Transformer, decoder and prediction head are frozen during this pilot, while only the image projector and image type embedding are trained.

Validation compares train-only PCA12, train-only PCA32, and a learned no-PCA projector using patient-eye-balanced MAE. NULL_IMAGE is attention-masked and must reproduce VF-only numerically. Modality dropout trains the image model to tolerate missing images. After validation selects one real-image variant, a separately trained shuffled-image control tests whether any gain is patient-specific. Test labels are not used for variant selection.

In [ ]:
# Internal image-token fusion: VF tokens + optional ROI token -> same compact Transformer.
# This trains 6 VF-only regional baselines, 3 projector variants, then 1 shuffled control.
# It is therefore longer-running than the previous shallow image diagnostics.
INTERNAL_IMAGE_SEEDS = (0,)
INTERNAL_IMAGE_FEATURE_MODE = 'retfound_roi_embedding'
INTERNAL_IMAGE_VARIANTS = (
    {'label': 'ROI PCA12 encoder token', 'pca_components': 12, 'image_hidden_dim': 32},
    {'label': 'ROI PCA32 encoder token', 'pca_components': 32, 'image_hidden_dim': 32},
    {'label': 'ROI learned encoder token', 'pca_components': 0, 'image_hidden_dim': 64},
)
INTERNAL_IMAGE_MODALITY_DROPOUT = 0.30
INTERNAL_IMAGE_MIN_VAL_GAIN_DB = 0.001
INTERNAL_IMAGE_EPOCHS = 20

%run -i ./notebooks2/retfound_internal_encoder_image_token.py

## Shared internal image module across all six VF regions

The previous encoder-token pilot used six independently trained image projectors and strongly overfit validation. This experiment implements the supervisor's diagram more literally: one shared ROI image projector creates one 32-dimensional image token, the same token enters all six frozen regional compact Transformers before self-attention, and one joint 59-point loss trains the shared image module.

Only two pre-specified low-capacity variants are compared: train-only PCA12 followed by one linear 12-to-32 projection, and a no-PCA 1024-to-16-to-32 bottleneck. Training uses eye-balanced sampling, AdamW weight decay, modality dropout, exact attention-masked NULL_IMAGE fallback, and a shuffled-image control for the validation-selected variant. The VF backbones and prediction heads remain fixed.

In [ ]:
# One shared image projector/token jointly controls all six frozen regional Transformers.
# Runtime: 6 regional baseline fits + 2 joint shared-projector fits + 1 shuffled joint fit.
SHARED_IMAGE_SEEDS = (0,)
SHARED_IMAGE_FEATURE_MODE = 'retfound_roi_embedding'
SHARED_IMAGE_VARIANTS = (
    {'label': 'Shared ROI PCA12 linear token', 'pca_components': 12, 'projector_kind': 'linear'},
    {'label': 'Shared ROI learned bottleneck token', 'pca_components': 0, 'projector_kind': 'bottleneck', 'bottleneck_dim': 16},
)
SHARED_IMAGE_MODALITY_DROPOUT = 0.30
SHARED_IMAGE_WEIGHT_DECAY = 1e-3
SHARED_IMAGE_MIN_VAL_GAIN_DB = 0.001
SHARED_IMAGE_EPOCHS = 25

%run -i ./notebooks2/retfound_shared_internal_image_module.py

## Outer patient-eye CV for the shared internal image module

The shared internal-token architecture and its two candidate projectors are now locked before evaluation. Three case-mix-stratified outer patient-eye folds are used. Within each outer-training cohort, a fresh inner validation subset selects PCA12 versus the learned 1024-to-16-to-32 bottleneck. Standardization/PCA, VF-only fine-tuning, image-module training, early stopping, and shuffled-control training are repeated independently in every fold.

Each untouched outer-test fold is evaluated once. The primary comparison is real shared image versus VF-only and shuffled image, reported as mean delta plus fold variability. NULL_IMAGE remains attention-masked. This experiment is intentionally longer-running but avoids further development against the repeatedly inspected fixed clinical test split.

In [ ]:
# Locked shared-image architecture under untouched outer patient-eye folds.
# Expected runtime is substantially longer than the fixed-split pilots.
SHARED_IMAGE_CV_OUTER_FOLDS = 3
SHARED_IMAGE_CV_SPLIT_SEED = 2026
SHARED_IMAGE_CV_INNER_VAL_FRACTION = 0.18
SHARED_IMAGE_CV_MODEL_SEED = 0
INTERNAL_IMAGE_EPOCHS = 15
SHARED_IMAGE_EPOCHS = 15
SHARED_IMAGE_VARIANTS = (
    {'label': 'Shared ROI PCA12 linear token', 'pca_components': 12, 'projector_kind': 'linear'},
    {'label': 'Shared ROI learned bottleneck token', 'pca_components': 0, 'projector_kind': 'bottleneck', 'bottleneck_dim': 16},
)
SHARED_IMAGE_MODALITY_DROPOUT = 0.30
SHARED_IMAGE_WEIGHT_DECAY = 1e-3
SHARED_IMAGE_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_shared_internal_image_outer_cv.py

## Spatial RETFound patch-token module inside the six-region VF backbone

The preceding shared-token experiment used one globally pooled 1024-dimensional RETFound vector. It therefore discarded where retinal patterns occurred. This experiment changes the **image representation**, not the validated VF backbone: each canonicalized full CFP is retained as a `14 x 14 x 1024` RETFound patch grid, with OS eyes horizontally flipped to the same retinal orientation as OD eyes.

Patch channels are standardized and reduced to 16 dimensions using training eyes only. One pre-specified lightweight shared spatial projector (`3 x 3 convolution -> 3 x 3 convolution -> 2 x 2 pooling -> 32-dimensional token`) is trained jointly across all six frozen regional Transformers. The token enters before Transformer self-attention; no 59-point late residual branch is used.

The experiment includes three safeguards: a shuffled-eye spatial-image control, an attention-masked `NULL_IMAGE` path that must numerically recover VF-only prediction, and validation-only activation with a minimum `0.001 dB` gain. This is a fixed-split representation audit; outer patient-eye CV is warranted only if real spatial pairing beats both VF-only and shuffled pairing.

In [ ]:
# One locked spatial CFP module shared across the six unchanged regional VF backbones.
# Prerequisite: the canonical RETFound patch cache generated earlier in this notebook.
SPATIAL_IMAGE_SEEDS = (0,)
SPATIAL_IMAGE_CHANNEL_COMPONENTS = 16
SPATIAL_IMAGE_PCA_PATCH_SAMPLE = 8192
SPATIAL_IMAGE_CONV_CHANNELS = 16
SPATIAL_IMAGE_MODALITY_DROPOUT = 0.30
SPATIAL_IMAGE_PROJECTOR_DROPOUT = 0.20
SPATIAL_IMAGE_WEIGHT_DECAY = 1e-3
SPATIAL_IMAGE_MIN_VAL_GAIN_DB = 0.001
SPATIAL_IMAGE_EPOCHS = 20

%run -i ./notebooks2/retfound_spatial_patch_internal_image_module.py

## Shared progression-proxy token with an unchanged VF backbone

This experiment combines the strongest inductive bias from the earlier exploratory image adapter with the stricter optional-module interface. The six GRAPE-fine-tuned regional VF Transformers are copied unchanged and frozen. A baseline ROI RETFound embedding is standardized and reduced to PCA12 using training patient-eyes only, then translated by a frozen multi-output Ridge model into seven annualized-progression values: global mean-TD change per year plus one change rate for each of the six VF regions.

Ridge regularization is selected by three-fold CV entirely inside the training patient-eyes. Multiple sequence windows from one eye are first aggregated to a median eye-level progression target. Training sequences receive cross-fitted out-of-fold proxy predictions, preventing the token model from seeing in-sample Ridge predictions; validation and test receive predictions from a Ridge refit on all training eyes.

Only one shared `7 -> 32` proxy-token projection is trainable. The token enters each frozen regional Transformer before self-attention, all 59 outputs are optimized jointly, and the previously motivated auxiliary regional-mean weight is fixed at `0.10`. There is no all-finetuning and no output-level gamma blending. `NULL_PROXY` must exactly recover VF-only prediction, and a separately trained shuffled-eye proxy control tests patient-specific information.

In [ ]:
# Locked compromise: old progression-proxy prior, current strict optional-token interface.
PROXY_TOKEN_SEEDS = (0,)
PROXY_TOKEN_PCA_COMPONENTS = 12
PROXY_TOKEN_INNER_FOLDS = 3
PROXY_TOKEN_AUX_WEIGHT = 0.10
PROXY_TOKEN_MODALITY_DROPOUT = 0.30
PROXY_TOKEN_WEIGHT_DECAY = 1e-3
PROXY_TOKEN_MIN_VAL_GAIN_DB = 0.001
PROXY_TOKEN_EPOCHS = 20

%run -i ./notebooks2/retfound_shared_progression_proxy_token.py

## History-length diagnostic for the shared progression-proxy token

The strict proxy-token model improved eye-balanced validation MAE but worsened held-out test MAE, and real pairing did not beat shuffled pairing. Before abandoning the short-history hypothesis, this diagnostic reruns the identical locked model and reports effects separately for `k=2`, `k=3`, and `k=4`.

Both validation and test deltas are shown with patient-eye-balanced aggregation. These rows are **post-hoc diagnostics only**: no history-length gate is selected from test results. A gate would be considered only if the same pre-specified history group benefits on validation and later repeated patient-eye splits.

In [ ]:
# Identical locked proxy-token experiment; only subgroup diagnostics are added.
PROXY_TOKEN_SEEDS = (0,)
PROXY_TOKEN_PCA_COMPONENTS = 12
PROXY_TOKEN_INNER_FOLDS = 3
PROXY_TOKEN_AUX_WEIGHT = 0.10
PROXY_TOKEN_MODALITY_DROPOUT = 0.30
PROXY_TOKEN_WEIGHT_DECAY = 1e-3
PROXY_TOKEN_MIN_VAL_GAIN_DB = 0.001
PROXY_TOKEN_EPOCHS = 20

%run -i ./notebooks2/retfound_shared_progression_proxy_token.py

## Canonical optic-disc ROI RETFound patch cache

The supplied GRAPE ROI images are optic-disc-centered crops rather than generic central crops. This cell extracts a frozen RETFound `14 x 14 x 1024` patch grid from each baseline ROI. OS images are horizontally flipped before feature extraction so temporal/nasal ONH directions match the OD-like canonical orientation. No VF labels or train/validation/test assignments are used during frozen feature extraction.

In [ ]:
# First run builds the cache; later runs load it directly.
RETFOUND_ROI_PATCH_BATCH_SIZE = 4
RETFOUND_ROI_PATCH_FORCE_REBUILD = False

%run -i ./notebooks2/retfound_roi_patch_embedding_pilot.py

## Garway-Heath-inspired anatomy-routed ROI tokens

This is a fixed anatomy-constrained pilot, not a claim of exact clinical Garway-Heath registration. Training-eye ROI patches are channel-standardized and reduced to PCA16. Six pre-specified ONH masks pool infero-temporal, infero-nasal, papillomacular-temporal, supero-temporal, supero-nasal, and nasal sectors.

Each frozen VF regional Transformer receives only its corresponding ONH sector token: superior VF routes from inferior ONH, inferior VF routes from superior ONH, macular VF routes from the temporal papillomacular sector, and temporal VF routes from nasal ONH. A single shared `16 -> 32` projector is trained jointly across all 59 outputs. No output correction, gamma blending, all-finetuning, or sector-map selection is performed. Real, shuffled, and attention-masked `NULL_ROI` controls are reported.

In [ ]:
# Locked low-capacity anatomy-routed optional image module.
ANATOMY_ROI_SEEDS = (0,)
ANATOMY_ROI_CHANNEL_COMPONENTS = 16
ANATOMY_ROI_PCA_PATCH_SAMPLE = 8192
ANATOMY_ROI_MODALITY_DROPOUT = 0.30
ANATOMY_ROI_WEIGHT_DECAY = 1e-3
ANATOMY_ROI_MIN_VAL_GAIN_DB = 0.001
ANATOMY_ROI_EPOCHS = 20

%run -i ./notebooks2/retfound_anatomy_routed_roi_token.py

## Anatomy-routed image modulation of regional VF velocity

This final constrained image experiment does not predict an arbitrary 59-point residual and does not alter the six-region VF Transformer. For each region, the frozen VF-only prediction is written as a change from the last observed VF. The matched optic-disc ROI sector may only multiply that predicted change by a bounded factor in `[0.75, 1.25]`. A missing image uses factor `1`, exactly recovering VF-only.

The six scale targets are derived from patient-eye cross-fitted training predictions, and Ridge strength is selected using training eyes only. Validation must show that real image scaling improves both VF-only and a no-image constant regional calibration before activation. Held-out test reports raw real-image, shuffled-pairing, constant-calibration, NULL, and policy-final results.

In [ ]:
# Final constrained image experiment; this retrains cross-fitted VF baselines.
VELOCITY_SEEDS = (0,)
VELOCITY_CROSSFIT_FOLDS = 3
VELOCITY_SCALE_MIN = 0.75
VELOCITY_SCALE_MAX = 1.25
VELOCITY_MIN_VAL_GAIN_DB = 0.001
VELOCITY_MIN_IMAGE_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/retfound_anatomy_velocity_scaling.py

## Does image help a weaker LSTM backbone?

The older Bern-pretrained, GRAPE-fine-tuned six-region LSTM achieved about `2.223 dB` VF-only MAE, compared with about `2.162 dB` for the compact Transformer in the strict image pilots. This experiment tests the pre-specified backbone-saturation hypothesis: image may have no measurable marginal value after the stronger Transformer has already extracted most predictable signal, but may compensate for unused capacity in LSTM.

Each regional LSTM is first fine-tuned normally and then frozen. One shared optional ROI image module applies bounded FiLM modulation to the six LSTM hidden states immediately before their unchanged decoders. Training-only PCA12 and a learned bottleneck are compared on validation. Real pairing, shuffled pairing, and exact `NULL_IMAGE` fallback are reported. A benefit only on LSTM means compensation for a weaker backbone; it does not override the Transformer result.

In [ ]:
# Backbone-saturation diagnostic using the old six-region LSTM protocol.
LSTM_IMAGE_SEEDS = (0,)
LSTM_BASE_EPOCHS = 20
LSTM_IMAGE_EPOCHS = 25
LSTM_BASE_LR = 5e-4
LSTM_IMAGE_LR = 3e-4
LSTM_IMAGE_FILM_SCALE = 0.10
LSTM_IMAGE_MODALITY_DROPOUT = 0.30
LSTM_IMAGE_WEIGHT_DECAY = 1e-3
LSTM_IMAGE_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_shared_lstm_image_film.py

## Locked three-seed confirmation of LSTM image-FiLM

The seed-0 pilot improved LSTM-only test MAE from `2.2223` to `2.2133 dB` (`-0.0090 dB`) and real pairing beat shuffled pairing by `0.0345 dB`. The benefit was concentrated in Supero_Nasal (`-0.0555 dB`).

This confirmation does not repeat architecture selection. The learned-bottleneck image-FiLM found in the pilot is now locked and rerun with seeds 0, 1, and 2 on the same clinical split. The required evidence is a negative mean image delta plus consistent real-versus-shuffled superiority. Only then should this architecture proceed to outer patient-eye cross-validation.

In [ ]:
# Locked architecture: no PCA-versus-bottleneck re-selection in confirmation.
LSTM_IMAGE_SEEDS = (0, 1, 2)
LSTM_IMAGE_VARIANTS = (
    {
        'label': 'LSTM + locked shared ROI learned-bottleneck image-FiLM',
        'pca_components': 0,
        'projector_kind': 'bottleneck',
        'bottleneck_dim': 16,
    },
)
LSTM_BASE_EPOCHS = 20
LSTM_IMAGE_EPOCHS = 25
LSTM_BASE_LR = 5e-4
LSTM_IMAGE_LR = 3e-4
LSTM_IMAGE_FILM_SCALE = 0.10
LSTM_IMAGE_MODALITY_DROPOUT = 0.30
LSTM_IMAGE_WEIGHT_DECAY = 1e-3
LSTM_IMAGE_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_shared_lstm_image_film.py

## Outer patient-eye CV for locked LSTM image-FiLM

The locked learned-bottleneck image-FiLM improved all three fixed-split validation runs and reduced mean test MAE from `2.2194` to `2.2124 dB` (`-0.0070 dB`). Real pairing outperformed shuffled pairing by `0.0382 dB`, and NULL_IMAGE was exact.

This final confirmation changes patient-eye composition rather than only optimization seed. Three stratified outer folds are built from all eligible eyes. Within each fold, the six-region Bern-to-GRAPE LSTM, image normalization, and learned bottleneck are refitted from scratch. The architecture is fixed; inner validation only decides activation, and each outer test fold remains untouched until reporting.

In [ ]:
# Final split-robustness test of the locked LSTM image module.
LSTM_IMAGE_CV_OUTER_FOLDS = 3
LSTM_IMAGE_CV_SPLIT_SEED = 2026
LSTM_IMAGE_CV_INNER_VAL_FRACTION = 0.18
LSTM_IMAGE_CV_MODEL_SEED = 0
LSTM_IMAGE_CV_LOCKED_VARIANT = {
    'label': 'Locked shared ROI learned-bottleneck image-FiLM',
    'pca_components': 0,
    'projector_kind': 'bottleneck',
    'bottleneck_dim': 16,
}
LSTM_BASE_EPOCHS = 20
LSTM_IMAGE_EPOCHS = 25
LSTM_BASE_LR = 5e-4
LSTM_IMAGE_LR = 3e-4
LSTM_IMAGE_FILM_SCALE = 0.10
LSTM_IMAGE_MODALITY_DROPOUT = 0.30
LSTM_IMAGE_WEIGHT_DECAY = 1e-3
LSTM_IMAGE_MIN_VAL_GAIN_DB = 0.001

%run -i ./notebooks2/retfound_shared_lstm_image_outer_cv.py

## Minimal all-finetune Transformer image test

Earlier Transformer image-adapter gains were obtained when the image adapter and Transformer were fine-tuned together. The later strict shared-token experiments froze the Transformer and did not generalize. This experiment changes only that training decision: the same shared ROI learned-bottleneck image token is retained, while the six regional Transformers update with a low learning rate.

Four paths are compared: original VF-only Transformer, real-image all-finetune, shuffled-image all-finetune, and `NULL_IMAGE` extra-finetune. The NULL control is essential because otherwise an apparent gain could come from simply training the Transformer for additional epochs. Validation activates the image route only if real image beats both the original baseline and the NULL extra-finetune control. Missing or rejected images are deployed through the separately retained original VF-only model.

In [ ]:
# One locked architecture; only frozen versus all-finetune is being tested.
TRANSFORMER_ALL_FT_SEEDS = (0,)
TRANSFORMER_ALL_FT_VARIANT = {
    'label': 'Shared ROI learned-bottleneck token all-finetune',
    'pca_components': 0,
    'projector_kind': 'bottleneck',
    'bottleneck_dim': 16,
}
TRANSFORMER_ALL_FT_EPOCHS = 15
TRANSFORMER_ALL_FT_IMAGE_LR = 3e-4
TRANSFORMER_ALL_FT_BACKBONE_LR = 1e-5
TRANSFORMER_ALL_FT_WEIGHT_DECAY = 1e-3
TRANSFORMER_ALL_FT_MODALITY_DROPOUT = 0.30
TRANSFORMER_ALL_FT_MIN_VAL_GAIN_DB = 0.001
TRANSFORMER_ALL_FT_MIN_IMAGE_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/retfound_shared_transformer_all_finetune.py

## Optional visit-aligned IOP module (unchanged VF backbone)

This experiment starts the IOP module independently from image and RNFL. The validated six-region compact Transformer is retained and frozen. A single small IOP projector is shared across all six regions and adds an IOP-derived update only to the VF tokens from the matching input visit. This preserves IOP as a dynamic longitudinal measurement rather than collapsing it into a static late correction.

`NULL_IOP` produces an exact zero update. A patient-eye-blocked shuffled-IOP model controls for extra parameters, and validation enables the real IOP module only if it improves over both VF-only and shuffled IOP. Test labels are not used for module activation.

In [ ]:
# Minimal supervisor-style IOP module: same backbone, one optional shared module.
# Locked three-seed replication: architecture and hyperparameters unchanged.
IOP_TOKEN_SEEDS = (0, 1, 2)
IOP_TOKEN_EPOCHS = 25
IOP_TOKEN_BATCH_SIZE = 64
IOP_TOKEN_LR = 5e-4
IOP_TOKEN_WEIGHT_DECAY = 1e-3
IOP_TOKEN_MODALITY_DROPOUT = 0.25
IOP_TOKEN_HIDDEN_DIM = 8
IOP_TOKEN_UPDATE_SCALE = 0.10
IOP_TOKEN_MIN_VAL_GAIN_DB = 0.001
IOP_TOKEN_MIN_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/optional_iop_visit_token_transformer.py

## Optional baseline-RNFL token (unchanged VF backbone)

This experiment evaluates RNFL independently from image and IOP. RNFL is defined strictly from the first observed visit for each patient-eye. The five measurements (`RNFL_Mean`, `RNFL_S`, `RNFL_N`, `RNFL_I`, `RNFL_T`) are standardized using unique training eyes only and projected into one shared structural token. The same token enters each of the six frozen regional compact Transformers.

`NULL_RNFL` is attention-masked and must exactly recover VF-only predictions. A patient-eye shuffled RNFL model controls for extra capacity. The real module is activated only when validation shows improvement over both VF-only and shuffled RNFL; test labels remain reporting-only. The architecture and thresholds are locked across three seeds.

In [ ]:
# Locked three-seed baseline-RNFL module test.
RNFL_TOKEN_SEEDS = (0, 1, 2)
RNFL_TOKEN_COLS = ('RNFL_Mean', 'RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T')
RNFL_TOKEN_EPOCHS = 25
RNFL_TOKEN_LR = 5e-4
RNFL_TOKEN_WEIGHT_DECAY = 1e-3
RNFL_TOKEN_MODALITY_DROPOUT = 0.25
RNFL_TOKEN_PROJECTOR_DROPOUT = 0.20
RNFL_TOKEN_HIDDEN_DIM = 8
RNFL_TOKEN_MIN_VAL_GAIN_DB = 0.001
RNFL_TOKEN_MIN_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/optional_rnfl_token_transformer.py

## IOP and RNFL on the weaker six-region LSTM backbone

This is a backbone-saturation diagnostic analogous to the shared image-FiLM experiment. The same Bern-to-GRAPE six-region LSTM baseline is trained once per seed and then frozen. IOP and RNFL are tested as two independent optional modules, never combined.

Visit-aligned IOP histories are encoded by one small shared GRU and converted into region-conditioned FiLM parameters. First-visit RNFL Mean/S/N/I/T values are converted by one shared bottleneck projector into the same type of region-conditioned FiLM modulation. Both experiments use three seeds, patient-eye shuffled controls, exact NULL fallback, and validation-only activation requiring improvement over both LSTM-only and shuffled data.

In [ ]:
# Locked LSTM backbone-saturation diagnostic; IOP and RNFL remain independent.
LSTM_CLINICAL_SEEDS = (0, 1, 2)
LSTM_CLINICAL_EPOCHS = 25
LSTM_CLINICAL_BATCH_SIZE = 64
LSTM_CLINICAL_LR = 3e-4
LSTM_CLINICAL_WEIGHT_DECAY = 1e-3
LSTM_CLINICAL_MODALITY_DROPOUT = 0.30
LSTM_CLINICAL_FILM_SCALE = 0.10
LSTM_CLINICAL_MIN_VAL_GAIN_DB = 0.001
LSTM_CLINICAL_MIN_SPECIFIC_GAIN_DB = 0.0005
LSTM_RNFL_COLS = ('RNFL_Mean', 'RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T')

%run -i ./notebooks2/optional_lstm_iop_rnfl_diagnostic.py

## Visit-aligned longitudinal ROI image module

This experiment uses every available historical ROI image rather than one static baseline image. Filenames and the clinical workbook jointly map each image to `(patient-eye, visit)`. A training-eye-only PCA12 and one small shared projector create a 32-dimensional update that is added only to VF tokens from the same visit. The six validated regional Transformers remain frozen. Missing visits contribute exactly zero, and real pairing must beat both VF-only and a patient-eye-shuffled longitudinal-image control on validation before activation.

In [ ]:
# Extend the existing 263-image baseline ROI cache with all follow-up ROI images.
# The extractor is incremental, so already cached baseline embeddings are reused.
RETFOUND_BATCH_SIZE = 4
%run -i ./notebooks2/retfound_image_embedding_pilot.py

In [ ]:
# Locked three-seed dynamic-image experiment.
DYNAMIC_IMAGE_SEEDS = (0, 1, 2)
DYNAMIC_IMAGE_EPOCHS = 25
DYNAMIC_IMAGE_LR = 5e-4
DYNAMIC_IMAGE_WEIGHT_DECAY = 1e-3
DYNAMIC_IMAGE_MODALITY_DROPOUT = 0.25
DYNAMIC_IMAGE_PCA_COMPONENTS = 12
DYNAMIC_IMAGE_HIDDEN_DIM = 8
DYNAMIC_IMAGE_UPDATE_SCALE = 0.10
DYNAMIC_IMAGE_MIN_VAL_GAIN_DB = 0.001
DYNAMIC_IMAGE_MIN_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/retfound_dynamic_visit_image_transformer.py

## Longitudinal ROI change-signal diagnostic

This diagnostic does not retrain or modify the Transformer. For each patient-eye with at least two available ROI images, it computes annualized RETFound embedding change between successive imaged visits and tests whether that change predicts concurrent global and six-region TD change. PCA and grouped Ridge selection use training patient-eyes only. Validation/test are compared with a training-mean predictor and a shuffled patient-eye image-change control.

In [ ]:
# Independent signal diagnostic; no Transformer weights are trained or changed.
IMAGE_CHANGE_PCA_COMPONENTS = 12
IMAGE_CHANGE_MIN_INTERVAL_YEARS = 0.25
IMAGE_CHANGE_INNER_FOLDS = 3
IMAGE_CHANGE_ALPHAS = np.logspace(-3, 6, 13)
IMAGE_CHANGE_SHUFFLE_SEED = 4401

%run -i ./notebooks2/retfound_longitudinal_image_change_diagnostic.py

## Visit-aligned longitudinal ROI image module on LSTM

This replaces the earlier static patient-level image-FiLM comparison with the same longitudinal image definition used by the dynamic Transformer experiment. Each available ROI image is matched to one historical VF visit, projected through training-only PCA12 and one shared small adapter, and added only to the encoded VF representation from that visit before LSTM recurrence. The six Bern-to-GRAPE regional LSTMs are then frozen. Real, patient-eye-shuffled, and exact NULL controls are repeated for seeds 0, 1, and 2; validation must approve both absolute and image-specific gain before deployment.

In [ ]:
# Locked three-seed LSTM test using visit-aligned longitudinal ROI images.
DYNAMIC_LSTM_IMAGE_SEEDS = (0, 1, 2)
DYNAMIC_LSTM_IMAGE_EPOCHS = 25
DYNAMIC_LSTM_IMAGE_BATCH_SIZE = 64
DYNAMIC_LSTM_IMAGE_LR = 3e-4
DYNAMIC_LSTM_IMAGE_WEIGHT_DECAY = 1e-3
DYNAMIC_LSTM_IMAGE_MODALITY_DROPOUT = 0.25
DYNAMIC_LSTM_IMAGE_PCA_COMPONENTS = 12
DYNAMIC_LSTM_IMAGE_HIDDEN_DIM = 8
DYNAMIC_LSTM_IMAGE_UPDATE_SCALE = 0.10
DYNAMIC_LSTM_IMAGE_MIN_VAL_GAIN_DB = 0.001
DYNAMIC_LSTM_IMAGE_MIN_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/retfound_dynamic_visit_image_lstm.py

## Supervisor-style no-PCA longitudinal ROI route

This is the strict spatial counterpart of the visit-aligned image experiment. Every available historical ROI is represented by its full frozen RETFound 14 x 14 x 1024 patch grid. A single shared learned 1 x 1 channel projection, spatial convolution and pooling module creates a bounded update for only the matching VF visit; no PCA or global image pooling is used. The six validated regional Transformers remain frozen. Real, patient-eye-shuffled and exact NULL controls are evaluated for three seeds, and activation uses validation labels only.

In [ ]:
# Incrementally extend the 263-image baseline patch cache to all longitudinal ROIs.
# Existing patch grids are reused; only missing follow-up images run through RETFound.
RETFOUND_LONGITUDINAL_ROI_PATCH_BATCH_SIZE = 4
RETFOUND_LONGITUDINAL_ROI_PATCH_FORCE_REBUILD = False
%run -i ./notebooks2/retfound_longitudinal_roi_patch_embedding.py

In [ ]:
# Locked three-seed test of the no-PCA convolution/pooling image route.
DYNAMIC_SPATIAL_IMAGE_SEEDS = (0, 1, 2)
DYNAMIC_SPATIAL_IMAGE_EPOCHS = 25
DYNAMIC_SPATIAL_IMAGE_BATCH_SIZE = 16
DYNAMIC_SPATIAL_IMAGE_CONV_CHANNELS = 8
DYNAMIC_SPATIAL_IMAGE_LR = 3e-4
DYNAMIC_SPATIAL_IMAGE_WEIGHT_DECAY = 1e-3
DYNAMIC_SPATIAL_IMAGE_MODALITY_DROPOUT = 0.25
DYNAMIC_SPATIAL_IMAGE_PROJECTOR_DROPOUT = 0.20
DYNAMIC_SPATIAL_IMAGE_UPDATE_SCALE = 0.10
DYNAMIC_SPATIAL_IMAGE_MIN_VAL_GAIN_DB = 0.001
DYNAMIC_SPATIAL_IMAGE_MIN_SPECIFIC_GAIN_DB = 0.0005

%run -i ./notebooks2/retfound_dynamic_visit_spatial_image_transformer.py

In [ ]:
%run -i ./notebooks2/rnfl_failure_diagnostic.py

In [ ]:
%run -i ./notebooks2/optional_rnfl_cross_attention_transformer.py

In [ ]:
RNFL_FUSION_ROUTES = ("post_self_attention_cross_attention",)

%run -i ./notebooks2/optional_rnfl_cross_attention_transformer.py

In [ ]:
%run -i ./notebooks2/optional_rnfl_cross_attention_transformer.py
%run -i ./notebooks2/rnfl_failure_diagnostic.py

In [ ]:
%run -i ./notebooks2/rnfl_failure_diagnostic.py

In [ ]:
%run -i ./notebooks2/optional_iop_cross_attention_transformer.py

In [ ]:
%run -i ./notebooks2/iop_failure_diagnostic.py

In [ ]:
%run -i ./notebooks2/optional_image_cross_attention_transformer.py

In [ ]:
"optional_image_cross_attention_outputs" in globals()

In [ ]:
%run -i image_failure_diagnostic.py

In [ ]:
%run -i optional_image_cross_attention_transformer.py

In [ ]:
%run -i image_failure_diagnostic.py

In [ ]:
%run -i optional_image_cross_attention_transformer.py

In [ ]:
%run -i image_fixed_checkpoint_figure.py

In [ ]:
%run -i image_fixed_checkpoint_figure.py

In [ ]:
%run -i image_data_rigor_diagnostic.py

In [ ]:
%run -i ./notebooks2/image_subject_grouped_sensitivity.py

In [ ]:
%run ./notebooks2/grape_visit_count_summary.py

In [ ]:
%run -i ./notebooks2/optional_image_cross_attention_transformer.py

import pandas as pd
summary = optional_image_cross_attention_outputs["summary"].copy()
region_order = ["Overall", "Supero_Nasal", "Supero_Temporal", "Macular", "Infero_Nasal", "Infero_Temporal", "Temporal"]
vf_regional = summary.loc[(summary["route"] == "post_self_attention_cross_attention") & (summary["Model"] == "VF-only six-region Transformer"), ["Region", "MAE_mean_db", "MAE_seed_std_db", "MAE_sample_std_db", "Samples", "#Pts"]].copy()
vf_regional["Region"] = pd.Categorical(vf_regional["Region"], categories=region_order, ordered=True)
vf_regional = vf_regional.sort_values("Region")
print(vf_regional.to_string(index=False))

In [ ]:
print789

In [ ]:
abc